# Setup 06 — Generate returns and promotions
Part of the retail-demo setup utility. Re-runnable (overwrite-by-design).

Builds refunds and promotion facts from the fulfilled sales (after setup-03).

This notebook runs its slice of the generation stage graph (see `STAGES` in
the engine); stages whose inputs are ready run concurrently. Inputs from
upstream notebooks are read from `gen__*` handoff tables in the Silver
staging schema, and outputs are written there for later notebooks.
`setup-10-publish-silver` validates and publishes everything atomically.
The engine cell below is generated by `scripts/build_notebooks.py` — edit
the package sources under `utility/src/retail_setup/`, never the cell.

In [ ]:
# The inlined engine below imports pydantic v2 on the driver. Newer Fabric runtimes
# preinstall pydantic and may already have its compiled pydantic_core loaded; a
# compiled module cannot be replaced inside a running process, so installing over
# it pairs new Python files with the old binary (ImportError: validate_core_schema).
# Reuse a working runtime pydantic v2; otherwise (older runtimes ship an OLD
# typing_extensions and no pydantic) install pinned versions. Why each step:
#  - subprocess pip (not %pip): a %pip kernel restart tears down the Spark session
#    mid pipeline run and fails the activity.
#  - pin pydantic <2.12: 2.12+ imports typing_extensions.Sentinel.
#  - drop the stale modules from sys.modules: the runtime already imported the old
#    typing_extensions, so the freshly installed copy is shadowed; evicting it lets
#    the engine cell re-import the upgraded versions (pydantic needs symbols such
#    as TypeIs that the runtime's typing_extensions lacks).
#  - pydantic runs only on the driver (engine builds rows, then
#    spark.createDataFrame), so a driver-side install is sufficient.
import importlib
import subprocess
import sys


def _runtime_pydantic_v2() -> bool:
    if "pydantic_core" in sys.modules or "pydantic" in sys.modules:
        loaded = sys.modules.get("pydantic")
        if loaded is not None and not str(getattr(loaded, "VERSION", "2.")).startswith("2."):
            raise RuntimeError("The runtime loaded pydantic v1; the engine requires v2.")
        return True
    probe = (
        "import pydantic, sys; from pydantic import BaseModel, Field, field_validator; "
        "sys.exit(0 if pydantic.VERSION.startswith('2.') else 1)"
    )
    return subprocess.run([sys.executable, "-c", probe], capture_output=True).returncode == 0


if _runtime_pydantic_v2():
    print("Using the Fabric runtime's pydantic v2.")
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "annotated-types==0.7.0", "pydantic==2.11.10",
        "pydantic-core==2.33.2", "typing-inspection==0.4.2",
        "typing_extensions==4.15.0",
    ])
    for _name in [m for m in list(sys.modules)
                  if m.split(".", 1)[0] in {"typing_extensions", "typing_inspection",
                                            "pydantic", "pydantic_core"}]:
        del sys.modules[_name]
    importlib.invalidate_caches()

In [ ]:
# PARAMETERS — rendered by `retail-setup render`; defaults work unrendered
def _param(value: str, default: str) -> str:
    return default if len(value) > 1 and value[0] == value[1] == "{" else value

LAKEHOUSE_NAME = _param("{{LAKEHOUSE_NAME}}", "retail_lakehouse")
SILVER_DB = _param("{{SILVER_DB}}", "ag")
GOLD_DB = _param("{{GOLD_DB}}", "au")
STORE_TYPE = _param("{{STORE_TYPE}}", "supercenter")
START_DATE = _param("{{START_DATE}}", "2025-01-01")
END_DATE = _param("{{END_DATE}}", "2025-03-31")
# Dimension anchor pinned by `retail-setup configure` (see GenerationConfig.anchor_date).
ANCHOR_DATE = _param("{{ANCHOR_DATE}}", START_DATE)
STORE_COUNT = int(_param("{{STORE_COUNT}}", "50"))
SEED = int(_param("{{SEED}}", "42"))
DICTIONARY_REF = _param("{{DICTIONARY_REF}}", "main")

spark.conf.set("spark.sql.session.timeZone", "UTC")  # engine timestamps depend on it
# Reorder seeded-draw math ((xxhash64(...)/1e12)*k -> (xxhash64(...)*k)/1e12) to
# cut rounding-error propagation and silence Fabric's Spark advisor. The rewrite
# shifts draws by <=1 ULP (generation is statistical, no exact-value contracts);
# set in every engine driver so dims and facts stay mutually consistent.
spark.conf.set("spark.advise.divisionExprConvertRule.enable", "true")

In [ ]:
# ENGINE SOURCE (generated — do not edit)
# Built by scripts/build_notebooks.py from utility/src/retail_setup/.

# --- retail_setup/dictionaries/models.py ---
"""Pydantic models for dictionary JSON files.

Entry models mirror the field names in
datagen/src/retail_datagen/shared/models.py (PascalCase) so the supercenter
conversion is verifiable 1:1. StoreTypeProfile is new: the behavioral knobs
that differentiate store types.
"""

import re
from decimal import Decimal
from typing import Literal

from pydantic import BaseModel, Field, field_validator, model_validator


class TaxJurisdictionEntry(BaseModel):
    StateCode: str = Field(..., min_length=2, max_length=2)
    County: str = Field(..., min_length=1)
    City: str = Field(..., min_length=1)
    CombinedRate: Decimal = Field(..., ge=0, le=Decimal("0.20"))

    @field_validator("StateCode")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("StateCode must be alphabetic")
        return v.upper()

    @field_validator("CombinedRate", mode="before")
    @classmethod
    def _rate_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("CombinedRate must be a valid decimal number") from None


class TaxRuleEntry(BaseModel):
    """A state's taxability rule: the share of the combined rate each class pays.

    ``GroceryTaxPct`` applies to grocery staples (products with taxability
    ``NON_TAXABLE``), ``ClothingTaxPct`` to clothing (``REDUCED_RATE``); items
    priced below ``ClothingExemptUnder`` dollars are clothing-exempt. States
    without a rule exempt groceries and tax clothing in full.
    """

    StateCode: str = Field(..., min_length=2, max_length=2)
    GroceryTaxPct: int = Field(default=0, ge=0, le=100)
    ClothingTaxPct: int = Field(default=100, ge=0, le=100)
    ClothingExemptUnder: Decimal | None = Field(default=None, gt=0)

    @field_validator("StateCode")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("StateCode must be alphabetic")
        return v.upper()


class GeographyEntry(BaseModel):
    City: str = Field(..., min_length=1)
    State: str = Field(..., min_length=2, max_length=2)
    Zip: str
    District: str = Field(..., min_length=1)
    Region: str = Field(..., min_length=1)

    @field_validator("State")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("State must be alphabetic")
        return v.upper()

    @field_validator("Zip")
    @classmethod
    def _zip_format(cls, v: str) -> str:
        if not re.match(r"^\d{5}(-\d{4})?$", v):
            raise ValueError("Zip must be 12345 or 12345-6789")
        return v


class NameEntry(BaseModel):
    """One first or last name (shared across store types).

    The ``Name`` field intentionally unifies the separate ``FirstNameDict`` and
    ``LastNameDict`` models from datagen — both source models carry a single
    name string, so one model suffices here.
    """

    Name: str = Field(..., min_length=1)


class ProductBrandEntry(BaseModel):
    Brand: str = Field(..., min_length=1)
    Company: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)


class ProductEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    BasePrice: Decimal = Field(..., gt=0)
    Department: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)
    Subcategory: str = Field(..., min_length=1)
    Tags: str | None = None
    # Optional explicit attributes. Preset catalogs omit them and rely on the
    # department/category name rules in generation.dims; custom catalogs set them
    # because their department names are not known in advance.
    Taxability: Literal["TAXABLE", "NON_TAXABLE", "REDUCED_RATE"] | None = None
    Refrigerated: bool | None = None

    @field_validator("BasePrice", mode="before")
    @classmethod
    def _price_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("BasePrice must be a valid decimal number") from None


class ProductTagEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    Tags: str = Field(..., min_length=1)


# Store formats, largest first. Volume classes pick among the formats their size
# suits (generation.dims.VOLUME_FORMATS); a profile may narrow the set.
STORE_FORMATS = ["hypermarket", "superstore", "standard", "neighborhood", "express", "boutique"]
_HOURS_PATTERN = re.compile(r"^(\d{1,2})-(\d{1,2})$")
# Derived defaults for profiles that leave the store-format knobs unset (custom
# layouts): traffic falls with ticket size around the supercenter reference.
_REFERENCE_TICKET = 85.0
_REFERENCE_TRANSACTIONS = 400
_TICKET_TRAFFIC_EXPONENT = 0.6
_MIN_DERIVED_TRANSACTIONS = 20
_MAX_DERIVED_TRANSACTIONS = 1000
# A custom profile's hours are the hours whose traffic weight reaches this
# share of its peak hour; a profile busy around the clock runs "24h".
_OPEN_HOUR_PEAK_SHARE = 0.10
# Above this average ticket a derived profile runs boutique stores.
_BOUTIQUE_TICKET = 500.0
# Derived trip frequency of an active customer around the supercenter reference.
_REFERENCE_VISITS = 8.0
_TICKET_VISITS_EXPONENT = 0.4
_MIN_DERIVED_VISITS = 1.0
_MAX_DERIVED_VISITS = 52.0


def parse_operating_hours(value: str) -> tuple[int, int]:
    """``[open_hour, close_hour)`` of an ``operating_hours`` value ("24h" or "H-H")."""
    text = value.strip()
    if text == "24h":
        return 0, 24
    match = _HOURS_PATTERN.fullmatch(text)
    if match is None:
        raise ValueError(f"operating hours {value!r} must be '24h' or 'open-close' hours")
    open_hour, close_hour = int(match.group(1)), int(match.group(2))
    if not 0 <= open_hour < close_hour <= 24:
        raise ValueError(f"operating hours {value!r} must open before closing, within 0-24")
    return open_hour, close_hour


class StoreTypeProfile(BaseModel):
    """Behavioral knobs that make a store type act differently.

    Weight lists are relative (normalized at use time, not here). The store
    knobs (transactions, hours, formats) are optional: presets set them, and
    custom layouts derive them from the ticket size and hourly weights.
    """

    store_type: str = Field(..., min_length=1)
    display_name: str = Field(..., min_length=1)
    basket_lambda: float = Field(..., gt=0, description="Nominal mean basket-line ceiling")
    avg_ticket_target: float = Field(..., gt=0, description="Basket calibration target, USD")
    hourly_weights: list[float] = Field(..., description="24 relative traffic weights")
    daily_weights: list[float] = Field(..., description="7 weights, Monday first")
    monthly_weights: list[float] = Field(..., description="12 seasonality weights, Jan first")
    department_weights: dict[str, float] = Field(..., min_length=1)
    promo_rate: float = Field(..., ge=0, le=1, description="Share of lines with a promotion")
    online_order_share: float = Field(..., ge=0, le=1)
    zones: list[str] = Field(..., min_length=1, description="Store footprint zones for BLE")
    transactions_per_store_day: int | None = Field(
        default=None, gt=0, le=5000, description="In-store receipts per store-day at multiplier 1"
    )
    operating_hours: list[str] | None = Field(
        default=None, min_length=1, description="Opening-hour patterns stores pick from"
    )
    store_formats: list[str] | None = Field(
        default=None, min_length=1, description="Store formats this store type runs"
    )
    customer_visits_per_year: float | None = Field(
        default=None, gt=0, le=156, description="Mean yearly trips of an active loyalty customer"
    )

    @field_validator("operating_hours")
    @classmethod
    def _hours_parse(cls, v: list[str] | None) -> list[str] | None:
        if v is None:
            return v
        for value in v:
            parse_operating_hours(value)
        return [value.strip() for value in v]

    @field_validator("store_formats")
    @classmethod
    def _known_formats(cls, v: list[str] | None) -> list[str] | None:
        if v is None:
            return v
        unknown = sorted(set(v) - set(STORE_FORMATS))
        if unknown:
            raise ValueError(f"unknown store formats {unknown}; known: {STORE_FORMATS}")
        return list(dict.fromkeys(v))

    def resolved_transactions_per_store_day(self) -> int:
        """Receipts per store-day: the profile's value, else derived from the ticket."""
        if self.transactions_per_store_day is not None:
            return self.transactions_per_store_day
        derived = (
            _REFERENCE_TRANSACTIONS
            * (_REFERENCE_TICKET / self.avg_ticket_target) ** _TICKET_TRAFFIC_EXPONENT
        )
        return int(min(_MAX_DERIVED_TRANSACTIONS, max(_MIN_DERIVED_TRANSACTIONS, round(derived))))

    def resolved_operating_hours(self) -> list[str]:
        """Opening-hour patterns: the profile's, else the hours its traffic weights use."""
        if self.operating_hours is not None:
            return list(self.operating_hours)
        floor = max(self.hourly_weights) * _OPEN_HOUR_PEAK_SHARE
        busy = [hour for hour, weight in enumerate(self.hourly_weights) if weight >= floor]
        open_hour, close_hour = busy[0], busy[-1] + 1
        if open_hour <= 1 and close_hour >= 23:
            return ["24h"]
        return [f"{open_hour}-{close_hour}"]

    def resolved_store_formats(self) -> list[str]:
        """Store formats: the profile's, else boutiques for high tickets, else every format."""
        if self.store_formats is not None:
            return list(self.store_formats)
        if self.avg_ticket_target >= _BOUTIQUE_TICKET:
            return ["boutique"]
        return [name for name in STORE_FORMATS if name != "boutique"]

    def resolved_customer_visits_per_year(self) -> float:
        """Mean yearly trips of an active customer: the profile's, else from the ticket.

        Shoppers return less often to stores with bigger tickets.
        """
        if self.customer_visits_per_year is not None:
            return self.customer_visits_per_year
        derived = _REFERENCE_VISITS * (_REFERENCE_TICKET / self.avg_ticket_target) ** (
            _TICKET_VISITS_EXPONENT
        )
        return float(min(_MAX_DERIVED_VISITS, max(_MIN_DERIVED_VISITS, round(derived, 1))))

    @model_validator(mode="after")
    def _hours_have_traffic(self) -> "StoreTypeProfile":
        # A store pattern whose open hours all carry zero weight would never sell.
        for value in self.resolved_operating_hours():
            open_hour, close_hour = parse_operating_hours(value)
            if sum(self.hourly_weights[open_hour:close_hour]) <= 0:
                raise ValueError(f"operating hours {value!r} have no hourly traffic weight")
        return self

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_nonneg(cls, v: list[float]) -> list[float]:
        if any(w < 0 for w in v):
            raise ValueError("weight values must be non-negative")
        return v

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_have_effect(cls, v: list[float]) -> list[float]:
        # An all-zero weight vector is a declared-but-unused control: it can't be
        # normalized and would silently suppress all traffic. Reject it so profile
        # controls provably drive generation (IMP-010 validated profile controls).
        if sum(v) <= 0:
            raise ValueError("weight values must not all be zero")
        return v

    @field_validator("hourly_weights")
    @classmethod
    def _24(cls, v: list[float]) -> list[float]:
        if len(v) != 24:
            raise ValueError("hourly_weights must have 24 entries")
        return v

    @field_validator("daily_weights")
    @classmethod
    def _7(cls, v: list[float]) -> list[float]:
        if len(v) != 7:
            raise ValueError("daily_weights must have 7 entries")
        return v

    @field_validator("monthly_weights")
    @classmethod
    def _12(cls, v: list[float]) -> list[float]:
        if len(v) != 12:
            raise ValueError("monthly_weights must have 12 entries")
        return v

    @field_validator("department_weights")
    @classmethod
    def _departments_have_effect(cls, v: dict[str, float]) -> dict[str, float]:
        # Every declared department must carry positive weight: a zero/negative
        # weight is an unused control that silently excludes a department from
        # every basket (IMP-010 validated profile controls).
        bad = sorted(k for k, w in v.items() if w <= 0)
        if bad:
            raise ValueError(f"department_weights must be positive; got <=0 for {bad}")
        return v

# --- retail_setup/dictionaries/loader.py ---
"""Load and validate dictionary JSON sets (shared + one store type)."""

import json
from dataclasses import dataclass, field
from importlib.resources import files
from pathlib import Path

from pydantic import BaseModel, ValidationError



@dataclass
class DictionarySet:
    store_type: str
    profile: StoreTypeProfile
    first_names: list[NameEntry] = field(default_factory=list)
    last_names: list[NameEntry] = field(default_factory=list)
    geographies: list[GeographyEntry] = field(default_factory=list)
    tax_rates: list[TaxJurisdictionEntry] = field(default_factory=list)
    tax_rules: list[TaxRuleEntry] = field(default_factory=list)
    products: list[ProductEntry] = field(default_factory=list)
    brands: list[ProductBrandEntry] = field(default_factory=list)
    tags: list[ProductTagEntry] = field(default_factory=list)


def default_dictionary_root() -> Path:
    """Return the packaged dictionary root, with a source-tree fallback."""
    packaged_root = files("retail_setup").joinpath("resources", "dictionaries")
    if packaged_root.is_dir():
        if not isinstance(packaged_root, Path):
            raise RuntimeError("Packaged dictionaries are not available as filesystem paths.")
        return packaged_root

    source_root = Path(__file__).resolve().parents[3] / "data" / "dictionaries"
    if source_root.is_dir():
        return source_root
    raise RuntimeError("Default dictionaries are missing from the installed package.")


def available_store_types(root: Path) -> list[str]:
    if not root.is_dir():
        return []
    return sorted(
        p.name
        for p in root.iterdir()
        if p.is_dir() and not p.name.startswith("_") and (p / "profile.json").exists()
    )


def load_list(path: Path, model: type[BaseModel]) -> list:
    """Load and validate a JSON array file, returning a list of model instances.

    Raises ValueError with filename and row index on validation failure.
    """
    try:
        raw = json.loads(path.read_text())
    except FileNotFoundError:
        raise ValueError(f"missing dictionary file: {path}") from None
    if not isinstance(raw, list):
        raise ValueError(f"{path.name}: expected a JSON array")
    result = []
    for i, row in enumerate(raw):
        try:
            result.append(model.model_validate(row))
        except ValidationError as exc:
            raise ValueError(f"{path.name}[{i}]: {exc}") from exc
    return result


# Keep the private alias for any callers that haven't migrated yet.
_load_list = load_list


def load_tax_rules(root: Path) -> list[TaxRuleEntry]:
    """State taxability rules (``_shared/tax_rules.json``); none when the file is absent."""
    path = root / "_shared" / "tax_rules.json"
    return load_list(path, TaxRuleEntry) if path.exists() else []


def load_dictionaries(root: Path, store_type: str) -> DictionarySet:
    type_dir = root / store_type
    if not (type_dir / "profile.json").exists():
        raise ValueError(
            f"unknown store type {store_type!r}; available: {available_store_types(root)}"
        )
    shared = root / "_shared"

    try:
        profile = StoreTypeProfile.model_validate(
            json.loads((type_dir / "profile.json").read_text())
        )
    except (json.JSONDecodeError, ValidationError) as exc:
        raise ValueError(f"profile.json: {exc}") from exc
    if profile.store_type != store_type:
        raise ValueError(
            f"profile.json store_type {profile.store_type!r} does not match folder {store_type!r}"
        )

    tags_path = type_dir / "tags.json"
    return DictionarySet(
        store_type=store_type,
        profile=profile,
        first_names=load_list(shared / "first_names.json", NameEntry),
        last_names=load_list(shared / "last_names.json", NameEntry),
        geographies=load_list(shared / "geographies.json", GeographyEntry),
        tax_rates=load_list(shared / "tax_rates.json", TaxJurisdictionEntry),
        tax_rules=load_tax_rules(root),
        products=load_list(type_dir / "products.json", ProductEntry),
        brands=load_list(type_dir / "brands.json", ProductBrandEntry),
        tags=load_list(tags_path, ProductTagEntry) if tags_path.exists() else [],
    )

# --- retail_setup/config/generation.py ---
"""Generation settings (utility/config.yaml). Environment settings live in deploy/config/."""

import calendar
import json
from datetime import date, timedelta
from pathlib import Path
from typing import Any

import yaml
from pydantic import BaseModel, Field, ValidationError, model_validator


# Minimum number of synthetic customers, regardless of store_count. The churn
# model (fabric/lakehouse/09-ml-churn-prediction.ipynb) needs at least two
# customers in each class (active / churned) to build a train/test split; very
# small customer pools can leave fewer than two churned customers and abort
# training. Flooring the count keeps small-store demos trainable.
MIN_CUSTOMER_COUNT = 5000
# Fallbacks when the store type's profile cannot be read (the store type is
# then reported as unknown by validation).
DEFAULT_TRANSACTIONS_PER_STORE_DAY = 400
DEFAULT_ONLINE_ORDER_SHARE = 0.12
DEFAULT_CUSTOMER_VISITS_PER_YEAR = 8.0
# Expected store daily_traffic_multiplier: the mean of generation.dims
# VOLUME_CLASSES (a test keeps the two in step).
MEAN_TRAFFIC_MULTIPLIER = 1.0525
# Upper bound on online orders per day. Order IDs widen past 99,999 orders a
# day, so this bounds volume (generation time, table size), not ID width.
MAX_ONLINE_ORDERS_PER_DAY = 1_000_000
# Share of all customers active on a typical day of an 18-month history at the
# default churn rate (generation.customers turns the base over by churn and
# acquisition); sizes the default customer count.
NOMINAL_ACTIVE_SHARE = 0.75


def default_online_orders_per_day(
    online_order_share: float, store_count: int, transactions_per_store_day: float
) -> int:
    """Network online orders per day so online is ``online_order_share`` of all orders.

    In-store volume is ``store_count * transactions_per_store_day`` at the mean
    traffic multiplier; online orders make up the given share of the total.
    """
    share = min(max(online_order_share, 0.0), 0.95)
    in_store = store_count * transactions_per_store_day * MEAN_TRAFFIC_MULTIPLIER
    return max(1, round(in_store * share / (1.0 - share)))


def default_customer_count(
    store_count: int,
    transactions_per_store_day: float,
    anonymous_share: float,
    visits_per_year: float,
) -> int:
    """Customers so the identified receipts give active customers ``visits_per_year`` trips.

    Identified receipts per year are ``store_count * transactions_per_store_day
    * 365 * (1 - anonymous_share)`` at the mean traffic multiplier, spread over
    the active customers, who are ``NOMINAL_ACTIVE_SHARE`` of all customers.
    """
    identified = (store_count * transactions_per_store_day * 365 * MEAN_TRAFFIC_MULTIPLIER) * (
        1.0 - anonymous_share
    )
    return max(1, round(identified / (NOMINAL_ACTIVE_SHARE * visits_per_year)))


def _profile_for(data: dict[str, Any]) -> StoreTypeProfile | None:
    """The configured store type's profile, or None when it cannot be read."""
    root = data.get("dictionary_root")
    base = Path(root) if root else default_dictionary_root()
    store_type = data.get("store_type", GenerationConfig.model_fields["store_type"].default)
    if not isinstance(store_type, str):
        return None
    try:
        raw = json.loads((base / store_type / "profile.json").read_text(encoding="utf-8"))
        return StoreTypeProfile.model_validate(raw)
    except (OSError, ValueError, ValidationError):
        return None


def _subtract_months(anchor: date, months: int) -> date:
    """Return the date ``months`` calendar months before ``anchor`` (day clamped)."""

    month_index = anchor.month - 1 - months
    year = anchor.year + month_index // 12
    month = month_index % 12 + 1
    day = min(anchor.day, calendar.monthrange(year, month)[1])
    return date(year, month, day)


class GenerationConfig(BaseModel):
    store_type: str = "supercenter"
    # Preferred input: number of months of history to generate. The window ends
    # *yesterday* so real-time streaming continues seamlessly from today.
    # ``start_date``/``end_date`` may be provided explicitly instead (back-compat);
    # they are derived from ``months`` only when not both already supplied.
    months: int | None = Field(default=None, ge=1, le=120)
    start_date: date
    end_date: date
    # Anchor for dimension dates (product launch dates). Dimensions depend on
    # the seed and this anchor only, never on the run date, so re-rendering a
    # ``months`` config on a later day regenerates identical dimensions.
    # `retail-setup configure` pins it; it defaults to ``start_date``.
    anchor_date: date
    store_count: int = Field(default=50, gt=0, le=2000)
    seed: int = 42
    silver_db: str = "ag"
    gold_db: str = "au"
    # Optional override for dictionary root; when None, default_dictionary_root() is used.
    # Pass an absolute path string to point at a custom dictionary tree (e.g. a Fabric
    # lakehouse Files mount) without mutating the package data directory.
    dictionary_root: str | None = None

    # scale knobs; omitted or None -> derived from store_count before validation
    dc_count: int = Field(gt=0, le=999)
    customer_count: int = Field(gt=0)
    # base in-store transactions per store-day at multiplier 1.0; profiles'
    # hourly/daily/monthly weights shape it, store daily_traffic_multiplier scales it.
    # Omitted -> the store type's profile value (luxury boutiques far below supercenters).
    transactions_per_store_day: int = Field(gt=0)
    # fraction of SALE receipts returned per day (Dec 26 spikes 6x, capped 10%)
    # nominal daily return share; Dec 26 applies a 6x spike capped at 0.10, so
    # values near the ceiling flatten the spike - keep this small (~0.01)
    return_rate: float = Field(default=0.01, ge=0.0, le=0.10)
    # network-wide online orders per day at multiplier 1.0; omitted -> derived so
    # online orders are the profile's online_order_share of all orders
    online_orders_per_day: int = Field(gt=0, le=MAX_ONLINE_ORDERS_PER_DAY)
    # number of category-matched branded SKUs generated per base catalog product
    # (datagen combinatorial SKUs); 1 = one SKU per dictionary row
    brands_per_product: int = Field(default=3, ge=1, le=10)
    # truck load capacity in units; a store-day shipment exceeding this is split
    # across multiple truck legs (datagen multi-truck shipments)
    truck_capacity: int = Field(default=15000, gt=0)
    # share of in-store receipts with no loyalty customer (cash and anonymous
    # shoppers); the rest carry a customer_id (generation.customers)
    anonymous_share: float = Field(default=0.45, ge=0.0, lt=1.0)
    # share of the customers active at a time who churn within a year; churn is
    # driven by segment, trip frequency and spend, with a tapering lead-in
    annual_churn_rate: float = Field(default=0.20, gt=0.0, lt=1.0)
    # year-over-year growth of store traffic and online orders, compounding from
    # anchor_date (1.0 there); prices carry their own inflation (pricing.py)
    annual_volume_growth: float = Field(default=0.03, ge=-0.5, le=1.0)

    @model_validator(mode="before")
    @classmethod
    def _derive_defaults(cls, data: Any) -> Any:
        """Fill the date window and scale knobs that derive from other settings."""

        if not isinstance(data, dict):
            return data
        data = dict(data)
        months = data.get("months")
        if months is not None and not (data.get("start_date") and data.get("end_date")):
            # Invalid months are left for field validation to report.
            if isinstance(months, int) and not isinstance(months, bool) and months >= 1:
                end = date.today() - timedelta(days=1)
                data["end_date"] = end
                data["start_date"] = _subtract_months(end, months)
        elif data.get("start_date") is None or data.get("end_date") is None:
            raise ValueError("provide `months`, or both `start_date` and `end_date`")
        if data.get("anchor_date") is None:
            data["anchor_date"] = data["start_date"]
        profile = _profile_for(data)
        if data.get("transactions_per_store_day") is None:
            data["transactions_per_store_day"] = (
                profile.resolved_transactions_per_store_day()
                if profile is not None
                else DEFAULT_TRANSACTIONS_PER_STORE_DAY
            )
        store_count = data.get("store_count", cls.model_fields["store_count"].default)
        transactions = data["transactions_per_store_day"]
        numeric = isinstance(transactions, (int, float)) and not isinstance(transactions, bool)
        if isinstance(store_count, int) and not isinstance(store_count, bool):
            if data.get("dc_count") is None:
                data["dc_count"] = max(1, store_count // 10)
            anonymous = data.get("anonymous_share", cls.model_fields["anonymous_share"].default)
            share_ok = isinstance(anonymous, (int, float)) and not isinstance(anonymous, bool)
            if data.get("customer_count") is None and numeric and share_ok:
                data["customer_count"] = default_customer_count(
                    store_count,
                    transactions,
                    anonymous,
                    profile.resolved_customer_visits_per_year()
                    if profile is not None
                    else DEFAULT_CUSTOMER_VISITS_PER_YEAR,
                )
            if data.get("online_orders_per_day") is None and numeric:
                data["online_orders_per_day"] = default_online_orders_per_day(
                    profile.online_order_share
                    if profile is not None
                    else DEFAULT_ONLINE_ORDER_SHARE,
                    store_count,
                    transactions,
                )
        return data

    @model_validator(mode="after")
    def _known_store_type(self) -> "GenerationConfig":
        root = Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()
        known = available_store_types(root)
        if self.store_type not in known:
            raise ValueError(f"store_type {self.store_type!r} not found; available: {known}")
        return self

    @property
    def resolved_dictionary_root(self) -> Path:
        """Resolved dictionary root path (explicit override or package default)."""
        return Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()

    @model_validator(mode="after")
    def _date_order(self) -> "GenerationConfig":
        if self.end_date < self.start_date:
            raise ValueError("end_date must be on or after start_date")
        return self

    @model_validator(mode="after")
    def _customer_floor(self) -> "GenerationConfig":
        # Guarantee enough customers for the churn model's two-class train/test
        # split; see MIN_CUSTOMER_COUNT above. Applied to the final value so an
        # explicitly small customer_count is lifted too.
        self.customer_count = max(self.customer_count, MIN_CUSTOMER_COUNT)
        return self


def generation_config_path(repo_root: Path, env: str) -> Path:
    """Return the environment-scoped generation configuration path."""

    return repo_root / "utility" / "config" / "environments" / f"{env}.yaml"


def resolve_generation_config_path(repo_root: Path, env: str) -> Path:
    """Prefer environment-scoped generation config, with a legacy fallback."""

    scoped = generation_config_path(repo_root, env)
    if scoped.is_file():
        return scoped
    return repo_root / "utility" / "config.yaml"


def load_generation_config(path: Path, *, dictionary_root: Path | None = None) -> GenerationConfig:
    """Load a generation YAML, optionally pointing it at a custom dictionary root."""

    data = yaml.safe_load(path.read_text())
    if dictionary_root is not None and isinstance(data, dict):
        data = {**data, "dictionary_root": str(dictionary_root)}
    return GenerationConfig.model_validate(data)

# --- retail_setup/generation/schemas.py ---
"""Authoritative output schemas for generated tables (Plan 2a + 2b scope).

Spark simple type strings. Dimension columns keep the legacy PascalCase names
because the semantic model TMDL binds sourceColumn to them (e.g. StoreNumber,
Cost, MSRP). Most fact tables are snake_case, but several 2b tables have
TMDL-bound PascalCase or mixed-case columns documented per table below.

The TMDL contract test (tests/generation/test_schema_contract.py) is the
arbiter: every sourceColumn in a table's TMDL must be present here with a
compatible type. Extra columns not in TMDL are allowed.

Columns added vs plan (from TMDL audit 2026-06-12):
  fact_receipts: added ("Subtotal", "string") — legacy trace/aggregate string
    column bound as sourceColumn: Subtotal in the semantic model. Not a
    snake_case column; appears to be a pre-existing legacy column the model
    still references.

  --- Plan 2b TMDL deltas (per-table, added to reconcile TMDL bindings) ---

  fact_ble_pings:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId (PascalCase),
      not customer_id. Plan listed customer_id (kept as extra); TMDL column added.
    - ("__index_level_0__", "long") — TMDL-bound legacy pandas-index column.

  fact_customer_zone_changes:
    - TMDL uses PascalCase for all data columns: StoreID, CustomerBLEId,
      FromZone, ToZone (plan listed snake_case equivalents, kept as extras).
    - ("StoreID", "long"), ("CustomerBLEId", "string"), ("FromZone", "string"),
      ("ToZone", "string") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_marketing:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId; plan's
      customer_id kept as extra.
    - ("CostCents", "long") — TMDL sourceColumn is CostCents; plan's cost_cents
      kept as extra.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_promo_lines:
    - TMDL uses PascalCase: ReceiptId, PromoCode, LineNumber, ProductID, Qty,
      DiscountAmount, DiscountCents (plan listed snake_case equivalents, kept
      as extras).
    - ("ReceiptId", "string"), ("PromoCode", "string"), ("LineNumber", "long"),
      ("ProductID", "long"), ("Qty", "long"), ("DiscountAmount", "string"),
      ("DiscountCents", "long") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_online_order_lines:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_reorders:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_moves:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_inventory:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_dc_inventory_txn:
    - ("Source", "string") — TMDL sourceColumn is PascalCase Source. The
      lowercase ``source`` that appears during construction is renamed to
      ``Source`` in the final select (inventory.py); keeping both would cause
      a case-insensitive collision that Delta on Fabric rejects.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_store_inventory_txn:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_stockouts:
    - ("__index_level_0__", "long") — TMDL-bound.

Columns in plan but NOT in TMDL (allowed — Direct Lake ignores extra columns):
  fact_receipts: trace_id, tender_type
  fact_payments: tender_type (not in payments TMDL at all)
  fact_ble_pings: customer_id (TMDL uses CustomerId), trace_id, event_ts,
    event_date
  fact_customer_zone_changes: store_id, customer_ble_id, from_zone, to_zone,
    event_ts, trace_id, event_date
  fact_marketing: customer_id, cost_cents, event_ts, trace_id, event_date,
    channel (extra beyond TMDL-bound channel — already in TMDL, fine)
  fact_promo_lines: receipt_id_ext, promo_code, line_number, product_id,
    quantity, discount_amount, discount_cents, event_ts, trace_id, event_date
  fact_dc_inventory_txn: trace_id, event_ts, event_date
    (lowercase source is renamed to Source in the final select; not an extra)
"""

# table -> list of (column, spark_type)
TABLES: dict[str, list[tuple[str, str]]] = {
    "dim_geographies": [
        ("ID", "long"),
        ("City", "string"),
        ("State", "string"),
        ("ZipCode", "string"),
        ("District", "string"),
        ("Region", "string"),
    ],
    "dim_stores": [
        ("ID", "long"),
        ("StoreNumber", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
        ("tax_rate", "double"),
        ("volume_class", "string"),
        ("store_format", "string"),
        ("operating_hours", "string"),
        ("daily_traffic_multiplier", "double"),
        # IANA time zone from the store geography's state. Generation works in
        # this local time; Silver timestamps are UTC (see timezones.py).
        ("TimeZone", "string"),
    ],
    "dim_distribution_centers": [
        ("ID", "long"),
        ("DCNumber", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
    ],
    "dim_trucks": [
        ("ID", "long"),
        ("LicensePlate", "string"),
        ("Refrigeration", "boolean"),
        # double, not long: NULL for pool trucks + Direct Lake nullability
        ("DCID", "double"),
    ],
    "dim_customers": [
        ("ID", "long"),
        ("FirstName", "string"),
        ("LastName", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
        ("LoyaltyCard", "string"),
        ("Phone", "string"),
        ("BLEId", "string"),
        ("AdId", "string"),
        # The store where the customer makes most visits (generation.customers).
        ("HomeStoreID", "long"),
    ],
    "dim_products": [
        ("ID", "long"),
        ("ProductName", "string"),
        ("Brand", "string"),
        ("Company", "string"),
        ("Department", "string"),
        ("Category", "string"),
        ("Subcategory", "string"),
        ("Cost", "double"),
        ("MSRP", "double"),
        ("SalePrice", "double"),
        ("RequiresRefrigeration", "boolean"),
        ("LaunchDate", "timestamp"),
        ("taxability", "string"),
        ("Tags", "string"),
    ],
    "dim_date": [
        ("date_key", "long"),
        ("date", "date"),
        ("year", "long"),
        ("quarter", "long"),
        ("month", "long"),
        ("month_name", "string"),
        ("day", "long"),
        ("day_of_week", "long"),
        ("day_name", "string"),
        ("week_of_year", "long"),
        ("is_weekend", "long"),
        ("fiscal_year", "long"),
        ("fiscal_quarter", "long"),
    ],
    "fact_receipts": [
        ("receipt_id_ext", "string"),
        ("trace_id", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("store_id", "long"),
        ("customer_id", "long"),
        ("receipt_type", "string"),
        ("tender_type", "string"),
        ("subtotal_cents", "long"),
        ("discount_amount", "string"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("subtotal_amount", "string"),
        ("tax_amount", "string"),
        ("total_amount", "string"),
        ("payment_method", "string"),
        # Legacy column bound by the semantic model (sourceColumn: Subtotal).
        # Present in fact_receipts.tmdl as a string column; added here to
        # satisfy the TMDL contract test (TMDL arbiter rule).
        ("Subtotal", "string"),
        # --- IMP-007 marketing attribution (additive, snake_case) ---
        # gross_subtotal_cents = subtotal_cents (net, unchanged) + discount_cents;
        # discount before tax is preserved (subtotal_cents stays the net amount).
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        # NULL unless this SALE receipt was selected for marketing attribution.
        ("attribution_journey_id", "string"),
        ("campaign_id", "string"),
        ("impression_id_ext", "string"),
        # Store-local hour (0-23) of the UTC event_ts, for hour-of-day reporting.
        ("local_hour", "int"),
        # RETURN rows: the sale they refund. NULL on SALE rows.
        ("original_receipt_id_ext", "string"),
    ],
    "fact_receipt_lines": [
        ("receipt_id_ext", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("line_num", "int"),
        ("product_id", "long"),
        ("quantity", "int"),
        ("unit_price", "string"),
        ("unit_cents", "long"),
        ("ext_price", "string"),
        ("ext_cents", "long"),
        ("promo_code", "string"),
    ],
    "fact_payments": [
        ("receipt_id_ext", "string"),
        ("order_id_ext", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("payment_method", "string"),
        ("amount_cents", "long"),
        ("amount", "string"),
        ("transaction_id", "string"),
        ("status", "string"),
        ("decline_reason", "string"),
        ("processing_time_ms", "long"),
        ("store_id", "long"),
        ("customer_id", "long"),
        # IMP-007: NULL unless this payment's receipt/order was attributed.
        ("attribution_journey_id", "string"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2b: 15 remaining fact tables
    # -----------------------------------------------------------------------
    "fact_store_ops": [
        # TMDL-bound: trace_id, store_id (int64→long), operation_type, event_date
        # Extra (not in TMDL, allowed): event_ts
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("operation_type", "string"),
        ("event_date", "date"),
    ],
    "fact_foot_traffic": [
        # TMDL-bound: count (int64→long), zone, dwell_seconds (int64→long),
        #   store_id (int64→long), sensor_id, event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("sensor_id", "string"),
        ("zone", "string"),
        ("dwell_seconds", "long"),
        ("count", "long"),
        ("event_date", "date"),
    ],
    "fact_ble_pings": [
        # TMDL-bound: beacon_id, rssi (int64→long), customer_ble_id, zone,
        #   store_id (int64→long), CustomerId (double), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("beacon_id", "string"),
        ("customer_ble_id", "string"),
        ("customer_id", "double"),
        # TMDL-bound PascalCase column (sourceColumn: CustomerId)
        ("CustomerId", "double"),
        ("rssi", "long"),
        ("zone", "string"),
        ("event_date", "date"),
        # TMDL-bound legacy pandas-index column
        ("__index_level_0__", "long"),
    ],
    "fact_customer_zone_changes": [
        # TMDL-bound PascalCase: StoreID (int64→long), CustomerBLEId (string),
        #   FromZone (string), ToZone (string), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   store_id, customer_ble_id, from_zone, to_zone (snake_case plan names)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("store_id", "long"),
        ("customer_ble_id", "string"),
        ("from_zone", "string"),
        ("to_zone", "string"),
        # TMDL-bound PascalCase columns
        ("StoreID", "long"),
        ("CustomerBLEId", "string"),
        ("FromZone", "string"),
        ("ToZone", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_marketing": [
        # TMDL-bound: device, cost, customer_ad_id, impression_id_ext, campaign_id,
        #   creative_id, channel, CustomerId (double), CostCents (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId),
        #   cost_cents (plan name; TMDL uses CostCents)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("channel", "string"),
        ("campaign_id", "string"),
        ("creative_id", "string"),
        ("customer_ad_id", "string"),
        # snake_case plan columns (extras)
        ("customer_id", "double"),
        ("cost_cents", "long"),
        # TMDL-bound PascalCase columns (sourceColumn: CustomerId, CostCents)
        ("CustomerId", "double"),
        ("CostCents", "long"),
        ("impression_id_ext", "string"),
        ("cost", "string"),
        ("device", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # IMP-007: NULL for organic/background impressions; set on the two
        # synthetic touches created for a selected purchase's journey.
        ("attribution_journey_id", "string"),
    ],
    "fact_promotions": [
        # TMDL-bound: receipt_id_ext, promo_code, discount_amount, discount_cents
        #   (int64→long), discount_type, product_count (int64→long), product_ids,
        #   store_id (int64→long), customer_id (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("receipt_id_ext", "string"),
        ("promo_code", "string"),
        ("discount_amount", "string"),
        ("discount_cents", "long"),
        ("discount_type", "string"),
        ("product_count", "long"),
        ("product_ids", "string"),
        ("store_id", "long"),
        ("customer_id", "long"),
        ("event_date", "date"),
        # IMP-007: NULL unless the underlying receipt was attributed.
        ("attribution_journey_id", "string"),
    ],
    "fact_promo_lines": [
        # TMDL-bound PascalCase: ReceiptId, PromoCode, LineNumber (int64→long),
        #   ProductID (int64→long), Qty (int64→long), DiscountAmount (string),
        #   DiscountCents (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   receipt_id_ext, promo_code, line_number, product_id, quantity,
        #   discount_amount, discount_cents (snake_case plan names)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("receipt_id_ext", "string"),
        ("promo_code", "string"),
        ("line_number", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("discount_amount", "string"),
        ("discount_cents", "long"),
        # TMDL-bound PascalCase columns
        ("ReceiptId", "string"),
        ("PromoCode", "string"),
        ("LineNumber", "long"),
        ("ProductID", "long"),
        ("Qty", "long"),
        ("DiscountAmount", "string"),
        ("DiscountCents", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_online_order_headers": [
        # TMDL-bound: customer_id (int64→long), subtotal_cents (int64→long),
        #   order_id_ext, payment_method, total_amount, total_cents (int64→long),
        #   tax_amount, subtotal_amount, tax_cents (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts
        ("order_id_ext", "string"),
        ("customer_id", "long"),
        ("subtotal_cents", "long"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("subtotal_amount", "string"),
        ("tax_amount", "string"),
        ("total_amount", "string"),
        ("payment_method", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        # --- IMP-007 marketing attribution (additive, snake_case) ---
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        ("attribution_journey_id", "string"),
        ("campaign_id", "string"),
        ("impression_id_ext", "string"),
    ],
    # -----------------------------------------------------------------------
    # IMP-007: dedicated marketing-attribution fact. One row per in-store SALE
    # receipt or online order, including attributed, unattributed, declined,
    # and reconciliation-failed outcomes (see attribution.py).
    # -----------------------------------------------------------------------
    "fact_marketing_attribution": [
        ("attribution_id", "string"),
        ("attribution_journey_id", "string"),
        ("attribution_status", "string"),
        ("attribution_model", "string"),
        ("attribution_window_days", "int"),
        ("impression_id_ext", "string"),
        ("campaign_id", "string"),
        ("creative_id", "string"),
        ("channel", "string"),
        ("customer_ad_id", "string"),
        ("customer_id", "long"),
        ("touch_ts", "timestamp"),
        ("purchase_ts", "timestamp"),
        ("lag_seconds", "long"),
        ("purchase_type", "string"),
        ("receipt_id_ext", "string"),
        ("order_id_ext", "string"),
        ("store_id", "long"),
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        ("net_subtotal_cents", "long"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("payment_cents", "long"),
        ("attributed_revenue_cents", "long"),
        ("event_date", "date"),
    ],
    "fact_online_order_lines": [
        # TMDL-bound: order_id, unit_cents (int64→long), fulfillment_mode,
        #   promo_code, node_type, product_id (int64→long), ext_price, ext_cents
        #   (int64→long), line_num (int64→long), fulfillment_status, unit_price,
        #   quantity (int64→long), node_id (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): picked_ts, shipped_ts, delivered_ts,
        #   event_ts, event_date
        ("order_id", "string"),
        ("product_id", "long"),
        ("line_num", "long"),
        ("quantity", "long"),
        ("unit_price", "string"),
        ("unit_cents", "long"),
        ("ext_price", "string"),
        ("ext_cents", "long"),
        ("promo_code", "string"),
        ("fulfillment_mode", "string"),
        ("fulfillment_status", "string"),
        ("node_type", "string"),
        ("node_id", "long"),
        ("picked_ts", "timestamp"),
        ("shipped_ts", "timestamp"),
        ("delivered_ts", "timestamp"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_reorders": [
        # TMDL-bound: store_id (int64→long), dc_id (int64→long), product_id
        #   (int64→long), current_quantity (int64→long), reorder_quantity
        #   (int64→long), reorder_point (int64→long), priority,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("dc_id", "long"),
        ("product_id", "long"),
        ("current_quantity", "long"),
        ("reorder_quantity", "long"),
        ("reorder_point", "long"),
        ("priority", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_truck_moves": [
        # TMDL-bound: status, shipment_id, dc_id (int64→long), truck_id
        #   (int64→long), store_id (int64→long), actual_unload_duration (double),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, eta, etd,
        #   departure_time, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("truck_id", "long"),
        ("dc_id", "long"),
        ("store_id", "long"),
        ("shipment_id", "string"),
        ("status", "string"),
        ("eta", "timestamp"),
        ("etd", "timestamp"),
        ("departure_time", "timestamp"),
        ("actual_unload_duration", "double"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # Planned vs actual arrival (incidents.plan_delivery). Moves before
        # the truck arrives show the plan in eta/etd with delay 0; moves from
        # ARRIVED on show the actual arrival, delay_minutes (actual minus
        # planned arrival) and delay_reason (DC_OUTAGE, HOLIDAY_CAPACITY,
        # LANE_DELAY, WEATHER, TRAFFIC; NULL when on time).
        ("planned_arrival_time", "timestamp"),
        ("delay_minutes", "double"),
        ("delay_reason", "string"),
        # Units on the shipment (drives the unload time).
        ("load_units", "long"),
    ],
    "fact_truck_inventory": [
        # TMDL-bound: truck_id (int64→long), shipment_id, product_id (int64→long),
        #   quantity (int64→long), action, location_id (int64→long), location_type,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("truck_id", "long"),
        ("shipment_id", "string"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("action", "string"),
        ("location_id", "long"),
        ("location_type", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_dc_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), dc_id (int64→long),
        #   balance (int64→long), product_id (int64→long), Source (PascalCase!),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   source (lowercase snake_case plan name)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("dc_id", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("balance", "long"),
        ("txn_type", "string"),
        # TMDL-bound PascalCase column (sourceColumn: Source). The lowercase
        # ``source`` used during construction is renamed AS "Source" in
        # inventory.py before the final select; keeping both would produce a
        # case-insensitive duplicate that Delta on Fabric rejects.
        ("Source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_store_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), balance (int64→long),
        #   product_id (int64→long), store_id (int64→long), source (lowercase OK),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("balance", "long"),
        ("txn_type", "string"),
        ("source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # Attempted demand is retained even when a whole line cannot be filled.
        # LOST_SALE changes no stock (quantity=0); all non-demand rows use zero.
        ("requested_quantity", "long"),
        ("unfulfilled_quantity", "long"),
    ],
    "fact_stockouts": [
        # PascalCase IS the contract: TMDL binds StoreID (double), DCID (double),
        #   ProductID (int64→long), LastKnownQuantity (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("StoreID", "double"),
        ("DCID", "double"),
        ("ProductID", "long"),
        ("LastKnownQuantity", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    # Logistics disruptions (incidents.INCIDENT_TYPES: DC_OUTAGE, LANE_DELAY,
    # WEATHER, HOLIDAY_CAPACITY, SUPPLIER_SHORT_SHIP): one row per incident,
    # resolved_at NULL while open. store_id is set for a lane incident and NULL
    # for DC-wide ones. History and the live stream (logistics_incident_started
    # / _resolved) plan incidents with the same model.
    "fact_logistics_incidents": [
        ("incident_id", "string"),
        ("incident_type", "string"),
        ("dc_id", "long"),
        ("store_id", "long"),
        ("started_at", "timestamp"),
        ("resolved_at", "timestamp"),
        ("severity", "string"),
        ("affected_store_count", "long"),
        ("description", "string"),
        ("trace_id", "string"),
        ("event_date", "date"),
    ],
    # Bridge: one row per incident and store it affects (every store the DC
    # serves, the lane's store, or the geography's stores).
    "fact_logistics_incident_stores": [
        ("incident_id", "string"),
        ("store_id", "long"),
        ("dc_id", "long"),
        ("incident_type", "string"),
        ("started_at", "timestamp"),
        ("resolved_at", "timestamp"),
        ("trace_id", "string"),
        ("event_date", "date"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2c: 10 Gold (au) aggregate tables
    # TMDL audit 2026-06-12: bindings match exactly; `computed_at` and `as_of`
    # are produced by the legacy transforms but unbound in TMDL (extras OK).
    # TMDL `day`/`ts` are dateTime — TYPE_COMPAT accepts timestamp|date.
    # -----------------------------------------------------------------------
    "sales_minute_store": [
        ("store_id", "long"),
        ("ts", "timestamp"),
        ("total_sales", "double"),
        ("receipts", "long"),
        ("avg_basket", "double"),
    ],
    "top_products_15m": [
        ("product_id", "long"),
        ("revenue", "double"),
        ("units", "long"),
        ("computed_at", "timestamp"),  # window end; bound as 'Window End' in TMDL
    ],
    "inventory_position_current": [
        ("store_id", "long"),
        ("product_id", "long"),
        ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "dc_inventory_position_current": [
        ("dc_id", "long"),
        ("product_id", "long"),
        ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "truck_dwell_daily": [
        ("site", "string"),
        ("day", "date"),
        ("avg_dwell_min", "double"),
        ("trucks", "long"),
    ],
    "online_sales_daily": [
        ("day", "date"),
        ("orders", "long"),
        ("subtotal", "double"),
        ("tax", "double"),
        ("total", "double"),
        ("avg_order_value", "double"),
    ],
    "zone_dwell_minute": [
        ("store_id", "long"),
        ("zone", "string"),
        ("ts", "timestamp"),
        ("avg_dwell", "double"),
        ("customers", "long"),
    ],
    "marketing_cost_daily": [
        ("campaign_id", "string"),
        ("day", "date"),
        ("impressions", "long"),
        ("cost", "double"),
    ],
    "campaign_performance_daily": [
        ("campaign_id", "string"),
        ("channel", "string"),
        ("day", "date"),
        ("impressions", "long"),
        ("spend_cents", "long"),
        ("conversions", "long"),
        ("attributed_revenue_cents", "long"),
        ("discount_cents", "long"),
        ("tax_cents", "long"),
        ("payment_cents", "long"),
        ("conversion_rate", "double"),
        ("roas", "double"),
    ],
    "tender_mix_daily": [
        ("day", "date"),
        ("payment_method", "string"),
        ("transactions", "long"),
        ("total_amount", "double"),
    ],
}


_SPARK_TYPE_MAP = None


def _type_map():
    """Lazy-import PySpark type map (avoids import cost when pyspark not installed)."""
    global _SPARK_TYPE_MAP
    if _SPARK_TYPE_MAP is None:
        from pyspark.sql.types import (
            BooleanType,
            DateType,
            DoubleType,
            IntegerType,
            LongType,
            StringType,
            TimestampType,
        )

        _SPARK_TYPE_MAP = {
            "long": LongType(),
            "int": IntegerType(),
            "string": StringType(),
            "double": DoubleType(),
            "boolean": BooleanType(),
            "timestamp": TimestampType(),
            "date": DateType(),
        }
    return _SPARK_TYPE_MAP


def spark_schema(table: str):
    """Build a StructType for createDataFrame with explicit types."""
    from pyspark.sql.types import StructField, StructType

    tmap = _type_map()
    fields = [StructField(name, tmap[typ], nullable=True) for name, typ in TABLES[table]]
    return StructType(fields)


def column_names(table: str) -> list[str]:
    return [name for name, _ in TABLES[table]]

# --- retail_setup/generation/runtime.py ---
"""Deterministic seeding + partition grids for the generation engine."""

import hashlib
import math
from collections.abc import Callable, Iterable
from datetime import date

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


# Spark's default of 200 shuffle partitions leaves billion-row full-demo shuffles
# (receipt lines, inventory events, BLE pings) at millions of rows per task: long
# spilling sorts and too few tasks to occupy a large pool. Target roughly one
# million rows of the dominant stream per partition; adaptive query execution
# still coalesces the many small shuffles back to its advisory size.
SHUFFLE_ROWS_PER_PARTITION = 1_000_000
MIN_SHUFFLE_PARTITIONS = 200
MAX_SHUFFLE_PARTITIONS = 8192


def shuffle_partitions_for(cfg: GenerationConfig, basket_lambda: float = 12.0) -> int:
    """Shuffle partition count sized to the configured receipt-line volume.

    Rounded up to a multiple of 64 so tasks spread evenly across executors.
    Generation output is independent of the partition count.
    """
    days = (cfg.end_date - cfg.start_date).days + 1
    lines = cfg.store_count * days * cfg.transactions_per_store_day * basket_lambda
    wanted = math.ceil(lines / SHUFFLE_ROWS_PER_PARTITION / 64) * 64
    return max(MIN_SHUFFLE_PARTITIONS, min(MAX_SHUFFLE_PARTITIONS, wanted))


class seeded_draws:
    """Deterministic draw expressions bound to a seed.

    u(cols, salt)        -> uniform [0,1) double
    gauss(cols, salt)    -> N(0,1) (Box-Muller)
    negative_binomial(cols, salt, mean, shape) -> count, variance mean + mean²/shape
    h64(cols, salt)      -> non-negative long hash
    pick_by_weights(cols, salt, [(value, weight), ...]) -> weighted categorical
    """

    _U_MOD = 10**12

    def __init__(self, seed: int):
        self.seed = seed

    def h64(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return F.pmod(F.xxhash64(*cols, F.lit(f"{salt}|{self.seed}")), F.lit(2**62))

    def u(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return (self.h64(cols, salt) % F.lit(self._U_MOD)) / F.lit(float(self._U_MOD))

    def gauss(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        radius = F.sqrt(-2.0 * F.log(F.lit(1.0) - self.u(cols, f"{salt}|g1")))
        return radius * F.cos(F.lit(6.283185307179586) * self.u(cols, f"{salt}|g2"))

    def negative_binomial(self, cols: list, salt: str, mean: Column, shape: int = 12) -> Column:
        """Exact sum of geometric draws; integer shape controls overdispersion."""
        if shape < 1:
            raise ValueError("negative-binomial shape must be positive")
        valid_mean = F.when((mean >= 0) & (mean < float("inf")), mean).otherwise(
            F.raise_error("negative-binomial mean must be finite and non-negative")
        )
        probability = valid_mean / (valid_mean + F.lit(float(shape)))
        draws = [
            F.when(valid_mean == 0, F.lit(0))
            .otherwise(
                F.floor(F.log(F.lit(1.0) - self.u(cols, f"{salt}|nb{i}")) / F.log(probability))
            )
            .cast("long")
            for i in range(shape)
        ]
        return sum(draws, F.lit(0).cast("long"))

    def pick_by_weights(self, cols: list, salt: str, weighted: list[tuple[str, float]]):
        from pyspark.sql import functions as F

        total = sum(w for _, w in weighted)
        uu = self.u(cols, salt)
        expr, acc = None, 0.0
        for value, w in weighted[:-1]:
            acc += w / total
            expr = expr.when(uu < acc, value) if expr is not None else F.when(uu < acc, value)
        return expr.otherwise(weighted[-1][0]) if expr is not None else F.lit(weighted[0][0])


def derive_seed(global_seed: int, section: str, key: int, day: date) -> int:
    """Stable 31-bit seed from (global_seed, section, key, day).

    Independent of Spark partitioning/execution order — safe for per-row
    or per-group RNG seeding.
    """
    payload = f"{global_seed}|{section}|{key}|{day.isoformat()}".encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "big") % (2**31)


def legacy_index(*key_cols):
    """Deterministic long for the legacy __index_level_0__ pandas column.

    The semantic model binds the column by name only; nothing consumes its
    values, so a hash beats a dense row_number — which forced a
    single-partition global sort at full volume.
    """
    from pyspark.sql import functions as F

    return F.pmod(F.xxhash64(*key_cols, F.lit("__legacy_index__")), F.lit(2**62))


def bind(value: Column, body: Callable[[Column], Column]) -> Column:
    """``body(value)`` with ``value`` evaluated once.

    Binds ``value`` to a lambda variable (a one-element ``transform``), so a
    sub-expression used several times in ``body`` is not copied into the
    plan; large copied expressions make query planning slow.
    """
    return F.element_at(F.transform(F.array(value), body), 1)


def case_when(branches: Iterable[tuple[Column, object]]) -> Column:
    """Chain `F.when` over `(condition, value)` branches (at least one)."""
    expr: Column | None = None
    for condition, value in branches:
        expr = F.when(condition, value) if expr is None else expr.when(condition, value)
    if expr is None:
        raise ValueError("case_when needs at least one branch")
    return expr


def fixed_width_decimal(value: Column, width: int, label: str) -> Column:
    """Zero-pad a non-negative decimal value and fail if it exceeds ``width``."""
    if width < 1:
        raise ValueError("width must be positive")
    text = value.cast("string")
    invalid = (value < F.lit(0)) | (F.length(text) > F.lit(width))
    error = F.raise_error(F.lit(f"{label} exceeds {width} digits"))
    return (
        F.when(value.isNull(), F.lit(None).cast("string"))
        .when(invalid, error)
        .otherwise(F.lpad(text, width, "0"))
    )


def minimum_width_decimal(value: Column, width: int) -> Column:
    """Zero-pad to at least ``width`` digits without truncating larger values."""
    return minimum_width_text(value.cast("string"), width)


def minimum_width_text(value: Column, width: int) -> Column:
    """Zero-pad text to at least ``width`` characters without truncation."""
    if width < 1:
        raise ValueError("width must be positive")
    text = value.cast("string")
    return F.when(F.length(text) >= width, text).otherwise(F.lpad(text, width, "0"))


def date_grid(
    spark: SparkSession,
    start: date,
    end: date,
    column: str = "day",
) -> DataFrame:
    """Inclusive Spark-native date range without a driver-materialized row list."""
    if end < start:
        raise ValueError("end must be on or after start")
    days = (end - start).days + 1
    return spark.range(days).select(F.date_add(F.lit(start), F.col("id").cast("int")).alias(column))


def store_day_grid(
    spark: SparkSession,
    store_ids: list[int],
    start: date,
    end: date,
    global_seed: int,
    section: str,
) -> DataFrame:
    """store_id x day cross grid.

    Only the small store-ID set is materialized on the driver. Dates and the
    Cartesian expansion remain Spark-native, so long histories do not create a
    large Python row list. Random draws derive from stable row keys rather than
    partition order.
    """
    del global_seed, section
    store_values = F.array(*[F.lit(int(store_id)).cast("long") for store_id in store_ids])
    stores = spark.range(len(store_ids)).select(
        F.element_at(store_values, F.col("id").cast("int") + 1).alias("store_id")
    )
    return stores.crossJoin(date_grid(spark, start, end))

# --- retail_setup/generation/vocabulary.py ---
"""Shared retail vocabularies and identifier formats.

Historical generation and the live stream must agree on these names and
formats. Batch modules import them from here, and ``build_notebooks.py``
inlines this module into the stream notebook, so both producers read one
definition.

The module holds:

- payment tenders, their mix, and the decline model (decline, then a retry on
  another tender or a voided sale)
- the named promotion catalog
- identifier formats: BLE IDs, ad IDs, beacon and sensor IDs, campaign IDs
- zone vocabularies and the visit path through a store type's zones
- marketing channels, their cost-per-impression bands, and device types
- online fulfillment modes

The live stream does not use every definition yet. The remaining differences
are listed, with their review IDs, in the known-gap allowlist of
``tests/generation/test_vocabulary.py``.

Column helpers here (tender, decline and BLE draws) are the single
implementation both producers call, keyed on each producer's own IDs.
"""

from dataclasses import dataclass
from datetime import date

from pyspark.sql import Column
from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Payments
# ---------------------------------------------------------------------------

# In-store tenders: (method, mix weight, decline multiplier, processing_ms lo,
# processing_ms hi). Weights are relative.
TENDERS: list[tuple[str, float, float, int, int]] = [
    ("CREDIT_CARD", 0.4, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.3, 0.8, 1200, 3500),
    ("CASH", 0.2, 0.0, 500, 2000),
    ("MOBILE_PAY", 0.1, 1.2, 800, 2500),
]

# Online tenders (no cash): 60% credit / 25% debit / 10% PayPal / 5% other.
ONLINE_TENDERS: list[tuple[str, float, float, int, int]] = [
    ("CREDIT_CARD", 0.60, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.25, 0.8, 1200, 3500),
    ("PAYPAL", 0.10, 1.1, 2000, 5000),
    ("OTHER", 0.05, 0.5, 1500, 4000),
]

# Decline model: P(decline) = BASE_DECLINE x the tender's decline multiplier.
BASE_DECLINE = 0.025
DECLINE_REASONS = [
    "INSUFFICIENT_FUNDS",
    "CARD_EXPIRED",
    "INVALID_CVV",
    "NETWORK_ERROR",
    "FRAUD_SUSPECTED",
    "CARD_BLOCKED",
    "LIMIT_EXCEEDED",
]


def tender_names(tenders: list[tuple[str, float, float, int, int]] = TENDERS) -> list[str]:
    """Tender method names in table order."""
    return [name for name, *_ in tenders]


def decline_probability(
    method: str, tenders: list[tuple[str, float, float, int, int]] = TENDERS
) -> float:
    """Probability that a payment with ``method`` is declined."""
    for name, _, multiplier, _, _ in tenders:
        if name == method:
            return BASE_DECLINE * multiplier
    raise ValueError(f"unknown tender {method!r}")


def tender_column(
    draws: seeded_draws,
    keys: list,
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
    salt: str = "tender",
) -> Column:
    """Tender drawn by the mix weights of ``tenders``."""
    return draws.pick_by_weights(keys, salt, [(name, weight) for name, weight, *_ in tenders])


def decline_probability_column(
    method: Column, tenders: list[tuple[str, float, float, int, int]] = TENDERS
) -> Column:
    """Column form of :func:`decline_probability` (0 for an unknown method)."""
    return case_when(
        (method == name, F.lit(BASE_DECLINE * multiplier)) for name, _, multiplier, _, _ in tenders
    ).otherwise(F.lit(0.0))


def payment_status_column(
    draws: seeded_draws,
    keys: list,
    method: Column,
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
    salt: str = "decline",
) -> Column:
    """``DECLINED`` with the tender's decline probability, else ``APPROVED``."""
    declined = draws.u(keys, salt) < decline_probability_column(method, tenders)
    return F.when(declined, F.lit("DECLINED")).otherwise(F.lit("APPROVED"))


# A declined shopper retries with another tender DECLINE_RETRY_SHARE of the
# time (the retry is approved); otherwise the sale is voided and never happens.
DECLINE_RETRY_SHARE = 0.8
APPROVED, RETRIED, VOIDED = "APPROVED", "RETRIED", "VOIDED"


def payment_outcome_column(
    draws: seeded_draws,
    keys: list,
    method: Column,
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
    salt: str = "decline",
) -> Column:
    """``APPROVED``, ``RETRIED`` (declined, then approved on a second tender) or ``VOIDED``.

    The first attempt declines with the tender's probability (the same draw as
    :func:`payment_status_column`); a declined sale is retried
    ``DECLINE_RETRY_SHARE`` of the time and voided otherwise.
    """
    declined = draws.u(keys, salt) < decline_probability_column(method, tenders)
    retried = draws.u(keys, f"{salt}-retry") < F.lit(DECLINE_RETRY_SHARE)
    return F.when(~declined, F.lit(APPROVED)).when(retried, F.lit(RETRIED)).otherwise(F.lit(VOIDED))


def retry_tender_column(
    draws: seeded_draws,
    keys: list,
    declined: Column,
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
    salt: str = "retry-tender",
) -> Column:
    """The second tender after ``declined``: drawn by the mix of the other tenders."""
    return case_when(
        (
            declined == name,
            draws.pick_by_weights(
                keys, salt, [(other, weight) for other, weight, *_ in tenders if other != name]
            ),
        )
        for name, *_ in tenders
    )


def outcome_shares(
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
) -> dict[str, float]:
    """Expected share of sales by payment outcome for a tender mix."""
    total = sum(weight for _, weight, *_ in tenders)
    declined = sum(weight * BASE_DECLINE * mult for _, weight, mult, _, _ in tenders) / total
    return {
        APPROVED: 1.0 - declined,
        RETRIED: declined * DECLINE_RETRY_SHARE,
        VOIDED: declined * (1.0 - DECLINE_RETRY_SHARE),
    }


def transaction_id_column(prefix: str, purchase_id: Column, attempt: Column | int) -> Column:
    """Payment transaction ID, unique by construction: ``<prefix>-<purchase>-<attempt>``."""
    attempt_col = attempt if isinstance(attempt, Column) else F.lit(attempt)
    return F.concat(F.lit(f"{prefix}-"), purchase_id, F.lit("-"), attempt_col.cast("string"))


def decline_reason_column(
    draws: seeded_draws, keys: list, status: Column, salt: str = "reason"
) -> Column:
    """A decline reason for declined payments, NULL otherwise."""
    index = (draws.h64(keys, salt) % len(DECLINE_REASONS)).cast("int")
    return F.when(
        status == "DECLINED",
        F.element_at(F.array(*[F.lit(reason) for reason in DECLINE_REASONS]), index + 1),
    )


def processing_ms_column(
    draws: seeded_draws,
    keys: list,
    method: Column,
    tenders: list[tuple[str, float, float, int, int]] = TENDERS,
    salt: str = "proc",
) -> Column:
    """Payment processing time in ms, uniform in the tender's range."""
    low: Column = F.lit(tenders[-1][3])
    high: Column = F.lit(tenders[-1][4])
    for name, _, _, lo, hi in tenders[:-1]:
        low = F.when(method == name, F.lit(lo)).otherwise(low)
        high = F.when(method == name, F.lit(hi)).otherwise(high)
    return (low + draws.u(keys, salt) * (high - low)).cast("long")


# ---------------------------------------------------------------------------
# Promotions
# ---------------------------------------------------------------------------

# Named promo catalog. A purchase with an offer gets at most one code for the
# whole basket (``baskets.basket_promotion``): seasonal codes apply only in
# their eligible months, minimums apply to the basket gross subtotal, and each
# code discounts only the lines in its department scope (``baskets.PROMO_SCOPES``).
# BOGO discounts every second unit and needs a line with at least two units.
# (code, discount_pct, eligible_months | None, min_cents, kind)
PROMO_CATALOG: list[tuple[str, int, list[int] | None, int, str]] = [
    ("SAVE10", 10, None, 0, "PCT"),
    ("SAVE15", 15, None, 2500, "PCT"),  # min $25
    ("SAVE20", 20, None, 5000, "PCT"),  # min $50
    ("SAVE25", 25, None, 7500, "PCT"),  # min $75
    ("CLEARANCE30", 30, None, 0, "PCT"),
    ("BOGO50", 50, None, 0, "BOGO"),  # buy one get one 50% off (qty >= 2)
    ("NEWYEAR15", 15, [1, 2], 0, "PCT"),
    ("SPRINGSALE20", 20, [3, 4, 5], 0, "PCT"),
    ("SUMMER25", 25, [6, 7, 8], 0, "PCT"),
    ("BACKTOSCHOOL20", 20, [8, 9], 0, "PCT"),
    ("BFRIDAY30", 30, [11], 0, "PCT"),
    ("BFRIDAY40", 40, [11], 10000, "PCT"),  # min $100
    ("HOLIDAY20", 20, [12], 0, "PCT"),
]


def promo_eligible(index: int, month: int, basket_cents: int, max_quantity: int) -> bool:
    """Whether ``PROMO_CATALOG[index]`` may apply to a basket.

    The code must be in season, the basket gross subtotal must meet any
    minimum, and BOGO needs a line with at least two units. Department scope
    is checked per line by ``baskets.basket_promotion``.
    """
    _, _, months, min_cents, kind = PROMO_CATALOG[index]
    if months is not None and month not in months:
        return False
    if basket_cents < min_cents:
        return False
    return kind != "BOGO" or max_quantity >= 2


# ---------------------------------------------------------------------------
# Identifier formats
# ---------------------------------------------------------------------------

BLE_ID_PREFIX = "BLE"
BLE_ID_BASE = 36
BLE_ID_WIDTH = 6
AD_ID_PREFIX = "AD"
AD_ID_WIDTH = 8
BEACON_ID_FORMAT = "BEACON_%03d_%s"
SENSOR_ID_FORMAT = "SENSOR_%03d_%s"
CAMPAIGN_ID_PREFIX = "CAMP"
CAMPAIGN_INDEX_WIDTH = 2

_BASE36 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def ble_id(customer_id: int) -> str:
    """BLE device ID of a customer: ``BLE`` + upper-case base-36 ID, padded to 6."""
    if customer_id < 0:
        raise ValueError("customer_id must be non-negative")
    digits = ""
    value = customer_id
    while True:
        value, remainder = divmod(value, BLE_ID_BASE)
        digits = _BASE36[remainder] + digits
        if value == 0:
            break
    return BLE_ID_PREFIX + digits.rjust(BLE_ID_WIDTH, "0")


def ble_id_column(customer_id: Column) -> Column:
    """Column form of :func:`ble_id`."""
    digits = F.upper(F.conv(customer_id.cast("string"), 10, BLE_ID_BASE))
    return F.concat(F.lit(BLE_ID_PREFIX), minimum_width_text(digits, BLE_ID_WIDTH))


def ad_id(customer_id: int) -> str:
    """Advertising ID of a customer: ``AD`` + decimal ID, padded to 8 digits."""
    return AD_ID_PREFIX + str(customer_id).rjust(AD_ID_WIDTH, "0")


def ad_id_column(customer_id: Column) -> Column:
    """Column form of :func:`ad_id`."""
    return F.concat(F.lit(AD_ID_PREFIX), minimum_width_decimal(customer_id, AD_ID_WIDTH))


def beacon_id(store_id: int, zone: str) -> str:
    return BEACON_ID_FORMAT % (store_id, zone)


def beacon_id_column(store_id: Column, zone: Column) -> Column:
    return F.format_string(BEACON_ID_FORMAT, store_id, zone)


def sensor_id(store_id: int, zone: str) -> str:
    return SENSOR_ID_FORMAT % (store_id, zone)


def sensor_id_column(store_id: Column, zone: Column) -> Column:
    return F.format_string(SENSOR_ID_FORMAT, store_id, zone)


# Share of in-store visits whose phone is identified (its BLE ID is in
# dim_customers); the rest are anonymous.
BLE_KNOWN_SHARE = 0.3
ANONYMOUS_BLE_PREFIX = "ANON-"


def anonymous_ble_id_column(store_id: Column, draw: Column) -> Column:
    """Anonymous BLE device ID: ``ANON-<store>-<6 digits>`` from a non-negative hash."""
    return F.concat(
        F.lit(ANONYMOUS_BLE_PREFIX),
        store_id.cast("string"),
        F.lit("-"),
        (draw % F.lit(900000) + F.lit(100000)).cast("string"),
    )


def campaign_id(campaign_start: date, archetype_index: int) -> str:
    """Campaign ID: ``CAMP`` + window start (yyyyMMdd) + 2-digit archetype index."""
    if not 0 <= archetype_index < 10**CAMPAIGN_INDEX_WIDTH:
        raise ValueError("archetype_index exceeds 2 digits")
    return f"{CAMPAIGN_ID_PREFIX}{campaign_start:%Y%m%d}{archetype_index:02d}"


def campaign_id_column(campaign_start: Column, archetype_index: Column) -> Column:
    """Column form of :func:`campaign_id`."""
    return F.concat(
        F.lit(CAMPAIGN_ID_PREFIX),
        F.date_format(campaign_start, "yyyyMMdd"),
        fixed_width_decimal(archetype_index, CAMPAIGN_INDEX_WIDTH, "campaign archetype index"),
    )


# ---------------------------------------------------------------------------
# Zones
# ---------------------------------------------------------------------------

# BLE beacon zones a shopper's phone is seen in.
BLE_ZONES = ["ENTRANCE", "ELECTRONICS", "GROCERY", "CLOTHING", "CHECKOUT"]

# Generic zone dwell ranges (seconds), which the store types' zones borrow.
FOOT_TRAFFIC_ZONE_DWELL: dict[str, tuple[int, int]] = {
    "ENTRANCE_MAIN": (45, 90),
    "ENTRANCE_SIDE": (30, 75),
    "AISLES_A": (180, 420),
    "AISLES_B": (120, 300),
    "CHECKOUT": (90, 240),
}


# A shopper's path starts at the entrance, visits some department zones in a
# random order, and passes checkout when the visit buys (visit_zone_path). Each
# zone borrows the dwell range of the generic zone it overlaps.
BLE_ENTRY_ZONE = "ENTRANCE"
BLE_CHECKOUT_ZONE = "CHECKOUT"
BLE_ZONE_DWELL: dict[str, tuple[int, int]] = {
    "ENTRANCE": FOOT_TRAFFIC_ZONE_DWELL["ENTRANCE_MAIN"],
    "ELECTRONICS": FOOT_TRAFFIC_ZONE_DWELL["AISLES_B"],
    "GROCERY": FOOT_TRAFFIC_ZONE_DWELL["AISLES_A"],
    "CLOTHING": FOOT_TRAFFIC_ZONE_DWELL["AISLES_B"],
    "CHECKOUT": FOOT_TRAFFIC_ZONE_DWELL["CHECKOUT"],
}


def ble_zones_for_profile(profile: StoreTypeProfile | None) -> list[str]:
    """BLE zones for a store type: the profile's ``zones`` when it declares any.

    Profile zone names are normalized to upper case so they match the rest of
    the zone vocabulary. Without a profile the default ``BLE_ZONES`` apply.
    History (``sensors.py``) and the live stream both use these zones.
    """
    if profile is None or not profile.zones:
        return list(BLE_ZONES)
    return list(dict.fromkeys(zone.strip().upper() for zone in profile.zones if zone.strip()))


BLE_EXIT_ZONE = "EXIT"
# Dwell of zones outside BLE_ZONE_DWELL: exits pass like entrances, every other
# store-type zone (produce, tools, jewelry, ...) dwells like a secondary aisle.
BLE_EXIT_DWELL = FOOT_TRAFFIC_ZONE_DWELL["ENTRANCE_MAIN"]
BLE_DEPARTMENT_DWELL = FOOT_TRAFFIC_ZONE_DWELL["AISLES_B"]


@dataclass(frozen=True)
class BleLayout:
    """A store type's BLE zones by their role in a shopper's path."""

    entry: str
    middle: list[str]
    checkout: str | None
    exit: str | None
    dwell: dict[str, tuple[int, int]]

    @property
    def zones(self) -> list[str]:
        ordered = [self.entry, *self.middle, self.checkout, self.exit]
        return list(dict.fromkeys(zone for zone in ordered if zone is not None))


def ble_layout(profile: StoreTypeProfile | None) -> BleLayout:
    """Entry, department, checkout and exit zones with their dwell ranges.

    The entry is ``ENTRANCE`` (else the first zone), checkout and exit are the
    ``CHECKOUT`` and ``EXIT`` zones when the store type has them, and every
    other zone is a department zone a shopper may browse.
    """
    zones = ble_zones_for_profile(profile)
    entry = BLE_ENTRY_ZONE if BLE_ENTRY_ZONE in zones else zones[0]
    checkout = BLE_CHECKOUT_ZONE if BLE_CHECKOUT_ZONE in zones else None
    exit_zone = BLE_EXIT_ZONE if BLE_EXIT_ZONE in zones and BLE_EXIT_ZONE != entry else None
    middle = [zone for zone in zones if zone not in (entry, checkout, exit_zone)]
    if not middle:
        middle = [zone for zone in zones if zone != entry] or [entry]
    dwell = {
        zone: BLE_ZONE_DWELL.get(
            zone, BLE_EXIT_DWELL if zone == BLE_EXIT_ZONE else BLE_DEPARTMENT_DWELL
        )
        for zone in zones
    }
    return BleLayout(entry, middle, checkout, exit_zone, dwell)


# A shopper's visit (history and the live stream): the entrance, 1-3 of the
# store type's department zones in a random order, checkout when the visit
# buys, and the exit. Foot-traffic sensors count every zone a visitor passes;
# beacons detect the phones of BLE_DETECTED_SHARE of visits, 2-5 pings a zone.
BLE_MAX_MIDDLE_ZONES = 3
BLE_PINGS_PER_ZONE = (2, 5)
BLE_DETECTED_SHARE = 0.25
# Pings carry an RSSI uniform in [-80, -29] dBm.
RSSI_FLOOR = -80
RSSI_SPAN = 52


def zone_bound(zone: Column, dwell: dict[str, tuple[int, int]], index: int) -> Column:
    """Lower (``index`` 0) or upper (1) dwell seconds of ``zone``."""
    return case_when((zone == name, F.lit(bounds[index])) for name, bounds in dwell.items())


def visit_zone_path(draws: seeded_draws, key: list, converts: Column, layout: BleLayout) -> Column:
    """Ordered zones of a visit: entrance, browsed departments, checkout (buyers), exit.

    ``key`` holds the visit's key columns; every draw is keyed on it, so history
    and the stream lay out a visit the same way.
    """
    shuffled = F.array_sort(
        F.array(
            *[
                F.struct(
                    draws.u([*key, F.lit(zone)], "zone-order").alias("draw"),
                    F.lit(zone).alias("zone"),
                )
                for zone in layout.middle
            ]
        )
    )
    browse = min(BLE_MAX_MIDDLE_ZONES, len(layout.middle))
    middle_count = (F.floor(draws.u(key, "zone-count") * browse) + 1).cast("int")
    middle = F.transform(F.slice(shuffled, 1, middle_count), lambda entry: entry["zone"])
    no_zone = F.array().cast("array<string>")
    checkout = (
        F.when(converts, F.array(F.lit(layout.checkout))).otherwise(no_zone)
        if layout.checkout is not None
        else no_zone
    )
    exit_zone = F.array(F.lit(layout.exit)) if layout.exit is not None else no_zone
    return F.concat(F.array(F.lit(layout.entry)), middle, checkout, exit_zone)


def visit_zone_dwell(draws: seeded_draws, key: list, zones: Column, layout: BleLayout) -> Column:
    """Dwell seconds in each zone of a visit's path, uniform in the zone's range."""
    return F.transform(
        zones,
        lambda zone, i: (
            zone_bound(zone, layout.dwell, 0)
            + F.floor(
                draws.u([*key, i], "zone-dwell")
                * (zone_bound(zone, layout.dwell, 1) - zone_bound(zone, layout.dwell, 0) + 1)
            )
        ).cast("long"),
    )


def visit_zone_starts(dwell: Column) -> Column:
    """Seconds from the visit start at which each zone of the path is entered."""
    return F.transform(
        dwell,
        lambda _dwell, i: F.aggregate(
            F.slice(dwell, 1, i), F.lit(0).cast("long"), lambda acc, x: acc + x
        ),
    )


def visit_ping_counts(draws: seeded_draws, key: list, zones: Column) -> Column:
    """Beacon pings in each zone of a detected visit (2-5)."""
    low, high = BLE_PINGS_PER_ZONE
    return F.transform(
        zones,
        lambda _zone, i: (F.floor(draws.u([*key, i], "pings") * (high - low + 1)) + low).cast(
            "int"
        ),
    )


def checkout_index(zones: Column, layout: BleLayout) -> Column:
    """1-based position of the zone a buyer pays in: the last before any exit."""
    return F.size(zones) - (1 if layout.exit is not None else 0)


def zone_passage_shares(layout: BleLayout) -> dict[str, float]:
    """Share of visitors that pass each non-checkout zone of the layout.

    Every visitor passes the entrance and the exit; a department zone is one
    of 1-``BLE_MAX_MIDDLE_ZONES`` browsed zones. Checkout is passed by buyers.
    """
    browse = min(BLE_MAX_MIDDLE_ZONES, len(layout.middle))
    department = (browse + 1) / 2 / len(layout.middle)
    shares = {zone: department for zone in layout.middle}
    shares[layout.entry] = 1.0
    if layout.exit is not None:
        shares[layout.exit] = 1.0
    return shares


# ---------------------------------------------------------------------------
# Marketing
# ---------------------------------------------------------------------------

# (archetype, channels, base impressions/day at the legacy 86-store fleet,
#  campaign duration in days). The archetype index in campaign IDs is the
#  1-based position in this list.
CAMPAIGN_ARCHETYPES: list[tuple[str, list[str], int, int]] = [
    ("seasonal_sale", ["FACEBOOK", "GOOGLE", "EMAIL"], 1000, 7),
    ("product_launch", ["INSTAGRAM", "YOUTUBE", "DISPLAY"], 2000, 14),
    ("loyalty_program", ["EMAIL", "SOCIAL"], 500, 30),
    ("flash_sale", ["SOCIAL", "SEARCH"], 5000, 1),  # runs ~1 day in 7
]

# channel -> (lo, hi) cost-per-impression band in dollars (uniform draw), at
# real CPMs ($2.50-$21 per thousand); the device multiplier keeps every
# impression between $0.002 and $0.03.
CHANNEL_COSTS: dict[str, tuple[float, float]] = {
    "EMAIL": (0.0025, 0.006),
    "DISPLAY": (0.003, 0.010),
    "SOCIAL": (0.005, 0.014),
    "SEARCH": (0.010, 0.025),
    "FACEBOOK": (0.006, 0.016),
    "GOOGLE": (0.008, 0.022),
    "INSTAGRAM": (0.006, 0.016),
    "YOUTUBE": (0.010, 0.025),
}
# Relative chance that an impression on a channel converts (attribution.py):
# search and email convert best, display worst.
CHANNEL_CONVERSION_WEIGHTS: dict[str, float] = {
    "SEARCH": 2.0,
    "EMAIL": 1.6,
    "GOOGLE": 1.5,
    "FACEBOOK": 1.0,
    "INSTAGRAM": 0.9,
    "SOCIAL": 0.8,
    "YOUTUBE": 0.7,
    "DISPLAY": 0.5,
}
# Each campaign rotates a small pool of creatives per channel.
CREATIVES_PER_CAMPAIGN = 4
CREATIVE_ID_PREFIX = "CRT"


def channel_conversion_weight(channel: Column) -> Column:
    """Column form of ``CHANNEL_CONVERSION_WEIGHTS`` (1.0 for an unknown channel)."""
    return case_when(
        (channel == name, F.lit(weight)) for name, weight in CHANNEL_CONVERSION_WEIGHTS.items()
    ).otherwise(F.lit(1.0))


def creative_id_column(
    draws: seeded_draws, keys: list, campaign: Column, channel: Column
) -> Column:
    """A creative from the campaign's small pool on the channel: ``CRT<campaign>-<channel>-<n>``."""
    index = (draws.h64(keys, "creative") % CREATIVES_PER_CAMPAIGN + 1).cast("int")
    return F.concat(
        F.lit(CREATIVE_ID_PREFIX),
        F.substring(campaign, len(CAMPAIGN_ID_PREFIX) + 1, 32),
        F.lit("-"),
        channel,
        F.lit("-"),
        index.cast("string"),
    )


def impression_cost_cents(draws: seeded_draws, keys: list, dollars: Column) -> Column:
    """Whole cents of a sub-cent cost, rounded at random so sums stay unbiased."""
    return F.floor(dollars * 100 + draws.u(keys, "cost-round")).cast("long")


MARKETING_CHANNELS = list(CHANNEL_COSTS)

# Device types are upper case; cost multipliers scale the channel band.
DEVICE_WEIGHTS: list[tuple[str, float]] = [("MOBILE", 0.6), ("DESKTOP", 0.3), ("TABLET", 0.1)]
DEVICE_MULTIPLIERS: dict[str, float] = {"MOBILE": 1.2, "DESKTOP": 0.8, "TABLET": 0.9}
DEVICE_TYPES = [device for device, _ in DEVICE_WEIGHTS]

# ---------------------------------------------------------------------------
# Online fulfillment
# ---------------------------------------------------------------------------

ONLINE_FULFILLMENT_MIX: list[tuple[str, float]] = [
    ("SHIP_FROM_DC", 0.60),
    ("SHIP_FROM_STORE", 0.30),
    ("BOPIS", 0.10),
]
FULFILLMENT_MODES = [mode for mode, _ in ONLINE_FULFILLMENT_MIX]

# Order-level outcomes and stage delays shared by history (online_orders.py)
# and the live stream. Cancelled orders have zero totals and zero line
# amounts, no payment and no fulfillment; backordered (never BOPIS) orders
# wait ONLINE_BACKORDER_SECONDS for stock between picking and shipping, then
# ship and deliver like any other order.
ONLINE_CANCEL_RATE = 0.02
ONLINE_BACKORDER_RATE = 0.08
ONLINE_BACKORDER_SECONDS = (3 * 86400, 14 * 86400)
# Promo codes with their percentage discount; each order puts a promo on a
# share of its lines drawn uniformly from ONLINE_PROMO_LINE_SHARE.
ONLINE_PROMOS: list[tuple[str, int]] = [("PROMO05", 5), ("PROMO10", 10), ("PROMO20", 20)]
ONLINE_PROMO_LINE_SHARE = (0.10, 0.30)
# Stage delays in seconds, uniform in [low, high): picking after the order
# (BOPIS orders are picked for the pickup counter, ship orders for the
# carrier), shipping after picking (plus the backorder wait), delivery after
# shipping, and the BOPIS customer pickup after the order is ready.
ONLINE_PICK_SECONDS = {"BOPIS": (4 * 3600, 24 * 3600), "SHIP": (30 * 60, 240 * 60)}
ONLINE_SHIP_SECONDS = (2 * 3600, 4 * 3600)
ONLINE_DELIVER_SECONDS = (86400, 3 * 86400)
ONLINE_PICKUP_SECONDS = (3600, 24 * 3600)
# Relative weight of online orders in each UTC hour (0-23). Most customers
# are in US time zones, so orders peak in the US evening (00-03 UTC) and are
# fewest in the US night (07-10 UTC).
ONLINE_HOURLY_WEIGHTS = [
    6.5, 6.5, 6.0, 5.0, 3.8, 2.6, 1.6, 1.0, 0.8, 0.9, 1.3, 2.0,
    2.9, 3.6, 4.1, 4.4, 4.6, 4.7, 4.7, 4.8, 5.0, 5.4, 5.9, 6.3,
]  # fmt: skip

# --- retail_setup/generation/calibration.py ---
"""Calibrated thresholds shared by the generators and their consumers.

Rules (``fabric/rules/definitions.kql``), KQL functions, the semantic model and
the live stream keep literal values in their own languages. Each literal must
equal the constant here; ``tests/generation/test_calibration.py`` parses those
sources and fails when one drifts.

The stream generator parameters below sit next to the thresholds they are
calibrated against, so a normal truck stays under the dwell SLA and a late
truck exceeds it.
"""

from math import ceil, sqrt


# ---------------------------------------------------------------------------
# Truck dwell (minutes from arrival to departure)
# ---------------------------------------------------------------------------

# One SLA for every consumer: the dwell-breach rule and the semantic model's
# breach-day measure. A typical consolidated load dwells 30-75 minutes, so a
# site-day averaging above 90 is a real outlier.
TRUCK_DWELL_SLA_MINUTES = 90
# Unload model shared by history and the live stream (network.unload_minutes):
# a truck dwells the base minutes plus its load at the site's unload rate
# (units per minute times a per-store dock factor), plus a little noise, never
# less than the minimum and never more than the maximum. A typical
# consolidated store load stays inside the normal range, under the SLA; only an
# unusually large load (a store served late after a backlog) breaches it.
TRUCK_NORMAL_DWELL_MINUTES_MIN = 30
TRUCK_NORMAL_DWELL_MINUTES_MAX = 75
TRUCK_UNLOAD_BASE_MINUTES = 25
TRUCK_UNLOAD_UNITS_PER_MINUTE = 120
TRUCK_UNLOAD_NOISE_MINUTES = 6
TRUCK_DOCK_FACTOR_RANGE = (0.8, 1.2)
TRUCK_UNLOAD_MINUTES_MAX = 180
# History: unloading (and the store's stock receipt) starts this long after
# the truck arrives.
TRUCK_CHECKIN_MINUTES = 15
# Rule evaluation window for completed lifecycles.
TRUCK_DWELL_RULE_WINDOW_MINUTES = 5

# ---------------------------------------------------------------------------
# Inventory
# ---------------------------------------------------------------------------

# A stockout is reported when a store's on-hand balance drops below this
# quantity, that is when a sale takes it to zero: the crossing history's
# fact_stockouts records. Reorder points and priorities come from the shared
# replenishment policy (inventory_policy.reorder_levels / reorder_priority).
STOCKOUT_QUANTITY_THRESHOLD = 1
# Rule evaluation windows.
STOCKOUT_RULE_WINDOW_MINUTES = 5
URGENT_REORDER_RULE_WINDOW_MINUTES = 10

# ---------------------------------------------------------------------------
# Truck logistics (history and stream_supply)
# ---------------------------------------------------------------------------

# Reorders wait at the DC for the store's daily delivery slot, so one truck
# carries a store's consolidated load; then the next free truck of the DC's
# fleet loads for TRUCK_LOAD_MINUTES, drives the lane (one-way minutes, a stable
# draw per DC and store) and returns.
TRUCK_LOAD_MINUTES = 45
TRUCK_TRANSIT_MINUTES_MIN = 60
TRUCK_TRANSIT_MINUTES_MAX = 240
# Delivery routes (network.plan_routes): each DC truck serves a fixed set of
# stores, one delivery a day each, at slots spaced by the longest possible
# trip (load, both legs of the lane, the maximum unload) plus this turnaround,
# which also absorbs the hour lost when clocks spring forward.
TRUCK_TURNAROUND_MINUTES = 60

# Unload extras (network.unload_minutes): a truck unloading in the store's
# busy hours (local) takes longer, one at night is quicker; a truck that
# arrives TRUCK_LATE_MINUTES or more after plan has missed its dock
# appointment and waits for a door.
TRUCK_UNLOAD_BUSY_HOURS = (11, 19)
TRUCK_UNLOAD_NIGHT_HOURS = (22, 6)
TRUCK_UNLOAD_BUSY_FACTOR = 1.15
TRUCK_UNLOAD_NIGHT_FACTOR = 0.9
TRUCK_LATE_DOCK_WAIT_MINUTES = 10

# ---------------------------------------------------------------------------
# Delivery delays and logistics incidents (incidents.py)
# ---------------------------------------------------------------------------

# A truck is late when it arrives this many minutes or more after its planned
# arrival (the late-truck rule, the semantic model's late-shipment measures and
# the unload model's missed dock appointment).
TRUCK_LATE_MINUTES = 30
# Traffic: a run's chance of being late grows with the lane's length (from the
# first value on the shortest lane to the second on the longest), scaled by
# the weekday (Monday first: Friday getaway traffic, quiet Sundays); a late run
# loses this share of its lane minutes.
TRUCK_TRAFFIC_LATE_SHARE = (0.05, 0.20)
TRUCK_TRAFFIC_DAY_FACTORS = (1.0, 1.0, 1.0, 1.1, 1.4, 0.8, 0.6)
TRUCK_TRAFFIC_DELAY_SHARE = (0.15, 0.50)
# Random incidents per DC and year, and their extent.
DC_OUTAGES_PER_DC_YEAR = 1.5
LANE_DELAYS_PER_DC_YEAR = 4.0
SUPPLIER_SHORT_SHIPS_PER_DC_YEAR = 2.0
DC_OUTAGE_HOURS = (6, 36)
LANE_DELAY_HOURS = (2, 12)
LANE_DELAY_MINUTES = (45, 180)
SUPPLIER_SHORT_SHIP_DAYS = (3, 10)
# Share of a supplier order that arrives while the supplier short-ships.
SUPPLIER_FILL_RATE = (0.2, 0.6)
# Products split into this many supplier groups; a short-ship hits one group.
SUPPLIER_GROUPS = 6
# Severe-weather days (stream_calendar.severe_weather_column) open a WEATHER
# incident on the lanes into the geography with this probability in the
# climate's roughest month, scaled down by its weather sensitivity in others.
WEATHER_INCIDENT_SHARE = 0.6
WEATHER_INCIDENT_MAX_DAYS = 4
WEATHER_DELAY_MINUTES = (60, 240)
# Peak holiday weekends (Black Friday to Cyber Monday, the days after
# Christmas) strain each DC's crews with this probability.
HOLIDAY_CAPACITY_SHARE = 0.7
HOLIDAY_CAPACITY_DELAY_MINUTES = (60, 240)
# Incident severity by the delay it adds (minutes) or the share of supply lost.
INCIDENT_HIGH_DELAY_MINUTES = 150
INCIDENT_MEDIUM_DELAY_MINUTES = 90
INCIDENT_HIGH_OUTAGE_HOURS = 24
INCIDENT_MEDIUM_OUTAGE_HOURS = 12
INCIDENT_HIGH_SHORT_SHARE = 0.65
INCIDENT_MEDIUM_SHORT_SHARE = 0.5
# Rules: a late truck alerts on arrival; a lane alerts when its trucks average
# this delay over the window; an incident alerts when stores it affects report
# this many stockouts.
TRUCK_LATE_RULE_WINDOW_MINUTES = 5
LANE_DELAY_RULE_MINUTES = 60
LANE_DELAY_RULE_WINDOW_MINUTES = 60
INCIDENT_RULE_WINDOW_MINUTES = 5
INCIDENT_STOCKOUT_RULE_WINDOW_MINUTES = 60
INCIDENT_STOCKOUT_RULE_MIN_STOCKOUTS = 3

# ---------------------------------------------------------------------------
# DC replenishment (inventory_policy, stream_supply)
# ---------------------------------------------------------------------------

# Each DC keeps stock of every product, fed by supplier orders that arrive a
# stable 3-7 days later at DC_SUPPLIER_RECEIPT_HOUR (DC local). The DC orders
# when its position (on hand plus open supplier orders) falls to its daily
# demand times (lead time + review day + safety days), up to the lead time plus
# DC_ORDER_UP_TO_DAYS. It opens with DC_OPENING_DAYS of demand.
DC_SUPPLIER_LEAD_DAYS = (3, 7)
DC_SAFETY_DAYS = 4
DC_ORDER_UP_TO_DAYS = 14
DC_OPENING_DAYS = (10, 16)
DC_SUPPLIER_RECEIPT_HOUR = 4
# Weight of the newest day in the DC's smoothed daily demand.
DC_DEMAND_SMOOTHING = 0.1
# Online ship-from-DC orders add about this share to a DC's store demand.
DC_ONLINE_DEMAND_SHARE = 0.1

# ---------------------------------------------------------------------------
# Zone arrivals
# ---------------------------------------------------------------------------

# Arrival-surge rule: per (store, zone) over the window, alert when arrivals
# exceed zone_surge_arrivals(). The live stream (stream_events.visit_events)
# emits one customer_entered per zone a visit passes, and every visit passes
# the entrance and the exit, so those zones carry the store's whole visit
# rate. The threshold sits ZONE_SURGE_SIGMAS Poisson standard deviations above
# the busiest window a normal store can have: the busiest hour, weekday and
# month of the store type, at the largest store traffic multiplier and in the
# best weather. Peak holidays are left out on purpose: a Black Friday crowd at
# a flagship store's entrance is a surge.
ZONE_SURGE_WINDOW_MINUTES = 5
ZONE_SURGE_SIGMAS = 4.0
ZONE_SURGE_MIN_ARRIVALS = 10
# Traffic model values mirrored from the generators (a test keeps them in
# step): visits per receipt (stream_calendar.VISIT_CONVERSION_CALIBRATION),
# the largest store daily_traffic_multiplier (dims.VOLUME_CLASSES) and the
# best-weather traffic factor (stream_calendar.WEATHER_FACTOR_RANGE). They are
# repeated here so deployment can size the rules without Spark.
TRAFFIC_VISIT_CONVERSION = 0.225
TRAFFIC_STORE_MULTIPLIER_MAX = 2.5
TRAFFIC_WEATHER_FACTOR_MAX = 1.2

# ---------------------------------------------------------------------------
# High-value receipts
# ---------------------------------------------------------------------------

# A receipt is high value when its total exceeds this multiple of the store
# type's calibrated average ticket (profile.avg_ticket_target). Baskets are
# heavy-tailed (mission mix, shopper budgets, long-tail prices): in the
# simulated live stream the median ticket is about 0.6 x the average, 6% of
# tickets exceed 3 x, 1% exceed 6 x and about 0.1% exceed 10 x, so 10 x keeps
# the alert to roughly one receipt in a thousand.
HIGH_VALUE_TICKET_MULTIPLE = 10.0
HIGH_VALUE_RULE_WINDOW_MINUTES = 5
# Store type whose thresholds the checked-in fn_rule_thresholds() carries;
# deployment writes the deployed store type's values (apply_kql).
REFERENCE_STORE_TYPE = "supercenter"


def high_value_receipt_threshold(profile: StoreTypeProfile) -> float:
    """Receipt total (dollars) above which a receipt counts as high value."""
    return round(HIGH_VALUE_TICKET_MULTIPLE * profile.avg_ticket_target, 2)


def _peak_hour_share(profile: StoreTypeProfile) -> float:
    """Largest share of a store-day's visits in one local hour, over the opening patterns."""
    best = 0.0
    for hours in profile.resolved_operating_hours() or ["24h"]:
        open_hour, close_hour = parse_operating_hours(hours)
        masked = [
            weight if open_hour <= hour < close_hour else 0.0
            for hour, weight in enumerate(profile.hourly_weights)
        ]
        if sum(masked) > 0:
            best = max(best, max(masked) / sum(masked))
    return best


def peak_store_visits_per_hour(
    profile: StoreTypeProfile, transactions_per_store_day: float
) -> float:
    """Visits in the busiest hour of a store at traffic multiplier 1.

    The stream's rate model (``stream_calendar.store_visits_per_hour``) at the
    store type's busiest hour, weekday and month, without holidays or weather.
    """
    daily_peak = max(profile.daily_weights) * 7 / sum(profile.daily_weights)
    monthly_peak = max(profile.monthly_weights) * 12 / sum(profile.monthly_weights)
    return (
        transactions_per_store_day
        / TRAFFIC_VISIT_CONVERSION
        * daily_peak
        * monthly_peak
        * _peak_hour_share(profile)
    )


def zone_peak_arrivals(profile: StoreTypeProfile, transactions_per_store_day: float) -> float:
    """Expected arrivals in one surge window at the busiest zone of the busiest store."""
    return (
        peak_store_visits_per_hour(profile, transactions_per_store_day)
        * TRAFFIC_STORE_MULTIPLIER_MAX
        * TRAFFIC_WEATHER_FACTOR_MAX
        * ZONE_SURGE_WINDOW_MINUTES
        / 60.0
    )


def zone_surge_arrivals(profile: StoreTypeProfile, transactions_per_store_day: float) -> int:
    """Arrivals per (store, zone) and window above which the surge rule alerts."""
    mean = zone_peak_arrivals(profile, transactions_per_store_day)
    return max(ZONE_SURGE_MIN_ARRIVALS, ceil(mean + ZONE_SURGE_SIGMAS * sqrt(mean)))


def rule_thresholds(
    profile: StoreTypeProfile, transactions_per_store_day: float
) -> dict[str, float | int]:
    """Store-type thresholds the live rules read from ``fn_rule_thresholds()``."""
    return {
        "high_value_receipt_total": high_value_receipt_threshold(profile),
        "zone_surge_arrivals": zone_surge_arrivals(profile, transactions_per_store_day),
    }


# ---------------------------------------------------------------------------
# Anomaly detection (series_decompose_anomalies)
# ---------------------------------------------------------------------------

ANOMALY_LOOKBACK_HOURS = 168
ANOMALY_THRESHOLD = 1.5
ANOMALY_SEASONALITY = -1  # autodetect
ANOMALY_TREND = "avg"
# Hours with stock movement a store's inventory series needs before it is
# scored: the stream moves stock in bursts (sales, reorders, unloads), so a
# series with only a few active hours makes every movement an outlier.
ANOMALY_MIN_ACTIVE_HOURS = 24
# Severity by |anomaly_score|, highest band first; below the last band is "low".
ANOMALY_SEVERITY_BANDS: list[tuple[str, float]] = [
    ("critical", 3.0),
    ("high", 2.0),
    ("medium", 1.5),
]


def anomaly_severity(score: float) -> str:
    """Severity label the anomaly functions assign to an anomaly score."""
    for label, bound in ANOMALY_SEVERITY_BANDS:
        if abs(score) >= bound:
            return label
    return "low"

# --- retail_setup/generation/timezones.py ---
"""Store time zones and the local-to-UTC publication convention.

Historical generation works in each store's local wall-clock time: opening
hours, hourly traffic curves and the inventory schedule are all local. Silver
stores every event timestamp as true UTC, the same convention the live stream
uses, and ``event_date`` is the UTC date of ``event_ts``.

Each store's IANA time zone (``dim_stores.TimeZone``) comes from its
geography's state. The generation stages convert their published outputs with
the helpers below:

- store events (receipts, payments, promotions, store ops, foot traffic, BLE,
  store inventory, reorders) use their store's zone
- receipt and promo lines use the store encoded in ``receipt_id_ext``
- a shipment's whole lifecycle (truck moves, truck loads and unloads, and the
  DC ledger rows that mirror the loads) uses the destination store's zone, so
  one shipment never mixes zones
- online orders and marketing impressions are not tied to store hours and are
  generated directly in UTC; they are not converted

``fact_receipts.local_hour`` keeps the store-local hour for hour-of-day
reporting.
"""

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

# Primary IANA zone per US state (plus DC and PR). States that span two zones
# use the zone of most of their population.
STATE_TIME_ZONES: dict[str, str] = {
    "AL": "America/Chicago",
    "AK": "America/Anchorage",
    "AZ": "America/Phoenix",
    "AR": "America/Chicago",
    "CA": "America/Los_Angeles",
    "CO": "America/Denver",
    "CT": "America/New_York",
    "DC": "America/New_York",
    "DE": "America/New_York",
    "FL": "America/New_York",
    "GA": "America/New_York",
    "HI": "Pacific/Honolulu",
    "IA": "America/Chicago",
    "ID": "America/Boise",
    "IL": "America/Chicago",
    "IN": "America/Indiana/Indianapolis",
    "KS": "America/Chicago",
    "KY": "America/New_York",
    "LA": "America/Chicago",
    "MA": "America/New_York",
    "MD": "America/New_York",
    "ME": "America/New_York",
    "MI": "America/Detroit",
    "MN": "America/Chicago",
    "MO": "America/Chicago",
    "MS": "America/Chicago",
    "MT": "America/Denver",
    "NC": "America/New_York",
    "ND": "America/Chicago",
    "NE": "America/Chicago",
    "NH": "America/New_York",
    "NJ": "America/New_York",
    "NM": "America/Denver",
    "NV": "America/Los_Angeles",
    "NY": "America/New_York",
    "OH": "America/New_York",
    "OK": "America/Chicago",
    "OR": "America/Los_Angeles",
    "PA": "America/New_York",
    "PR": "America/Puerto_Rico",
    "RI": "America/New_York",
    "SC": "America/New_York",
    "SD": "America/Chicago",
    "TN": "America/Chicago",
    "TX": "America/Chicago",
    "UT": "America/Denver",
    "VA": "America/New_York",
    "VT": "America/New_York",
    "WA": "America/Los_Angeles",
    "WI": "America/Chicago",
    "WV": "America/New_York",
    "WY": "America/Denver",
}

TIME_ZONE_COLUMN = "_tz"
_TZ_STORE = "_tz_store_id"


def state_time_zone(state: str) -> str:
    """IANA time zone of a US state code."""
    try:
        return STATE_TIME_ZONES[state.strip().upper()]
    except KeyError:
        raise ValueError(f"no time zone is defined for state {state!r}") from None


def receipt_store_id(receipt_id_ext: Column) -> Column:
    """Store ID encoded in a receipt ID (``RCP``/``RET`` + yyyyMMddHHmm + store4 + seq6)."""
    return F.substring(receipt_id_ext, 16, 4).cast("long")


def with_store_time_zone(
    df: DataFrame,
    dims: dict[str, DataFrame],
    store_id: Column,
    *,
    required: Column | None = None,
) -> DataFrame:
    """Add the store's time zone as ``_tz``.

    Rows matching ``required`` (default: every row) fail when their store has
    no time zone; other rows get a NULL zone.
    """
    zones = dims["dim_stores"].select(
        F.col("ID").cast("long").alias(_TZ_STORE), F.col("TimeZone").alias(TIME_ZONE_COLUMN)
    )
    joined = df.join(F.broadcast(zones), store_id == F.col(_TZ_STORE), "left").drop(_TZ_STORE)
    zone = F.col(TIME_ZONE_COLUMN)
    missing = F.raise_error(F.lit("event references a store without a time zone"))
    checked = F.when(zone.isNotNull(), zone)
    if required is not None:
        checked = checked.when(~required, F.lit(None).cast("string"))
    return joined.withColumn(TIME_ZONE_COLUMN, checked.otherwise(missing))


def local_to_utc(df: DataFrame, columns: list[str], *, convert: Column | None = None) -> DataFrame:
    """Convert local wall-clock ``columns`` to UTC using ``_tz``.

    ``event_date`` is re-derived from the converted ``event_ts``. ``convert``
    limits the conversion to matching rows (other rows are already UTC).
    """
    for name in columns:
        utc = F.to_utc_timestamp(F.col(name), F.col(TIME_ZONE_COLUMN))
        df = df.withColumn(
            name, utc if convert is None else F.when(convert, utc).otherwise(F.col(name))
        )
    if "event_ts" in columns and "event_date" in df.columns:
        df = df.withColumn("event_date", F.to_date("event_ts"))
    return df


def localize_by_store(
    df: DataFrame,
    dims: dict[str, DataFrame],
    columns: list[str],
    *,
    store_id: Column | None = None,
    convert: Column | None = None,
) -> DataFrame:
    """Convert a frame of store-local timestamps to UTC, keeping its columns."""
    original = df.columns
    store = F.col("store_id") if store_id is None else store_id
    zoned = with_store_time_zone(df, dims, store, required=convert)
    return local_to_utc(zoned, columns, convert=convert).select(*original)


def localize_receipt_lines(df: DataFrame, dims: dict[str, DataFrame]) -> DataFrame:
    """Convert lines keyed by ``receipt_id_ext`` (receipt or promo lines) to UTC."""
    return localize_by_store(
        df, dims, ["event_ts"], store_id=receipt_store_id(F.col("receipt_id_ext"))
    )


def local_hour(event_ts: Column, time_zone: Column) -> Column:
    """Store-local hour (0-23) of a UTC timestamp."""
    return F.hour(F.from_utc_timestamp(event_ts, time_zone)).cast("int")


def with_local_hour(df: DataFrame, dims: dict[str, DataFrame]) -> DataFrame:
    """Set ``local_hour`` of a UTC store frame from its store's time zone."""
    original = df.columns
    zoned = with_store_time_zone(df, dims, F.col("store_id"))
    return zoned.withColumn(
        "local_hour", local_hour(F.col("event_ts"), F.col(TIME_ZONE_COLUMN))
    ).select(*original)

# --- retail_setup/generation/stream_calendar.py ---
"""Store calendars, holidays, weather and live-stream volume calibration.

Historical traffic (``traffic.py``) draws visitors per store, local day and
local hour from the profile's hourly, daily and monthly weights, the US
holiday calendar, the local weather, the store's ``daily_traffic_multiplier``
and its operating hours. The live stream uses the same weights, holidays and
weather, evaluated in each store's own time zone, so live volume and its
time-of-day curve continue the history.

- **Holidays.** :func:`holiday_dates` lists the US retail calendar of a year
  (New Year, Easter, Memorial Day, July 4, Labor Day, Thanksgiving, Black
  Friday, Cyber Monday, Christmas Eve, Christmas, ...). Each holiday scales a
  day's store traffic by the store type's class (:func:`store_class`: a
  hardware store's spring holidays are busier, a grocer's Thanksgiving eve,
  a general merchant's Black Friday) and online orders by its own factor. A
  factor of 0 closes the stores: Christmas for everyone, Thanksgiving for all
  but grocers.
- **Weather.** Each geography has a daily weather index, an average of the
  last ``WEATHER_MEMORY_DAYS`` days' shocks with geometric weights, so weather
  persists for days (lag-1 autocorrelation about ``WEATHER_PERSISTENCE``).
  The index moves traffic more in its climate's rough season (snow-belt
  winters, southern storm summers). :func:`severe_weather_column` marks the
  days the logistics incident model can treat as weather disruptions.

The stream is driven by a ``rate`` source. Each source row is assigned to one
lane (store visits, online orders, marketing) and is
kept with an emit probability, so the kept rows form a Poisson process with
the calibrated rate (thinning). :func:`plan_lanes` sizes the rate and the
lane shares so the emit probability never needs to exceed 1 at peak.
Inventory and logistics have no lane: they follow from the emitted purchases
(``stream_supply``).

Pure Python plus Column helpers; ``build_notebooks.py`` inlines this module
into the stream notebook.
"""

from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from math import ceil, sqrt
from zoneinfo import ZoneInfo

from pyspark.sql import Column
from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Operating hours
# ---------------------------------------------------------------------------

# Unknown formats fall back to standard hours.
DEFAULT_OPERATING_WINDOW = (6, 22)
# A 24-hour store "closes" at 23:59 and reopens at midnight, like history.
ALL_DAY_CLOSE_SECONDS = 23 * 3600 + 59 * 60


def operating_window(operating_hours: str | None) -> tuple[int, int]:
    """``[open_hour, close_hour)`` of an ``operating_hours`` value."""
    if operating_hours is None:
        return DEFAULT_OPERATING_WINDOW
    try:
        return parse_operating_hours(operating_hours)
    except ValueError:
        return DEFAULT_OPERATING_WINDOW


# ---------------------------------------------------------------------------
# Holiday calendar
# ---------------------------------------------------------------------------

# Store-type classes with their own holiday behavior (:func:`store_class`).
GENERAL, GROCERY, HARDWARE, LUXURY = "general", "grocery", "hardware", "luxury"
STORE_CLASSES = (GENERAL, GROCERY, HARDWARE, LUXURY)
# Department-name tokens and the department-weight share that make a store
# type a grocer or a home-improvement store; luxury goes by ticket size.
GROCERY_TOKENS = (
    "grocery", "fresh", "food", "produce", "dairy", "meat", "bakery", "deli", "frozen",
    "pantry", "beverage", "snack",
)  # fmt: skip
HARDWARE_TOKENS = (
    "hardware", "tool", "building", "lumber", "paint", "garden", "lawn", "plumbing",
    "electrical",
)  # fmt: skip
GROCERY_SHARE = 0.75
HARDWARE_SHARE = 0.40
LUXURY_TICKET = 500.0

# Holiday -> daily volume multiplier for (general, grocery, hardware, luxury)
# store traffic and for online orders. 0 closes stores.
HOLIDAY_EFFECTS: dict[str, tuple[float, float, float, float, float]] = {
    "NEW_YEARS_DAY": (0.55, 0.65, 0.50, 0.40, 0.90),
    "PRESIDENTS_DAY": (1.10, 1.00, 1.15, 1.10, 1.10),
    "VALENTINES_DAY": (1.05, 1.15, 1.00, 1.60, 1.10),
    "EASTER_SATURDAY": (1.15, 1.30, 1.30, 1.05, 1.00),
    "EASTER_SUNDAY": (0.55, 0.70, 0.75, 0.45, 0.80),
    "MOTHERS_DAY": (1.00, 1.10, 1.25, 1.30, 0.90),
    "MEMORIAL_SATURDAY": (1.15, 1.20, 1.45, 1.05, 1.10),
    "MEMORIAL_DAY": (1.20, 1.10, 1.60, 1.10, 1.30),
    "FATHERS_DAY": (1.05, 1.05, 1.30, 1.05, 1.00),
    "JULY_3": (1.15, 1.40, 1.15, 0.95, 1.00),
    "INDEPENDENCE_DAY": (0.75, 0.85, 0.80, 0.55, 0.80),
    "LABOR_DAY": (1.20, 1.05, 1.40, 1.10, 1.30),
    "HALLOWEEN": (1.10, 1.10, 1.00, 0.95, 1.00),
    "THANKSGIVING_EVE": (1.30, 1.80, 0.95, 0.95, 1.20),
    "THANKSGIVING": (0.00, 0.45, 0.00, 0.00, 1.60),
    "BLACK_FRIDAY": (2.20, 1.00, 1.70, 1.80, 2.20),
    "BLACK_FRIDAY_SATURDAY": (1.35, 1.05, 1.20, 1.30, 1.50),
    "CYBER_MONDAY": (1.00, 1.00, 1.00, 1.00, 3.00),
    "CHRISTMAS_EVE": (1.30, 1.50, 0.70, 1.40, 0.80),
    "CHRISTMAS": (0.00, 0.00, 0.00, 0.00, 0.50),
    "DAY_AFTER_CHRISTMAS": (1.50, 0.90, 0.95, 1.30, 1.30),
    "NEW_YEARS_EVE": (1.10, 1.30, 0.90, 0.90, 0.90),
}
_ONLINE_INDEX = len(STORE_CLASSES)


def store_class(profile: StoreTypeProfile) -> str:
    """The holiday class of a store type: luxury by ticket, else by department mix."""
    if profile.avg_ticket_target >= LUXURY_TICKET:
        return LUXURY
    total = sum(profile.department_weights.values())

    def share(tokens: tuple[str, ...]) -> float:
        return (
            sum(
                weight
                for name, weight in profile.department_weights.items()
                if any(token in name.lower() for token in tokens)
            )
            / total
        )

    if share(GROCERY_TOKENS) >= GROCERY_SHARE:
        return GROCERY
    if share(HARDWARE_TOKENS) >= HARDWARE_SHARE:
        return HARDWARE
    return GENERAL


def easter_sunday(year: int) -> date:
    """Western Easter Sunday (anonymous Gregorian algorithm)."""
    a, b, c = year % 19, year // 100, year % 100
    d, e = b // 4, b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = c // 4, c % 4
    l_ = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l_) // 451
    n = h + l_ - 7 * m + 114
    return date(year, n // 31, n % 31 + 1)


def _nth_weekday(year: int, month: int, weekday: int, n: int) -> date:
    """The ``n``-th (1-based; -1 = last) ``weekday`` (Monday 0) of a month."""
    if n > 0:
        first = date(year, month, 1)
        return first + timedelta(days=(weekday - first.weekday()) % 7 + 7 * (n - 1))
    last = date(year + month // 12, month % 12 + 1, 1) - timedelta(days=1)
    return last - timedelta(days=(last.weekday() - weekday) % 7)


# A plain dict, not functools.lru_cache: Spark workers of the setup notebooks
# (which inline this module into __main__) call it through is_closed_date, and
# cloudpickle ships lru_cache wrappers by reference.
_HOLIDAYS: dict[int, dict[date, str]] = {}


def holiday_dates(year: int) -> dict[date, str]:
    """The retail holidays of ``year`` (US calendar), by date."""
    found = _HOLIDAYS.get(year)
    if found is None:
        found = _HOLIDAYS[year] = _holiday_dates(year)
    return found


def _holiday_dates(year: int) -> dict[date, str]:
    easter = easter_sunday(year)
    memorial = _nth_weekday(year, 5, 0, -1)
    thanksgiving = _nth_weekday(year, 11, 3, 4)
    one = timedelta(days=1)
    return {
        date(year, 1, 1): "NEW_YEARS_DAY",
        _nth_weekday(year, 2, 0, 3): "PRESIDENTS_DAY",
        date(year, 2, 14): "VALENTINES_DAY",
        easter - one: "EASTER_SATURDAY",
        easter: "EASTER_SUNDAY",
        _nth_weekday(year, 5, 6, 2): "MOTHERS_DAY",
        memorial - 2 * one: "MEMORIAL_SATURDAY",
        memorial: "MEMORIAL_DAY",
        _nth_weekday(year, 6, 6, 3): "FATHERS_DAY",
        date(year, 7, 3): "JULY_3",
        date(year, 7, 4): "INDEPENDENCE_DAY",
        _nth_weekday(year, 9, 0, 1): "LABOR_DAY",
        date(year, 10, 31): "HALLOWEEN",
        thanksgiving - one: "THANKSGIVING_EVE",
        thanksgiving: "THANKSGIVING",
        thanksgiving + one: "BLACK_FRIDAY",
        thanksgiving + 2 * one: "BLACK_FRIDAY_SATURDAY",
        thanksgiving + 4 * one: "CYBER_MONDAY",
        date(year, 12, 24): "CHRISTMAS_EVE",
        date(year, 12, 25): "CHRISTMAS",
        date(year, 12, 26): "DAY_AFTER_CHRISTMAS",
        date(year, 12, 31): "NEW_YEARS_EVE",
    }


def holiday_name(day: date) -> str | None:
    """The holiday on ``day``, or None."""
    return holiday_dates(day.year).get(day)


def holiday_factor(day: date, klass: str = GENERAL) -> float:
    """Store-traffic multiplier of ``day`` for a store class (0 when closed)."""
    name = holiday_name(day)
    return 1.0 if name is None else HOLIDAY_EFFECTS[name][STORE_CLASSES.index(klass)]


def online_holiday_factor(day: date) -> float:
    """Online-order multiplier of ``day``."""
    name = holiday_name(day)
    return 1.0 if name is None else HOLIDAY_EFFECTS[name][_ONLINE_INDEX]


def is_closed_date(day: date, klass: str = GENERAL) -> bool:
    """True on dates stores of this class are closed."""
    return holiday_factor(day, klass) == 0.0


def max_holiday_factor(klass: str = GENERAL) -> float:
    """The busiest holiday's store-traffic multiplier for a class (at least 1)."""
    index = STORE_CLASSES.index(klass)
    return max(1.0, *(effects[index] for effects in HOLIDAY_EFFECTS.values()))


def max_online_holiday_factor() -> float:
    """The busiest holiday's online-order multiplier (at least 1)."""
    return max(1.0, *(effects[_ONLINE_INDEX] for effects in HOLIDAY_EFFECTS.values()))


def _easter_column(year: Column) -> Column:
    """Column form of :func:`easter_sunday` for an integer year column."""

    def div(x: Column, n: int) -> Column:
        return F.floor(x / n)

    a, b, c = F.pmod(year, F.lit(19)), div(year, 100), F.pmod(year, F.lit(100))
    d, e = div(b, 4), F.pmod(b, F.lit(4))
    g = div(b - div(b + 8, 25) + 1, 3)
    h = F.pmod(19 * a + b - d - g + 15, F.lit(30))
    i, k = div(c, 4), F.pmod(c, F.lit(4))
    l_ = F.pmod(32 + 2 * e + 2 * i - h - k, F.lit(7))
    m = div(a + 11 * h + 22 * l_, 451)
    n = h + l_ - 7 * m + 114
    return F.make_date(year, div(n, 31).cast("int"), (F.pmod(n, F.lit(31)) + 1).cast("int"))


def holiday_name_column(day: Column) -> Column:
    """Column form of :func:`holiday_name` for a date column (NULL off holidays)."""

    def name(parts: Column) -> Column:
        m, dom, wd = parts["m"], parts["dom"], parts["wd"]
        week = F.floor((dom - 1) / 7) + 1

        def fixed(month: int, day_of_month: int) -> Column:
            return (m == month) & (dom == day_of_month)

        def nth(month: int, weekday: int, n: int) -> Column:
            return (m == month) & (wd == weekday) & (week == n)

        def between(month: int, weekday: int, first: int, last: int) -> Column:
            return (m == month) & (wd == weekday) & (dom >= first) & (dom <= last)

        # ISO weekdays (Monday 1). Thanksgiving is Thursday 22-28 November.
        rules = [
            (fixed(1, 1), "NEW_YEARS_DAY"),
            (nth(2, 1, 3), "PRESIDENTS_DAY"),
            (fixed(2, 14), "VALENTINES_DAY"),
            (F.date_add(parts["day"], 1) == parts["easter"], "EASTER_SATURDAY"),
            (parts["day"] == parts["easter"], "EASTER_SUNDAY"),
            (nth(5, 7, 2), "MOTHERS_DAY"),
            (between(5, 6, 23, 29), "MEMORIAL_SATURDAY"),
            (between(5, 1, 25, 31), "MEMORIAL_DAY"),
            (nth(6, 7, 3), "FATHERS_DAY"),
            (fixed(7, 3), "JULY_3"),
            (fixed(7, 4), "INDEPENDENCE_DAY"),
            (nth(9, 1, 1), "LABOR_DAY"),
            (fixed(10, 31), "HALLOWEEN"),
            (between(11, 3, 21, 27), "THANKSGIVING_EVE"),
            (between(11, 4, 22, 28), "THANKSGIVING"),
            (between(11, 5, 23, 29), "BLACK_FRIDAY"),
            (between(11, 6, 24, 30), "BLACK_FRIDAY_SATURDAY"),
            (between(11, 1, 26, 30) | between(12, 1, 1, 2), "CYBER_MONDAY"),
            (fixed(12, 24), "CHRISTMAS_EVE"),
            (fixed(12, 25), "CHRISTMAS"),
            (fixed(12, 26), "DAY_AFTER_CHRISTMAS"),
            (fixed(12, 31), "NEW_YEARS_EVE"),
        ]
        return case_when((condition, F.lit(label)) for condition, label in rules)

    parts = F.struct(
        day.alias("day"),
        F.month(day).alias("m"),
        F.dayofmonth(day).alias("dom"),
        (F.pmod(F.dayofweek(day) + 5, F.lit(7)) + 1).alias("wd"),
        _easter_column(F.year(day)).alias("easter"),
    )
    return bind(parts, name)


def _holiday_lookup(day: Column, index: int) -> Column:
    def factor(name: Column) -> Column:
        branches = [
            (name == F.lit(holiday), F.lit(float(effects[index])))
            for holiday, effects in HOLIDAY_EFFECTS.items()
            if effects[index] != 1.0
        ]
        return case_when(branches).otherwise(F.lit(1.0))

    return bind(holiday_name_column(day), factor)


def holiday_factor_column(day: Column, klass: str = GENERAL) -> Column:
    """Column form of :func:`holiday_factor` for a date column."""
    return _holiday_lookup(day, STORE_CLASSES.index(klass))


def online_holiday_factor_column(day: Column) -> Column:
    """Column form of :func:`online_holiday_factor` for a date column."""
    return _holiday_lookup(day, _ONLINE_INDEX)


def closed_date_column(local_day: Column, klass: str = GENERAL) -> Column:
    """Column form of :func:`is_closed_date`."""
    return holiday_factor_column(local_day, klass) == F.lit(0.0)


# ---------------------------------------------------------------------------
# Weather
# ---------------------------------------------------------------------------

# A geography's weather index weights the last WEATHER_MEMORY_DAYS daily shocks
# by WEATHER_PERSISTENCE ** age, so consecutive days are correlated.
WEATHER_PERSISTENCE = 0.7
WEATHER_MEMORY_DAYS = 8
# Climates by state; other states are "temperate".
WEATHER_CLIMATES: dict[str, frozenset[str]] = {
    "cold": frozenset(
        {
            "AK",
            "CO",
            "CT",
            "IA",
            "ID",
            "IL",
            "IN",
            "MA",
            "ME",
            "MI",
            "MN",
            "MT",
            "ND",
            "NE",
            "NH",
            "NY",
            "OH",
            "PA",
            "RI",
            "SD",
            "UT",
            "VT",
            "WI",
            "WY",
        }
    ),  # fmt: skip
    "hot": frozenset({"AL", "AR", "AZ", "FL", "GA", "LA", "MS", "NM", "NV", "OK", "SC", "TX"}),
    "mild": frozenset({"CA", "HI", "OR", "WA"}),
}
# Log-scale traffic sensitivity to the weather index by climate and month
# (January first): snow-belt winters and southern storm seasons move traffic most.
WEATHER_SIGMA: dict[str, list[float]] = {
    "cold": [0.22, 0.22, 0.16, 0.10, 0.08, 0.08, 0.08, 0.08, 0.08, 0.10, 0.14, 0.20],
    "hot": [0.08, 0.08, 0.08, 0.10, 0.12, 0.15, 0.15, 0.16, 0.16, 0.10, 0.08, 0.08],
    "mild": [0.10, 0.10, 0.08, 0.06, 0.05, 0.04, 0.04, 0.04, 0.05, 0.06, 0.08, 0.10],
    "temperate": [0.15, 0.15, 0.12, 0.10, 0.08, 0.08, 0.08, 0.08, 0.08, 0.09, 0.11, 0.14],
}
# Weather scales traffic within this range: bad days lose more than good days gain.
WEATHER_FACTOR_RANGE = (0.4, 1.2)
# An index at or above this marks a severe-weather day (about 2% of days).
SEVERE_WEATHER_INDEX = 2.0


def weather_climate(state: str | None) -> str:
    """The climate of a state code (``temperate`` when unknown)."""
    for climate, states in WEATHER_CLIMATES.items():
        if state in states:
            return climate
    return "temperate"


def weather_lag1_autocorrelation() -> float:
    """Lag-1 autocorrelation of the weather index implied by its weights."""
    weights = [WEATHER_PERSISTENCE**age for age in range(WEATHER_MEMORY_DAYS)]
    return sum(a * b for a, b in zip(weights, weights[1:])) / sum(w * w for w in weights)


def weather_index_column(draws: seeded_draws, geography_id: Column, day: Column) -> Column:
    """A geography's standard-normal weather index on a date (higher is worse)."""
    weights = [WEATHER_PERSISTENCE**age for age in range(WEATHER_MEMORY_DAYS)]
    norm = sqrt(sum(w * w for w in weights))
    geography = geography_id.cast("long")
    total: Column = F.lit(0.0)
    for age, weight in enumerate(weights):
        shock = draws.gauss([geography, F.date_sub(day, age)], "weather")
        total = total + F.lit(weight / norm) * shock
    return total


def _climate_column(state: Column) -> Column:
    return case_when(
        (state.isin(*sorted(states)), F.lit(climate))
        for climate, states in WEATHER_CLIMATES.items()
    ).otherwise(F.lit("temperate"))


def weather_factor_column(
    draws: seeded_draws, geography_id: Column, state: Column, day: Column
) -> Column:
    """Traffic multiplier of a geography's weather on a date (mean about 1)."""
    climate = _climate_column(state)
    sigma = case_when(
        (climate == F.lit(name), F.element_at(F.array(*[F.lit(v) for v in values]), F.month(day)))
        for name, values in WEATHER_SIGMA.items()
    )
    low, high = WEATHER_FACTOR_RANGE

    def factor(inputs: Column) -> Column:
        s = inputs["sigma"]
        return F.least(F.lit(high), F.greatest(F.lit(low), F.exp(-s * inputs["index"] - s * s / 2)))

    return bind(
        F.struct(
            sigma.alias("sigma"),
            weather_index_column(draws, geography_id, day).alias("index"),
        ),
        factor,
    )


def severe_weather_column(draws: seeded_draws, geography_id: Column, day: Column) -> Column:
    """True on a geography's severe-weather days.

    The hook for logistics weather incidents: the incident model can open a
    ``WEATHER`` incident on the lanes into a geography on these days, so
    deliveries slow when store traffic drops.
    """
    return weather_index_column(draws, geography_id, day) >= F.lit(SEVERE_WEATHER_INDEX)


def _close_seconds(close_hour: int) -> int:
    return ALL_DAY_CLOSE_SECONDS if close_hour == 24 else close_hour * 3600


# ---------------------------------------------------------------------------
# Traffic weights (the historical model in traffic.py)
# ---------------------------------------------------------------------------

# traffic.py draws visitors at ``transactions_per_store_day / 0.225`` and
# converts about this share of them into receipts.
VISIT_CONVERSION_CALIBRATION = 0.225
# Visit conversion: logistic in a store effect and a store/day/hour effect.
CONVERSION_LOGIT_INTERCEPT = 1.2657
CONVERSION_STORE_SD = 0.25
CONVERSION_HOUR_SD = 0.20


def hour_shares(hourly_weights: list[float], operating_hours: str | None) -> list[float]:
    """Share of a store-day's visits in each local hour (0 while closed)."""
    open_hour, close_hour = operating_window(operating_hours)
    masked = [w if open_hour <= h < close_hour else 0.0 for h, w in enumerate(hourly_weights)]
    total = sum(masked)
    return [w / total if total > 0 else 0.0 for w in masked]


def daily_factors(profile: StoreTypeProfile) -> list[float]:
    """Day-of-week multipliers, Monday first, averaging 1."""
    return [w * 7 / sum(profile.daily_weights) for w in profile.daily_weights]


def monthly_factors(profile: StoreTypeProfile) -> list[float]:
    """Month multipliers, January first, averaging 1."""
    return [w * 12 / sum(profile.monthly_weights) for w in profile.monthly_weights]


def online_hour_factors() -> list[float]:
    """Online-order multiplier of each UTC hour, averaging 1."""
    total = sum(ONLINE_HOURLY_WEIGHTS)
    return [w * 24 / total for w in ONLINE_HOURLY_WEIGHTS]


def online_hour_factor_column(ts: Column) -> Column:
    """Column form of :func:`online_hour_factors` for a UTC timestamp column."""
    return F.element_at(F.array(*[F.lit(w) for w in online_hour_factors()]), F.hour(ts) + 1)


def daily_factor_column(profile: StoreTypeProfile, day: Column) -> Column:
    """Column form of :func:`daily_factors` for a date column."""
    return F.element_at(
        F.array(*[F.lit(w) for w in daily_factors(profile)]),
        F.pmod(F.dayofweek(day) + 5, F.lit(7)) + 1,
    )


def monthly_factor_column(profile: StoreTypeProfile, day: Column) -> Column:
    """Column form of :func:`monthly_factors` for a date column."""
    return F.element_at(F.array(*[F.lit(w) for w in monthly_factors(profile)]), F.month(day))


def visit_conversion(draws: seeded_draws, store_id: Column, day: Column, hour: Column) -> Column:
    """Probability that a visit in this store, local day and local hour buys."""
    return 1.0 / (
        1.0
        + F.exp(
            F.lit(CONVERSION_LOGIT_INTERCEPT)
            - CONVERSION_STORE_SD * draws.gauss([store_id], "conversion-store")
            - CONVERSION_HOUR_SD * draws.gauss([store_id, day, hour], "conversion-hour")
        )
    )


def store_visits_per_hour(
    profile: StoreTypeProfile,
    transactions_per_store_day: float,
    multiplier: float,
    operating_hours: str | None,
    local: datetime,
) -> float:
    """Expected visits in the local hour starting at ``local`` (weather excluded)."""
    holiday = holiday_factor(local.date(), store_class(profile))
    if holiday == 0.0:
        return 0.0
    return (
        transactions_per_store_day
        / VISIT_CONVERSION_CALIBRATION
        * holiday
        * daily_factors(profile)[local.weekday()]
        * monthly_factors(profile)[local.month - 1]
        * multiplier
        * hour_shares(profile.hourly_weights, operating_hours)[local.hour]
    )


# ---------------------------------------------------------------------------
# Store open/close boundaries
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class StoreHours:
    store_id: int
    operating_hours: str
    time_zone: str
    # Holiday class of the store type (closures), see :func:`store_class`.
    store_class: str = GENERAL


def store_boundaries(
    store: StoreHours, after: datetime, until: datetime
) -> list[tuple[datetime, str]]:
    """Open and close instants (UTC) of ``store`` in ``(after, until]``, in time order.

    Hours are local wall-clock times, so the UTC instants move with daylight
    saving time. Every open day has exactly one ``opened`` and one ``closed``.
    """
    if after.tzinfo is None or until.tzinfo is None:
        raise ValueError("boundary window must be timezone-aware")
    zone = ZoneInfo(store.time_zone)
    open_hour, close_hour = operating_window(store.operating_hours)
    day = after.astimezone(zone).date() - timedelta(days=1)
    last = until.astimezone(zone).date()
    found: list[tuple[datetime, str]] = []
    while day <= last:
        if not is_closed_date(day, store.store_class):
            for seconds, operation in (
                (open_hour * 3600, "opened"),
                (_close_seconds(close_hour), "closed"),
            ):
                # Wall-clock arithmetic: add the offset to local midnight, then
                # normalize through UTC so DST days keep the local hour.
                local = datetime(day.year, day.month, day.day) + timedelta(seconds=seconds)
                instant = local.replace(tzinfo=zone).astimezone(timezone.utc)
                if after < instant <= until:
                    found.append((instant, operation))
        day += timedelta(days=1)
    return sorted(found)


def is_open_at(store: StoreHours, instant: datetime) -> bool:
    """True when ``store`` is open at the UTC ``instant``."""
    local = instant.astimezone(ZoneInfo(store.time_zone))
    if is_closed_date(local.date(), store.store_class):
        return False
    open_hour, close_hour = operating_window(store.operating_hours)
    second_of_day = local.hour * 3600 + local.minute * 60 + local.second
    return open_hour * 3600 <= second_of_day < _close_seconds(close_hour)


# ---------------------------------------------------------------------------
# Lane plan: rate-source sizing and per-lane emit probabilities
# ---------------------------------------------------------------------------

# Marketing volumes in vocabulary.CAMPAIGN_ARCHETYPES are per day at this
# legacy fleet size (marketing.py scales them by store_count / 86).
MARKETING_LEGACY_FLEET_SIZE = 86

STREAM_LANES = ("visits", "online", "marketing")


@dataclass(frozen=True)
class LanePlan:
    """How rate-source rows are split into lanes and thinned.

    ``shares[lane]`` is the fraction of source rows assigned to a lane (the
    rest are idle). A lane row is kept with probability
    ``min(1, factors[lane] * time_factor)``, where the time factor is the
    lane's own calendar weight at the row's time (1 for flat lanes).
    """

    rows_per_second: float
    shares: dict[str, float]
    factors: dict[str, float]
    peaks: dict[str, float]
    clipped: bool


def _visit_peak_factor(profile: StoreTypeProfile, operating_hours: list[str]) -> float:
    patterns = sorted(set(operating_hours)) or ["24h"]
    peak_share = max(max(hour_shares(profile.hourly_weights, hours)) for hours in patterns)
    return max(daily_factors(profile)) * max(monthly_factors(profile)) * peak_share


def plan_lanes(
    profile: StoreTypeProfile,
    *,
    store_count: int,
    multiplier_total: float,
    operating_hours: list[str],
    transactions_per_store_day: float,
    online_orders_per_day: float,
    marketing_impressions_per_day: float,
    volume_scale: float = 1.0,
    rows_per_second: float | None = None,
    growth: float = 1.0,
    calendar_peaks: bool = True,
) -> LanePlan:
    """Size the rate source and the lanes from the historical configuration.

    ``multiplier_total`` is the sum of the stores' ``daily_traffic_multiplier``
    and ``marketing_impressions_per_day`` the legacy-fleet total of every
    campaign archetype. The default rate is the smallest whole number of rows
    per second that covers the peak of every lane; an explicit
    ``rows_per_second`` only changes the resolution, not the volume, unless it
    is below the peak (``clipped``). ``growth`` is the volume trend multiplier
    (``pricing.volume_growth``) of the stream's dates; the lanes apply it per
    row, so it only sizes the peaks. ``calendar_peaks`` sizes the peaks for the
    busiest holiday and the best weather too; without it those rare hours are
    clipped.
    """
    if volume_scale <= 0:
        raise ValueError("volume_scale must be positive")
    visit_peak_factor = _visit_peak_factor(profile, operating_hours)
    online_peak_factor = max(monthly_factors(profile)) * max(online_hour_factors())
    if calendar_peaks:
        visit_peak_factor *= max_holiday_factor(store_class(profile)) * WEATHER_FACTOR_RANGE[1]
        online_peak_factor *= max_online_holiday_factor()
    visits_per_second = (
        transactions_per_store_day / VISIT_CONVERSION_CALIBRATION / 3600.0 * multiplier_total
    )
    online_per_second = online_orders_per_day / 86400.0
    marketing_per_second = (
        store_count / MARKETING_LEGACY_FLEET_SIZE * marketing_impressions_per_day / 86400.0
    )
    # Mean events per second of each lane at time factor 1, and its peak factor.
    base = {
        "visits": (visits_per_second, visit_peak_factor * growth),
        "online": (online_per_second, online_peak_factor * growth),
        "marketing": (marketing_per_second, 1.0),
    }
    peaks = {lane: rate * peak * volume_scale for lane, (rate, peak) in base.items()}
    total_peak = sum(peaks.values())
    rate = float(rows_per_second) if rows_per_second else float(max(1, ceil(total_peak)))
    if rate <= 0:
        raise ValueError("rows_per_second must be positive")
    capacity = max(rate, total_peak)
    shares = {lane: peaks[lane] / capacity for lane in STREAM_LANES}
    factors = {
        lane: (base[lane][0] * volume_scale / (rate * shares[lane]) if shares[lane] > 0 else 0.0)
        for lane in STREAM_LANES
    }
    return LanePlan(rate, shares, factors, peaks, rate < total_peak)

# --- retail_setup/generation/pricing.py ---
"""Shelf prices over time and the demand response to them.

Each SKU's shelf price is a pure function of the seed, the SKU and the week,
shared by history (receipts, online orders, the published price history) and
the live stream:

- **Trend.** Regular prices follow a per-department inflation trend
  (``DEPARTMENT_PRICING``), anchored on ``dim_products.SalePrice`` at the
  configuration's anchor date.
- **Price reviews.** Every ``PRICE_REVIEW_WEEKS`` weeks (each SKU on its own
  phase) the regular price is reset within ``REGULAR_PRICE_SPREAD`` of trend.
- **Markdowns.** ``MARKDOWN_PROBABILITY`` of review periods carry a one-to-three
  week markdown of 10-35%.
- **Price endings.** Regular prices end in .49 or .99 (9 cents below $1,
  :func:`charm_cents`); markdown prices end in 9 cents and always undercut the
  regular price (:func:`markdown_cents`).

Volume also trends: store traffic and online orders compound
``annual_volume_growth`` a year from the anchor date (:func:`volume_growth`).

Demand responds to price: within a department, a SKU's choice weight scales
with ``(price / trend price) ** elasticity``, with department elasticities
between about -0.8 (staples) and -2.5 (electronics). Markdowns and expensive
review periods therefore move units, which is what elasticity models measure.

Weeks count from ``PRICE_EPOCH`` (a Monday), not from the history window, so
the same SKU has the same price on the same day in every run.
"""

from dataclasses import dataclass
from datetime import date
from math import floor

from pyspark.sql import Column
from pyspark.sql import functions as F


PRICE_EPOCH = date(2000, 1, 3)
PRICE_REVIEW_WEEKS = 13
REGULAR_PRICE_SPREAD = 0.06
MARKDOWN_PROBABILITY = 0.3
MARKDOWN_WEEKS = (1, 3)
MARKDOWN_DEPTH = (0.10, 0.35)
DAYS_PER_YEAR = 365.25

# (department-name token, annual inflation, price elasticity of choice). The
# first token found in a lower-cased department name applies.
DEPARTMENT_PRICING: list[tuple[str, float, float]] = [
    ("electronic", -0.03, -2.5),
    ("jewel", 0.04, -0.8),
    ("watch", 0.04, -0.8),
    ("toy", 0.02, -2.0),
    ("seasonal", 0.02, -2.2),
    ("footwear", 0.02, -1.8),
    ("apparel", 0.02, -1.8),
    ("clothing", 0.02, -1.8),
    ("accessor", 0.03, -1.4),
    ("sport", 0.02, -1.8),
    ("garden", 0.03, -1.6),
    ("home", 0.025, -1.6),
    ("automotive", 0.03, -1.2),
    ("office", 0.02, -1.4),
    ("tool", 0.035, -1.2),
    ("building", 0.04, -1.0),
    ("electrical", 0.035, -1.1),
    ("plumbing", 0.035, -1.1),
    ("paint", 0.035, -1.2),
    ("hardware", 0.035, -1.2),
    ("pet", 0.03, -1.0),
    ("baby", 0.03, -0.9),
    ("health", 0.03, -1.1),
    ("beauty", 0.03, -1.3),
    ("household", 0.03, -1.0),
    ("fresh", 0.035, -0.9),
    ("grocery", 0.03, -0.8),
]
DEFAULT_INFLATION = 0.025
DEFAULT_ELASTICITY = -1.5

# (department-name token, gross margin low, high): the share of the sale price
# above cost, drawn per SKU (dims.compute_pricing). First token found applies.
DEPARTMENT_MARGINS: list[tuple[str, float, float]] = [
    ("electronic", 0.10, 0.25),
    ("jewel", 0.45, 0.65),
    ("watch", 0.40, 0.55),
    ("fragrance", 0.40, 0.60),
    ("toy", 0.30, 0.45),
    ("seasonal", 0.35, 0.55),
    ("footwear", 0.40, 0.55),
    ("apparel", 0.45, 0.60),
    ("clothing", 0.40, 0.55),
    ("accessor", 0.45, 0.60),
    ("sport", 0.30, 0.45),
    ("garden", 0.30, 0.45),
    ("home", 0.30, 0.50),
    ("automotive", 0.25, 0.40),
    ("office", 0.30, 0.50),
    ("tool", 0.30, 0.45),
    ("building", 0.20, 0.35),
    ("electrical", 0.30, 0.45),
    ("plumbing", 0.30, 0.45),
    ("paint", 0.40, 0.55),
    ("hardware", 0.35, 0.50),
    ("pet", 0.25, 0.40),
    ("baby", 0.20, 0.35),
    ("health", 0.25, 0.40),
    ("beauty", 0.35, 0.55),
    ("household", 0.20, 0.35),
    ("fresh", 0.25, 0.40),
    ("grocery", 0.15, 0.30),
]
DEFAULT_MARGIN = (0.25, 0.45)


def margin_band(department: str) -> tuple[float, float]:
    """(low, high) gross margin of a department name."""
    name = department.lower()
    for token, low, high in DEPARTMENT_MARGINS:
        if token in name:
            return low, high
    return DEFAULT_MARGIN


def charm_cents(cents: float) -> int:
    """The nearest charm price: .49 or .99 from $1, a 9-cent ending below it."""
    if cents >= 100:
        return max(99, int(floor((cents + 1) / 50 + 0.5)) * 50 - 1)
    return max(9, int(floor((cents + 1) / 10 + 0.5)) * 10 - 1)


def markdown_cents(cents: float, regular_cents: int) -> int:
    """A markdown price: the nearest 9-cent ending, below the regular price."""
    return max(1, min(regular_cents - 1, int(floor((cents + 1) / 10 + 0.5)) * 10 - 1))


def charm_cents_column(cents: Column) -> Column:
    """Column form of :func:`charm_cents` (long)."""
    return (
        F.when(
            cents >= 100, F.greatest(F.lit(99), F.floor((cents + 1) / 50 + 0.5) * 50 - 1)
        ).otherwise(F.greatest(F.lit(9), F.floor((cents + 1) / 10 + 0.5) * 10 - 1))
    ).cast("long")


def markdown_cents_column(cents: Column, regular_cents: Column) -> Column:
    """Column form of :func:`markdown_cents` (long)."""
    dime = F.floor((cents + 1) / 10 + 0.5) * 10 - 1
    return F.greatest(F.lit(1), F.least(regular_cents - 1, dime)).cast("long")


def department_pricing(department: str) -> tuple[float, float]:
    """(annual inflation, elasticity) of a department name."""
    name = department.lower()
    for token, inflation, elasticity in DEPARTMENT_PRICING:
        if token in name:
            return inflation, elasticity
    return DEFAULT_INFLATION, DEFAULT_ELASTICITY


@dataclass(frozen=True)
class PriceSchedule:
    """Price schedule anchored on ``dim_products.SalePrice`` at ``anchor_date``."""

    seed: int
    anchor_date: date


@dataclass(frozen=True)
class ShelfPrice:
    """A SKU's prices on one day (pure-Python form of :func:`shelf_price_struct`)."""

    regular_cents: int
    price_cents: int
    trend_cents: float
    markdown: bool


def week_index(day: date) -> int:
    return (day - PRICE_EPOCH).days // 7


def markdown_window(
    review_draw: float, length_draw: float, start_draw: float
) -> tuple[int, int] | None:
    """``[first, last)`` weeks into a review period of its markdown, or None."""
    if review_draw >= MARKDOWN_PROBABILITY:
        return None
    low, high = MARKDOWN_WEEKS
    length = low + int(length_draw * (high - low + 1))
    first = int(start_draw * (PRICE_REVIEW_WEEKS - length + 1))
    return first, first + length


def shelf_price(
    base_cents: int,
    inflation: float,
    anchor: date,
    day: date,
    phase: int,
    review_draw: float,
    markdown_draws: tuple[float, float, float, float],
) -> ShelfPrice:
    """A SKU's regular and shelf price on ``day`` given its draws.

    ``phase`` offsets the SKU's review calendar; ``review_draw`` sets the
    period's regular price within the spread; ``markdown_draws`` are (has
    markdown, length, start, depth).
    """
    week = week_index(day)
    period = (week + phase) // PRICE_REVIEW_WEEKS
    period_start = period * PRICE_REVIEW_WEEKS - phase
    years = (period_start * 7 + (PRICE_EPOCH - anchor).days) / DAYS_PER_YEAR
    trend_years = (day - anchor).days / DAYS_PER_YEAR
    regular = (
        base_cents * (1 + inflation) ** years * (1 + REGULAR_PRICE_SPREAD * (2 * review_draw - 1))
    )
    window = markdown_window(*markdown_draws[:3])
    marked = window is not None and window[0] <= week - period_start < window[1]
    low, high = MARKDOWN_DEPTH
    depth = low + markdown_draws[3] * (high - low) if marked else 0.0
    regular_cents = charm_cents(regular)
    return ShelfPrice(
        regular_cents=regular_cents,
        price_cents=markdown_cents(regular * (1 - depth), regular_cents)
        if marked
        else regular_cents,
        trend_cents=base_cents * (1 + inflation) ** trend_years,
        markdown=marked,
    )


def _department_column(department: Column, index: int, default: float) -> Column:
    branches = [
        (F.lower(department).contains(token), F.lit(values[index]))
        for token, *values in DEPARTMENT_PRICING
    ]
    return case_when(branches).otherwise(F.lit(default))


def inflation_column(department: Column) -> Column:
    return _department_column(department, 0, DEFAULT_INFLATION)


def elasticity_column(department: Column) -> Column:
    return _department_column(department, 1, DEFAULT_ELASTICITY)


def shelf_price_struct(
    schedule: PriceSchedule,
    product_id: Column,
    base_price: Column,
    inflation: Column,
    day: Column,
) -> Column:
    """``struct<regular_cents long, price_cents long, trend_cents double, markdown boolean>``.

    Column form of :func:`shelf_price` for a SKU (``base_price`` in dollars,
    ``dim_products.SalePrice``; ``inflation`` its department's yearly trend,
    :func:`inflation_column`) on a date column.
    """
    draws = seeded_draws(schedule.seed)
    epoch_offset = (PRICE_EPOCH - schedule.anchor_date).days
    base_cents = F.round(base_price * 100)
    growth = F.lit(1.0) + inflation
    week = F.floor(F.datediff(day, F.lit(PRICE_EPOCH)) / 7)
    phase = draws.h64([product_id], "price-phase") % PRICE_REVIEW_WEEKS
    calendar = F.struct(
        week.alias("week"),
        F.floor((week + phase) / PRICE_REVIEW_WEEKS).alias("period"),
        phase.alias("phase"),
    )

    def in_period(c: Column) -> Column:
        period_start = c["period"] * PRICE_REVIEW_WEEKS - c["phase"]
        keys = [product_id, c["period"]]
        period_draws = F.struct(
            draws.u(keys, "price-review").alias("review"),
            draws.u(keys, "markdown").alias("markdown"),
            draws.u(keys, "markdown-length").alias("length"),
            draws.u(keys, "markdown-start").alias("start"),
            draws.u(keys, "markdown-depth").alias("depth"),
        )

        def price(r: Column) -> Column:
            years = (period_start * 7 + F.lit(epoch_offset)) / DAYS_PER_YEAR
            low, high = MARKDOWN_WEEKS
            length = low + F.floor(r["length"] * (high - low + 1))
            first = F.floor(r["start"] * (PRICE_REVIEW_WEEKS - length + 1))
            into = c["week"] - period_start
            marked = (
                (r["markdown"] < F.lit(MARKDOWN_PROBABILITY))
                & (into >= first)
                & (into < first + length)
            )
            depth_low, depth_high = MARKDOWN_DEPTH
            return F.struct(
                (
                    base_cents
                    * F.pow(growth, years)
                    * (1 + REGULAR_PRICE_SPREAD * (2 * r["review"] - 1))
                ).alias("regular"),
                F.when(marked, F.lit(depth_low) + r["depth"] * (depth_high - depth_low))
                .otherwise(F.lit(0.0))
                .alias("depth"),
                marked.alias("markdown"),
            )

        return bind(period_draws, price)

    def shelf(r: Column) -> Column:
        trend_years = F.datediff(day, F.lit(schedule.anchor_date)) / DAYS_PER_YEAR
        regular_cents = charm_cents_column(r["regular"])
        return F.struct(
            regular_cents.alias("regular_cents"),
            F.when(
                r["markdown"],
                markdown_cents_column(r["regular"] * (1 - r["depth"]), regular_cents),
            )
            .otherwise(regular_cents)
            .alias("price_cents"),
            (base_cents * F.pow(growth, trend_years)).alias("trend_cents"),
            r["markdown"].alias("markdown"),
        )

    # Each step binds its inputs once, which keeps the expression small.
    return bind(bind(calendar, in_period), shelf)


def demand_response(price_cents: Column, trend_cents: Column, elasticity: Column) -> Column:
    """Choice-weight multiplier of a SKU priced off its trend: ``(price / trend) ** elasticity``."""
    return F.pow(price_cents / trend_cents, elasticity)


def volume_growth(annual_growth: float, anchor: date, day: date) -> float:
    """Year-over-year volume trend: 1 at ``anchor``, compounding ``annual_growth`` a year."""
    return (1.0 + annual_growth) ** ((day - anchor).days / DAYS_PER_YEAR)


def volume_growth_column(annual_growth: float, anchor: date, day: Column) -> Column:
    """Column form of :func:`volume_growth` for a date column."""
    return F.pow(F.lit(1.0 + annual_growth), F.datediff(day, F.lit(anchor)) / DAYS_PER_YEAR)

# --- retail_setup/generation/behavior.py ---
"""Shared historical/live shopping behavior, expressed as deterministic Spark columns."""

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def repeat_purchase_probability(department: Column) -> Column:
    """Consumables permit multi-unit purchases more often than durable goods."""
    consumable = F.lower(department).rlike(
        "grocery|fresh|food|household|personal|health|beauty|pet|baby"
    )
    return F.when(consumable, F.lit(0.22)).otherwise(F.lit(0.04))


def retail_quantity(draws: seeded_draws, keys: list, department: Column) -> Column:
    return (
        F.lit(1)
        + F.floor(
            F.log(F.lit(1.0) - draws.u(keys, "retail-quantity"))
            / F.log(repeat_purchase_probability(department))
        )
    ).cast("int")


def retail_basket_size(
    draws: seeded_draws,
    purchase_keys: list,
    shopper: Column,
    mean: Column,
    offer: Column,
    promo_rate: float,
) -> Column:
    """Mission mixtures with a calibrated mean and persistent customer budgets.

    ``shopper`` keys the budget: the customer (``customers.shopper_key``), so
    a customer's baskets share one budget, or the purchase when anonymous.
    """
    trip = draws.u(purchase_keys, "retail-mission")
    mission = (
        F.when(trip < 0.30, 0.30).when(trip < 0.70, 0.90).when(trip < 0.90, 1.60).otherwise(2.50)
    ) / F.lit(1.02)
    budget = F.exp(0.35 * draws.gauss([shopper], "retail-budget") - F.lit(0.35**2 / 2))
    # Exposure is decided before demand: coupons increase additional-item demand.
    lift = (F.lit(1.0) + offer.cast("double") * 0.20) / (1.0 + promo_rate * 0.20)
    extra = F.greatest(mean - 1.0, F.lit(0.0)) * mission * budget * lift
    return (1 + draws.negative_binomial(purchase_keys, "retail-basket", extra, 3)).cast("int")


def retail_catalog(
    products: DataFrame, dates: DataFrame, draws: seeded_draws, schedule: PriceSchedule
) -> DataFrame:
    """One launch-eligible weighted product CDF per day and department.

    Popularity is persistent per SKU, with cross-sectional price sensitivity,
    a long tail, and a demand response to the SKU's shelf price that day
    (markdowns and price reviews, ``pricing.demand_response``). Each entry
    carries the day's shelf price (``unit_cents``) and the SKU's complementary
    partner priced the same day (``baskets.basket_products``).
    Arrays keep selection a bounded lookup rather than a receipt x catalog join.
    """
    eligible = (
        dates.select("event_date")
        .distinct()
        .join(F.broadcast(products), F.col("launch_date") <= F.col("event_date"))
    )
    day = F.col("event_date")
    eligible = eligible.withColumn(
        "_price",
        shelf_price_struct(
            schedule, F.col("product_id"), F.col("SalePrice"), F.col("inflation"), day
        ),
    ).withColumn(
        "_partner_price",
        shelf_price_struct(
            schedule, F.col("partner_id"), F.col("partner_price"), F.col("inflation"), day
        ),
    )
    eligible = eligible.withColumn(
        "_popularity",
        F.exp(0.8 * draws.gauss(["product_id"], "retail-popularity") - 0.32)
        / F.pow(F.col("SalePrice"), 1.15)
        * demand_response(
            F.col("_price.price_cents"), F.col("_price.trend_cents"), F.col("elasticity")
        ),
    ).withColumn("unit_cents", F.col("_price.price_cents"))
    # Floating-point sums run in product_id order over the whole department, so
    # they are bit-identical whatever order the shuffle delivers rows in.
    by_product = Window.partitionBy("event_date", "department").orderBy("product_id")
    ordered = by_product.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    whole = by_product.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
    # Calibration uses the default state rule: clothing (REDUCED_RATE) pays
    # the full rate and groceries (NON_TAXABLE) none (baskets.line_tax_cents).
    taxable_share = F.when(F.col("taxability").isin("TAXABLE", "REDUCED_RATE"), 1.0).otherwise(0.0)
    eligible = (
        eligible.withColumn("_total_weight", F.sum("_popularity").over(whole))
        .withColumn("_probability", F.col("_popularity") / F.col("_total_weight"))
        .withColumn("_cdf", F.sum("_probability").over(ordered))
        .withColumn(
            "_unit_value",
            F.col("unit_cents") / 100.0 / (1.0 - repeat_purchase_probability(F.col("department"))),
        )
        .withColumn(
            "_expected_line_value",
            F.sum(F.col("_probability") * F.col("_unit_value")).over(whole),
        )
        .withColumn(
            "_expected_taxable_value",
            F.sum(F.col("_probability") * F.col("_unit_value") * taxable_share).over(whole),
        )
        .withColumn(
            "partner_ok",
            F.col("partner_id").isNotNull() & (F.col("partner_launch") <= F.col("event_date")),
        )
        .withColumn("partner_cents", F.col("_partner_price.price_cents"))
    )
    return eligible.groupBy("event_date", "department").agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    "_cdf",
                    "product_id",
                    "SalePrice",
                    "taxability",
                    "unit_cents",
                    "partner_id",
                    "partner_cents",
                    "partner_taxability",
                    "partner_ok",
                )
            )
        ).alias("_catalog"),
        F.first("_expected_line_value").alias("_expected_line_value"),
        F.first("_expected_taxable_value").alias("_expected_taxable_value"),
    )


def pick_retail_product(catalog: Column, uniform: Column) -> Column:
    """Binary CDF search avoids scanning the catalog for every basket line."""
    size = F.size(catalog)
    initial = F.struct(F.lit(1).alias("lo"), size.alias("hi"))

    def narrow(bounds: Column, _step: Column) -> Column:
        middle = F.floor((bounds["lo"] + bounds["hi"]) / 2).cast("int")
        left = F.element_at(catalog, middle)["_cdf"] > uniform
        return F.struct(
            F.when(left, bounds["lo"]).otherwise(F.least(middle + 1, size)).alias("lo"),
            F.when(left, middle).otherwise(bounds["hi"]).alias("hi"),
        )

    steps = F.sequence(F.lit(1), (F.ceil(F.log2(size)) + 1).cast("int"))
    selected = F.aggregate(steps, initial, narrow)["lo"]
    # The final entry absorbs floating-point CDF accumulation error.
    return F.when(size > 0, F.element_at(catalog, selected)).otherwise(
        F.raise_error("Cannot select a product from an empty catalog")
    )


def calibrated_basket_mean(
    expected_line: Column,
    expected_taxable: Column,
    tax_rate: Column,
    target: float,
    promo_rate: float,
    nominal_lines: float,
    complement_rate: float = 0.0,
) -> Column:
    """Drawn basket lines so the average ticket meets ``target``.

    ``complement_rate`` is the share of drawn lines that bring a complementary
    product along (``baskets.with_complements``); those extra lines count
    towards the ticket too.
    """
    affordable_lines = F.lit(target) / (
        (expected_line + expected_taxable * tax_rate)
        * (1.0 - promo_rate * 0.20)
        * (1.0 + complement_rate)
    )
    return F.when(
        (expected_line > 0)
        & (expected_line < float("inf"))
        & (expected_taxable >= 0)
        & (tax_rate >= 0),
        F.least(affordable_lines, F.lit(nominal_lines)),
    ).otherwise(F.raise_error("Basket calibration requires valid catalog prices and tax"))

# --- retail_setup/generation/customers.py ---
"""Customer lifecycle shared by history and the live stream.

Every customer has, as pure functions of the seed and their ID:

- a **home store** (``dim_customers.HomeStoreID``), drawn by store traffic, where
  most of their visits happen;
- a **visit frequency**: a long-tailed annual trip rate around the store type's
  ``customer_visits_per_year``, capped at three trips a week, replicated as
  selection tickets;
- a **segment** and a persistent **basket budget** (the draw baskets use);
- a **signup day**: pre-history customers are already active when history
  starts, the rest sign up uniformly across the history (the acquisition
  curve), so the base turns over instead of starting complete;
- a **churn day** from a feature-driven hazard: budget and convenience
  shoppers, infrequent visitors and small spenders churn sooner. The base
  hazard is calibrated so ``annual_churn_rate`` of the customers active at a
  given time churn within a year;
- a **taper**: most churners visit less and less over their last
  ``TAPER_DAYS``, the early-warning signal churn models learn from; the rest
  stop abruptly.

Receipts thin visits by this lifecycle: a store visit is identified with
``identify_probability``, picks a ticket from its store's pool (or, for
``1 - HOME_STORE_SHARE`` of visits, another store's), and keeps the customer
with their activity that day; otherwise the receipt is anonymous. So each
customer's trips form a thinned Poisson process at their own rate, and the
anonymous share of receipts is ``anonymous_share`` on average.

``build_notebooks.py`` inlines this module into the setup and stream notebooks.
"""

from dataclasses import dataclass
from datetime import date
from functools import cached_property, lru_cache
from math import exp

import numpy as np
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Shopping segments (datagen CustomerJourney distribution) and their relative
# churn hazard.
SEGMENT_WEIGHTS: list[tuple[str, float]] = [
    ("BUDGET", 0.35),
    ("CONVENIENCE", 0.25),
    ("QUALITY", 0.20),
    ("BRAND_LOYAL", 0.20),
]
SEGMENT_HAZARD = {"BUDGET": 1.3, "CONVENIENCE": 1.1, "QUALITY": 0.8, "BRAND_LOYAL": 0.6}
# Trip rate: lognormal around the store type's mean, capped at three a week.
VISIT_RATE_SIGMA = 1.0
MAX_VISITS_PER_YEAR = 156.0
# Selection tickets of a customer at the mean trip rate (ticket resolution).
TICKETS_PER_MEAN_VISITOR = 6
# Hazard elasticities: infrequent visitors and small spenders churn sooner.
FREQUENCY_HAZARD_EXPONENT = -0.35
SPEND_HAZARD_COEFFICIENT = -0.25
# Share of a customer's identified visits made at their home store.
HOME_STORE_SHARE = 0.85
# Most churners taper: activity falls linearly to TAPER_FLOOR over the last
# TAPER_DAYS before churn.
TAPER_DAYS = 60
TAPER_FLOOR = 0.15
TAPERED_SHARE = 0.75
# Pre-history customers signed up up to this many days before history.
PRE_HISTORY_MAX_TENURE_DAYS = 1500
PRE_HISTORY_MIN_TENURE_DAYS = 30
# Share of all customers active at a time for a typical 18-month history at the
# default churn rate; sizes the default customer count (config.generation).
NOMINAL_ACTIVE_SHARE = 0.75
DAYS_PER_YEAR = 365.25
# Online orders try this many customers for one active that day.
ONLINE_CUSTOMER_ATTEMPTS = 4

_GAUSS_NODES, _GAUSS_WEIGHTS = np.polynomial.hermite_e.hermegauss(24)
_GAUSS_WEIGHTS = _GAUSS_WEIGHTS / _GAUSS_WEIGHTS.sum()
_BUDGET_NODES, _BUDGET_WEIGHTS = np.polynomial.hermite_e.hermegauss(9)
_BUDGET_WEIGHTS = _BUDGET_WEIGHTS / _BUDGET_WEIGHTS.sum()


def relative_visit_rate(visit_gauss: float) -> float:
    """Trip rate relative to the mean for a standard-normal propensity draw."""
    return exp(VISIT_RATE_SIGMA * visit_gauss - VISIT_RATE_SIGMA**2 / 2)


def hazard_multiplier(segment: str, relative_rate: float, budget_gauss: float) -> float:
    """Relative churn hazard of a customer's segment, trip rate and basket budget."""
    return (
        SEGMENT_HAZARD[segment]
        * relative_rate**FREQUENCY_HAZARD_EXPONENT
        * exp(SPEND_HAZARD_COEFFICIENT * budget_gauss)
    )


def activity(day: int, signup_day: int, churn_day: int, tapered: bool) -> float:
    """Share of a customer's trips that happen on ``day`` (days since history start)."""
    if day < signup_day or day >= churn_day:
        return 0.0
    taper_start = churn_day - TAPER_DAYS
    if tapered and day >= taper_start:
        return 1.0 - (1.0 - TAPER_FLOOR) * (day - taper_start) / TAPER_DAYS
    return 1.0


def _customer_types() -> list[tuple[float, float, float]]:
    """(headcount weight, ticket weight, hazard multiplier) over the population."""
    types = []
    for segment, segment_weight in SEGMENT_WEIGHTS:
        for g, g_weight in zip(_GAUSS_NODES, _GAUSS_WEIGHTS):
            rate = relative_visit_rate(float(g))
            for b, b_weight in zip(_BUDGET_NODES, _BUDGET_WEIGHTS):
                weight = segment_weight * float(g_weight) * float(b_weight)
                types.append((weight, rate, hazard_multiplier(segment, rate, float(b))))
    return types


@lru_cache(maxsize=16)
def calibrate_base_hazard(annual_churn_rate: float) -> float:
    """Base yearly hazard so ``annual_churn_rate`` of active customers churn in a year."""
    if not 0.0 < annual_churn_rate < 1.0:
        raise ValueError("annual_churn_rate must be between 0 and 1")
    types = _customer_types()

    def churned(base: float) -> float:
        return sum(w * (1.0 - exp(-base * m)) for w, _, m in types)

    low, high = 1e-9, 50.0
    for _ in range(200):
        middle = (low + high) / 2
        if churned(middle) < annual_churn_rate:
            low = middle
        else:
            high = middle
    return (low + high) / 2


@dataclass(frozen=True)
class CustomerLifecycle:
    """Lifecycle parameters for one history window."""

    seed: int
    history_start: date
    history_end: date
    annual_churn_rate: float
    anonymous_share: float
    visits_per_year: float

    @property
    def history_days(self) -> int:
        return (self.history_end - self.history_start).days + 1

    @cached_property
    def base_hazard(self) -> float:
        return calibrate_base_hazard(self.annual_churn_rate)

    @cached_property
    def pre_history_share(self) -> float:
        """Customers already active at history start; the rest join during history.

        Sized so acquisitions over the history replace the customers who churn.
        """
        mean_hazard = self.base_hazard * sum(w * m for w, _, m in _customer_types())
        return 1.0 / (1.0 + mean_hazard * self.history_days / DAYS_PER_YEAR)

    @property
    def ticket_unit(self) -> float:
        return self.visits_per_year / TICKETS_PER_MEAN_VISITOR

    def expected_active_share(self) -> float:
        """Ticket-weighted share of customers active on an average history day."""
        types = _customer_types()
        base, pre = self.base_hazard, self.pre_history_share
        years = self.history_days / DAYS_PER_YEAR
        taper_years = TAPER_DAYS / DAYS_PER_YEAR
        total = sum(w * rate for w, rate, _ in types)
        steps = 48
        share = 0.0
        for step in range(steps):
            t = years * (step + 0.5) / steps
            for weight, rate, multiplier in types:
                hazard = base * multiplier
                taper_loss = 1.0 - TAPERED_SHARE * (1.0 - TAPER_FLOOR) * hazard * taper_years / 2
                pre_active = exp(-hazard * t)
                joined_active = (1.0 - exp(-hazard * t)) / (hazard * years)
                share += weight * rate * taper_loss * (pre * pre_active + (1 - pre) * joined_active)
        return share / total / steps

    @cached_property
    def identify_probability(self) -> float:
        """Chance a store visit tries a customer, so receipts are ``anonymous_share`` anonymous."""
        return min(1.0, (1.0 - self.anonymous_share) / self.expected_active_share())


def lifecycle_for(cfg, visits_per_year: float) -> CustomerLifecycle:
    """The lifecycle of a generation config (``GenerationConfig``)."""
    return CustomerLifecycle(
        seed=cfg.seed,
        history_start=cfg.start_date,
        history_end=cfg.end_date,
        annual_churn_rate=cfg.annual_churn_rate,
        anonymous_share=cfg.anonymous_share,
        visits_per_year=visits_per_year,
    )


# ---------------------------------------------------------------------------
# Column forms
# ---------------------------------------------------------------------------


def customer_key(customer_id: Column) -> Column:
    """Hash key of a customer's persistent draws (shared with basket budgets)."""
    return customer_id.cast("string")


def shopper_key(customer_id: Column, purchase_key: Column) -> Column:
    """Basket-draw key: the customer, or the purchase itself when anonymous."""
    return F.coalesce(customer_key(customer_id), F.concat(F.lit("anon-"), purchase_key))


def budget_gauss(draws: seeded_draws, key: Column) -> Column:
    """Persistent basket budget draw (``behavior.retail_basket_size``)."""
    return draws.gauss([key], "retail-budget")


def relative_rate_column(draws: seeded_draws, customer_id: Column) -> Column:
    """Column form of :func:`relative_visit_rate` for a customer."""
    gauss = draws.gauss([customer_key(customer_id)], "visit-propensity")
    return F.exp(F.lit(VISIT_RATE_SIGMA) * gauss - VISIT_RATE_SIGMA**2 / 2)


def visit_tickets(lc: CustomerLifecycle, customer_id: Column) -> Column:
    """Selection tickets: the customer's trip rate in units of ``ticket_unit``."""
    draws = seeded_draws(lc.seed)
    rate = F.least(
        relative_rate_column(draws, customer_id) * lc.visits_per_year,
        F.lit(MAX_VISITS_PER_YEAR),
    )
    return F.greatest(F.lit(1), F.round(rate / lc.ticket_unit)).cast("int")


def _segment_hazard(segment_draw: Column) -> Column:
    """Hazard factor of the segment a uniform draw picks (``SEGMENT_WEIGHTS`` order)."""
    total = sum(weight for _, weight in SEGMENT_WEIGHTS)
    cumulative, branches = 0.0, []
    for name, weight in SEGMENT_WEIGHTS[:-1]:
        cumulative += weight / total
        branches.append((segment_draw < F.lit(cumulative), F.lit(SEGMENT_HAZARD[name])))
    return case_when(branches).otherwise(F.lit(SEGMENT_HAZARD[SEGMENT_WEIGHTS[-1][0]]))


def lifecycle_struct(lc: CustomerLifecycle, customer_id: Column) -> Column:
    """``struct<signup_day int, churn_day long, tapered boolean>`` (days since history start)."""
    draws = seeded_draws(lc.seed)
    key = [customer_key(customer_id)]
    pre = lc.pre_history_share
    inputs = F.struct(
        draws.u(key, "customer-signup").alias("join"),
        draws.u(key, "customer-tenure").alias("tenure"),
        draws.u(key, "customer-churn").alias("churn"),
        draws.u(key, "customer-taper").alias("taper"),
        draws.u(key, "customer-segment").alias("segment"),
        relative_rate_column(draws, customer_id).alias("rate"),
        budget_gauss(draws, customer_key(customer_id)).alias("budget"),
    )

    def build(r: Column) -> Column:
        hazard = (
            F.lit(lc.base_hazard)
            * _segment_hazard(r["segment"])
            * F.pow(r["rate"], F.lit(FREQUENCY_HAZARD_EXPONENT))
            * F.exp(F.lit(SPEND_HAZARD_COEFFICIENT) * r["budget"])
        )
        is_pre = r["join"] < F.lit(pre)
        tenure = PRE_HISTORY_MIN_TENURE_DAYS + F.floor(
            r["tenure"] * (PRE_HISTORY_MAX_TENURE_DAYS - PRE_HISTORY_MIN_TENURE_DAYS)
        )
        joined = F.floor((r["join"] - F.lit(pre)) / F.lit(1.0 - pre) * lc.history_days)
        signup_day = F.when(is_pre, -tenure).otherwise(joined).cast("int")
        # Exponential lifetime from the hazard, counted from history start for
        # pre-history customers (memoryless) and from signup for the rest.
        lifetime_days = F.floor(-F.log(F.lit(1.0) - r["churn"]) / hazard * DAYS_PER_YEAR)
        churn_start = F.when(is_pre, F.lit(0)).otherwise(joined).cast("long")
        return F.struct(
            signup_day.alias("signup_day"),
            (churn_start + lifetime_days).cast("long").alias("churn_day"),
            (r["taper"] < F.lit(TAPERED_SHARE)).alias("tapered"),
        )

    # The draws are evaluated once and bound to a lambda variable, which
    # keeps the expression (and query planning) small.
    return bind(inputs, build)


def activity_column(lc: CustomerLifecycle, customer_id: Column, day: Column) -> Column:
    """Column form of :func:`activity` for a customer and a date column (0 when NULL)."""
    offset = F.datediff(day, F.lit(lc.history_start))

    def from_life(life: Column) -> Column:
        taper_start = life["churn_day"] - TAPER_DAYS
        tapered = F.lit(1.0) - F.lit(1.0 - TAPER_FLOOR) * (offset - taper_start) / F.lit(
            float(TAPER_DAYS)
        )
        return (
            F.when(
                customer_id.isNull()
                | (offset < life["signup_day"])
                | (offset >= life["churn_day"]),
                F.lit(0.0),
            )
            .when(life["tapered"] & (offset >= taper_start), tapered)
            .otherwise(F.lit(1.0))
        )

    return bind(lifecycle_struct(lc, customer_id), from_life)


# ---------------------------------------------------------------------------
# Ticket pools
# ---------------------------------------------------------------------------


def store_picker(stores: list[tuple[int, float]]) -> Column:
    """A literal CDF over ``(store_id, multiplier)`` for store picks weighted by traffic.

    A literal (at most ``store_count`` entries) rather than a joined frame:
    joining a one-row frame lets the optimizer push upstream filters into the
    join condition, which copies large expressions into the plan.
    """
    if not stores:
        raise ValueError("customer lifecycles need at least one store")
    total = sum(weight for _, weight in stores)
    cumulative, entries = 0.0, []
    for store_id, weight in sorted(stores):
        cumulative += weight / total
        entries.append(
            F.struct(
                F.lit(cumulative).alias("_cdf"), F.lit(int(store_id)).cast("long").alias("store_id")
            )
        )
    return F.array(*entries)


def picked_store(picker: Column, draw: Column) -> Column:
    """The store of a :func:`store_picker` CDF at a uniform draw."""
    return bind(picker, lambda cdf: pick_retail_product(cdf, draw)["store_id"]).cast("long")


def with_home_store(seed: int, customers: DataFrame, picker: Column) -> DataFrame:
    """Add ``HomeStoreID``, drawn in proportion to store traffic, to customers with ``ID``.

    Depends on the seed and the stores only, so dimensions stay independent
    of the history window.
    """
    draws = seeded_draws(seed)
    draw = draws.u([customer_key(F.col("ID"))], "customer-home-store")
    return customers.withColumn("HomeStoreID", picked_store(picker, draw))


def ticket_pool(lc: CustomerLifecycle, customers: DataFrame) -> DataFrame:
    """``pool_store, rank, customer_id``: each customer's tickets, ranked within their home store.

    ``customers`` has ``ID`` and ``HomeStoreID``.
    """
    tickets = customers.select(
        F.col("ID").cast("long").alias("customer_id"),
        F.col("HomeStoreID").cast("long").alias("pool_store"),
    ).where(F.col("pool_store").isNotNull())
    tickets = tickets.withColumn(
        "_ticket", F.explode(F.sequence(F.lit(1), visit_tickets(lc, F.col("customer_id"))))
    )
    order = Window.partitionBy("pool_store").orderBy("customer_id", "_ticket")
    return tickets.select("pool_store", F.row_number().over(order).alias("rank"), "customer_id")


def pool_sizes(pool: DataFrame) -> DataFrame:
    """``pool_store, pool_tickets``."""
    return pool.groupBy("pool_store").agg(F.count("*").alias("pool_tickets"))


def assign_customers(
    lc: CustomerLifecycle,
    visits: DataFrame,
    pool: DataFrame,
    sizes: DataFrame,
    picker: Column,
    key: str,
    store: str,
    day: str,
) -> DataFrame:
    """Add ``customer_id`` to store visits (NULL for anonymous receipts).

    ``key`` names the visit's unique ID, ``store`` its store and ``day`` its
    local date. Away visits use a store from ``picker`` (:func:`store_picker`).
    """
    draws = seeded_draws(lc.seed)
    k = [F.col(key)]
    frame = visits.withColumn(
        "_pool_store",
        F.when(draws.u(k, "customer-home-visit") < F.lit(HOME_STORE_SHARE), F.col(store)).otherwise(
            picked_store(picker, draws.u(k, "customer-away-store"))
        ),
    ).join(F.broadcast(sizes.withColumnRenamed("pool_store", "_pool_store")), "_pool_store", "left")
    frame = frame.withColumn(
        "_rank", (draws.h64(k, "customer-ticket") % F.col("pool_tickets") + 1).cast("long")
    )
    picked = pool.select(
        F.col("pool_store").alias("_pool_store"),
        F.col("rank").alias("_rank"),
        F.col("customer_id").alias("_picked"),
    )
    frame = frame.join(picked, ["_pool_store", "_rank"], "left")
    keep = (draws.u(k, "customer-identified") < F.lit(lc.identify_probability)) & (
        draws.u(k, "customer-active") < activity_column(lc, F.col("_picked"), F.col(day))
    )
    return frame.withColumn("customer_id", F.when(keep, F.col("_picked")).cast("long")).drop(
        "_pool_store", "pool_tickets", "_rank", "_picked"
    )


def assign_online_customers(
    lc: CustomerLifecycle,
    orders: DataFrame,
    pool: DataFrame,
    sizes: DataFrame,
    picker: Column,
    key: str,
    day: str,
) -> DataFrame:
    """Add ``customer_id`` to online orders: the first of a few tries active that day.

    Orders whose tries all land on inactive customers are dropped (about
    ``(1 - active share) ** ONLINE_CUSTOMER_ATTEMPTS`` of them).
    """
    draws = seeded_draws(lc.seed)
    tries = orders.select(
        F.col(key).alias("_order"),
        F.col(day).alias("_day"),
        F.explode(F.sequence(F.lit(1), F.lit(ONLINE_CUSTOMER_ATTEMPTS))).alias("_try"),
    )
    k = [F.col("_order"), F.col("_try")]
    tries = tries.withColumn(
        "_pool_store", picked_store(picker, draws.u(k, "online-customer-store"))
    ).join(F.broadcast(sizes.withColumnRenamed("pool_store", "_pool_store")), "_pool_store")
    tries = tries.withColumn(
        "_rank", (draws.h64(k, "online-customer-ticket") % F.col("pool_tickets") + 1).cast("long")
    ).join(
        pool.select(
            F.col("pool_store").alias("_pool_store"),
            F.col("rank").alias("_rank"),
            F.col("customer_id").alias("_picked"),
        ),
        ["_pool_store", "_rank"],
    )
    active = draws.u(k, "online-customer-active") < activity_column(
        lc, F.col("_picked"), F.col("_day")
    )
    chosen = (
        tries.where(active)
        .groupBy("_order")
        .agg(F.min_by("_picked", "_try").cast("long").alias("customer_id"))
        .withColumnRenamed("_order", key)
    )
    return orders.join(chosen, key)

# --- retail_setup/generation/baskets.py ---
"""Basket-level line handling shared by history and the live stream.

Both producers build a purchase's lines as an array of structs and pass it
through the same steps:

1. :func:`with_complements` - each line may bring its complementary product
   (a fixed partner in the same department, see :func:`complement_partners`),
   so baskets carry real product affinities.
2. :func:`merge_lines` - repeated SKUs collapse into one line with the summed
   quantity, numbered in first-seen order.
3. :func:`basket_promotion` - an offered purchase gets at most one promotion
   code: one of the ``PROMO_CATALOG`` codes in season whose minimum is met by
   the basket subtotal and that has an eligible line. The code discounts only
   the lines in its department scope (``promo_scopes``); BOGO discounts every
   second unit of eligible lines with two or more units.

Line structs carry ``line_num``, ``product_id``, ``quantity``, ``unit_cents``,
``taxability`` and ``department``; complements also need ``partner_id``,
``partner_cents``, ``partner_taxability`` and ``partner_ok`` (the partner is
launched).
"""

from collections.abc import Callable, Iterable

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Share of lines that bring their complementary product along.
COMPLEMENT_RATE = 0.15
# Promotion code -> department-name tokens it applies to; None applies to all.
# A code whose tokens match none of a store type's departments applies to all.
PROMO_SCOPES: dict[str, list[str] | None] = {
    "SAVE10": None,
    "SAVE15": None,
    "SAVE20": None,
    "SAVE25": None,
    "CLEARANCE30": [
        "seasonal", "clothing", "apparel", "footwear", "accessor", "home", "toy",
        "electronic", "sport", "garden", "lawn",
    ],
    "BOGO50": [
        "grocery", "fresh", "food", "household", "personal", "health", "beauty", "pet", "baby",
    ],
    "NEWYEAR15": ["health", "sport", "beauty", "fitness"],
    "SPRINGSALE20": ["garden", "lawn", "home", "clothing", "apparel", "paint"],
    "SUMMER25": ["garden", "lawn", "sport", "seasonal", "outdoor", "clothing", "apparel"],
    "BACKTOSCHOOL20": ["office", "clothing", "apparel", "electronic", "baby", "kids"],
    "BFRIDAY30": [
        "electronic", "toy", "home", "clothing", "apparel", "jewel", "accessor", "tool",
    ],
    "BFRIDAY40": [
        "electronic", "toy", "home", "clothing", "apparel", "jewel", "accessor", "tool",
    ],
    "HOLIDAY20": ["toy", "seasonal", "electronic", "jewel", "fragrance", "home", "accessor"],
}  # fmt: skip
# Seasonal codes are favored while in season.
SEASONAL_CODE_WEIGHT = 3.0


def promo_scopes(departments: list[str]) -> dict[str, list[str] | None]:
    """Each code's department names among ``departments`` (None = every department)."""
    scopes: dict[str, list[str] | None] = {}
    for code, *_rest in PROMO_CATALOG:
        tokens = PROMO_SCOPES.get(code)
        matched = (
            [name for name in departments if any(t in name.lower() for t in tokens)]
            if tokens is not None
            else []
        )
        scopes[code] = sorted(matched) if matched else None
    return scopes


def receipt_offer(draws: seeded_draws, key: Column, promo_rate: float) -> Column:
    """Whether a purchase presents a promotion (``promo_rate`` of purchases)."""
    return draws.u([key], "offer") < F.lit(promo_rate)


def _in_scope(department: Column, scope: list[str] | None) -> Column:
    return F.lit(True) if scope is None else department.isin(*scope)


def _discountable(scope: list[str] | None, kind: str) -> Callable[[Column], Column]:
    """Whether a line can take a code of this scope and kind (BOGO needs 2+ units)."""

    def check(line: Column) -> Column:
        units = F.lit(True) if kind != "BOGO" else line["quantity"] >= 2
        return _in_scope(line["department"], scope) & units

    return check


def basket_promotion(
    draws: seeded_draws,
    key: Column,
    lines: Column,
    month: Column,
    offer: Column,
    profile: StoreTypeProfile,
) -> Column:
    """Lines with ``promo_code`` and ``discount_cents`` from one basket-level code.

    The code is drawn among the codes eligible for the basket (in season,
    basket gross subtotal at or above the code's minimum, at least one
    discountable line); lines outside its scope, and BOGO lines with one unit,
    keep no code and no discount. Purchases without an offer get no codes.
    """
    scopes = promo_scopes(list(profile.department_weights))
    gross = F.aggregate(
        lines, F.lit(0).cast("long"), lambda acc, line: acc + line["unit_cents"] * line["quantity"]
    )

    def choose(basket_cents: Column) -> Column:
        candidates = []
        for code, _pct, months, min_cents, kind in PROMO_CATALOG:
            eligible = F.exists(lines, _discountable(scopes[code], kind)) & (
                basket_cents >= F.lit(min_cents)
            )
            if months is not None:
                eligible = eligible & month.isin(*months)
            weight = SEASONAL_CODE_WEIGHT if months is not None else 1.0
            # Weighted random choice: the largest u ** (1 / weight) among eligible codes.
            score = F.when(eligible, F.pow(draws.u([key, F.lit(code)], "pcode"), 1.0 / weight))
            candidates.append(F.struct(score.alias("score"), F.lit(code).alias("code")))
        eligible_codes = F.filter(F.array(*candidates), lambda entry: entry["score"].isNotNull())
        return F.when(offer, F.array_max(eligible_codes)["code"])

    def priced(line: Column, code: Column) -> Column:
        line_cents = line["unit_cents"] * line["quantity"]
        discount: Column = F.lit(0).cast("long")
        for name, pct, _months, _min_cents, kind in PROMO_CATALOG:
            applies = (code == F.lit(name)) & _in_scope(line["department"], scopes[name])
            if kind == "BOGO":
                amount = F.floor(
                    F.floor(line["quantity"] / 2) * line["unit_cents"] * pct / 100.0 + 0.5
                )
            else:
                amount = F.floor(line_cents * pct / 100.0 + 0.5)
            discount = F.when(applies, amount.cast("long")).otherwise(discount)
        discount = F.coalesce(discount, F.lit(0).cast("long"))
        return line.withField("promo_code", F.when(discount > 0, code)).withField(
            "discount_cents", discount
        )

    # The subtotal and the chosen code are evaluated once per basket and bound
    # to lambda variables, which keeps the expression (and planning) small.
    code = bind(gross, choose)
    return bind(code, lambda chosen: F.transform(lines, lambda line: priced(line, chosen)))


def merge_lines(lines: Column) -> Column:
    """Collapse repeated SKUs into one line (summed quantity), renumbered 1..n."""
    products = F.array_distinct(F.transform(lines, lambda line: line["product_id"]))

    def merged(product_id: Column, index: Column) -> Column:
        same = F.filter(lines, lambda line: line["product_id"] == product_id)
        quantity = F.aggregate(same, F.lit(0), lambda acc, line: acc + line["quantity"].cast("int"))
        first = F.element_at(same, 1)
        return first.withField("line_num", (index + 1).cast("int")).withField(
            "quantity", quantity.cast("int")
        )

    return F.transform(products, merged)


def with_complements(draws: seeded_draws, key: Column, lines: Column) -> Column:
    """Lines followed, for ``COMPLEMENT_RATE`` of them, by one unit of their partner."""

    def expand(line: Column) -> Column:
        brings = line["partner_ok"] & (
            draws.u([key, line["line_num"]], "complement") < F.lit(COMPLEMENT_RATE)
        )
        partner = (
            line.withField("product_id", line["partner_id"])
            .withField("quantity", F.lit(1))
            .withField("unit_cents", line["partner_cents"])
            .withField("taxability", line["partner_taxability"])
        )
        return F.when(brings, F.array(line, partner)).otherwise(F.array(line))

    return F.flatten(F.transform(lines, expand))


def complement_partners(products: DataFrame, seed: int) -> DataFrame:
    """``product_id, partner_id``: a fixed complementary product in the same department.

    Products are ranked within their department by a seeded hash; a product's
    partner sits a seeded, non-zero offset further round the ranking, so each
    pair is stable for the whole history and the live stream. Departments
    with one product have no partner.
    """
    draws = seeded_draws(seed)
    ranked = products.select(
        F.col("product_id").cast("long").alias("product_id"), "department"
    ).withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("department").orderBy(
                draws.h64([F.col("product_id")], "complement-rank"), "product_id"
            )
        ),
    )
    ranked = ranked.withColumn("_size", F.count("*").over(Window.partitionBy("department")))
    offset = draws.h64([F.col("product_id")], "complement-offset") % (F.col("_size") - 1) + 1
    wanted = ranked.where(F.col("_size") > 1).withColumn(
        "_partner_rank", F.pmod(F.col("_rank") - 1 + offset, F.col("_size")) + 1
    )
    partners = ranked.select(
        "department",
        F.col("_rank").alias("_partner_rank"),
        F.col("product_id").alias("partner_id"),
    )
    return wanted.join(partners, ["department", "_partner_rank"]).select("product_id", "partner_id")


def basket_products(dim_products: DataFrame, seed: int) -> DataFrame:
    """Catalog input of ``behavior.retail_catalog``: SKUs with their complementary partner.

    ``product_id, SalePrice, taxability, department, launch_date, inflation,
    elasticity`` plus the partner's ``partner_id, partner_price, partner_taxability, partner_launch``
    (NULL for a department of one product).
    """
    products = dim_products.select(
        F.col("ID").cast("long").alias("product_id"),
        "SalePrice",
        "taxability",
        F.col("Department").alias("department"),
        F.to_date("LaunchDate").alias("launch_date"),
        # Department price trend and elasticity (pricing.DEPARTMENT_PRICING),
        # resolved once per SKU here instead of in every price expression.
        inflation_column(F.col("Department")).alias("inflation"),
        elasticity_column(F.col("Department")).alias("elasticity"),
    )
    partner = products.select(
        F.col("product_id").alias("partner_id"),
        F.col("SalePrice").alias("partner_price"),
        F.col("taxability").alias("partner_taxability"),
        F.col("launch_date").alias("partner_launch"),
    )
    return products.join(complement_partners(products, seed), "product_id", "left").join(
        partner, "partner_id", "left"
    )


# Product taxability classes: TAXABLE pays the full combined rate everywhere;
# NON_TAXABLE (grocery staples) and REDUCED_RATE (clothing) pay the share of
# the rate their state's rule sets (``_shared/tax_rules.json``).
GROCERY_TAXABILITY = "NON_TAXABLE"
CLOTHING_TAXABILITY = "REDUCED_RATE"
# (grocery %, clothing %, clothing exempt under cents) of a state without a rule.
DEFAULT_TAX_RULE: tuple[int, int, int | None] = (0, 100, None)
TaxRules = dict[str, tuple[int, int, int | None]]


def tax_rules_by_state(entries: Iterable[TaxRuleEntry]) -> TaxRules:
    """``{state: (grocery %, clothing %, clothing exempt under cents)}`` of rule entries."""
    return {
        entry.StateCode: (
            entry.GroceryTaxPct,
            entry.ClothingTaxPct,
            None if entry.ClothingExemptUnder is None else int(entry.ClothingExemptUnder * 100),
        )
        for entry in entries
    }


def tax_percent(
    taxability: str, unit_cents: int, rule: tuple[int, int, int | None] = DEFAULT_TAX_RULE
) -> int:
    """Share (percent) of the combined rate a line pays under a state rule."""
    grocery, clothing, exempt_under = rule
    if taxability == GROCERY_TAXABILITY:
        return grocery
    if taxability == CLOTHING_TAXABILITY:
        return 0 if exempt_under is not None and unit_cents < exempt_under else clothing
    return 100 if taxability == "TAXABLE" else 0


def _rule_struct(rule: tuple[int, int, int | None]) -> Column:
    grocery, clothing, exempt_under = rule
    return F.struct(
        F.lit(grocery).cast("int").alias("grocery_pct"),
        F.lit(clothing).cast("int").alias("clothing_pct"),
        F.lit(exempt_under).cast("long").alias("clothing_exempt_under_cents"),
    )


def tax_rule_column(rules: TaxRules, state: Column) -> Column:
    """The state rule of a state-code column as a struct (the default when none)."""
    default = _rule_struct(DEFAULT_TAX_RULE)
    if not rules:
        return default
    return case_when(
        (state == F.lit(code), _rule_struct(rule)) for code, rule in sorted(rules.items())
    ).otherwise(default)


def line_tax_cents(
    net_cents: Column,
    taxability: Column,
    bps: Column,
    rule: Column | None = None,
    unit_cents: Column | None = None,
) -> Column:
    """Integer basis-point tax of a line's net cents (datagen ``_tax_cents``).

    ``rule`` is the selling (or destination) state's :func:`tax_rule_column`;
    without it groceries are exempt and clothing pays the full rate. Clothing
    priced (``unit_cents``) below the rule's threshold is exempt.
    """
    rule = _rule_struct(DEFAULT_TAX_RULE) if rule is None else rule
    exempt = rule["clothing_exempt_under_cents"]
    clothing = rule["clothing_pct"]
    if unit_cents is not None:
        clothing = F.when(exempt.isNotNull() & (unit_cents < exempt), F.lit(0)).otherwise(clothing)
    multiplier = (
        F.when(taxability == "TAXABLE", F.lit(100))
        .when(taxability == GROCERY_TAXABILITY, rule["grocery_pct"])
        .when(taxability == CLOTHING_TAXABILITY, clothing)
        .otherwise(F.lit(0))
    ).cast("long")
    return F.floor((net_cents * bps * multiplier + F.lit(500_000)) / F.lit(1_000_000)).cast("long")


def store_tax_rules(
    dim_stores: DataFrame, dim_geographies: DataFrame | None, rules: TaxRules
) -> DataFrame:
    """``store_id, tax_rule``: each store's state rule (the default without geographies)."""
    stores = dim_stores.select(F.col("ID").cast("long").alias("store_id"), "GeographyID")
    if dim_geographies is None:
        return stores.select("store_id", _rule_struct(DEFAULT_TAX_RULE).alias("tax_rule"))
    states = dim_geographies.select(F.col("ID").alias("GeographyID"), "State")
    return stores.join(states, "GeographyID", "left").select(
        "store_id", tax_rule_column(rules, F.col("State")).alias("tax_rule")
    )


def price_basket(
    draws: seeded_draws,
    key: Column,
    lines: Column,
    month: Column,
    offer: Column,
    bps: Column,
    profile: StoreTypeProfile,
    rule: Column | None = None,
) -> Column:
    """Lines with the basket promotion, ``ext_cents`` (net) and ``tax_cents``.

    ``rule`` is the store's state taxability rule (:func:`tax_rule_column`).
    """
    promoted = basket_promotion(draws, key, lines, month, offer, profile)

    def finish(line: Column) -> Column:
        gross = line["unit_cents"] * line["quantity"]
        net = F.greatest(F.lit(0).cast("long"), gross - line["discount_cents"])
        return line.withField("ext_cents", net).withField(
            "tax_cents", line_tax_cents(net, line["taxability"], bps, rule, line["unit_cents"])
        )

    return F.transform(promoted, finish)


def sum_lines(lines: Column, field: str) -> Column:
    """Sum of a long field over a basket's lines."""
    return F.aggregate(lines, F.lit(0).cast("long"), lambda acc, line: acc + line[field])

# --- retail_setup/generation/traffic.py ---
"""Visit-first store traffic shared by receipt generation and footfall facts.

Visitors per store, local day and local hour follow the profile's hourly,
daily and monthly weights, the holiday calendar of the store type's class
(closed days have no hours), the geography's weather, the volume trend and the
store's traffic multiplier (``stream_calendar``, shared with the live stream).
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def generate_hourly_visits(
    spark: SparkSession,
    stores: DataFrame,
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
    geographies: DataFrame | None = None,
) -> DataFrame:
    """``store_id, day, hour, hour_ts, visitors, n_receipts`` per open store-hour.

    ``geographies`` (``dim_geographies``) gives each store's state for its
    weather climate; without it every store has the temperate climate.
    """
    draws = seeded_draws(cfg.seed)
    if geographies is None:
        stores = stores.withColumn("State", F.lit(None).cast("string"))
    else:
        stores = stores.join(
            geographies.select(F.col("ID").alias("GeographyID"), "State"), "GeographyID", "left"
        )
    hours = stores.select(
        F.col("ID").alias("store_id"),
        "daily_traffic_multiplier",
        "operating_hours",
    )
    is_24 = F.col("operating_hours") == "24h"
    parts = F.split("operating_hours", "-")
    hours = hours.withColumn(
        "hour",
        F.explode(
            F.sequence(
                F.when(is_24, 0).otherwise(parts.getItem(0).cast("int")),
                F.when(is_24, 24).otherwise(parts.getItem(1).cast("int")) - 1,
            )
        ),
    ).withColumn(
        "_hour_weight",
        F.element_at(F.array(*[F.lit(w) for w in profile.hourly_weights]), F.col("hour") + 1),
    )
    hours = hours.withColumn(
        "_hour_share",
        F.col("_hour_weight") / F.sum("_hour_weight").over(Window.partitionBy("store_id")),
    )
    # Calendar and weather factors are evaluated once per day and store-day,
    # not per hour; closed days have no hours at all.
    days = (
        date_grid(spark, cfg.start_date, cfg.end_date)
        .withColumn("_holiday", holiday_factor_column(F.col("day"), store_class(profile)))
        .filter(F.col("_holiday") > 0)
        .withColumn(
            "_day_factor",
            F.col("_holiday")
            * daily_factor_column(profile, F.col("day"))
            * monthly_factor_column(profile, F.col("day"))
            * volume_growth_column(cfg.annual_volume_growth, cfg.anchor_date, F.col("day")),
        )
        .drop("_holiday")
    )
    # Nearby stores (one geography) share weather; store/hour conversion
    # remains heterogeneous.
    store_days = (
        stores.select(F.col("ID").alias("store_id"), "GeographyID", "State")
        .crossJoin(days)
        .withColumn(
            "_weather",
            weather_factor_column(draws, F.col("GeographyID"), F.col("State"), F.col("day")),
        )
        .select("store_id", "day", (F.col("_day_factor") * F.col("_weather")).alias("_factor"))
    )
    grid = hours.join(store_days, "store_id")
    conversion = visit_conversion(draws, F.col("store_id"), F.col("day"), F.col("hour"))
    grid = grid.withColumn(
        "_visits_mean",
        F.lit(cfg.transactions_per_store_day / VISIT_CONVERSION_CALIBRATION)
        * F.col("_factor")
        * F.col("daily_traffic_multiplier")
        * F.col("_hour_share"),
    ).withColumn("_conversion", conversion)
    grid = grid.withColumn(
        "visitors",
        draws.negative_binomial(["store_id", "day", "hour"], "visitors", F.col("_visits_mean"), 24),
    )
    visitor_ids = F.when(
        F.col("visitors") > 0, F.sequence(F.lit(1).cast("long"), F.col("visitors"))
    ).otherwise(F.array().cast("array<long>"))
    grid = grid.withColumn(
        "n_receipts",
        F.aggregate(
            visitor_ids,
            F.lit(0).cast("long"),
            lambda count, visitor: (
                count
                + (
                    draws.u(["store_id", "day", "hour", visitor], "visit-conversion")
                    < F.col("_conversion")
                ).cast("long")
            ),
        ),
    )
    return grid.withColumn(
        "hour_ts",
        F.timestamp_seconds(
            F.unix_timestamp(F.col("day").cast("timestamp")) + F.col("hour") * 3600
        ),
    ).select("store_id", "day", "hour", "hour_ts", "visitors", "n_receipts")

# --- retail_setup/generation/inventory_policy.py ---
"""Causal, streaming inventory policy, independent of Spark.

Each SKU learns an EWMA of attempted daily demand (including lost sales).
There are no backorders: rejected whole lines have no stock commitment.
Inventory position includes outstanding deliveries and accepted restock returns.

Deliveries follow the store's delivery slot, as in the live stream: every SKU a
store reorders when a day closes ships on that store's next consolidated load,
which leaves the DC at the store's daily slot (``network.plan_routes``), drives
the lane and is unloaded on arrival. ``StoreSku`` holds one store-SKU's state;
``dc_supply.simulate_dc_sku`` runs every store of a DC for one SKU against the
DC's own stock (supplier lead times, short-ships, incidents), and
:func:`simulate_sku` runs one store against an unlimited DC with loads on plan.
``inventory.py`` consolidates the shipments into capacity-bounded legs and puts
them on trucks. The DC replenishment policy (lead time, reorder levels,
allocation) is shared with the live stream. This is a stock/service
simulation, not a vehicle-routing model.
"""

from dataclasses import dataclass
from datetime import date, datetime, timedelta
from datetime import time as inventory_time
from hashlib import sha256
from heapq import heappop, heappush
from math import ceil
from typing import Iterable, Iterator


CASE_PACK = 12


def stable_int(seed: int, *keys: object) -> int:
    """Stable across processes, Python hash seeds, and Spark partition counts."""
    value = "|".join(str(key) for key in (seed, *keys))
    return int.from_bytes(sha256(value.encode("utf-8")).digest()[:8], "big")


def opening_stock(seed: int, store_id: int, product_id: int, rate: float = 1.0) -> int:
    """Opening shelf stock for a SKU expected to sell ``rate`` units a day.

    A stable draw between the reorder point and the order-up-to target, so a
    store opens with 4-8 days of cover and its SKUs do not all reorder on the
    first day. ``rate`` is a prior from the demand model, never generated demand.
    """
    point, target = reorder_levels(rate)
    return point + 1 + stable_int(seed, store_id, product_id, "opening") % (target - point)


def delivery_slot_minute(seed: int, store_id: int) -> int:
    """Fallback daily delivery slot (minute of the day) of a store without a route."""
    return stable_int(seed, int(store_id), "slot") % 1440


def lane_transit_minutes(seed: int, dc_id: int, store_id: int) -> int:
    """One-way drive between a store and its DC (a stable draw per lane)."""
    span = TRUCK_TRANSIT_MINUTES_MAX - TRUCK_TRANSIT_MINUTES_MIN + 1
    return TRUCK_TRANSIT_MINUTES_MIN + stable_int(seed, int(dc_id), int(store_id), "lane") % span


# The replenishment policy, shared by history (``simulate_sku``) and the live
# stream (``stream_supply``).


def smooth_demand(rate: float, attempted: int) -> float:
    """Close a day: fold its attempted demand into the EWMA daily demand rate."""
    return 0.75 * rate + 0.25 * attempted


def decay_demand(rate: float, quiet_days: int) -> float:
    """EWMA rate after ``quiet_days`` closed days without any attempted demand."""
    return rate * 0.75 ** max(0, quiet_days)


def reorder_levels(rate: float) -> tuple[int, int]:
    """(reorder point, order-up-to target) for a smoothed daily demand ``rate``."""
    point = max(CASE_PACK, ceil(rate * 4))
    target = max(point + CASE_PACK, ceil(rate * 8))
    return point, target


def reorder_quantity(target: int, position: int) -> int:
    """Whole cases that lift the inventory position to the target."""
    return ceil((target - position) / CASE_PACK) * CASE_PACK


def reorder_priority(on_hand: int, rate: float) -> str:
    return "URGENT" if on_hand == 0 else ("HIGH" if on_hand < rate * 2 else "NORMAL")


@dataclass(frozen=True)
class Demand:
    event_ts: datetime
    receipt_id: str
    line_num: int
    quantity: int
    return_ts: datetime | None = None
    return_id: str | None = None
    # Units the return brings back (a partial return); None means the whole line.
    return_quantity: int | None = None

    @property
    def trace_id(self) -> str:
        return f"TRC-INV-{self.receipt_id}-{self.line_num}"


@dataclass
class InventoryEvent:
    # NULL for the DC's own rows (``kind`` DC_*), which carry ``dc_id``.
    store_id: int | None
    product_id: int
    event_ts: datetime
    trace_id: str
    kind: str
    quantity: int
    source: str
    balance: int = 0
    receipt_id: str | None = None
    line_num: int | None = None
    requested_quantity: int = 0
    unfulfilled_quantity: int = 0
    reorder_point: int | None = None
    priority: str | None = None
    shipment_id: str | None = None
    leg: int | None = None
    scheduled_ts: datetime | None = None
    loading_ts: datetime | None = None
    depart_ts: datetime | None = None
    eta: datetime | None = None
    unloading_ts: datetime | None = None
    etd: datetime | None = None
    dc_id: int | None = None
    # Planned arrival and the delay (minutes, reason) of a shipment's load.
    planned_eta: datetime | None = None
    delay_minutes: int = 0
    delay_reason: str | None = None


def shipment_base_id(dispatch: datetime, store_id: int) -> str:
    """A store's consolidated load of one dispatch day (``inventory.py`` adds the leg)."""
    return f"SHP-{dispatch:%Y%m%d}-{int(store_id):04d}"


def shipment_event(
    store_id: int,
    product_id: int,
    day: date,
    quantity: int,
    *,
    slot_minute: int,
    transit_minutes: int,
) -> InventoryEvent:
    """The SKU's line on its store's next load: the day after the reorder, at the slot.

    Timing depends only on the store's slot and lane, so every SKU the store
    reorders on ``day`` shares one dispatch and one arrival. Stock is received
    when unloading starts; ``inventory.py`` adds the unload duration (``etd``).
    """
    if not 0 <= slot_minute < 1440 or transit_minutes < 0:
        raise ValueError("invalid delivery slot or lane time")
    loading = datetime.combine(day + timedelta(days=1), inventory_time()) + timedelta(
        minutes=slot_minute
    )
    departure = loading + timedelta(minutes=TRUCK_LOAD_MINUTES)
    arrival = departure + timedelta(minutes=transit_minutes)
    shipment_id = shipment_base_id(loading, store_id)
    return InventoryEvent(
        store_id,
        product_id,
        datetime.combine(day, inventory_time(23, 59, 59)),
        f"TRC-PLAN-{shipment_id}-{product_id}",
        "SHIPMENT",
        quantity,
        shipment_id,
        shipment_id=shipment_id,
        leg=0,
        scheduled_ts=datetime.combine(day + timedelta(days=1), inventory_time()),
        loading_ts=loading,
        depart_ts=departure,
        eta=arrival,
        unloading_ts=arrival + timedelta(minutes=TRUCK_CHECKIN_MINUTES),
        planned_eta=arrival,
    )


class StoreSku:
    """One store-SKU's stock, demand estimate and pending receipts (the store policy).

    Shared by the single-store simulation (:func:`simulate_sku`) and the DC
    simulation (``dc_supply``), which decides how much of each reorder ships.
    """

    def __init__(
        self, *, store_id: int, product_id: int, seed: int, start_date: date, prior: float
    ) -> None:
        self.store_id = int(store_id)
        self.product_id = int(product_id)
        self.on_hand = opening_stock(seed, store_id, product_id, prior)
        self.rate = prior
        self.inbound = 0
        self.expected_returns = 0
        self.attempted = 0
        self.start_date = start_date
        self.pending: list[tuple[datetime, str, InventoryEvent]] = []
        self.previous: tuple[datetime, str] | None = None

    def opening(self) -> InventoryEvent:
        return InventoryEvent(
            self.store_id,
            self.product_id,
            datetime.combine(self.start_date, inventory_time()),
            f"TRC-INIT-ST-{self.store_id}-{self.product_id}",
            "INITIAL",
            self.on_hand,
            "SEED",
            balance=self.on_hand,
        )

    def _receive(self) -> InventoryEvent:
        _, _, event = heappop(self.pending)
        self.on_hand += event.quantity
        if event.kind == "RETURN":
            self.expected_returns -= event.quantity
        else:
            self.inbound -= event.quantity
        event.balance = self.on_hand
        return event

    def consume(self, candidate: Demand) -> Iterator[InventoryEvent]:
        """Receipts due before the demand line, then the sale or lost sale."""
        key = (candidate.event_ts, candidate.trace_id)
        if candidate.event_ts.date() < self.start_date or (self.previous and key <= self.previous):
            raise ValueError("demand must have unique, chronological line keys")
        if candidate.quantity <= 0:
            raise ValueError("candidate demand quantity must be positive")
        self.previous = key
        while self.pending and self.pending[0][:2] <= key:
            yield self._receive()
        self.attempted += candidate.quantity
        fulfilled = candidate.quantity <= self.on_hand
        if fulfilled:
            self.on_hand -= candidate.quantity
        yield InventoryEvent(
            self.store_id,
            self.product_id,
            candidate.event_ts,
            candidate.trace_id,
            "SALE" if fulfilled else "LOST_SALE",
            -candidate.quantity if fulfilled else 0,
            "CUSTOMER_PURCHASE",
            balance=self.on_hand,
            receipt_id=candidate.receipt_id,
            line_num=candidate.line_num,
            requested_quantity=candidate.quantity,
            unfulfilled_quantity=0 if fulfilled else candidate.quantity,
        )
        if fulfilled and candidate.return_ts is not None:
            if candidate.return_ts <= candidate.event_ts or not candidate.return_id:
                raise ValueError("restock returns must identify a later return")
            returned = InventoryEvent(
                self.store_id,
                self.product_id,
                candidate.return_ts,
                f"TRC-INV-{candidate.return_id}-{candidate.line_num}",
                "RETURN",
                min(candidate.quantity, candidate.return_quantity or candidate.quantity),
                candidate.return_id,
            )
            heappush(self.pending, (returned.event_ts, returned.trace_id, returned))
            self.expected_returns += returned.quantity

    def release(self, closing: datetime) -> Iterator[InventoryEvent]:
        """Receipts and returns due by ``closing``."""
        while self.pending and self.pending[0][0] <= closing:
            yield self._receive()

    def close_day(self, closing: datetime) -> tuple[int, int, str] | None:
        """Fold the day's demand into the rate; ``(quantity, point, priority)`` to reorder."""
        self.rate = smooth_demand(self.rate, self.attempted)
        self.attempted = 0
        point, target = reorder_levels(self.rate)
        # Only near-term returns reduce the order. A distant return cannot
        # cover demand before the service lead time has elapsed.
        near_returns = (
            sum(
                event.quantity
                for ts, _, event in self.pending
                if event.kind == "RETURN" and ts <= closing + timedelta(days=4)
            )
            if self.expected_returns
            else 0
        )
        position = self.on_hand + self.inbound + near_returns
        if position > point:
            return None
        return (
            reorder_quantity(target, position),
            point,
            reorder_priority(self.on_hand, self.rate),
        )

    def reorder_event(self, day: date, quantity: int, point: int, priority: str) -> InventoryEvent:
        return InventoryEvent(
            self.store_id,
            self.product_id,
            datetime.combine(day, inventory_time(23, 59, 59)),
            f"TRC-RO-{self.store_id}-{day}-{self.product_id}",
            "REORDER",
            quantity,
            "POLICY",
            balance=self.on_hand,
            reorder_point=point,
            priority=priority,
        )

    def expect(self, shipment: InventoryEvent) -> InventoryEvent:
        """Book a shipment's receipt (when its unloading starts); returns the receipt."""
        if shipment.unloading_ts is None:
            raise RuntimeError(f"shipment {shipment.source} has no unloading time")
        delivery = InventoryEvent(
            self.store_id,
            self.product_id,
            shipment.unloading_ts,
            f"TRC-SI-{shipment.source}-{self.product_id}",
            "INBOUND_SHIPMENT",
            shipment.quantity,
            shipment.source,
        )
        heappush(self.pending, (delivery.event_ts, delivery.trace_id, delivery))
        self.inbound += delivery.quantity
        return delivery

    def add_to(self, delivery: InventoryEvent, quantity: int) -> None:
        """Add units to a receipt already booked (a load that took a later reorder)."""
        delivery.quantity += quantity
        self.inbound += quantity


def simulate_sku(
    demand: Iterable[Demand],
    *,
    store_id: int,
    product_id: int,
    start_date: date,
    end_date: date,
    seed: int,
    daily_prior: float = 1.0,
    slot_minute: int | None = None,
    transit_minutes: int | None = None,
) -> Iterator[InventoryEvent]:
    """One store-SKU supplied by an unlimited DC, with loads always on plan.

    Consumes sorted demand with O(one SKU + pending deliveries/returns) state.
    ``daily_prior`` is the SKU's expected daily demand: it sizes the opening
    stock and starts the demand estimate, so history has no start-up stockout
    wave. ``slot_minute`` and ``transit_minutes`` are the store's delivery slot
    and lane (``network.plan_routes``); they default to the stable fallbacks.
    History runs the DC-limited, incident-aware ``dc_supply.simulate_dc_sku``.
    """
    if daily_prior < 0 or end_date < start_date:
        raise ValueError("invalid inventory simulation bounds")
    if slot_minute is None:
        slot_minute = delivery_slot_minute(seed, store_id)
    if transit_minutes is None:
        transit_minutes = lane_transit_minutes(seed, 0, store_id)
    sku = StoreSku(
        store_id=store_id,
        product_id=product_id,
        seed=seed,
        start_date=start_date,
        prior=daily_prior,
    )
    yield sku.opening()
    iterator = iter(demand)
    candidate = next(iterator, None)
    day = start_date
    while day <= end_date:
        closing = datetime.combine(day, inventory_time(23, 59, 59))
        while candidate is not None and candidate.event_ts <= closing:
            yield from sku.consume(candidate)
            candidate = next(iterator, None)
        yield from sku.release(closing)
        order = sku.close_day(closing)
        if order is not None:
            quantity, point, priority = order
            yield sku.reorder_event(day, quantity, point, priority)
            shipment = shipment_event(
                store_id,
                product_id,
                day,
                quantity,
                slot_minute=slot_minute,
                transit_minutes=transit_minutes,
            )
            yield shipment
            sku.expect(shipment)
        day += timedelta(days=1)
    if candidate is not None:
        raise ValueError("candidate demand falls outside the simulation history")


# ---------------------------------------------------------------------------
# DC replenishment (shared by dc_supply and stream_supply)
# ---------------------------------------------------------------------------


def dc_lead_days(seed: int, dc_id: int, product_id: int) -> int:
    """Supplier lead time (days) of a DC's product (a stable draw)."""
    low, high = DC_SUPPLIER_LEAD_DAYS
    return low + stable_int(seed, int(dc_id), int(product_id), "lead") % (high - low + 1)


def dc_case_floor(stores: int) -> int:
    """Smallest useful DC stock: a store reorders whole cases, so a case per store."""
    return max(1, int(stores)) * CASE_PACK


def dc_opening_stock(
    seed: int, dc_id: int, product_id: int, rate: float, floor: int = CASE_PACK
) -> int:
    """A DC's opening stock: a stable 10-16 days of its expected daily demand, in cases.

    At least two of ``floor`` (``dc_case_floor``): a slow mover still covers a
    first case to each store and the next.
    """
    low, high = DC_OPENING_DAYS
    days = low + stable_int(seed, int(dc_id), int(product_id), "dc-opening") % (high - low + 1)
    return max(2 * floor, ceil(rate * days / CASE_PACK) * CASE_PACK)


def smooth_dc_demand(rate: float, outbound: int) -> float:
    """Close a DC day: fold its requested units into the smoothed daily demand."""
    return (1.0 - DC_DEMAND_SMOOTHING) * rate + DC_DEMAND_SMOOTHING * outbound


def dc_reorder_levels(rate: float, lead_days: int, floor: int = CASE_PACK) -> tuple[int, int]:
    """(reorder point, order-up-to target) of a DC product with ``lead_days`` supply.

    ``floor`` (``dc_case_floor``) keeps a case per store in reserve for slow movers.
    """
    point = max(floor, ceil(rate * (lead_days + 1 + DC_SAFETY_DAYS)))
    target = max(point + floor, ceil(rate * (lead_days + DC_ORDER_UP_TO_DAYS)))
    return point, target


def supplier_receipt(ordered: int, fill_rate: float) -> int:
    """Units of a supplier order that arrive (whole cases) at a fill rate."""
    if fill_rate >= 1.0:
        return ordered
    return int(ordered * fill_rate) // CASE_PACK * CASE_PACK


def allocate(available: int, requests: Iterable[tuple[object, int]]) -> list[tuple[object, int]]:
    """Ship requests in order from ``available`` units: ``[(key, shipped)]``."""
    shipped = []
    for key, quantity in requests:
        ship = max(0, min(int(quantity), available))
        available -= ship
        shipped.append((key, ship))
    return shipped


PRIORITY_ORDER = {"URGENT": 0, "HIGH": 1, "NORMAL": 2}

# --- retail_setup/generation/incidents.py ---
"""Logistics incidents and delivery timing, shared by history and the live stream.

**Incidents.** :func:`plan_incidents` lays out, deterministically from the seed,
every disruption of the supply network between two dates:

- ``DC_OUTAGE``: a DC stops dispatching for 6-36 hours (power, systems,
  equipment, labor); loads wait for the stores' next delivery slot after it.
- ``LANE_DELAY``: one DC-to-store lane runs slow for 2-12 hours (closures,
  accidents, construction); trucks on it arrive 45-180 minutes late.
- ``WEATHER``: severe weather in a geography (the severe-weather days of
  ``stream_calendar``, opened more often in the climate's rough season) slows
  the lanes into its stores for one to four days.
- ``HOLIDAY_CAPACITY``: strained DC crews over the Black Friday weekend and
  the days after Christmas delay loading.
- ``SUPPLIER_SHORT_SHIP``: one supplier group (a sixth of the products) ships
  only 20-60% of the DC's orders for 3-10 days, so the DC runs short.

Each incident has a start, an end, a severity, its DC (and lane store or
geography) and the set of stores it affects: every store the DC serves, the
lane's store, or the geography's stores. Random incidents occur a few times a
year per DC (``calibration``).

**Delivery timing.** :func:`plan_delivery` turns a store's load into a
planned and an actual schedule in the store's local time. The plan ships at
the store's next delivery slot whose arrival day the store is open (no
deliveries on holiday closures), loads, and drives the lane. The actual run
waits out DC outages (to the next slot after the outage), loads late under
holiday capacity limits, and arrives late from traffic (more likely on long
lanes and Fridays), lane incidents and weather. ``delay_minutes`` is the
actual minus the planned arrival; ``delay_reason`` names the largest cause.

History (``inventory_policy``) and the live stream (``stream_supply``) both
use these functions, so they disrupt the same lanes on the same days. Pure
Python; ``build_notebooks.py`` inlines this module into the notebooks.
"""

from collections.abc import Callable, Iterable, Mapping
from dataclasses import dataclass, field
from datetime import date, datetime, time, timedelta, timezone
from zoneinfo import ZoneInfo


INCIDENT_TYPES = (
    "DC_OUTAGE",
    "LANE_DELAY",
    "WEATHER",
    "HOLIDAY_CAPACITY",
    "SUPPLIER_SHORT_SHIP",
)
# delay_reason values of a late truck (TRAFFIC has no incident).
DELAY_REASONS = ("DC_OUTAGE", "HOLIDAY_CAPACITY", "LANE_DELAY", "WEATHER", "TRAFFIC")
SEVERITIES = ("LOW", "MEDIUM", "HIGH")
# An incident planned this many days before a window can still be open in it
# (the longest incident is a supplier short-ship).
INCIDENT_LOOKBACK_DAYS = SUPPLIER_SHORT_SHIP_DAYS[1] + 1
# Severe-weather days needed after a window to size its last weather incident.
WEATHER_LOOKAHEAD_DAYS = WEATHER_INCIDENT_MAX_DAYS
# Holiday weekends that strain DC crews, and how many days each lasts.
HOLIDAY_CAPACITY_WINDOWS = {"BLACK_FRIDAY": 4, "DAY_AFTER_CHRISTMAS": 2}

_OUTAGE_CAUSES = (
    "power failure",
    "warehouse system outage",
    "dock equipment failure",
    "labor shortage",
)
_LANE_CAUSES = (
    "highway closure",
    "multi-vehicle accident",
    "road construction",
    "bridge restriction",
)
_TYPE_CODES = {
    "DC_OUTAGE": "OUT",
    "LANE_DELAY": "LANE",
    "WEATHER": "WX",
    "HOLIDAY_CAPACITY": "HOL",
    "SUPPLIER_SHORT_SHIP": "SUP",
}
_INCIDENT_UNIT = 1_000_000


def _incident_unit(seed: int, *keys: object) -> float:
    """Stable uniform draw in [0, 1)."""
    return stable_int(seed, *keys) % _INCIDENT_UNIT / _INCIDENT_UNIT


def _incident_between(seed: int, bounds: tuple[int, int], *keys: object) -> int:
    """Stable integer draw in ``[low, high]``."""
    low, high = bounds
    return low + stable_int(seed, *keys) % (high - low + 1)


def supplier_group(seed: int, product_id: int) -> int:
    """The supplier group a product comes from."""
    return stable_int(seed, int(product_id), "supplier") % SUPPLIER_GROUPS


def delay_severity(minutes: int) -> str:
    if minutes >= INCIDENT_HIGH_DELAY_MINUTES:
        return "HIGH"
    return "MEDIUM" if minutes >= INCIDENT_MEDIUM_DELAY_MINUTES else "LOW"


def outage_severity(hours: float) -> str:
    if hours >= INCIDENT_HIGH_OUTAGE_HOURS:
        return "HIGH"
    return "MEDIUM" if hours >= INCIDENT_MEDIUM_OUTAGE_HOURS else "LOW"


def short_ship_severity(fill_rate: float) -> str:
    lost = 1.0 - fill_rate
    if lost >= INCIDENT_HIGH_SHORT_SHARE:
        return "HIGH"
    return "MEDIUM" if lost >= INCIDENT_MEDIUM_SHORT_SHARE else "LOW"


@dataclass(frozen=True)
class Incident:
    """One logistics incident; times are UTC (aware)."""

    incident_id: str
    incident_type: str
    dc_id: int | None
    store_id: int | None
    started_at: datetime
    resolved_at: datetime
    severity: str
    affected_stores: tuple[int, ...]
    description: str
    # Minutes a delay incident adds to a run (loading or transit).
    delay_minutes: int = 0
    # Share of supplier orders a short-shipping supplier delivers.
    fill_rate: float = 1.0
    supplier_group: int | None = None
    geography_id: int | None = None

    def active_at(self, at: datetime) -> bool:
        return self.started_at <= at < self.resolved_at


@dataclass(frozen=True)
class IncidentNetwork:
    """The network the incidents run on: store routing, zones and geographies."""

    store_dc: Mapping[int, int]
    store_zones: Mapping[int, str] = field(default_factory=dict)
    store_geo: Mapping[int, int] = field(default_factory=dict)
    geo_state: Mapping[int, str] = field(default_factory=dict)
    dc_zones: Mapping[int, str] = field(default_factory=dict)

    def dc_ids(self) -> list[int]:
        return sorted({int(dc) for dc in self.store_dc.values()})

    def stores_of(self, dc_id: int) -> tuple[int, ...]:
        return tuple(sorted(int(s) for s, dc in self.store_dc.items() if int(dc) == int(dc_id)))

    def zone_of_store(self, store_id: int) -> str:
        return self.store_zones.get(int(store_id)) or "UTC"

    def zone_of_dc(self, dc_id: int) -> str:
        zone = self.dc_zones.get(int(dc_id))
        if zone:
            return zone
        stores = self.stores_of(dc_id)
        return self.zone_of_store(stores[0]) if stores else "UTC"


def local_instant(day: date, minute: int, zone: str) -> datetime:
    """UTC instant of a local wall-clock minute of ``day`` in ``zone``."""
    start = datetime.combine(day, time(), tzinfo=ZoneInfo(zone))
    return (start + timedelta(minutes=minute)).astimezone(timezone.utc)


def to_local(at: datetime, zone: str) -> datetime:
    """Naive local wall-clock time of a UTC instant in ``zone``."""
    return at.astimezone(ZoneInfo(zone)).replace(tzinfo=None)


def to_utc(local: datetime, zone: str) -> datetime:
    """UTC instant of a naive local wall-clock time in ``zone``."""
    return local.replace(tzinfo=ZoneInfo(zone)).astimezone(timezone.utc)


def _incident_id(kind: str, day: date, *parts: int) -> str:
    keys = "-".join(f"{int(part):03d}" for part in parts)
    return f"INC-{_TYPE_CODES[kind]}-{keys}-{day:%Y%m%d}"


def _dc_incidents(seed: int, network: IncidentNetwork, dc_id: int, day: date) -> list[Incident]:
    stores = network.stores_of(dc_id)
    zone = network.zone_of_dc(dc_id)
    found: list[Incident] = []
    if _incident_unit(seed, dc_id, day, "outage") < DC_OUTAGES_PER_DC_YEAR / 365.0:
        hours = _incident_between(seed, DC_OUTAGE_HOURS, dc_id, day, "outage-hours")
        start = local_instant(day, stable_int(seed, dc_id, day, "outage-minute") % 1440, zone)
        cause = _OUTAGE_CAUSES[stable_int(seed, dc_id, day, "outage-cause") % len(_OUTAGE_CAUSES)]
        found.append(
            Incident(
                _incident_id("DC_OUTAGE", day, dc_id),
                "DC_OUTAGE",
                dc_id,
                None,
                start,
                start + timedelta(hours=hours),
                outage_severity(hours),
                stores,
                f"DC {dc_id} outage ({cause}): no dispatches for {hours}h",
            )
        )
    if stores and _incident_unit(seed, dc_id, day, "lane") < LANE_DELAYS_PER_DC_YEAR / 365.0:
        store = stores[stable_int(seed, dc_id, day, "lane-store") % len(stores)]
        hours = _incident_between(seed, LANE_DELAY_HOURS, dc_id, day, "lane-hours")
        minutes = _incident_between(seed, LANE_DELAY_MINUTES, dc_id, day, "lane-minutes")
        # Lane incidents start in the daytime, 05:00-20:59 local.
        minute = 300 + stable_int(seed, dc_id, day, "lane-minute") % 960
        start = local_instant(day, minute, network.zone_of_store(store))
        cause = _LANE_CAUSES[stable_int(seed, dc_id, day, "lane-cause") % len(_LANE_CAUSES)]
        found.append(
            Incident(
                _incident_id("LANE_DELAY", day, dc_id, store),
                "LANE_DELAY",
                dc_id,
                store,
                start,
                start + timedelta(hours=hours),
                delay_severity(minutes),
                (store,),
                f"Lane DC {dc_id} to store {store} ({cause}): trucks {minutes} min late",
                delay_minutes=minutes,
            )
        )
    if _incident_unit(seed, dc_id, day, "supplier") < SUPPLIER_SHORT_SHIPS_PER_DC_YEAR / 365.0:
        days = _incident_between(seed, SUPPLIER_SHORT_SHIP_DAYS, dc_id, day, "supplier-days")
        low, high = SUPPLIER_FILL_RATE
        fill = round(low + (high - low) * _incident_unit(seed, dc_id, day, "supplier-fill"), 2)
        group = stable_int(seed, dc_id, day, "supplier-group") % SUPPLIER_GROUPS
        start = local_instant(day, 0, zone)
        found.append(
            Incident(
                _incident_id("SUPPLIER_SHORT_SHIP", day, dc_id),
                "SUPPLIER_SHORT_SHIP",
                dc_id,
                None,
                start,
                local_instant(day + timedelta(days=days), 0, zone),
                short_ship_severity(fill),
                stores,
                f"Supplier group {group} short-shipping DC {dc_id}: "
                f"{round(fill * 100)}% of orders arrive for {days} days",
                fill_rate=fill,
                supplier_group=group,
            )
        )
    name = holiday_name(day)
    if name in HOLIDAY_CAPACITY_WINDOWS and (
        _incident_unit(seed, dc_id, day, "holiday") < HOLIDAY_CAPACITY_SHARE
    ):
        minutes = _incident_between(
            seed, HOLIDAY_CAPACITY_DELAY_MINUTES, dc_id, day, "holiday-minutes"
        )
        days = HOLIDAY_CAPACITY_WINDOWS[name]
        found.append(
            Incident(
                _incident_id("HOLIDAY_CAPACITY", day, dc_id),
                "HOLIDAY_CAPACITY",
                dc_id,
                None,
                local_instant(day, 0, zone),
                local_instant(day + timedelta(days=days), 0, zone),
                delay_severity(minutes),
                stores,
                f"DC {dc_id} holiday peak ({name.replace('_', ' ').lower()}): "
                f"loading runs {minutes} min late",
                delay_minutes=minutes,
            )
        )
    return found


def _weather_incidents(
    seed: int, network: IncidentNetwork, geo: int, day: date, severe: set[tuple[int, date]]
) -> list[Incident]:
    if (geo, day) not in severe or (geo, day - timedelta(days=1)) in severe:
        return []
    sigma = WEATHER_SIGMA[weather_climate(network.geo_state.get(geo))]
    share = WEATHER_INCIDENT_SHARE * sigma[day.month - 1] / max(sigma)
    if _incident_unit(seed, geo, day, "weather-incident") >= share:
        return []
    days = 1
    while days < WEATHER_INCIDENT_MAX_DAYS and (geo, day + timedelta(days=days)) in severe:
        days += 1
    minutes = _incident_between(seed, WEATHER_DELAY_MINUTES, geo, day, "weather-minutes")
    by_dc: dict[int, list[int]] = {}
    for store, store_geo in network.store_geo.items():
        if int(store_geo) == geo and int(store) in network.store_dc:
            by_dc.setdefault(int(network.store_dc[int(store)]), []).append(int(store))
    found = []
    for dc_id, stores in sorted(by_dc.items()):
        zone = network.zone_of_store(min(stores))
        found.append(
            Incident(
                _incident_id("WEATHER", day, geo, dc_id),
                "WEATHER",
                dc_id,
                None,
                local_instant(day, 0, zone),
                local_instant(day + timedelta(days=days), 0, zone),
                delay_severity(minutes),
                tuple(sorted(stores)),
                f"Severe weather in geography {geo} ({network.geo_state.get(geo) or 'unknown'}): "
                f"lanes from DC {dc_id} run {minutes} min late for {days} day(s)",
                delay_minutes=minutes,
                geography_id=geo,
            )
        )
    return found


def plan_incidents(
    seed: int,
    network: IncidentNetwork,
    start: date,
    end: date,
    severe_days: Iterable[tuple[int, date]] = (),
) -> list[Incident]:
    """Every incident that starts between ``start`` and ``end`` (inclusive).

    ``severe_days`` holds ``(geography_id, date)`` pairs of severe weather
    (``stream_calendar.severe_weather_column``); it must cover the window plus
    ``WEATHER_LOOKAHEAD_DAYS`` after it and the day before it. Each incident
    depends only on the seed and its own day, so overlapping windows agree.
    """
    if end < start:
        return []
    severe = {(int(geo), day) for geo, day in severe_days}
    geos = sorted(
        {int(geo) for store, geo in network.store_geo.items() if int(store) in network.store_dc}
    )
    found: list[Incident] = []
    day = start
    while day <= end:
        for dc_id in network.dc_ids():
            found.extend(_dc_incidents(seed, network, dc_id, day))
        for geo in geos:
            found.extend(_weather_incidents(seed, network, geo, day, severe))
        day += timedelta(days=1)
    return sorted(found, key=lambda incident: (incident.started_at, incident.incident_id))


class IncidentCalendar:
    """Queries over a set of incidents at UTC instants."""

    def __init__(self, incidents: Iterable[Incident]) -> None:
        self.incidents = tuple(
            sorted(incidents, key=lambda incident: (incident.started_at, incident.incident_id))
        )

    def active(self, at: datetime) -> list[Incident]:
        return [incident for incident in self.incidents if incident.active_at(at)]

    def _of(self, kind: str, dc_id: int | None = None) -> list[Incident]:
        return [
            incident
            for incident in self.incidents
            if incident.incident_type == kind and (dc_id is None or incident.dc_id == dc_id)
        ]

    def outage_end(self, dc_id: int, at: datetime) -> datetime | None:
        """End of the DC outage (chained outages included) holding ``at``, else None."""
        end = None
        current = at
        outages = self._of("DC_OUTAGE", int(dc_id))
        while True:
            covering = [o.resolved_at for o in outages if o.active_at(current)]
            if not covering:
                return end
            end = current = max(covering)

    def held_dcs(self, at: datetime) -> frozenset[int]:
        """DCs in an outage at ``at``."""
        return frozenset(
            int(incident.dc_id)
            for incident in self._of("DC_OUTAGE")
            if incident.dc_id is not None and incident.active_at(at)
        )

    def supplier_fill(self, dc_id: int, group: int, at: datetime) -> float:
        """Share of a supplier order for the group that arrives at the DC at ``at``."""
        fills = [
            incident.fill_rate
            for incident in self._of("SUPPLIER_SHORT_SHIP", int(dc_id))
            if incident.supplier_group == group and incident.active_at(at)
        ]
        return min(fills, default=1.0)

    def store_view(self, store_id: int, dc_id: int, zone: str) -> "StoreIncidentView":
        """The incidents that time one store's loads, in its local time."""

        def windows(kinds: Iterable[Incident]) -> tuple[tuple[datetime, datetime, int], ...]:
            return tuple(
                (
                    to_local(incident.started_at, zone),
                    to_local(incident.resolved_at, zone),
                    incident.delay_minutes,
                )
                for incident in kinds
            )

        store_id, dc_id = int(store_id), int(dc_id)
        return StoreIncidentView(
            outages=windows(self._of("DC_OUTAGE", dc_id)),
            capacity=windows(self._of("HOLIDAY_CAPACITY", dc_id)),
            lanes=windows(i for i in self._of("LANE_DELAY", dc_id) if i.store_id == store_id),
            weather=windows(i for i in self._of("WEATHER") if store_id in i.affected_stores),
        )

    def dc_view(self, dc_id: int, zone: str) -> "DcIncidentView":
        """The supplier short-ships of one DC, in its local time."""
        return DcIncidentView(
            short_ships=tuple(
                (
                    to_local(incident.started_at, zone),
                    to_local(incident.resolved_at, zone),
                    int(incident.supplier_group or 0),
                    incident.fill_rate,
                )
                for incident in self._of("SUPPLIER_SHORT_SHIP", int(dc_id))
            )
        )


IncidentWindow = tuple[datetime, datetime, int]


def _window_delay(windows: Iterable[IncidentWindow], at: datetime) -> int:
    return max((minutes for start, end, minutes in windows if start <= at < end), default=0)


@dataclass(frozen=True)
class StoreIncidentView:
    """A store's incident windows (local naive start, end, delay minutes)."""

    outages: tuple[IncidentWindow, ...] = ()
    capacity: tuple[IncidentWindow, ...] = ()
    lanes: tuple[IncidentWindow, ...] = ()
    weather: tuple[IncidentWindow, ...] = ()

    def outage_end(self, at: datetime) -> datetime | None:
        end = None
        current = at
        while True:
            covering = [stop for start, stop, _ in self.outages if start <= current < stop]
            if not covering:
                return end
            end = current = max(covering)


@dataclass(frozen=True)
class DcIncidentView:
    """A DC's supplier short-ships (local naive start, end, group, fill rate)."""

    short_ships: tuple[tuple[datetime, datetime, int, float], ...] = ()

    def fill_rate(self, group: int, at: datetime) -> float:
        return min(
            (fill for start, end, g, fill in self.short_ships if g == group and start <= at < end),
            default=1.0,
        )


@dataclass(frozen=True)
class DeliveryTiming:
    """A load's planned and actual schedule (store-local naive times).

    ``slot`` is the delivery slot the load actually leaves at (after any
    outage hold); loading starts there unless holiday capacity delays it.
    """

    planned_loading: datetime
    slot: datetime
    loading: datetime
    departure: datetime
    planned_arrival: datetime
    arrival: datetime
    delay_minutes: int
    delay_reason: str | None


def traffic_delay_minutes(
    seed: int, store_id: int, dc_id: int, transit_minutes: int, planned_loading: datetime
) -> int:
    """Minutes traffic adds to a run (0 for most runs), keyed by its planned loading."""
    span = TRUCK_TRANSIT_MINUTES_MAX - TRUCK_TRANSIT_MINUTES_MIN
    length = min(1.0, max(0.0, (transit_minutes - TRUCK_TRANSIT_MINUTES_MIN) / span))
    low, high = TRUCK_TRAFFIC_LATE_SHARE
    chance = (low + (high - low) * length) * TRUCK_TRAFFIC_DAY_FACTORS[planned_loading.weekday()]
    key = (int(store_id), int(dc_id), f"{planned_loading:%Y%m%d%H%M}")
    if _incident_unit(seed, *key, "traffic") >= chance:
        return 0
    low_share, high_share = TRUCK_TRAFFIC_DELAY_SHARE
    share = low_share + (high_share - low_share) * _incident_unit(seed, *key, "traffic-share")
    return max(1, round(transit_minutes * share))


def next_local_slot(since: datetime, slot_minute: int) -> datetime:
    """The first local slot (minute of the day) at or after ``since``."""
    slot = datetime.combine(since.date(), time()) + timedelta(minutes=int(slot_minute))
    return slot if slot >= since else slot + timedelta(days=1)


def plan_delivery(
    seed: int,
    *,
    store_id: int,
    dc_id: int,
    ready: datetime,
    slot_minute: int,
    transit_minutes: int,
    view: StoreIncidentView | None = None,
    closed: Callable[[date], bool] | None = None,
) -> DeliveryTiming:
    """Planned and actual schedule of a load that is ready at ``ready`` (local).

    The plan ships at the first slot at or after ``ready`` whose arrival day
    the store is open. A DC outage moves loading to the first open slot after
    it; holiday capacity limits delay loading; traffic, lane incidents (at
    departure) and weather (at the planned arrival) delay the run.
    """
    run = timedelta(minutes=TRUCK_LOAD_MINUTES + int(transit_minutes))

    def open_slot(since: datetime) -> datetime:
        slot = next_local_slot(since, slot_minute)
        for _ in range(14):
            if closed is None or not closed((slot + run).date()):
                break
            slot = next_local_slot(slot + timedelta(minutes=1), slot_minute)
        return slot

    planned = open_slot(ready)
    slot = planned
    if view is not None:
        for _ in range(60):
            end = view.outage_end(slot)
            if end is None:
                break
            slot = open_slot(end)
    hold = round((slot - planned).total_seconds() / 60)
    capacity = _window_delay(view.capacity, slot) if view is not None else 0
    loading = slot + timedelta(minutes=capacity)
    departure = loading + timedelta(minutes=TRUCK_LOAD_MINUTES)
    planned_arrival = planned + run
    traffic = traffic_delay_minutes(seed, store_id, dc_id, transit_minutes, planned)
    lane = _window_delay(view.lanes, departure) if view is not None else 0
    weather = (
        _window_delay(view.weather, departure + timedelta(minutes=transit_minutes))
        if view is not None
        else 0
    )
    arrival = departure + timedelta(minutes=int(transit_minutes) + traffic + lane + weather)
    delay = round((arrival - planned_arrival).total_seconds() / 60)
    causes = [
        ("DC_OUTAGE", hold),
        ("HOLIDAY_CAPACITY", capacity),
        ("LANE_DELAY", lane),
        ("WEATHER", weather),
        ("TRAFFIC", traffic),
    ]
    reason = max(causes, key=lambda cause: cause[1])[0] if delay > 0 else None
    return DeliveryTiming(
        planned, slot, loading, departure, planned_arrival, arrival, delay, reason
    )

# --- retail_setup/generation/network.py ---
"""Store-to-DC routing, delivery routes and DC truck fleets.

History (``inventory.py``) and the live stream (``stream_supply``) share all of
it, so both deliver the same way:

- **Routing.** Each store is served by a DC in its own state, else in its
  region, else any DC. Among those candidates the store goes to the DC serving
  the fewest stores so far (stores in ID order, ties to the lower DC ID), so DCs
  in one state share the load instead of the lowest ID serving everything.
- **Routes.** Each DC truck runs a fixed daily route: a set of stores in one
  time zone, each with a daily delivery slot. Slots on a route are spaced by
  the longest possible trip (load, the lane both ways, the maximum unload, a
  turnaround), so a truck on its route is never in two places at once. A
  store's reorders of one day leave on its next slot as one consolidated load.
- **Fleets.** ``dim_trucks`` gives each DC one truck per route (at least
  ``MIN_DC_FLEET``) plus a shared pool (``DCID`` NULL) for overflow loads; a DC
  without assigned trucks falls back to the pool.
- **Unloading.** Dwell is driven by load size, a per-store dock factor, the
  store-local hour of arrival and a missed-appointment wait for late trucks
  (``unload_minutes``), capped at ``TRUCK_UNLOAD_MINUTES_MAX``.
"""

from collections.abc import Iterable, Mapping
from dataclasses import dataclass
from datetime import date, datetime, time, timedelta, timezone
from zoneinfo import ZoneInfo

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


MIN_DC_FLEET = 2
POOL_TRUCK_SHARE = 0.15
MINUTES_PER_DAY = 1440


def assign_dcs(
    stores: Iterable[tuple[int, str, str]], dcs: Iterable[tuple[int, str, str]]
) -> dict[int, int]:
    """``{store_id: dc_id}``: same state, else same region, else any DC, least loaded.

    ``stores`` and ``dcs`` hold ``(id, state, region)``.
    """
    sites = sorted((int(dc), state, region) for dc, state, region in dcs)
    if not sites:
        raise ValueError("routing requires at least one distribution center")
    load = {dc: 0 for dc, _, _ in sites}
    assigned: dict[int, int] = {}
    for store_id, state, region in sorted((int(s), st, rg) for s, st, rg in stores):
        candidates = (
            [dc for dc, dc_state, _ in sites if dc_state == state]
            or [dc for dc, _, dc_region in sites if dc_region == region]
            or [dc for dc, _, _ in sites]
        )
        best = min(candidates, key=lambda dc: (load[dc], dc))
        load[best] += 1
        assigned[store_id] = best
    return assigned


@dataclass(frozen=True)
class DeliveryRoute:
    """A store's place on its DC's delivery routes."""

    dc_id: int
    route: int
    slot_minute: int
    transit_minutes: int


def route_budget_minutes(transit_minutes: int) -> int:
    """Longest possible round trip to a store, plus the turnaround."""
    return (
        TRUCK_LOAD_MINUTES
        + 2 * transit_minutes
        + TRUCK_UNLOAD_MINUTES_MAX
        + TRUCK_TURNAROUND_MINUTES
    )


def plan_routes(
    seed: int, store_dc: Mapping[int, int], store_zones: Mapping[int, str]
) -> dict[int, DeliveryRoute]:
    """Daily delivery routes of every DC, packed first-fit into trucks.

    Stores of one DC and time zone (in ID order) fill a truck's day with their
    round-trip budgets; a route starts at a stable minute and each store's slot
    follows the previous store's budget, wrapping around midnight. Slots are in
    the store's local time.
    """
    groups: dict[int, dict[str, list[int]]] = {}
    for store_id, dc_id in store_dc.items():
        zone = store_zones.get(int(store_id), "UTC")
        groups.setdefault(int(dc_id), {}).setdefault(zone, []).append(int(store_id))
    routes: dict[int, DeliveryRoute] = {}
    for dc_id in sorted(groups):
        index = 0
        for zone in sorted(groups[dc_id]):
            bins: list[list[tuple[int, int, int]]] = []
            used: list[int] = []
            for store_id in sorted(groups[dc_id][zone]):
                transit = lane_transit_minutes(seed, dc_id, store_id)
                budget = route_budget_minutes(transit)
                slot = next(
                    (b for b, minutes in enumerate(used) if minutes + budget <= MINUTES_PER_DAY),
                    None,
                )
                if slot is None:
                    bins.append([])
                    used.append(0)
                    slot = len(bins) - 1
                bins[slot].append((store_id, transit, budget))
                used[slot] += budget
            for offset, members in enumerate(bins):
                minute = stable_int(seed, dc_id, zone, offset, "route") % MINUTES_PER_DAY
                for store_id, transit, budget in members:
                    routes[store_id] = DeliveryRoute(dc_id, index + offset, minute, transit)
                    minute = (minute + budget) % MINUTES_PER_DAY
            index += len(bins)
    return routes


def routes_per_dc(routes: Mapping[int, DeliveryRoute]) -> dict[int, int]:
    """Number of daily routes each DC runs."""
    counts: dict[int, int] = {}
    for route in routes.values():
        counts[route.dc_id] = max(counts.get(route.dc_id, 0), route.route + 1)
    return counts


def fleet_sizes(dc_routes: Mapping[int, int], dc_ids: Iterable[int]) -> dict[int, int]:
    """Trucks assigned to each DC: one per route, at least ``MIN_DC_FLEET``."""
    return {int(dc): max(MIN_DC_FLEET, int(dc_routes.get(int(dc), 0))) for dc in sorted(dc_ids)}


def pool_size(assigned_trucks: int) -> int:
    """Shared pool trucks so that they are POOL_TRUCK_SHARE of the fleet."""
    return max(1, round(assigned_trucks * POOL_TRUCK_SHARE / (1 - POOL_TRUCK_SHARE)))


def route_truck(route: DeliveryRoute, fleets: Mapping[int, tuple[int, ...]]) -> int:
    """The DC truck that runs a store's route."""
    fleet = fleets[route.dc_id]
    return fleet[route.route % len(fleet)]


def next_slot_ms(since_ms: int, slot_minute: int, zone: str | None = None) -> int:
    """First instant at or after ``since_ms`` whose wall clock in ``zone`` is the slot."""
    tz = ZoneInfo(zone) if zone else timezone.utc
    local_day = datetime.fromtimestamp(since_ms / 1000, tz).date()
    for days in range(3):
        start = datetime.combine(local_day + timedelta(days=days), time(), tzinfo=tz)
        candidate = round((start + timedelta(minutes=slot_minute)).timestamp() * 1000)
        if candidate >= since_ms:
            return candidate
    raise RuntimeError("no delivery slot within two days")


def dock_factor(seed: int, store_id: int) -> float:
    """How slowly a store's dock unloads relative to the calibrated rate."""
    low, high = TRUCK_DOCK_FACTOR_RANGE
    return low + (high - low) * (stable_int(seed, int(store_id), "dock") % 1001) / 1000


def unload_hour_factor(hour: int | None) -> float:
    """How the store's local hour scales unloading: slower when busy, quicker at night."""
    if hour is None:
        return 1.0
    busy_from, busy_to = TRUCK_UNLOAD_BUSY_HOURS
    night_from, night_to = TRUCK_UNLOAD_NIGHT_HOURS
    if busy_from <= hour < busy_to:
        return TRUCK_UNLOAD_BUSY_FACTOR
    if hour >= night_from or hour < night_to:
        return TRUCK_UNLOAD_NIGHT_FACTOR
    return 1.0


def estimated_unload_minutes(seed: int, store_id: int, units: int, hour: int | None = None) -> int:
    """Planned dwell of a load: base time plus the units at the site's rate and hour.

    ``hour`` is the store-local hour of arrival (None ignores the hour).
    """
    handling = (
        units
        / TRUCK_UNLOAD_UNITS_PER_MINUTE
        * dock_factor(seed, store_id)
        * unload_hour_factor(hour)
    )
    return min(
        TRUCK_UNLOAD_MINUTES_MAX,
        max(TRUCK_NORMAL_DWELL_MINUTES_MIN, round(TRUCK_UNLOAD_BASE_MINUTES + handling)),
    )


def unload_minutes(
    seed: int,
    store_id: int,
    units: int,
    shipment_id: str,
    hour: int | None = None,
    delay_minutes: int = 0,
) -> int:
    """Actual dwell: the estimate plus a missed-appointment wait and a little noise.

    A truck ``TRUCK_LATE_MINUTES`` or more late waits ``TRUCK_LATE_DOCK_WAIT_MINUTES``
    for a door. The result stays within the dwell bounds.
    """
    span = 2 * TRUCK_UNLOAD_NOISE_MINUTES + 1
    noise = stable_int(seed, shipment_id, "unload") % span - TRUCK_UNLOAD_NOISE_MINUTES
    wait = TRUCK_LATE_DOCK_WAIT_MINUTES if delay_minutes >= TRUCK_LATE_MINUTES else 0
    return min(
        TRUCK_UNLOAD_MINUTES_MAX,
        max(
            TRUCK_NORMAL_DWELL_MINUTES_MIN,
            estimated_unload_minutes(seed, store_id, units, hour) + wait + noise,
        ),
    )


def _store_sites(
    dim_stores: DataFrame, dim_geographies: DataFrame, dim_distribution_centers: DataFrame
) -> tuple[list[tuple[int, str, str]], list[tuple[int, str, str]], dict[int, str]]:
    geo = {
        int(row["ID"]): (row["State"], row["Region"])
        for row in dim_geographies.select("ID", "State", "Region").collect()
    }
    stores, zones = [], {}
    for row in dim_stores.select("ID", "GeographyID", "TimeZone").collect():
        state, region = geo[int(row["GeographyID"])]
        stores.append((int(row["ID"]), state, region))
        zones[int(row["ID"])] = row["TimeZone"] or "UTC"
    dcs = []
    for row in dim_distribution_centers.select("ID", "GeographyID").collect():
        state, region = geo[int(row["GeographyID"])]
        dcs.append((int(row["ID"]), state, region))
    return stores, dcs, zones


def store_routes(
    seed: int,
    dim_stores: DataFrame,
    dim_geographies: DataFrame,
    dim_distribution_centers: DataFrame,
) -> dict[int, DeliveryRoute]:
    """Every store's DC and delivery route, from the (small) dimensions."""
    stores, dcs, zones = _store_sites(dim_stores, dim_geographies, dim_distribution_centers)
    return plan_routes(seed, assign_dcs(stores, dcs), zones)


def store_dc_map(
    dim_stores: DataFrame, dim_geographies: DataFrame, dim_distribution_centers: DataFrame
) -> DataFrame:
    """One ``(store_id, dc_id)`` row per store (see :func:`assign_dcs`)."""
    stores, dcs, _ = _store_sites(dim_stores, dim_geographies, dim_distribution_centers)
    spark = SparkSession.getActiveSession() or dim_stores.sparkSession
    rows = sorted(assign_dcs(stores, dcs).items())
    return spark.createDataFrame(rows, "store_id long, dc_id long").select(
        F.col("store_id"), F.col("dc_id")
    )


def dc_fleets(
    trucks: Iterable[tuple[int, int | float | None]], dc_ids: Iterable[int]
) -> dict[int, tuple[int, ...]]:
    """Truck IDs of each DC's fleet from ``(truck_id, DCID)`` pairs.

    A DC with no assigned trucks uses the pool trucks (``DCID`` is NULL).
    """
    assigned: dict[int, list[int]] = {}
    pool: list[int] = []
    for truck_id, dc_id in trucks:
        if dc_id is None:
            pool.append(int(truck_id))
        else:
            assigned.setdefault(int(dc_id), []).append(int(truck_id))
    fleets: dict[int, tuple[int, ...]] = {}
    for dc in sorted(int(dc) for dc in dc_ids):
        fleet = sorted(assigned.get(dc) or pool)
        if not fleet:
            raise ValueError(f"DC {dc} has neither assigned nor shared trucks")
        fleets[dc] = tuple(fleet)
    return fleets


def incident_network(
    dim_stores: DataFrame, dim_geographies: DataFrame, dim_distribution_centers: DataFrame
) -> IncidentNetwork:
    """The routing, zones and geographies the incident model runs on."""
    stores, dcs, zones = _store_sites(dim_stores, dim_geographies, dim_distribution_centers)
    store_geo = {
        int(row["ID"]): int(row["GeographyID"])
        for row in dim_stores.select("ID", "GeographyID").collect()
    }
    geo_state = {
        int(row["ID"]): row["State"] for row in dim_geographies.select("ID", "State").collect()
    }
    return IncidentNetwork(
        store_dc=assign_dcs(stores, dcs),
        store_zones=zones,
        store_geo=store_geo,
        geo_state=geo_state,
    )


def severe_weather_days(
    spark: SparkSession, seed: int, geography_ids: Iterable[int], start: date, end: date
) -> set[tuple[int, date]]:
    """``(geography_id, date)`` severe-weather days (``severe_weather_column``)."""
    geos = sorted({int(geo) for geo in geography_ids})
    if not geos or end < start:
        return set()
    grid = date_grid(spark, start, end).crossJoin(
        spark.createDataFrame([(geo,) for geo in geos], "geo long")
    )
    severe = grid.where(severe_weather_column(seeded_draws(seed), F.col("geo"), F.col("day")))
    return {(int(row["geo"]), row["day"]) for row in severe.select("geo", "day").collect()}


def network_incidents(
    spark: SparkSession, seed: int, network: IncidentNetwork, start: date, end: date
) -> list[Incident]:
    """Incidents that can be open between ``start`` and ``end`` (planned from earlier days)."""
    first = start - timedelta(days=INCIDENT_LOOKBACK_DAYS)
    severe = severe_weather_days(
        spark,
        seed,
        network.store_geo.values(),
        first - timedelta(days=1),
        end + timedelta(days=WEATHER_LOOKAHEAD_DAYS),
    )
    return [
        incident
        for incident in plan_incidents(seed, network, first, end, severe)
        if incident.resolved_at.date() >= start
    ]

# --- retail_setup/generation/dims.py ---
"""Dimension generation using bounded driver metadata and Spark-native rows.

Semantics ported from datagen master_generators (ID schemes, tax lookup,
pricing rules); column names/types from schemas.TABLES, which the TMDL
contract test guards.
"""

from datetime import date, datetime, timedelta

import numpy as np
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


# datagen StoreProfiler equivalents (volume class -> traffic multiplier range).
# Five tiers incl. KIOSK, matching datagen's StoreVolumeClass (the kiosk tier
# was dropped in the first port; restored here for store-volume variety).
VOLUME_CLASSES = [
    ("flagship", 0.05, (1.8, 2.5)),
    ("high_volume", 0.15, (1.3, 1.8)),
    ("standard", 0.55, (0.8, 1.3)),
    ("low_volume", 0.20, (0.4, 0.8)),
    ("kiosk", 0.05, (0.25, 0.35)),
]
# Formats each volume class's size suits; a store picks among those its
# store type runs (StoreTypeProfile.resolved_store_formats), else the store
# type's smallest (small classes) or largest format. store_activity carries a
# zone-share row for every format in models.STORE_FORMATS.
VOLUME_FORMATS = {
    "flagship": ["hypermarket", "superstore", "boutique"],
    "high_volume": ["hypermarket", "superstore", "standard", "boutique"],
    "standard": ["superstore", "standard", "neighborhood", "boutique"],
    "low_volume": ["standard", "neighborhood", "express", "boutique"],
    "kiosk": ["express", "boutique"],
}
SMALL_VOLUME_CLASSES = {"low_volume", "kiosk"}
# Fixed day for the dimension RNG seed: dimensions depend on the seed alone.
DIMENSION_SEED_DAY = date(2000, 1, 1)
DEFAULT_TAX_RATE = 0.07407  # datagen receipts_mixin fallback
REFRIGERATED_CATEGORIES = {
    "Produce",
    "Dairy & Eggs",
    "Dairy & Alternatives",
    "Meat & Poultry",
    "Meat & Seafood",
    "Seafood",
    "Frozen",
}

# Map a product department/category token to a brand-dictionary Category so a
# product's brand fits its department (datagen product_generator parity: a
# grocery item never carries a hardware brand). Tokens are matched lowercased;
# substring matches (e.g. "home" in "home & garden") are tried before falling
# back to the whole brand pool.
_BRAND_CAT_SYNONYMS = {
    "grocery": "food",
    "fresh": "food",
    "pantry": "food",
    "snacks": "food",
    "beverages": "food",
    "health & beauty": "health",
    "home & garden": "home",
    "office supplies": "office",
    "pet supplies": "pet",
    "sports & recreation": "sports",
    "apparel": "clothing",
}


def _match_brand_category(key: str, brands_by_cat: dict[str, list]) -> list | None:
    """Resolve a product department/category to a brand pool, or None."""
    k = key.strip().lower()
    if k in brands_by_cat:
        return brands_by_cat[k]
    syn = _BRAND_CAT_SYNONYMS.get(k)
    if syn and syn in brands_by_cat:
        return brands_by_cat[syn]
    for cat, pool in brands_by_cat.items():
        if cat and (cat in k or k in cat):
            return pool
    return None


def compute_pricing(
    base_price: float, rng: np.random.Generator, department: str = ""
) -> tuple[float, float, float]:
    """Return ``(cost, msrp, sale_price)`` for a base price.

    After datagen ``PricingCalculator`` (shared/validators/pricing.py):

    - MSRP = BasePrice +/-15%, at the nearest charm price (.49/.99)
    - SalePrice = MSRP (60% of the time) OR MSRP discounted 5-35% (40%), at
      the nearest charm price not above MSRP
    - Cost leaves the department's gross margin (``pricing.margin_band``)

    Always guarantees ``Cost < SalePrice <= MSRP``.
    """
    # A positive cost and a strict one-cent margin require a minimum sale price
    # of two cents. This keeps the documented Cost < SalePrice invariant even
    # for adversarial one-cent source prices.
    msrp_cents = charm_cents(max(2.0, base_price * 100 * (1.0 + rng.uniform(-0.15, 0.15))))
    if rng.random() < 0.60:
        sale_cents = msrp_cents
    else:
        discounted = msrp_cents * (1.0 - rng.uniform(0.05, 0.35))
        sale_cents = min(msrp_cents, charm_cents(discounted))
    low, high = margin_band(department)
    sale = sale_cents / 100
    cost = round(sale * (1.0 - rng.uniform(low, high)), 2)
    cost = max(0.01, min(cost, round(sale - 0.01, 2)))
    return float(cost), float(msrp_cents / 100), float(sale)


def rebranded_name(name: str, brand: str, brands: list[str]) -> str:
    """``name`` sold under ``brand``: a name that starts with a brand takes this one.

    ``brands`` is every catalog brand, longest first, so the longest matching
    prefix is replaced. Unbranded (generic) names are kept.
    """
    for known in brands:
        if name.startswith(f"{known} "):
            return f"{brand} {name[len(known) + 1 :]}"
    return name


def balanced_geography_sample(
    states: list[str], size: int, rng: np.random.Generator, *, replace: bool = False
) -> list[int]:
    """Indices of ``size`` geographies spread evenly over their states.

    States are visited in a shuffled order, one city each per round, so a
    dictionary with most of its cities in a few states still yields a sample
    covering every state about equally. With ``replace`` a round may revisit a
    state's cities once all are used.
    """
    by_state: dict[str, list[int]] = {}
    for index, state in enumerate(states):
        by_state.setdefault(state, []).append(index)
    remaining = {state: list(rng.permutation(cities)) for state, cities in sorted(by_state.items())}
    order = [str(state) for state in rng.permutation(sorted(remaining))]
    picked: list[int] = []
    while len(picked) < size:
        progressed = False
        for state in order:
            if len(picked) == size:
                break
            if not remaining[state]:
                if not replace:
                    continue
                remaining[state] = list(rng.permutation(by_state[state]))
            picked.append(int(remaining[state].pop()))
            progressed = True
        if not progressed:
            raise ValueError(f"cannot sample {size} distinct geographies from {len(states)}")
    return picked


def _addr(rng: np.random.Generator) -> str:
    return (
        f"{int(rng.integers(100, 9999))} "
        f"{str(rng.choice(['Main', 'Oak', 'Maple', 'Market', 'Commerce', 'Liberty']))} "
        f"{str(rng.choice(['St', 'Ave', 'Blvd', 'Rd']))}"
    )


def generate_dimensions(
    spark: SparkSession, dicts: DictionarySet, cfg: GenerationConfig
) -> dict[str, DataFrame]:
    rng = np.random.default_rng(derive_seed(cfg.seed, "dims", 0, DIMENSION_SEED_DAY))
    out: dict[str, DataFrame] = {}

    # --- geographies: sample from dictionary, sequential IDs. The shared
    #     dictionary is skewed (most cities sit in five states), so the sample
    #     takes one city per state in turn (geo_rng); the main generator still
    #     makes its legacy draw so products and prices stay unchanged.
    n_geo = min(len(dicts.geographies), max(cfg.store_count * 2, cfg.dc_count * 2, 20))
    rng.choice(len(dicts.geographies), size=n_geo, replace=False)
    geo_rng = np.random.default_rng(derive_seed(cfg.seed, "geographies", 0, DIMENSION_SEED_DAY))
    geo_idx = balanced_geography_sample([g.State for g in dicts.geographies], n_geo, geo_rng)
    geos = [dicts.geographies[i] for i in geo_idx]
    geo_rows = [(i + 1, g.City, g.State, g.Zip, g.District, g.Region) for i, g in enumerate(geos)]
    out["dim_geographies"] = spark.createDataFrame(geo_rows, spark_schema("dim_geographies"))

    # tax lookup hierarchy (datagen TaxCalculator parity):
    #   exact (State, City) -> (State, County) average -> State average -> default.
    # County is resolved from the tax dictionary; geographies carry no county of
    # their own, so the county tier only applies to cities present in the dict.
    city_acc: dict[tuple[str, str], list[float]] = {}
    city_county: dict[tuple[str, str], str] = {}
    county_acc: dict[tuple[str, str], list[float]] = {}
    state_rates: dict[str, list[float]] = {}
    for t in dicts.tax_rates:
        rate = float(t.CombinedRate)
        city_acc.setdefault((t.StateCode, t.City), []).append(rate)
        city_county[(t.StateCode, t.City)] = t.County
        county_acc.setdefault((t.StateCode, t.County), []).append(rate)
        state_rates.setdefault(t.StateCode, []).append(rate)
    by_city = {k: float(np.mean(v)) for k, v in city_acc.items()}
    by_county = {k: float(np.mean(v)) for k, v in county_acc.items()}

    def tax_for(state: str, city: str) -> float:
        if (state, city) in by_city:
            return by_city[(state, city)]
        county = city_county.get((state, city))
        if county is not None and (state, county) in by_county:
            return by_county[(state, county)]
        if state in state_rates:
            return float(np.mean(state_rates[state]))
        return DEFAULT_TAX_RATE

    # --- DCs first (datagen: stores constrained to DC states), spread over states
    rng.choice(n_geo, size=cfg.dc_count, replace=cfg.dc_count > n_geo)
    dc_geo_idx = balanced_geography_sample(
        [g.State for g in geos], cfg.dc_count, geo_rng, replace=cfg.dc_count > n_geo
    )
    dc_rows = [
        (i + 1, f"DC{i + 1:03d}", _addr(rng), int(dc_geo_idx[i]) + 1) for i in range(cfg.dc_count)
    ]
    out["dim_distribution_centers"] = spark.createDataFrame(
        dc_rows, spark_schema("dim_distribution_centers")
    )

    # --- stores in DC states; formats and hours come from the store type
    # A store picks a DC (so its state) uniformly, then a city in that state:
    # each DC gets a similar number of nearby stores.
    dc_state_list = [geos[int(g)].State for g in dc_geo_idx]
    eligible = [i for i, g in enumerate(geos) if g.State in set(dc_state_list)] or list(
        range(n_geo)
    )
    cities_by_state: dict[str, list[int]] = {}
    for i in eligible:
        cities_by_state.setdefault(geos[i].State, []).append(i)
    profile_formats = [f for f in STORE_FORMATS if f in dicts.profile.resolved_store_formats()]
    profile_hours = dicts.profile.resolved_operating_hours()
    store_rows = []
    store_geo_indices: list[int] = []
    for sid in range(1, cfg.store_count + 1):
        rng.choice(eligible)
        state_cities = cities_by_state.get(
            dc_state_list[int(geo_rng.integers(len(dc_state_list)))], eligible
        )
        gi = int(state_cities[int(geo_rng.integers(len(state_cities)))])
        store_geo_indices.append(gi)
        g = geos[gi]
        classes, probs = zip(*[(c, p) for c, p, _ in VOLUME_CLASSES])
        vc = str(rng.choice(classes, p=probs))
        lo, hi = next(r for c, _, r in VOLUME_CLASSES if c == vc)
        formats = [f for f in VOLUME_FORMATS[vc] if f in profile_formats] or [
            profile_formats[-1] if vc in SMALL_VOLUME_CLASSES else profile_formats[0]
        ]
        store_rows.append(
            (
                sid,
                f"S{sid:06d}",
                _addr(rng),
                gi + 1,
                tax_for(g.State, g.City),
                vc,
                str(rng.choice(formats)),
                str(rng.choice(profile_hours)),
                float(np.round(rng.uniform(lo, hi), 2)),
                state_time_zone(g.State),
            )
        )
    out["dim_stores"] = spark.createDataFrame(store_rows, spark_schema("dim_stores"))

    # --- trucks: each DC runs one truck per daily delivery route over the
    #     stores it serves (network.assign_dcs / plan_routes, the routing the
    #     inventory chain and the live stream use), plus a shared pool with no
    #     DC (DCID NULL) for overflow loads; 40% are refrigerated.
    dc_sites = [
        (i + 1, geos[int(dc_geo_idx[i])].State, geos[int(dc_geo_idx[i])].Region)
        for i in range(cfg.dc_count)
    ]
    store_dc = assign_dcs(
        [(sid, geos[gi].State, geos[gi].Region) for sid, gi in enumerate(store_geo_indices, 1)],
        dc_sites,
    )
    routes = plan_routes(
        cfg.seed,
        store_dc,
        {row[0]: row[9] for row in store_rows},
    )
    fleets = fleet_sizes(routes_per_dc(routes), range(1, cfg.dc_count + 1))
    truck_dcs: list[float | None] = [float(dc) for dc, size in fleets.items() for _ in range(size)]
    truck_dcs += [None] * pool_size(len(truck_dcs))
    # The first trucks consume the main generator exactly as the earlier fixed
    # fleet did, so products (drawn after trucks) stay unchanged; any further
    # trucks draw from their own generator.
    legacy = [bool(rng.random() < 0.4) for _ in range(max(cfg.dc_count * 3, 6))]
    extra = np.random.default_rng(derive_seed(cfg.seed, "trucks", 0, DIMENSION_SEED_DAY))
    truck_rows = []
    for tid, dcid in enumerate(truck_dcs, start=1):
        plate = f"TRK{tid:04d}{chr(65 + tid % 26)}"
        refrig = legacy[tid - 1] if tid <= len(legacy) else bool(extra.random() < 0.4)
        truck_rows.append((tid, plate, refrig, dcid))
    out["dim_trucks"] = spark.createDataFrame(truck_rows, spark_schema("dim_trucks"))

    # --- customers; use Spark range + deterministic hashes so customer scale
    #     does not materialize one Python tuple per customer on the driver.
    #     Each customer has a home store, drawn by store traffic, and lives in
    #     its geography (generation.customers).
    first = [n.Name for n in dicts.first_names]
    last = [n.Name for n in dicts.last_names]
    customer_draws = seeded_draws(cfg.seed)
    picker = store_picker([(row[0], row[8]) for row in store_rows])
    customers = with_home_store(
        cfg.seed,
        spark.range(1, cfg.customer_count + 1).select(F.col("id").cast("long").alias("ID")),
        picker,
    )
    first_names = F.array(*[F.lit(str(name)) for name in first])
    last_names = F.array(*[F.lit(str(name)) for name in last])
    store_geographies = F.array(*[F.lit(index + 1).cast("long") for index in store_geo_indices])
    geography = F.element_at(store_geographies, F.col("HomeStoreID").cast("int"))

    street_names = F.array(
        *[F.lit(name) for name in ["Main", "Oak", "Maple", "Market", "Commerce", "Liberty"]]
    )
    street_types = F.array(*[F.lit(name) for name in ["St", "Ave", "Blvd", "Rd"]])
    first_index = (customer_draws.h64(["ID"], "customer-first") % F.lit(len(first)) + 1).cast(
        "long"
    )
    last_index = (customer_draws.h64(["ID"], "customer-last") % F.lit(len(last)) + 1).cast("long")
    street_number = (customer_draws.h64(["ID"], "customer-street-number") % F.lit(9900) + 100).cast(
        "long"
    )
    street_name = F.element_at(
        street_names,
        (customer_draws.h64(["ID"], "customer-street-name") % F.lit(6) + 1).cast("int"),
    )
    street_type = F.element_at(
        street_types,
        (customer_draws.h64(["ID"], "customer-street-type") % F.lit(4) + 1).cast("int"),
    )
    loyalty_suffix = (customer_draws.h64(["ID"], "customer-loyalty") % F.lit(1000)).cast("long")
    phone_exchange = (
        customer_draws.h64(["ID"], "customer-phone-exchange") % F.lit(799) + 200
    ).cast("long")
    phone_line = (customer_draws.h64(["ID"], "customer-phone-line") % F.lit(8999) + 1000).cast(
        "long"
    )

    customers = customers.select(
        "ID",
        F.element_at(first_names, first_index.cast("int")).alias("FirstName"),
        F.element_at(last_names, last_index.cast("int")).alias("LastName"),
        F.format_string("%d %s %s", street_number, street_name, street_type).alias("Address"),
        geography.alias("GeographyID"),
        F.concat(
            F.lit("LC"),
            minimum_width_decimal(F.col("ID"), 6),
            fixed_width_decimal(loyalty_suffix, 3, "loyalty suffix"),
        ).alias("LoyaltyCard"),
        F.format_string("555-%03d-%04d", phone_exchange, phone_line).alias("Phone"),
        ble_id_column(F.col("ID")).alias("BLEId"),
        ad_id_column(F.col("ID")).alias("AdId"),
        "HomeStoreID",
    )
    out["dim_customers"] = customers.select(
        *[field.name for field in spark_schema("dim_customers").fields]
    )

    # --- products: each base product is offered by up to brands_per_product
    #     category-matched brands (datagen combinatorial SKUs). Pricing/launch
    #     are re-rolled per branded variant for realistic price spread.
    brands_by_cat: dict[str, list] = {}
    for b in dicts.brands:
        brands_by_cat.setdefault(b.Category.strip().lower(), []).append(b)
    all_brands = list(dicts.brands)
    brand_names = sorted({b.Brand for b in all_brands}, key=lambda name: (-len(name), name))
    tags_by_product = {t.ProductName: t.Tags for t in dicts.tags}
    # Use naive UTC datetimes — Spark session timezone is UTC (set in conftest fixture).
    # Launch dates are placed around the pinned anchor, not the run window, so
    # they do not move when the same seed is rendered on another day.
    hist_start = datetime.combine(cfg.anchor_date, datetime.min.time())
    # A department's pinned product must launch before the first sale day even
    # when the history starts before the anchor.
    latest_pinned_launch = datetime.combine(cfg.start_date, datetime.min.time()) - timedelta(days=1)
    # Guarantee every department has at least one product available from the
    # first day of history: without this, an adversarial seed could push every
    # product in a department past `hist_start`, leaving a sale-eligible
    # department with no launched product and forcing receipts.py to either drop
    # lines or violate the no-pre-launch-sales invariant (IMP-010). The first
    # generated variant per department is pinned to a pre-history launch.
    dept_covered: set[str] = set()
    prod_rows = []
    pid = 0
    for p in dicts.products:
        pool = (
            _match_brand_category(p.Department, brands_by_cat)
            or _match_brand_category(p.Category, brands_by_cat)
            or all_brands
        )
        k = min(cfg.brands_per_product, len(pool))
        brand_idx = rng.choice(len(pool), size=k, replace=False)
        taxability = p.Taxability or (
            "NON_TAXABLE"
            if p.Department in {"Fresh", "Grocery"} and "Candy" not in p.Category
            else "REDUCED_RATE"
            if p.Department in {"Clothing", "Apparel"}
            else "TAXABLE"
        )
        refrigerated = (
            p.Refrigerated
            if p.Refrigerated is not None
            else bool(p.Category in REFRIGERATED_CATEGORIES)
        )
        tags = p.Tags or tags_by_product.get(p.ProductName)
        for j in brand_idx:
            pid += 1
            chosen = pool[int(j)]
            launch_r = float(rng.random())  # 60% before history, 30% first half, 10% later
            if p.Department not in dept_covered:
                # First product seen for this department: pin pre-history so the
                # department is sale-eligible on day one. Draw the RNG regardless
                # (above) to keep every other product's launch stream unchanged.
                launch = min(
                    hist_start - timedelta(days=int(rng.integers(30, 1500))),
                    latest_pinned_launch,
                )
                dept_covered.add(p.Department)
            elif launch_r < 0.6:
                launch = hist_start - timedelta(days=int(rng.integers(30, 1500)))
            elif launch_r < 0.9:
                launch = hist_start + timedelta(days=int(rng.integers(0, 183)))
            else:
                launch = hist_start + timedelta(days=int(rng.integers(183, 366)))
            cost, msrp, sale = compute_pricing(float(p.BasePrice), rng, p.Department)
            prod_rows.append(
                (
                    pid,
                    # A branded title carries the SKU's own brand.
                    rebranded_name(p.ProductName, chosen.Brand, brand_names),
                    chosen.Brand,
                    chosen.Company,
                    p.Department,
                    p.Category,
                    p.Subcategory,
                    cost,
                    msrp,
                    sale,
                    refrigerated,
                    launch,
                    taxability,
                    tags,
                )
            )
    out["dim_products"] = spark.createDataFrame(prod_rows, spark_schema("dim_products"))
    return out


def generate_dim_date(spark: SparkSession, start: date, end: date) -> DataFrame:
    """The calendar dimension (ISO weeks; fiscal years start in July).

    ``day_of_week`` and ``week_of_year`` follow ISO 8601: Monday is day 1 and
    week 1 holds the year's first Thursday, so the first days of January can
    fall in week 52 or 53 of the previous ISO year. ``fiscal_year`` runs July
    to June and is labelled by the calendar year it starts in (July 2025 to
    June 2026 is fiscal year 2025).
    """
    dates = date_grid(spark, start, end, "date")
    month = F.month("date")
    iso_day = F.pmod(F.dayofweek("date") + F.lit(5), F.lit(7)) + F.lit(1)
    week_of_year = F.weekofyear("date")
    return dates.select(
        F.date_format("date", "yyyyMMdd").cast("long").alias("date_key"),
        "date",
        F.year("date").cast("long").alias("year"),
        F.quarter("date").cast("long").alias("quarter"),
        month.cast("long").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.dayofmonth("date").cast("long").alias("day"),
        iso_day.cast("long").alias("day_of_week"),
        F.date_format("date", "EEEE").alias("day_name"),
        week_of_year.cast("long").alias("week_of_year"),
        F.when(iso_day >= 6, 1).otherwise(0).cast("long").alias("is_weekend"),
        F.when(month >= 7, F.year("date"))
        .otherwise(F.year("date") - 1)
        .cast("long")
        .alias("fiscal_year"),
        (F.floor(F.pmod(month - F.lit(7), F.lit(12)) / F.lit(3)) + F.lit(1))
        .cast("long")
        .alias("fiscal_quarter"),
    )

# --- retail_setup/generation/receipts.py ---
"""Receipts fact group, Spark-native.

Randomness: every stochastic decision derives a uniform double from
xxhash64(key columns, salt) via `runtime.seeded_draws` — partition-
arrangement-independent, so output is deterministic for a (config, seed) pair
regardless of cluster shape. The seed is folded into the salt delimiter inside
`seeded_draws`, keeping generators decoupled from one another.

Visits precede purchase decisions. Negative-binomial baskets combine shopping
missions and persistent budgets, calibrated against the eligible catalog.
Products are chosen by price-elastic popularity at the day's shelf price
(pricing.py); each basket then gains complementary products, merges repeated
SKUs and takes at most one promotion code (baskets.py, shared with the live
stream). Customers follow their lifecycle (customers.py); about 45% of
receipts are anonymous.

Money is integer cents end-to-end; tax replicates datagen's basis-point
integer formula exactly:
    rate_bps = round(rate * 10000); mult = percent of the rate the line pays;
    tax = (ext_cents * rate_bps * mult + 500_000) // 1_000_000
implemented with Spark integer `DIV` so no float rounding is involved. The
percent is 100 for taxable goods; groceries and clothing follow the store's
state rule (``baskets.line_tax_cents``, ``_shared/tax_rules.json``).
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Per-department seasonal demand multipliers by month (Jan..Dec), keyed by a
# lowercase token matched as a substring of the department name so the one
# table works across store types (datagen SeasonalPatterns category effects).
SEASONAL_LIFT: dict[str, list[float]] = {
    "electronics": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.0, 1.0, 2.5, 2.0],
    "seasonal": [0.6, 0.7, 0.9, 1.0, 1.3, 1.5, 1.5, 1.2, 1.0, 1.4, 2.0, 2.5],
    "garden": [0.5, 0.6, 1.3, 1.8, 2.0, 1.8, 1.4, 1.0, 0.8, 0.6, 0.5, 0.5],
    "home": [0.8, 0.8, 1.2, 1.5, 1.6, 1.4, 1.2, 1.0, 0.9, 0.8, 1.0, 1.1],
    "sport": [0.9, 0.9, 1.1, 1.3, 1.5, 1.6, 1.6, 1.3, 1.1, 1.0, 1.1, 1.2],
    "clothing": [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "apparel": [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "office": [1.1, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.6, 1.4, 1.0, 1.0, 1.1],
    "toys": [0.8, 0.8, 0.8, 0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.1, 2.0, 3.0],
    "baby": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.2],
    "grocery": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.3, 1.4],
    "fresh": [1.0, 1.0, 1.0, 1.0, 1.05, 1.1, 1.1, 1.05, 1.0, 1.05, 1.3, 1.4],
    "health": [1.3, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.1],
    "beauty": [1.2, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.2, 1.5],
    "automotive": [1.0, 1.0, 1.05, 1.1, 1.2, 1.2, 1.2, 1.1, 1.0, 1.0, 1.0, 1.0],
}


def _seasonal_factor(dept_name: str, month_col: Column) -> Column:
    """Monthly demand multiplier for a department (1.0 if no token matches)."""
    key = dept_name.lower()
    lifts = next((m for token, m in SEASONAL_LIFT.items() if token in key), None)
    if lifts is None:
        return F.lit(1.0)
    expr = case_when((month_col == F.lit(m), F.lit(lifts[m - 1])) for m in range(1, 13))
    return expr.otherwise(F.lit(1.0))


def _with_seasonal_department(
    df: DataFrame, u_col: Column, dept_weights: dict[str, float]
) -> DataFrame:
    """Pick a department per row via inverse-CDF over base weights scaled by the
    seasonal lift for the row's month.

    Intermediate per-department weight (``_w*``) and cumulative-CDF (``_c*``)
    columns are materialized so Catalyst codegen stays small even for store
    types with many departments (an inline single expression overflows the
    64 KB JVM method limit and forces interpreted fallback). They are dropped
    before returning.
    """
    depts = list(dept_weights)
    month = F.month(F.col("event_date"))
    out = df
    for i, dn in enumerate(depts):
        out = out.withColumn(f"_w{i}", F.lit(float(dept_weights[dn])) * _seasonal_factor(dn, month))
    wt = F.col("_w0")
    for i in range(1, len(depts)):
        wt = wt + F.col(f"_w{i}")
    out = out.withColumn("_wt", wt)
    prev: str | None = None
    for i in range(len(depts) - 1):
        term = F.col(f"_w{i}") / F.col("_wt")
        out = out.withColumn(f"_c{i}", term if prev is None else F.col(prev) + term)
        prev = f"_c{i}"
    expr: Column | None = None
    for i, dn in enumerate(depts[:-1]):
        cond = u_col < F.col(f"_c{i}")
        expr = F.when(cond, F.lit(dn)) if expr is None else expr.when(cond, F.lit(dn))
    dept_col = expr.otherwise(F.lit(depts[-1])) if expr is not None else F.lit(depts[0])
    out = out.withColumn("department", dept_col)
    drop_cols = (
        [f"_w{i}" for i in range(len(depts))] + ["_wt"] + [f"_c{i}" for i in range(len(depts) - 1)]
    )
    return out.drop(*drop_cols)


def customer_pools(
    lifecycle: CustomerLifecycle, dims: dict[str, DataFrame]
) -> tuple[DataFrame, DataFrame, Column]:
    """(ticket pool, pool sizes, store picker) of the customer dimension.

    Replicated selection tickets give persistent, long-tailed visit frequency;
    the dimension-sized expansion avoids receipt x customer inequality joins.
    """
    pool = ticket_pool(lifecycle, dims["dim_customers"])
    stores = dims["dim_stores"].select("ID", "daily_traffic_multiplier").collect()
    picker = store_picker(
        [(int(row["ID"]), float(row["daily_traffic_multiplier"])) for row in stores]
    )
    return pool, pool_sizes(pool), picker


def _assign_customers(
    receipts: DataFrame, dims: dict[str, DataFrame], lifecycle: CustomerLifecycle
) -> DataFrame:
    """Resolve each receipt's customer_id (NULL for anonymous shoppers).

    Most identified visits come from the store's own home customers, the
    rest from another store's; each customer is kept with their lifecycle
    activity on the receipt's local date (generation.customers).
    """
    pool, sizes, picker = customer_pools(lifecycle, dims)
    return assign_customers(
        lifecycle, receipts, pool, sizes, picker, "receipt_id_ext", "store_id", "event_date"
    )


def _fmt(cents_col: Column) -> Column:
    """Integer cents -> 'XX.XX' string (no thousands separators)."""
    return F.format_string("%.2f", cents_col / F.lit(100.0))


def _open_close(operating_hours: str) -> tuple[int, int]:
    """Parse an ``operating_hours`` string into ``[open, close)`` integer hours."""
    if operating_hours == "24h":
        return 0, 24
    lo, hi = operating_hours.split("-")
    return int(lo), int(hi)


def _open_close_cols(operating_hours: Column) -> tuple[Column, Column]:
    """Column form of :func:`_open_close` for ``[open, close)`` hour bounds.

    ``"24h"`` maps to ``[0, 24)``; ``"7-22"`` maps to ``[7, 22)``. The close hour
    is exclusive, so a ``"7-22"`` store transacts in hours 7..21 inclusive.
    """
    is_24 = operating_hours == "24h"
    parts = F.split(operating_hours, "-")
    open_h = F.when(is_24, F.lit(0)).otherwise(parts.getItem(0).cast("int"))
    close_h = F.when(is_24, F.lit(24)).otherwise(parts.getItem(1).cast("int"))
    return open_h, close_h


def payment_attempts(
    draws: seeded_draws,
    purchases: DataFrame,
    key: str,
    tenders: list[tuple[str, float, float, int, int]] | None = None,
    salt: str = "",
) -> DataFrame:
    """``fact_payments`` rows of purchases with a payment outcome (decline model).

    ``purchases`` holds ``key`` (``receipt_id_ext`` or ``order_id_ext``),
    ``event_ts``, ``store_id``, ``customer_id``, ``total_cents``, the first
    tender (``_first_tender``), the approved tender (``tender_type``) and
    ``_outcome``. Every purchase gets an attempt-1 payment, DECLINED when the
    sale was retried; a retried sale gets an APPROVED attempt 2 a minute later.
    Transaction IDs derive from the purchase ID and attempt, so they are unique.
    """

    tenders = TENDERS if tenders is None else tenders
    retried = F.col("_outcome") == F.lit(RETRIED)
    attempts = F.when(retried, F.array(F.lit(1), F.lit(2))).otherwise(F.array(F.lit(1)))
    second = F.col("_attempt") == 2
    rows = (
        purchases.withColumn("_attempt", F.explode(attempts))
        .withColumn(
            "payment_method", F.when(second, F.col("tender_type")).otherwise(F.col("_first_tender"))
        )
        .withColumn(
            "status",
            F.when(retried & ~second, F.lit("DECLINED")).otherwise(F.lit("APPROVED")),
        )
        .withColumn(
            "decline_reason", decline_reason_column(draws, [key], F.col("status"), f"{salt}reason")
        )
        .withColumn(
            "processing_time_ms",
            processing_ms_column(
                draws, [key, F.col("_attempt")], F.col("payment_method"), tenders, f"{salt}proc"
            ),
        )
        .withColumn(
            "event_ts",
            F.when(second, F.col("event_ts") + F.expr("INTERVAL 1 MINUTE")).otherwise(
                F.col("event_ts")
            ),
        )
        .withColumn("event_date", F.to_date("event_ts"))
        .withColumn("transaction_id", transaction_id_column("TXN", F.col(key), F.col("_attempt")))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
    )
    other = "order_id_ext" if key == "receipt_id_ext" else "receipt_id_ext"
    return rows.withColumn(other, F.lit(None).cast("string")).select(*column_names("fact_payments"))


def basket_calibration(catalog: DataFrame, profile: StoreTypeProfile) -> DataFrame:
    """``event_date, _line_value, _taxable_value``: expected line values of a day's baskets.

    The department-weighted (seasonal) average of ``behavior.retail_catalog``'s
    expected line and taxable values, which ``calibrated_basket_mean`` turns
    into the basket size that meets the profile's ticket target.
    """
    department_weight = sum(
        (
            F.when(
                F.col("department") == name,
                F.lit(weight) * _seasonal_factor(name, F.month("event_date")),
            ).otherwise(0.0)
            for name, weight in profile.department_weights.items()
        ),
        F.lit(0.0),
    )
    return (
        catalog.withColumn("_weight", department_weight)
        .groupBy("event_date")
        .agg(
            (F.sum(F.col("_weight") * F.col("_expected_line_value")) / F.sum("_weight")).alias(
                "_line_value"
            ),
            (F.sum(F.col("_weight") * F.col("_expected_taxable_value")) / F.sum("_weight")).alias(
                "_taxable_value"
            ),
        )
    )


def draw_basket_lines(
    purchases: DataFrame,
    key: str,
    catalog: DataFrame,
    d: seeded_draws,
    profile: StoreTypeProfile,
    carry: list[str] | None = None,
) -> DataFrame:
    """``key, *carry, _lines``: each purchase's drawn lines, before complements and merging.

    ``purchases`` has ``key``, ``event_date``, ``basket_n``, ``_shopper`` and
    the ``carry`` columns, which are passed through (one value per purchase).
    Lines follow shopping missions (a favorite or mission department for most
    lines), then a price- and promo-elastic product within the department and
    a category quantity. Receipts and online orders draw lines the same way.
    """
    carry = carry or []
    exploded = purchases.withColumn("line_num", F.explode(F.sequence(F.lit(1), F.col("basket_n"))))
    exploded = _with_seasonal_department(
        exploded,
        F.when(
            d.u([key, "line_num"], "mission-coherence") < 0.60,
            F.when(
                d.u([key], "favorite-mission") < 0.50,
                d.u(["_shopper"], "favorite-department"),
            ).otherwise(d.u([key], "mission-department")),
        ).otherwise(d.u([key, "line_num"], "dept")),
        profile.department_weights,
    )
    line_key = [key, "line_num"]
    return (
        exploded.join(catalog, ["event_date", "department"])
        .withColumn("_product", pick_retail_product(F.col("_catalog"), d.u(line_key, "prod")))
        .select(
            key,
            *carry,
            F.struct(
                F.col("line_num").cast("int").alias("line_num"),
                F.col("_product.product_id").alias("product_id"),
                retail_quantity(d, line_key, F.col("department")).alias("quantity"),
                F.col("_product.unit_cents").alias("unit_cents"),
                F.col("_product.taxability").alias("taxability"),
                F.col("department"),
                F.col("_product.partner_id").alias("partner_id"),
                F.col("_product.partner_cents").alias("partner_cents"),
                F.col("_product.partner_taxability").alias("partner_taxability"),
                F.col("_product.partner_ok").alias("partner_ok"),
            ).alias("_line"),
        )
        .groupBy(key, *carry)
        .agg(F.array_sort(F.collect_list("_line")).alias("_lines"))
    )


def generate_receipts_group(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate fact_receipts, fact_receipt_lines, fact_payments (in-store only)."""

    d = seeded_draws(cfg.seed)

    rules = tax_rules_by_state(load_tax_rules(cfg.resolved_dictionary_root))
    stores = (
        dims["dim_stores"]
        .select(
            F.col("ID").alias("store_id"),
            "tax_rate",
            "daily_traffic_multiplier",
            "operating_hours",
            F.col("GeographyID").alias("store_geo_id"),
        )
        .join(store_tax_rules(dims["dim_stores"], dims.get("dim_geographies"), rules), "store_id")
    )
    grid = generate_hourly_visits(
        spark, dims["dim_stores"], profile, cfg, dims.get("dim_geographies")
    ).join(stores, "store_id")
    earlier_hours = (
        Window.partitionBy("store_id", "day")
        .orderBy("hour")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    grid = grid.withColumn(
        "_prior_receipts", F.coalesce(F.sum("n_receipts").over(earlier_hours), F.lit(0))
    )
    receipts = (
        grid.filter(F.col("n_receipts") > 0)
        .withColumn("_hour_seq", F.explode(F.sequence(F.lit(1), F.col("n_receipts"))))
        .withColumn("seq", F.col("_prior_receipts") + F.col("_hour_seq"))
        .withColumn("minute", (d.h64(["store_id", "day", "seq"], "min") % 60).cast("int"))
        .withColumn("second", (d.h64(["store_id", "day", "seq"], "sec") % 60).cast("int"))
        .withColumn(
            "event_ts",
            F.make_timestamp(
                F.year("day"),
                F.month("day"),
                F.dayofmonth("day"),
                F.col("hour"),
                F.col("minute"),
                F.col("second"),
            ),
        )
        .withColumn("event_date", F.col("day"))
        # RCP(3) + yyyyMMddHHmm(12) + store(4) + seq(6) = 25 chars, unique by
        # construction (store_id, day, seq) is a key
        .withColumn(
            "receipt_id_ext",
            F.concat(
                F.lit("RCP"),
                F.date_format("event_ts", "yyyyMMddHHmm"),
                fixed_width_decimal(F.col("store_id"), 4, "receipt store ID"),
                fixed_width_decimal(F.col("seq"), 6, "daily receipt sequence"),
            ),
        )
        .withColumn("trace_id", F.concat(F.lit("TRC"), F.col("receipt_id_ext")))
    )

    # home-store customers by lifecycle activity, or anonymous shoppers
    lifecycle = lifecycle_for(cfg, profile.resolved_customer_visits_per_year())
    receipts = _assign_customers(receipts, dims, lifecycle).withColumn(
        "_shopper", shopper_key(F.col("customer_id"), F.col("receipt_id_ext"))
    )

    # Payment outcome (vocabulary decline model): a declined shopper retries on
    # another tender or walks away, so a voided sale never becomes a receipt.
    receipts = (
        receipts.withColumn("_offer", receipt_offer(d, F.col("receipt_id_ext"), profile.promo_rate))
        .withColumn("_first_tender", tender_column(d, ["receipt_id_ext"]))
        .withColumn(
            "_outcome", payment_outcome_column(d, ["receipt_id_ext"], F.col("_first_tender"))
        )
        .filter(F.col("_outcome") != F.lit(VOIDED))
        .withColumn(
            "tender_type",
            F.when(
                F.col("_outcome") == F.lit(RETRIED),
                retry_tender_column(d, ["receipt_id_ext"], F.col("_first_tender")),
            ).otherwise(F.col("_first_tender")),
        )
    )

    # --- lines: explode baskets, weighted department -> price- and
    # promo-elastic product within department, restricted to products already
    # launched on the sale day (IMP-010 no-pre-launch-sales invariant).
    products = basket_products(dims["dim_products"], cfg.seed)

    product_departments = {r.department for r in products.select("department").distinct().collect()}
    unknown = set(profile.department_weights) - product_departments
    if unknown:
        raise ValueError(
            f"profile department_weights reference departments missing from the catalog: {sorted(unknown)}"
        )

    # Catalog days come from the history window, not from the receipts: deriving
    # them from receipts lets the optimizer push the visit filter into the
    # traffic grid and copy its large expressions into the plan.
    sale_dates = date_grid(spark, cfg.start_date, cfg.end_date, "event_date")
    catalog = retail_catalog(products, sale_dates, d, PriceSchedule(cfg.seed, cfg.anchor_date))
    calibration = basket_calibration(catalog, profile)
    receipts = (
        receipts.join(calibration, "event_date")
        .withColumn(
            "_basket_mean",
            calibrated_basket_mean(
                F.col("_line_value"),
                F.col("_taxable_value"),
                F.col("tax_rate"),
                profile.avg_ticket_target,
                profile.promo_rate,
                profile.basket_lambda,
                COMPLEMENT_RATE,
            ),
        )
        .withColumn(
            "basket_n",
            retail_basket_size(
                d,
                ["receipt_id_ext"],
                F.col("_shopper"),
                F.col("_basket_mean"),
                F.col("_offer"),
                profile.promo_rate,
            ),
        )
    )

    drawn = draw_basket_lines(
        receipts.select("receipt_id_ext", "event_date", "basket_n", "_shopper"),
        "receipt_id_ext",
        catalog,
        d,
        profile,
    )
    # Basket-level steps shared with the live stream: complementary products,
    # repeated SKUs merged into one line, and at most one promotion code whose
    # threshold the basket meets; tax is integer basis-point math on the net.
    # Each step is its own projection so the optimizer keeps the array
    # expressions small instead of inlining one into the next.
    receipt_key = F.col("receipt_id_ext")
    receipts = (
        receipts.join(drawn, "receipt_id_ext")
        .withColumn("_lines", with_complements(d, receipt_key, F.col("_lines")))
        .withColumn("_lines", merge_lines(F.col("_lines")))
        .withColumn(
            "_lines",
            price_basket(
                d,
                receipt_key,
                F.col("_lines"),
                F.month(F.col("event_date")),
                F.col("_offer"),
                F.round(F.col("tax_rate") * 10000).cast("long"),
                profile,
                F.col("tax_rule"),
            ),
        )
    )

    fact_receipt_lines = (
        receipts.select("receipt_id_ext", "event_ts", "event_date", F.explode("_lines").alias("_l"))
        .select(
            "receipt_id_ext",
            "event_ts",
            "event_date",
            F.col("_l.line_num").alias("line_num"),
            F.col("_l.product_id").alias("product_id"),
            F.col("_l.quantity").alias("quantity"),
            _fmt(F.col("_l.unit_cents")).alias("unit_price"),
            F.col("_l.unit_cents").alias("unit_cents"),
            _fmt(F.col("_l.ext_cents")).alias("ext_price"),
            F.col("_l.ext_cents").alias("ext_cents"),
            F.col("_l.promo_code").alias("promo_code"),
        )
        .select(*column_names("fact_receipt_lines"))
    )

    # --- header rollup from the priced basket
    receipts = (
        receipts.withColumn("subtotal_cents", sum_lines(F.col("_lines"), "ext_cents"))
        .withColumn("discount_cents", sum_lines(F.col("_lines"), "discount_cents"))
        .withColumn("tax_cents", sum_lines(F.col("_lines"), "tax_cents"))
    )
    receipts = receipts.withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
    fact_receipts = (
        receipts.withColumn("receipt_type", F.lit("SALE"))
        .withColumn("payment_method", F.col("tender_type"))
        # IMP-007: discount is applied before tax, so subtotal_cents (kept as
        # the existing net subtotal) plus discount_cents recovers the
        # pre-discount gross subtotal. attribution fields default NULL here;
        # attribution.py enriches the converted purchases.
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
        .select(
            "receipt_id_ext",
            "trace_id",
            "event_ts",
            "event_date",
            "store_id",
            "customer_id",
            "receipt_type",
            "tender_type",
            "subtotal_cents",
            _fmt(F.col("discount_cents")).alias("discount_amount"),
            "tax_cents",
            "total_cents",
            _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
            _fmt(F.col("tax_cents")).alias("tax_amount"),
            _fmt(F.col("total_cents")).alias("total_amount"),
            "payment_method",
            # Legacy semantic-model column (sourceColumn: Subtotal); same value
            # as subtotal_amount per the TMDL contract.
            _fmt(F.col("subtotal_cents")).alias("Subtotal"),
            "gross_subtotal_cents",
            "discount_cents",
            "attribution_journey_id",
            "campaign_id",
            "impression_id_ext",
            F.hour("event_ts").alias("local_hour"),
            F.lit(None).cast("string").alias("original_receipt_id_ext"),
        )
        .select(*column_names("fact_receipts"))
    )

    # --- payments (in-store, so order_id_ext is NULL): the first attempt, and
    # for a retried sale the approved attempt on the second tender a minute later.
    fact_payments = payment_attempts(
        d,
        receipts.select(
            "receipt_id_ext",
            "event_ts",
            "store_id",
            "customer_id",
            "total_cents",
            "_first_tender",
            "tender_type",
            "_outcome",
        ),
        "receipt_id_ext",
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_receipt_lines": fact_receipt_lines,
        "fact_payments": fact_payments,
    }

# --- retail_setup/generation/returns.py ---
"""Returns fact group, Spark-native (unioned into the receipts contract).

Semantics: sample ~``cfg.return_rate`` of SALE receipts per day (Dec 26 spikes
6x, capped at 10% of the day's receipts), favouring baskets with categories
customers return more (``RETURN_CATEGORY_WEIGHTS``: apparel and electronics
far more than groceries). Each return posts 1..``RETURN_WINDOW_DAYS`` days
*after* its sale (IMP-010: no same-day returns), clamped to ``cfg.end_date``,
at a random time inside the store's hours (store-local).

A return is partial: it takes a subset of the receipt's lines, each with a
probability that follows its category (at least one line), and of a multi-unit
line some or all of its units (``returned_lines``). The refund keeps the
sale's customer and goes back to the tender the sale was paid with; its
header links the sale (``original_receipt_id_ext``). Lines keep their own
promo code (``receipt_type`` marks the return). Refund cents are the returned
share of each line's net, with integer basis-point tax on it. The return ID
has the sales layout (``RET`` + yyyyMMddHHmm + store4 + seq6); the refund
payment is APPROVED with a transaction ID derived from the return ID.

All randomness comes from `runtime.seeded_draws` keyed on the original
receipt and line, so output is deterministic for a (config, seed) pair.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Returns post 1..RETURN_WINDOW_DAYS days after the originating sale.
RETURN_WINDOW_DAYS = 14
# Relative return propensity by department token (lowercase substring match,
# first match wins); other departments use DEFAULT_RETURN_WEIGHT.
RETURN_CATEGORY_WEIGHTS: list[tuple[str, float]] = [
    ("clothing", 1.8),
    ("apparel", 1.8),
    ("shoe", 1.8),
    ("fitting", 1.6),
    ("electronics", 1.5),
    ("jewelry", 1.4),
    ("accessories", 1.4),
    ("toys", 1.2),
    ("home", 1.0),
    ("garden", 1.0),
    ("tools", 1.0),
    ("hardware", 1.0),
    ("sport", 1.0),
    ("beauty", 0.7),
    ("health", 0.5),
    ("pharmacy", 0.3),
    ("grocery", 0.25),
    ("food", 0.25),
    ("fresh", 0.15),
    ("produce", 0.15),
]
DEFAULT_RETURN_WEIGHT = 0.8
# A line of a returned receipt is returned with LINE_RETURN_SHARE x its weight
# (at most MAX_LINE_RETURN_SHARE); the receipt's most likely line always is.
LINE_RETURN_SHARE = 0.5
MAX_LINE_RETURN_SHARE = 0.95
_RETURNED_LINE = "array<struct<line_num:int,quantity:int>>"


def return_weight(department: Column) -> Column:
    """Return propensity of a department (``RETURN_CATEGORY_WEIGHTS``)."""
    lower = F.lower(F.coalesce(department, F.lit("")))
    return case_when(
        (lower.contains(token), F.lit(weight)) for token, weight in RETURN_CATEGORY_WEIGHTS
    ).otherwise(F.lit(DEFAULT_RETURN_WEIGHT))


def _receipt_lines(
    sales_group: dict[str, DataFrame], dims: dict[str, DataFrame] | None
) -> DataFrame | None:
    """Lines of each receipt with their return weight, as one sorted array."""
    lines = sales_group.get("fact_receipt_lines")
    if lines is None:
        return None
    lines = lines.select("receipt_id_ext", "line_num", "product_id", "quantity")
    if dims is not None and "dim_products" in dims:
        departments = dims["dim_products"].select(
            F.col("ID").alias("product_id"), F.col("Department").alias("_department")
        )
        lines = lines.join(departments, "product_id", "left")
    else:
        lines = lines.withColumn("_department", F.lit(None).cast("string"))
    return lines.groupBy("receipt_id_ext").agg(
        F.array_sort(
            F.collect_list(
                F.struct(
                    F.col("line_num").cast("int").alias("line_num"),
                    F.col("quantity").cast("int").alias("quantity"),
                    return_weight(F.col("_department")).alias("weight"),
                )
            )
        ).alias("_lines")
    )


def returned_lines_column(d: seeded_draws, key: Column, lines: Column) -> Column:
    """The lines (and units) a return takes from ``lines`` (line_num, quantity, weight).

    Each line is returned with ``LINE_RETURN_SHARE`` x its weight, and the
    line with the largest weighted draw always is, so a return is never
    empty. A multi-unit line returns 1..all of its units.
    """
    anchor = F.array_max(
        F.transform(
            lines,
            lambda line: F.struct(
                (line["weight"] * d.u([key, line["line_num"]], "return-anchor")).alias("score"),
                line["line_num"].alias("line_num"),
            ),
        )
    )["line_num"]
    chosen = F.filter(
        lines,
        lambda line: (
            (
                d.u([key, line["line_num"]], "return-line")
                < F.least(F.lit(MAX_LINE_RETURN_SHARE), line["weight"] * LINE_RETURN_SHARE)
            )
            | (line["line_num"] == anchor)
        ),
    )
    return F.transform(
        chosen,
        lambda line: F.struct(
            line["line_num"].alias("line_num"),
            F.when(
                line["quantity"] > 1,
                1 + F.floor(d.u([key, line["line_num"]], "return-units") * line["quantity"]),
            )
            .otherwise(F.lit(1))
            .cast("int")
            .alias("quantity"),
        ),
    )


def build_return_headers(
    sales_group: dict[str, DataFrame],
    cfg: GenerationConfig,
    dims: dict[str, DataFrame] | None = None,
    profile: StoreTypeProfile | None = None,
) -> DataFrame:
    """Sample SALE receipts for return and date each strictly after its sale.

    Returns the RETURN header plan, which retains ``orig_receipt_id_ext``,
    ``orig_event_date`` (the sale day) and ``returned_lines`` (the lines and
    units returned; NULL means the whole receipt when the plan was built
    without lines) alongside the return ``event_date``/``event_ts``. With
    ``dims`` the sample favours returnable categories and the return time falls
    inside the store's hours; without them it is noon. Deterministic per
    (config, seed).
    """
    d = seeded_draws(cfg.seed)
    if profile is None:
        profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile
    klass = store_class(profile)

    sales = sales_group["fact_receipts"].filter(
        (F.col("receipt_type") == "SALE")
        & (F.col("event_date") < F.lit(cfg.end_date))
        & ~closed_date_column(F.col("event_date"), klass)
    )
    receipt_lines = _receipt_lines(sales_group, dims)
    if receipt_lines is not None:
        sales = sales.join(receipt_lines, "receipt_id_ext", "left")
        weight = F.coalesce(
            F.array_max(F.transform("_lines", lambda line: line["weight"])),
            F.lit(DEFAULT_RETURN_WEIGHT),
        )
    else:
        sales = sales.withColumn(
            "_lines", F.lit(None).cast("array<struct<line_num:int,quantity:int,weight:double>>")
        )
        weight = F.lit(DEFAULT_RETURN_WEIGHT)

    # --- sample: per-day rate, 6x on Dec 26, hard-capped at 10%.
    # Accumulate fractional daily quotas before rounding so low-volume days do
    # not each lose their fraction independently. Within a day, receipts race
    # on an exponential draw scaled by their basket's return weight, so baskets
    # of returnable categories are returned more often.
    day_rate = F.least(
        F.lit(0.10),
        F.lit(cfg.return_rate)
        * F.when(
            (F.month("event_date") == 12) & (F.dayofmonth("event_date") == 26),
            F.lit(6.0),
        ).otherwise(F.lit(1.0)),
    )
    daily = (
        sales.groupBy("event_date")
        .agg(F.count("*").alias("_day_count"))
        .withColumn("_expected", F.col("_day_count") * day_rate)
    )
    cumulative = Window.orderBy("event_date").rowsBetween(
        Window.unboundedPreceding, Window.currentRow
    )
    prior = Window.orderBy("event_date").rowsBetween(Window.unboundedPreceding, -1)
    quotas = (
        daily.withColumn("_cumulative_expected", F.sum("_expected").over(cumulative))
        .withColumn(
            "_prior_expected",
            F.coalesce(F.sum("_expected").over(prior), F.lit(0.0)),
        )
        .withColumn(
            "_n_keep",
            (
                F.floor(F.col("_cumulative_expected") + F.lit(1e-9))
                - F.floor(F.col("_prior_expected") + F.lit(1e-9))
            ).cast("long"),
        )
        .select("event_date", "_n_keep")
    )
    race = -F.log(F.lit(1.0) - d.u(["receipt_id_ext"], "return")) / weight
    day_w = Window.partitionBy("event_date")
    sampled = (
        sales.join(quotas, "event_date")
        .withColumn("_race", race)
        .withColumn("_ret_rank", F.row_number().over(day_w.orderBy("_race", "receipt_id_ext")))
        .filter(F.col("_ret_rank") <= F.col("_n_keep"))
        .drop("_n_keep", "_ret_rank", "_race")
        .withColumn(
            "returned_lines",
            F.when(
                F.col("_lines").isNotNull(),
                returned_lines_column(d, F.col("receipt_id_ext"), F.col("_lines")),
            ).cast(_RETURNED_LINE),
        )
        .drop("_lines")
    )

    sequenced = (
        sampled.withColumnRenamed("receipt_id_ext", "orig_receipt_id_ext")
        .withColumnRenamed("event_date", "orig_event_date")
        .withColumn(
            "_max_delay",
            F.least(
                F.lit(RETURN_WINDOW_DAYS),
                F.datediff(F.lit(cfg.end_date), F.col("orig_event_date")),
            ).cast("int"),
        )
        .filter(F.col("_max_delay") >= 1)
        .withColumn(
            "_return_delay",
            (
                F.lit(1)
                + F.floor(d.u(["orig_receipt_id_ext"], "return_delay") * F.col("_max_delay"))
            ).cast("int"),
        )
        .withColumn(
            "_candidate_date",
            F.date_add("orig_event_date", F.col("_return_delay")),
        )
        .withColumn(
            "_latest_return_date",
            F.date_add("orig_event_date", F.col("_max_delay")),
        )
        .withColumn(
            "event_date",
            # Returns move off a closed day (stores never close two days running).
            F.when(
                closed_date_column(F.col("_candidate_date"), klass),
                F.when(
                    F.date_add("_candidate_date", 1) <= F.col("_latest_return_date"),
                    F.date_add("_candidate_date", 1),
                ).otherwise(F.date_add("_candidate_date", -1)),
            ).otherwise(F.col("_candidate_date")),
        )
        .filter(
            (F.col("event_date") > F.col("orig_event_date"))
            & (F.col("event_date") <= F.col("_latest_return_date"))
        )
    )
    # Return time: a random minute inside the store's (local) hours.
    key = ["orig_receipt_id_ext"]
    if dims is not None and "dim_stores" in dims:
        hours = dims["dim_stores"].select(F.col("ID").alias("store_id"), "operating_hours")
        sequenced = sequenced.join(hours, "store_id", "left")
        open_h, close_h = _open_close_cols(F.col("operating_hours"))
        open_h, close_h = F.coalesce(open_h, F.lit(12)), F.coalesce(close_h, F.lit(13))
    else:
        sequenced = sequenced.withColumn("operating_hours", F.lit(None).cast("string"))
        open_h, close_h = F.lit(12), F.lit(13)
    minute_of_day = open_h * 60 + F.floor(d.u(key, "return-time") * (close_h - open_h) * 60).cast(
        "int"
    )
    sequenced = sequenced.withColumn(
        "event_ts",
        F.make_timestamp(
            F.year("event_date"),
            F.month("event_date"),
            F.dayofmonth("event_date"),
            F.floor(minute_of_day / 60).cast("int"),
            (minute_of_day % 60).cast("int"),
            F.floor(d.u(key, "return-second") * 60).cast("int"),
        ),
    ).drop("operating_hours")
    seq_w = Window.partitionBy("store_id", "event_date").orderBy("orig_receipt_id_ext")
    return (
        sequenced.withColumn("seq", F.row_number().over(seq_w))
        .withColumn(
            "receipt_id_ext",
            F.concat(
                F.lit("RET"),
                F.date_format("event_ts", "yyyyMMddHHmm"),
                fixed_width_decimal(F.col("store_id"), 4, "return store ID"),
                fixed_width_decimal(F.col("seq"), 6, "daily return sequence"),
            ),
        )
        .withColumn("trace_id", F.concat(F.lit("TRC"), F.col("receipt_id_ext")))
        .withColumn("receipt_type", F.lit("RETURN"))
        # The refund goes back to the tender the sale was paid with.
        .withColumn(
            "tender_type",
            F.col("payment_method")
            if "payment_method" in sequenced.columns
            else F.lit(None).cast("string"),
        )
        .withColumn("payment_method", F.col("tender_type"))
        .withColumn("subtotal_cents", -F.col("subtotal_cents"))
        .withColumn("tax_cents", -F.col("tax_cents"))
        .withColumn("total_cents", -F.col("total_cents"))
        .withColumn("discount_cents", -F.col("discount_cents"))
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        # Returns are never attribution candidates (SALE-only per contract).
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
    )


def returned_quantity(returned_lines: Column, line_num: Column, quantity: Column) -> Column:
    """Units of ``line_num`` a return takes (the whole line when the plan has no lines)."""
    picked = F.aggregate(
        returned_lines,
        F.lit(0),
        lambda units, r: units + F.when(r["line_num"] == line_num, r["quantity"]).otherwise(0),
    )
    return F.when(returned_lines.isNull(), quantity).otherwise(F.least(picked, quantity))


def generate_returns(
    spark: SparkSession,
    sales_group: dict[str, DataFrame],
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    *,
    return_headers: DataFrame | None = None,
) -> dict[str, DataFrame]:
    """Generate refunds, optionally retaining a pre-fulfillment sampling plan.

    Planned dates/IDs/lines survive fulfillment; refunds are priced from the
    accepted lines, so a returned line the store never filled is not refunded
    and a return with nothing left disappears.
    """

    d = seeded_draws(cfg.seed)

    header = (
        build_return_headers(sales_group, cfg, dims) if return_headers is None else return_headers
    )
    if "returned_lines" not in header.columns:
        header = header.withColumn("returned_lines", F.lit(None).cast(_RETURNED_LINE))
    sold = sales_group["fact_receipt_lines"].select(
        F.col("receipt_id_ext").alias("orig_receipt_id_ext"),
        "line_num",
        "product_id",
        F.col("quantity").alias("_sold"),
        "unit_cents",
        F.col("ext_cents").alias("_sold_ext"),
        "promo_code",
    )
    taxability = dims["dim_products"].select(F.col("ID").alias("product_id"), "taxability")
    store_bps = (
        dims["dim_stores"]
        .select(
            F.col("ID").alias("store_id"),
            F.round(F.col("tax_rate") * 10000).cast("long").alias("_bps"),
        )
        .join(
            store_tax_rules(
                dims["dim_stores"],
                dims.get("dim_geographies"),
                tax_rules_by_state(load_tax_rules(cfg.resolved_dictionary_root)),
            ),
            "store_id",
        )
    )
    units = F.col("_units").cast("long")
    sold_units = F.col("_sold").cast("long")
    lines = (
        header.select(
            "orig_receipt_id_ext",
            "receipt_id_ext",
            "event_ts",
            "event_date",
            "store_id",
            "returned_lines",
        )
        .join(sold, "orig_receipt_id_ext")
        .withColumn(
            "_units", returned_quantity(F.col("returned_lines"), F.col("line_num"), F.col("_sold"))
        )
        .filter(F.col("_units") > 0)
        .join(taxability, "product_id", "left")
        .join(store_bps, "store_id", "left")
        # The returned share of the line's net, rounded half up, and its tax.
        .withColumn(
            "_net",
            F.floor((F.col("_sold_ext") * units * 2 + sold_units) / (sold_units * 2)).cast("long"),
        )
        .withColumn("_gross", F.col("unit_cents") * units)
        .withColumn(
            "_tax",
            line_tax_cents(
                F.col("_net"),
                F.col("taxability"),
                F.col("_bps"),
                F.col("tax_rule"),
                F.col("unit_cents"),
            ),
        )
    )
    totals = lines.groupBy("receipt_id_ext").agg(
        (-F.sum("_net")).cast("long").alias("subtotal_cents"),
        (-F.sum(F.col("_gross") - F.col("_net"))).cast("long").alias("discount_cents"),
        (-F.sum("_tax")).cast("long").alias("tax_cents"),
    )
    money = ["subtotal_cents", "discount_cents", "tax_cents", "total_cents", "gross_subtotal_cents"]
    header = (
        header.drop(*money)
        .join(totals, "receipt_id_ext")
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
    )

    fact_receipts = header.select(
        "receipt_id_ext",
        "trace_id",
        "event_ts",
        "event_date",
        "store_id",
        "customer_id",
        "receipt_type",
        "tender_type",
        "subtotal_cents",
        _fmt(F.col("discount_cents")).alias("discount_amount"),
        "tax_cents",
        "total_cents",
        _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
        _fmt(F.col("tax_cents")).alias("tax_amount"),
        _fmt(F.col("total_cents")).alias("total_amount"),
        F.col("tender_type").alias("payment_method"),
        # Legacy semantic-model column mirrors subtotal_amount (TMDL contract).
        _fmt(F.col("subtotal_cents")).alias("Subtotal"),
        "gross_subtotal_cents",
        "discount_cents",
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
        F.hour("event_ts").alias("local_hour"),
        F.col("orig_receipt_id_ext").alias("original_receipt_id_ext"),
    ).select(*column_names("fact_receipts"))

    # --- lines: the returned units of each line, negated; the line keeps its
    # own promo code (receipt_type marks the return)
    fact_receipt_lines = lines.select(
        "receipt_id_ext",
        "event_ts",
        "event_date",
        "line_num",
        "product_id",
        (-units).cast("int").alias("quantity"),
        _fmt(F.col("unit_cents")).alias("unit_price"),
        "unit_cents",
        _fmt(-F.col("_net")).alias("ext_price"),
        (-F.col("_net")).alias("ext_cents"),
        "promo_code",
    ).select(*column_names("fact_receipt_lines"))

    # --- payments: one negative APPROVED refund per return, to the sale's tender
    fact_payments = (
        fact_receipts.withColumn("order_id_ext", F.lit(None).cast("string"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .withColumn("transaction_id", transaction_id_column("TXN", F.col("receipt_id_ext"), 1))
        .withColumn("status", F.lit("APPROVED"))
        .withColumn("decline_reason", F.lit(None).cast("string"))
        .withColumn(
            "processing_time_ms",
            processing_ms_column(d, ["receipt_id_ext"], F.col("payment_method")),
        )
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_receipt_lines": fact_receipt_lines,
        "fact_payments": fact_payments,
    }

# --- retail_setup/generation/dc_supply.py ---
"""DC-limited, incident-aware supply of one SKU across a DC's stores (history).

:func:`simulate_dc_sku` runs every store a DC serves for one product, day by
day, against the DC's own stock:

- **Stores.** Each store-SKU follows the store policy (``inventory_policy``):
  sales and lost sales, restocking returns, the smoothed daily demand, and a
  reorder when a day closes with the inventory position at the reorder point.
- **Allocation.** When a day closes the DC ships the day's reorders from the
  stock it has, most urgent first (URGENT, HIGH, NORMAL, then store ID). A
  reorder the DC cannot cover is short-shipped: only the shipped units are
  inbound to the store, so the store reorders again the next day and runs out
  if the DC stays empty. DC stockouts therefore reach store shelves.
- **Supplier.** The DC orders from its supplier when its position (on hand plus
  open orders) falls to the reorder point (``inventory_policy.dc_reorder_levels``);
  an order arrives after the product's stable lead time at
  ``DC_SUPPLIER_RECEIPT_HOUR``. During a ``SUPPLIER_SHORT_SHIP`` incident on
  the product's supplier group only part of it arrives.
- **Online.** Ship-from-DC online lines take stock when they ship; a line the
  DC cannot fill is covered by an expedited receipt just before it.
- **Delivery.** Each shipment follows ``incidents.plan_delivery``: the store's
  next open delivery slot, held through DC outages, delayed by holiday
  capacity, traffic, lane incidents and weather. Reorders whose loads leave at
  the same slot (a load held by an outage) travel as one load.

All times are local: store rows in the store's zone, DC rows (receipts, online
lines, expedites, the opening stock) in the DC's zone. Supplier receipts
arrive early in the DC's day and allocations only use stock received by the
previous close, so the DC's physical balance never goes negative in true time.
"""

from collections.abc import Iterable, Iterator, Mapping
from dataclasses import dataclass
from datetime import date, datetime, time, timedelta


DC_KINDS = ("DC_INITIAL", "DC_RECEIPT", "DC_ONLINE", "DC_EXPEDITE")


@dataclass(frozen=True)
class StoreLane:
    """A store's place in a DC simulation: slot, lane, demand prior and incidents."""

    slot_minute: int
    transit_minutes: int
    prior: float = 1.0
    view: StoreIncidentView | None = None


@dataclass(frozen=True)
class DcDemand:
    """A ship-from-DC online order line (DC-local ship time)."""

    event_ts: datetime
    order_id: str
    line_num: int
    quantity: int

    @property
    def trace_id(self) -> str:
        return f"TRC-DCO-{self.order_id}-{self.line_num}"


# A plain dict, not functools.lru_cache: the setup notebooks inline this module
# into __main__, and cloudpickle ships an lru_cache wrapper to Spark workers by
# reference, where __main__ has no such name.
_TIMINGS: dict[tuple[object, ...], DeliveryTiming] = {}
_TIMINGS_MAX = 1 << 17


def delivery_timing(
    seed: int,
    store_id: int,
    dc_id: int,
    day: date,
    slot_minute: int,
    transit_minutes: int,
    view: StoreIncidentView | None,
    closed_class: str | None,
) -> DeliveryTiming:
    """Timing of the load that carries a store's reorders of ``day`` (cached per process)."""
    key = (seed, store_id, dc_id, day, slot_minute, transit_minutes, view, closed_class)
    found = _TIMINGS.get(key)
    if found is None:
        if len(_TIMINGS) >= _TIMINGS_MAX:
            _TIMINGS.clear()
        found = _TIMINGS[key] = plan_delivery(
            seed,
            store_id=store_id,
            dc_id=dc_id,
            ready=datetime.combine(day, time(23, 59, 59)),
            slot_minute=slot_minute,
            transit_minutes=transit_minutes,
            view=view,
            closed=None if closed_class is None else lambda d: is_closed_date(d, closed_class),
        )
    return found


class DcSku:
    """One DC product's stock, smoothed demand and open supplier orders."""

    def __init__(
        self,
        *,
        dc_id: int,
        product_id: int,
        seed: int,
        start_date: date,
        prior: float,
        view: DcIncidentView | None = None,
        stores: int = 1,
    ) -> None:
        self.dc_id = int(dc_id)
        self.product_id = int(product_id)
        self.lead = dc_lead_days(seed, dc_id, product_id)
        self.group = supplier_group(seed, product_id)
        self.floor = dc_case_floor(stores)
        self.rate = prior
        self.on_hand = dc_opening_stock(seed, dc_id, product_id, prior, self.floor)
        self.start_date = start_date
        self.view = view
        self.orders: list[tuple[datetime, int, str]] = []
        self.on_order = 0

    def _event(
        self, at: datetime, trace: str, kind: str, quantity: int, source: str, **extra: object
    ) -> InventoryEvent:
        return InventoryEvent(
            None,
            self.product_id,
            at,
            trace,
            kind,
            quantity,
            source,
            balance=self.on_hand,
            dc_id=self.dc_id,
            **extra,  # type: ignore[arg-type]
        )

    def opening(self) -> InventoryEvent:
        return self._event(
            datetime.combine(self.start_date, time()),
            f"TRC-INIT-DC-{self.dc_id}-{self.product_id}",
            "DC_INITIAL",
            self.on_hand,
            "SEED",
        )

    def receive(self, until: datetime) -> Iterator[InventoryEvent]:
        """Supplier orders that arrive by ``until`` (short while the supplier short-ships)."""
        while self.orders and self.orders[0][0] <= until:
            at, ordered, source = self.orders.pop(0)
            self.on_order -= ordered
            fill = 1.0 if self.view is None else self.view.fill_rate(self.group, at)
            received = supplier_receipt(ordered, fill)
            if received <= 0:
                continue
            self.on_hand += received
            yield self._event(
                at,
                f"TRC-SUP-{source}",
                "DC_RECEIPT",
                received,
                source,
                requested_quantity=ordered,
                unfulfilled_quantity=ordered - received,
            )

    def fill_online(self, line: DcDemand) -> Iterator[InventoryEvent]:
        """Ship an online line, expediting any units the DC lacks just before it."""
        short = line.quantity - self.on_hand
        if short > 0:
            self.on_hand += short
            yield self._event(
                line.event_ts - timedelta(seconds=1),
                f"TRC-EXP-{line.order_id}-{line.line_num}",
                "DC_EXPEDITE",
                short,
                f"EXPEDITE-{line.order_id}",
            )
        self.on_hand -= line.quantity
        yield self._event(
            line.event_ts,
            line.trace_id,
            "DC_ONLINE",
            -line.quantity,
            line.order_id,
            requested_quantity=line.quantity,
        )

    def ship(self, requested: int) -> int:
        shipped = max(0, min(int(requested), self.on_hand))
        self.on_hand -= shipped
        return shipped

    def close_day(self, day: date, outbound: int) -> None:
        """Fold the day's requests into the demand estimate and order from the supplier."""
        self.rate = smooth_dc_demand(self.rate, outbound)
        point, target = dc_reorder_levels(self.rate, self.lead, self.floor)
        position = self.on_hand + self.on_order
        if position > point:
            return
        ordered = reorder_quantity(target, position)
        arrival = datetime.combine(day + timedelta(days=self.lead), time(DC_SUPPLIER_RECEIPT_HOUR))
        source = f"SUPPLIER-{self.dc_id}-{self.product_id}-{day:%Y%m%d}"
        self.orders.append((arrival, ordered, source))
        self.orders.sort()
        self.on_order += ordered


def load_event(
    store_id: int, product_id: int, dc_id: int, timing: DeliveryTiming, quantity: int
) -> InventoryEvent:
    """The SKU's line on the store's load that leaves at ``timing``'s slot."""
    shipment_id = shipment_base_id(timing.slot, store_id)
    scheduled = datetime.combine(timing.planned_loading.date(), time())
    return InventoryEvent(
        store_id,
        product_id,
        scheduled - timedelta(seconds=1),
        f"TRC-PLAN-{shipment_id}-{product_id}",
        "SHIPMENT",
        quantity,
        shipment_id,
        shipment_id=shipment_id,
        leg=0,
        scheduled_ts=scheduled,
        loading_ts=timing.loading,
        depart_ts=timing.departure,
        eta=timing.arrival,
        unloading_ts=timing.arrival + timedelta(minutes=TRUCK_CHECKIN_MINUTES),
        dc_id=dc_id,
        planned_eta=timing.planned_arrival,
        delay_minutes=timing.delay_minutes,
        delay_reason=timing.delay_reason,
    )


def simulate_dc_sku(
    rows: Iterable[tuple[int | None, Demand | DcDemand]],
    *,
    dc_id: int,
    product_id: int,
    lanes: Mapping[int, StoreLane],
    start_date: date,
    end_date: date,
    seed: int,
    default_prior: float = 1.0,
    dc_prior: float | None = None,
    dc_view: DcIncidentView | None = None,
    closed_class: str | None = None,
) -> Iterator[InventoryEvent]:
    """Every store of a DC for one product, against the DC's stock.

    ``rows`` are ``(store_id, Demand)`` store lines and ``(None, DcDemand)``
    online lines in local time order. A store with demand but no lane uses
    ``default_prior`` and the stable fallback slot and lane.
    """
    if end_date < start_date:
        raise ValueError("invalid inventory simulation bounds")
    lanes = dict(lanes)
    prior = (
        dc_prior
        if dc_prior is not None
        else sum(lane.prior for lane in lanes.values()) * (1.0 + DC_ONLINE_DEMAND_SHARE)
    )
    dc = DcSku(
        dc_id=dc_id,
        product_id=product_id,
        seed=seed,
        start_date=start_date,
        prior=max(prior, default_prior),
        view=dc_view,
        stores=len(lanes),
    )
    yield dc.opening()
    skus: dict[int, StoreSku] = {}

    def store(store_id: int) -> StoreSku:
        if store_id not in skus:
            lane = lanes.setdefault(store_id, _fallback_lane(seed, dc_id, store_id, default_prior))
            skus[store_id] = StoreSku(
                store_id=store_id,
                product_id=product_id,
                seed=seed,
                start_date=start_date,
                prior=lane.prior,
            )
        return skus[store_id]

    for store_id in sorted(lanes):
        yield store(store_id).opening()

    def timing(store_id: int, day: date) -> DeliveryTiming:
        lane = lanes[store_id]
        return delivery_timing(
            seed,
            store_id,
            dc_id,
            day,
            lane.slot_minute,
            lane.transit_minutes,
            lane.view,
            closed_class,
        )

    def load_timing(store_id: int, day: date) -> DeliveryTiming:
        """The load's plan is that of the earliest reorder day that ships in it."""
        current = timing(store_id, day)
        earlier = day - timedelta(days=1)
        while earlier >= start_date:
            candidate = timing(store_id, earlier)
            if candidate.slot != current.slot:
                break
            current, earlier = candidate, earlier - timedelta(days=1)
        return current

    open_loads: dict[tuple[int, datetime], tuple[InventoryEvent, InventoryEvent]] = {}
    iterator = iter(rows)
    candidate = next(iterator, None)
    day = start_date
    while day <= end_date:
        closing = datetime.combine(day, time(23, 59, 59))
        online = 0
        while candidate is not None and candidate[1].event_ts <= closing:
            row_store, line = candidate
            if row_store is None:
                if not isinstance(line, DcDemand):
                    raise ValueError("DC rows must be online lines")
                if line.event_ts.date() < start_date or line.quantity <= 0:
                    raise ValueError("online DC lines must be positive and inside history")
                yield from dc.receive(line.event_ts)
                yield from dc.fill_online(line)
                online += line.quantity
            else:
                if not isinstance(line, Demand):
                    raise ValueError("store rows must be demand lines")
                if int(row_store) not in skus:
                    yield store(int(row_store)).opening()
                yield from skus[int(row_store)].consume(line)
            candidate = next(iterator, None)
        yield from dc.receive(closing)
        requests = []
        for store_id in sorted(skus):
            sku = skus[store_id]
            yield from sku.release(closing)
            order = sku.close_day(closing)
            if order is not None:
                requests.append(((PRIORITY_ORDER.get(order[2], 3), store_id), order))
        requested = 0
        for (_, store_id), (quantity, point, priority) in sorted(requests):
            sku = skus[store_id]
            yield sku.reorder_event(day, quantity, point, priority)
            requested += quantity
            shipped = dc.ship(quantity)
            if shipped <= 0:
                continue
            plan = load_timing(store_id, day)
            key = (store_id, plan.slot)
            if key in open_loads:
                shipment, delivery = open_loads[key]
                shipment.quantity += shipped
                sku.add_to(delivery, shipped)
            else:
                shipment = load_event(store_id, product_id, dc_id, plan, shipped)
                open_loads[key] = (shipment, sku.expect(shipment))
        dc.close_day(day, requested + online)
        for key in sorted(key for key in open_loads if key[1].date() <= day):
            yield open_loads.pop(key)[0]
        day += timedelta(days=1)
    for key in sorted(open_loads):
        yield open_loads.pop(key)[0]
    if candidate is not None:
        raise ValueError("candidate demand falls outside the simulation history")


def _fallback_lane(seed: int, dc_id: int, store_id: int, prior: float) -> StoreLane:
    return StoreLane(
        delivery_slot_minute(seed, store_id), lane_transit_minutes(seed, dc_id, store_id), prior
    )

# --- retail_setup/generation/fulfillment.py ---
"""Spark adapter for the causal SKU policy and fulfilled receipt finances."""

from dataclasses import dataclass, fields
from datetime import date
from functools import partial
from itertools import chain, groupby
from operator import attrgetter
from typing import Any, Iterator

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F


_EVENT_SCHEMA = """
    store_id long, product_id long, event_ts timestamp, trace_id string,
    kind string, quantity long, source string, balance long,
    receipt_id string, line_num long, requested_quantity long,
    unfulfilled_quantity long, reorder_point long, priority string,
    shipment_id string, leg long, scheduled_ts timestamp, loading_ts timestamp,
    depart_ts timestamp, eta timestamp, unloading_ts timestamp, etd timestamp,
    dc_id long, planned_eta timestamp, delay_minutes long, delay_reason string
"""
_FINANCE_COLUMNS = [
    "subtotal_cents",
    "discount_cents",
    "gross_subtotal_cents",
    "tax_cents",
    "total_cents",
    "subtotal_amount",
    "discount_amount",
    "tax_amount",
    "total_amount",
    "Subtotal",
]


@dataclass
class FulfillmentResult:
    sales: dict[str, DataFrame]
    return_headers: DataFrame
    events: DataFrame


_EVENT_FIELDS = tuple(field.name for field in fields(InventoryEvent))
_EVENT_TYPES = dict(
    (name, kind)
    for name, kind in (column.split() for column in _EVENT_SCHEMA.replace("\n", " ").split(","))
)
_DEMAND_COLUMNS = (
    "store_id",
    "product_id",
    "event_ts",
    "receipt_id_ext",
    "line_num",
    "quantity",
    "return_ts",
    "return_id",
    "return_quantity",
    "prior_rate",
)
# Bounded output batches keep Python worker memory flat for very busy SKUs.
_ARROW_BATCH_ROWS = 10_000


def _arrow_event_schema(time_zone: str) -> Any:
    import pyarrow as pa

    arrow_types = {
        "long": pa.int64(),
        "string": pa.string(),
        "timestamp": pa.timestamp("us", tz=time_zone),
    }
    return pa.schema([(name, arrow_types[_EVENT_TYPES[name]]) for name in _EVENT_FIELDS])


def _simulate_arrow_partition(
    batches: Iterator[Any],
    *,
    start_date: date,
    end_date: date,
    seed: int,
    daily_prior: float,
    time_zone: str,
    routes: dict[int, tuple[int, int]] | None = None,
    views: dict[int, StoreIncidentView] | None = None,
    dc_views: dict[int, DcIncidentView] | None = None,
    closed_class: str | None = None,
) -> Iterator[Any]:
    """Stream sorted (DC, SKU) demand through the DC simulation for ``mapInArrow``.

    Arrow moves rows between the JVM and Python in column batches instead of
    pickling one Row/dict per demand line and inventory event, which dominated
    this stage. Timestamps cross the boundary as UTC instants and are handled
    as naive datetimes (local wall clock), exactly as the policy expects.

    Rows of one DC and product are one group (``dc_id`` defaults to 0): store
    seed rows (no receipt) register each store with its demand prior
    (``prior_rate``, else ``daily_prior``), store rows are demand, and rows
    without a store are ship-from-DC online lines. ``routes`` maps a store to
    its (delivery slot, lane minutes), ``views`` and ``dc_views`` hold the
    incidents of each store and DC, and ``closed_class`` the holiday closures.
    """
    import pyarrow as pa

    schema = _arrow_event_schema(time_zone)
    naive = pa.timestamp("us")
    values = attrgetter(*_EVENT_FIELDS)
    columns_in = (*_DEMAND_COLUMNS, "dc_id")

    def demand_rows() -> Iterator[tuple]:
        for batch in batches:
            columns = []
            for name in columns_in:
                index = batch.schema.get_field_index(name)
                if index < 0:
                    columns.append([None] * batch.num_rows)
                    continue
                column = batch.column(index)
                if pa.types.is_timestamp(column.type):
                    column = column.cast(naive)
                columns.append(column.to_pylist())
            yield from zip(*columns)

    def flush(rows: list[tuple]) -> Any:
        return pa.RecordBatch.from_arrays(
            [
                pa.array(column, type=schema.field(index).type)
                for index, column in enumerate(zip(*rows))
            ],
            schema=schema,
        )

    pending: list[tuple] = []
    routes = routes or {}
    views = views or {}
    dc_views = dc_views or {}
    for (dc_id, product_id), group in groupby(
        demand_rows(),
        key=lambda row: (row[10] or 0, row[1]),
    ):
        lanes: dict[int, StoreLane] = {}
        first = None
        for row in group:
            store_id, receipt_id, prior = row[0], row[3], row[9]
            if receipt_id is None and store_id is not None:
                slot, transit = routes.get(store_id, (None, None))
                lanes[store_id] = StoreLane(
                    delivery_slot_minute(seed, store_id) if slot is None else slot,
                    lane_transit_minutes(seed, dc_id, store_id) if transit is None else transit,
                    daily_prior if prior is None else prior,
                    views.get(store_id),
                )
                continue
            first = row
            break
        rows = chain([first], group) if first is not None else iter(())
        demand = (
            (
                store_id,
                Demand(event_ts, receipt_id, line_num, quantity, return_ts, return_id, returned)
                if store_id is not None
                else DcDemand(event_ts, receipt_id, line_num, quantity),
            )
            for (
                store_id,
                _,
                event_ts,
                receipt_id,
                line_num,
                quantity,
                return_ts,
                return_id,
                returned,
                _,
                _,
            ) in rows
            if receipt_id is not None
        )
        for event in simulate_dc_sku(
            demand,
            dc_id=dc_id,
            product_id=product_id,
            lanes=lanes,
            start_date=start_date,
            end_date=end_date,
            seed=seed,
            default_prior=daily_prior,
            dc_view=dc_views.get(dc_id),
            closed_class=closed_class,
        ):
            pending.append(values(event))
            if len(pending) >= _ARROW_BATCH_ROWS:
                yield flush(pending)
                pending = []
    if pending:
        yield flush(pending)


def _fulfilled_finances(
    sales: dict[str, DataFrame],
    events: DataFrame,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    seed: int,
    tax_rules: TaxRules | None = None,
) -> dict[str, DataFrame]:
    """Receipts, lines and payments of the lines the policy could fill.

    The basket is priced again over its accepted lines (``baskets.price_basket``,
    as for candidates and the live stream), so a promotion whose minimum the
    fulfilled basket no longer meets is replaced or dropped.
    """
    draws = seeded_draws(seed)
    accepted = events.filter(F.col("kind") == "SALE").select(
        F.col("receipt_id").alias("receipt_id_ext"),
        "line_num",
    )
    lines = (
        sales["fact_receipt_lines"]
        .join(accepted, ["receipt_id_ext", "line_num"], "left_semi")
        .join(
            dims["dim_products"].select(
                F.col("ID").alias("product_id"),
                "taxability",
                F.col("Department").alias("department"),
            ),
            "product_id",
        )
        .groupBy("receipt_id_ext")
        .agg(
            F.array_sort(
                F.collect_list(
                    F.struct(
                        "line_num",
                        "product_id",
                        "quantity",
                        "unit_cents",
                        "taxability",
                        "department",
                    )
                )
            ).alias("_lines")
        )
    )
    key = F.col("receipt_id_ext")
    priced = (
        sales["fact_receipts"]
        .select("receipt_id_ext", "store_id", "event_ts", "event_date")
        .join(lines, "receipt_id_ext")
        .join(dims["dim_stores"].select(F.col("ID").alias("store_id"), "tax_rate"), "store_id")
        .join(
            store_tax_rules(dims["dim_stores"], dims.get("dim_geographies"), tax_rules or {}),
            "store_id",
        )
        .withColumn(
            "_lines",
            price_basket(
                draws,
                key,
                F.col("_lines"),
                F.month("event_date"),
                receipt_offer(draws, key, profile.promo_rate),
                F.round(F.col("tax_rate") * 10000).cast("long"),
                profile,
                F.col("tax_rule"),
            ),
        )
    )
    fulfilled_lines = priced.select(
        "receipt_id_ext", "event_ts", "event_date", F.explode("_lines").alias("_l")
    ).select(
        "receipt_id_ext",
        "event_ts",
        "event_date",
        F.col("_l.line_num").alias("line_num"),
        F.col("_l.product_id").alias("product_id"),
        F.col("_l.quantity").alias("quantity"),
        _fmt(F.col("_l.unit_cents")).alias("unit_price"),
        F.col("_l.unit_cents").alias("unit_cents"),
        _fmt(F.col("_l.ext_cents")).alias("ext_price"),
        F.col("_l.ext_cents").alias("ext_cents"),
        F.col("_l.promo_code").alias("promo_code"),
    )
    totals = priced.select(
        "receipt_id_ext",
        sum_lines(F.col("_lines"), "ext_cents").alias("subtotal_cents"),
        sum_lines(F.col("_lines"), "discount_cents").alias("discount_cents"),
        sum_lines(F.col("_lines"), "tax_cents").alias("tax_cents"),
    )
    header = (
        sales["fact_receipts"]
        .drop(*_FINANCE_COLUMNS)
        .join(totals, "receipt_id_ext")
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
    )
    for name in ("subtotal", "discount", "tax", "total"):
        header = header.withColumn(f"{name}_amount", _fmt(F.col(f"{name}_cents")))
    header = header.withColumn("Subtotal", F.col("subtotal_amount")).select(
        *column_names("fact_receipts"),
    )
    payments = (
        sales["fact_payments"]
        .drop("amount_cents", "amount")
        .join(
            header.select("receipt_id_ext", F.col("total_cents").alias("amount_cents")),
            "receipt_id_ext",
        )
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .select(*column_names("fact_payments"))
    )
    return {
        "fact_receipts": header,
        "fact_receipt_lines": fulfilled_lines.select(*column_names("fact_receipt_lines")),
        "fact_payments": payments,
    }


def sku_demand_priors(
    dims: dict[str, DataFrame], profile: StoreTypeProfile, cfg: GenerationConfig
) -> DataFrame:
    """``(store_id, product_id, prior_rate)``: expected daily units of every SKU.

    A theoretical prior from the demand model at the first history day, never a
    look-ahead at generated demand: the store's receipts a day (volume, store
    multiplier, month and growth) times the lines per basket (the ticket target
    over the expected line value, as ``behavior.calibrated_basket_mean``), the
    department's seasonal weight, the SKU's share of its launched department
    (``behavior.retail_catalog`` popularity at the list price) and the mean
    units per line. ``prior_rate`` is NULL where the catalog lacks the inputs.
    """
    draws = seeded_draws(cfg.seed)
    start = cfg.start_date
    weights = {}
    for name, weight in profile.department_weights.items():
        lifts = next((m for token, m in SEASONAL_LIFT.items() if token in name.lower()), None)
        weights[name] = weight * (lifts[start.month - 1] if lifts else 1.0)
    total_weight = sum(weights.values()) or 1.0
    department = F.col("department")
    products = (
        dims["dim_products"]
        .select(
            F.col("ID").cast("long").alias("product_id"),
            F.col("Department").alias("department"),
            "SalePrice",
            F.to_date("LaunchDate").alias("launch_date"),
        )
        .withColumn(
            "_launched",
            F.coalesce(F.col("launch_date") <= F.lit(start), F.lit(True)),
        )
        .withColumn(
            "_popularity",
            F.when(~F.col("_launched"), F.lit(0.0)).when(
                F.col("SalePrice") > 0,
                F.exp(0.8 * draws.gauss(["product_id"], "retail-popularity") - 0.32)
                / F.pow(F.col("SalePrice"), 1.15),
            ),
        )
        .withColumn(
            "_department_share",
            case_when(
                (department == name, F.lit(weight / total_weight))
                for name, weight in weights.items()
            ).otherwise(F.when(department.isNotNull(), F.lit(0.0))),
        )
        .withColumn("_units_per_line", 1.0 / (1.0 - repeat_purchase_probability(department)))
    )
    by_department = Window.partitionBy("department")
    products = products.withColumn(
        "_share",
        F.col("_department_share")
        * F.col("_popularity")
        / F.sum("_popularity").over(by_department),
    )
    line_value = products.agg(
        F.sum(F.col("_share") * F.col("SalePrice") * F.col("_units_per_line"))
    ).collect()[0][0]
    tax_rate = dims["dim_stores"].agg(F.avg("tax_rate")).collect()[0][0] or 0.0
    lines = None
    if line_value:
        affordable = profile.avg_ticket_target / (
            line_value
            * (1.0 + tax_rate)
            * (1.0 - profile.promo_rate * 0.20)
            * (1.0 + COMPLEMENT_RATE)
        )
        lines = max(1.0, min(affordable, profile.basket_lambda)) * (1.0 + COMPLEMENT_RATE)
    month = monthly_factors(profile)[start.month - 1]
    receipts_per_day = (
        cfg.transactions_per_store_day
        * month
        * volume_growth(cfg.annual_volume_growth, cfg.anchor_date, start)
    )
    units = (
        F.lit(lines) * F.col("_share") * F.col("_units_per_line")
        if lines is not None
        else F.lit(None).cast("double")
    )
    stores = dims["dim_stores"].select(
        F.col("ID").alias("store_id"),
        F.coalesce(F.col("daily_traffic_multiplier"), F.lit(1.0)).alias("_multiplier"),
    )
    result = stores.crossJoin(products.select("product_id", units.alias("_units"))).select(
        "store_id",
        "product_id",
        (F.lit(receipts_per_day) * F.col("_multiplier") * F.col("_units")).alias("prior_rate"),
    )
    return result


def _online_dc_demand(
    online_lines: DataFrame, network: IncidentNetwork, cfg: GenerationConfig
) -> DataFrame:
    """Ship-from-DC online lines as DC demand rows in DC-local time (no store)."""
    zones = online_lines.sparkSession.createDataFrame(
        [(dc, network.zone_of_dc(dc)) for dc in network.dc_ids()], "dc_id long, _zone string"
    )
    local = F.from_utc_timestamp(F.col("shipped_ts"), F.col("_zone"))
    return (
        online_lines.where(
            (F.col("node_type") == "DC") & F.col("shipped_ts").isNotNull() & (F.col("quantity") > 0)
        )
        .select(
            F.col("node_id").cast("long").alias("dc_id"),
            F.col("product_id").cast("long").alias("product_id"),
            "shipped_ts",
            "order_id",
            F.col("line_num").cast("long").alias("line_num"),
            F.col("quantity").cast("long").alias("quantity"),
        )
        .join(F.broadcast(zones), "dc_id")
        .withColumn("event_ts", local)
        .where(
            (F.to_date("event_ts") >= F.lit(cfg.start_date))
            & (F.to_date("event_ts") <= F.lit(cfg.end_date))
        )
        .select(
            F.lit(None).cast("long").alias("store_id"),
            "product_id",
            "event_ts",
            F.col("order_id").alias("receipt_id_ext"),
            "line_num",
            "quantity",
            F.lit(None).cast("timestamp").alias("return_ts"),
            F.lit(None).cast("string").alias("return_id"),
            F.lit(None).cast("long").alias("return_quantity"),
            F.lit(None).cast("double").alias("prior_rate"),
            F.concat(F.lit("TRC-DCO-"), "order_id", F.lit("-"), "line_num").alias("_sort"),
            "dc_id",
        )
    )


def resolve_fulfillment(
    sales_group: dict[str, DataFrame],
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    *,
    return_headers: DataFrame | None = None,
    profile: StoreTypeProfile | None = None,
    online_lines: DataFrame | None = None,
) -> FulfillmentResult:
    """Resolve candidate demand before any receipt-dependent fact generation.

    Only small dimension metadata is counted on the driver. Sales are shuffled
    by (DC, SKU), sorted, and streamed on workers (``dc_supply``), never
    collected/grouped into Python lists. ``online_lines`` (the online stage's
    ``fact_online_order_lines``, UTC) adds ship-from-DC demand to its DC.
    ``events`` is cached because all output ledgers and the fulfilled receipt
    filter must consume the exact same policy execution.
    """
    if profile is None:
        profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile
    headers = (
        build_return_headers(sales_group, cfg, dims, profile)
        if return_headers is None
        else return_headers
    ).cache()
    # The lines (and units) each return takes back; a plan without
    # returned_lines returns the whole receipt.
    returned = headers.select(
        F.col("orig_receipt_id_ext").alias("receipt_id_ext"),
        F.col("receipt_id_ext").alias("return_id"),
        F.col("event_ts").alias("return_ts"),
        (
            F.col("returned_lines")
            if "returned_lines" in headers.columns
            else F.lit(None).cast("array<struct<line_num:int,quantity:int>>")
        ).alias("returned_lines"),
    )
    demand = (
        sales_group["fact_receipt_lines"]
        .join(sales_group["fact_receipts"].select("receipt_id_ext", "store_id"), "receipt_id_ext")
        .join(returned, "receipt_id_ext", "left")
        .withColumn(
            "return_quantity",
            F.when(
                F.col("return_id").isNotNull(),
                returned_quantity(F.col("returned_lines"), F.col("line_num"), F.col("quantity")),
            ),
        )
    )
    restock = (
        seeded_draws(cfg.seed).u(
            [F.col("return_id"), F.col("line_num").cast("string")],
            "restock",
        )
        < 0.5
    ) & (F.col("return_quantity") > 0)
    demand = demand.select(
        "store_id",
        "product_id",
        "event_ts",
        "receipt_id_ext",
        "line_num",
        "quantity",
        F.when(restock, F.col("return_ts")).alias("return_ts"),
        F.when(restock, F.col("return_id")).alias("return_id"),
        F.when(restock, F.col("return_quantity")).cast("long").alias("return_quantity"),
        F.lit(None).cast("double").alias("prior_rate"),
        F.concat(F.lit("TRC-INV-"), "receipt_id_ext", F.lit("-"), "line_num").alias("_sort"),
    )
    n_products = dims["dim_products"].count()
    if n_products < 1:
        raise ValueError("fulfillment requires a nonempty product catalog")
    # A uniform basket prior covers SKUs the demand model cannot price.
    uniform_prior = cfg.transactions_per_store_day * 10.0 / n_products
    # Seed every catalog SKU, not only products that happen to sell in this
    # window, with its expected daily demand (sizes opening stock).
    seeds = sku_demand_priors(dims, profile, cfg).select(
        "store_id",
        "product_id",
        F.to_timestamp(F.lit(f"{cfg.start_date} 00:00:00")).alias("event_ts"),
        F.lit(None).cast("string").alias("receipt_id_ext"),
        F.lit(0).alias("line_num"),
        F.lit(0).alias("quantity"),
        F.lit(None).cast("timestamp").alias("return_ts"),
        F.lit(None).cast("string").alias("return_id"),
        F.lit(None).cast("long").alias("return_quantity"),
        F.coalesce(F.col("prior_rate"), F.lit(uniform_prior)).alias("prior_rate"),
        F.lit("").alias("_sort"),
    )
    spark = sales_group["fact_receipts"].sparkSession
    site_dims = (dims["dim_stores"], dims["dim_geographies"], dims["dim_distribution_centers"])
    routes = store_routes(cfg.seed, *site_dims)
    network = incident_network(*site_dims)
    calendar = IncidentCalendar(
        network_incidents(spark, cfg.seed, network, cfg.start_date, cfg.end_date)
    )
    store_dc = spark.createDataFrame(
        sorted((store, route.dc_id) for store, route in routes.items()),
        "store_id long, dc_id long",
    )
    rows = demand.unionByName(seeds).join(F.broadcast(store_dc), "store_id")
    if online_lines is not None:
        rows = rows.unionByName(_online_dc_demand(online_lines, network, cfg))
    ordered = rows.repartition("dc_id", "product_id").sortWithinPartitions(
        "dc_id",
        "product_id",
        "event_ts",
        "_sort",
    )
    time_zone = spark.conf.get("spark.sql.session.timeZone")
    if time_zone is None:
        raise RuntimeError("spark.sql.session.timeZone is not set")
    worker = partial(
        _simulate_arrow_partition,
        start_date=cfg.start_date,
        end_date=cfg.end_date,
        seed=cfg.seed,
        daily_prior=uniform_prior,
        time_zone=time_zone,
        routes={
            store: (route.slot_minute, route.transit_minutes) for store, route in routes.items()
        },
        views={
            store: calendar.store_view(store, route.dc_id, network.zone_of_store(store))
            for store, route in routes.items()
        },
        dc_views={dc: calendar.dc_view(dc, network.zone_of_dc(dc)) for dc in network.dc_ids()},
        closed_class=store_class(profile),
    )
    events = (
        ordered.select(*_DEMAND_COLUMNS, "dc_id")
        .mapInArrow(worker, _EVENT_SCHEMA)
        .withColumn(
            "event_date",
            F.to_date("event_ts"),
        )
        .cache()
    )
    return FulfillmentResult(
        _fulfilled_finances(
            sales_group,
            events,
            dims,
            profile,
            cfg.seed,
            tax_rules_by_state(load_tax_rules(cfg.resolved_dictionary_root)),
        ),
        headers,
        events,
    )

# --- retail_setup/generation/store_activity.py ---
"""Store ops + foot traffic facts, Spark-native.

fact_store_ops: exactly two rows (opened/closed) per open store-day, with the
open/close hours parsed from `dim_stores.operating_hours` ("24h" or
"open-close", from the store type's profile). Days the holiday calendar
closes the store type's stores (Christmas, and Thanksgiving for all but
grocers; ``stream_calendar.closed_date_column``) are skipped entirely.

Footfall uses the same independently generated visitors as receipt conversion
and counts zone passages, not shares: the store type's zones
(``vocabulary.ble_layout``), passed as in a visit's path
(``vocabulary.visit_zone_path``, shared with BLE and the live stream). Every
visitor passes the entrance and the exit, checkout counts the hour's buyers,
and each department zone the visitors who browse it.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F



def generate_store_ops(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    profile: StoreTypeProfile | None = None,
) -> DataFrame:
    """Two rows (opened/closed) per store-day, skipping days the stores are closed."""
    if profile is None:
        profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile

    stores = dims["dim_stores"].select(F.col("ID").alias("store_id"), "operating_hours")
    store_ids = [r.store_id for r in stores.select("store_id").collect()]

    grid = (
        store_day_grid(spark, store_ids, cfg.start_date, cfg.end_date, cfg.seed, "store_ops")
        .filter(~closed_date_column(F.col("day"), store_class(profile)))
        .join(stores, "store_id")
    )

    # operating_hours ("24h" or "open-close") -> open/close hour; unknown
    # formats fall back to standard hours instead of silent NULL event_ts
    open_hour, close_hour = _open_close_cols(F.col("operating_hours"))
    default_open, default_close = DEFAULT_OPERATING_WINDOW
    grid = grid.withColumn("open_hour", F.coalesce(open_hour, F.lit(default_open))).withColumn(
        "close_hour", F.coalesce(close_hour, F.lit(default_close))
    )

    day_ts = F.unix_timestamp(F.col("day").cast("timestamp"))
    open_ts = F.timestamp_seconds(day_ts + F.col("open_hour") * 3600)
    # 24h stores "close" at 23:59:00 the same day; others at close_hour:00
    close_secs = F.when(F.col("close_hour") == 24, F.lit(23 * 3600 + 59 * 60)).otherwise(
        F.col("close_hour") * 3600
    )
    close_ts = F.timestamp_seconds(day_ts + close_secs)

    ops = grid.select(
        "store_id",
        F.col("day").alias("event_date"),
        F.explode(
            F.array(
                F.struct(F.lit("opened").alias("operation_type"), open_ts.alias("event_ts")),
                F.struct(F.lit("closed").alias("operation_type"), close_ts.alias("event_ts")),
            )
        ).alias("op"),
    ).select(
        F.col("op.event_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-OPS-"),
            F.col("store_id"),
            F.lit("-"),
            F.col("event_date"),
            F.lit("-"),
            F.col("op.operation_type"),
        ).alias("trace_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("op.operation_type").alias("operation_type"),
        "event_date",
    )
    return ops.select(*column_names("fact_store_ops"))


def generate_foot_traffic(
    spark: SparkSession,
    receipts: DataFrame,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Per store-hour-zone passage counts of the hour's visitors and buyers."""

    d = seeded_draws(cfg.seed)

    del receipts
    profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile
    layout = ble_layout(profile)
    shares = zone_passage_shares(layout)
    grid = generate_hourly_visits(
        spark, dims["dim_stores"], profile, cfg, dims.get("dim_geographies")
    )

    zone_structs = [
        F.struct(
            F.lit(zone).alias("zone"),
            F.lit(shares.get(zone, -1.0)).alias("share"),
            F.lit(layout.dwell[zone][0]).alias("dwell_lo"),
            F.lit(layout.dwell[zone][1]).alias("dwell_hi"),
        )
        for zone in layout.zones
    ]
    ft = grid.select(
        "store_id",
        "hour_ts",
        "visitors",
        "n_receipts",
        F.explode(F.array(*zone_structs)).alias("z"),
    )
    visitors = F.col("visitors").cast("double")
    share = F.col("z.share")
    # Department passages: binomial counts (normal approximation) of the
    # hour's visitors; checkout counts the hour's buyers (share -1).
    spread = F.sqrt(visitors * share * (1.0 - share))
    browsed = F.least(
        visitors,
        F.greatest(
            F.lit(0.0),
            F.round(
                visitors * share
                + spread
                * d.gauss([F.col("store_id"), F.col("hour_ts"), F.col("z.zone")], "ft_count")
            ),
        ),
    )
    count = (
        F.when(share < 0, F.col("n_receipts").cast("double"))
        .when(share >= 1.0, visitors)
        .otherwise(browsed)
    )
    dwell_u = d.u([F.col("store_id"), F.col("hour_ts"), F.col("z.zone")], "ft_dwell")
    ft = ft.select(
        F.col("hour_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-FT-"),
            F.col("store_id"),
            F.lit("-"),
            F.date_format("hour_ts", "yyyyMMddHH"),
            F.lit("-"),
            F.col("z.zone"),
        ).alias("trace_id"),
        F.col("store_id").cast("long").alias("store_id"),
        sensor_id_column(F.col("store_id"), F.col("z.zone")).alias("sensor_id"),
        F.col("z.zone").alias("zone"),
        F.floor(F.col("z.dwell_lo") + dwell_u * (F.col("z.dwell_hi") - F.col("z.dwell_lo") + 1))
        .cast("long")
        .alias("dwell_seconds"),
        count.cast("long").alias("count"),
        F.to_date("hour_ts").alias("event_date"),
    )
    return ft.select(*column_names("fact_foot_traffic"))

# --- retail_setup/generation/online_orders.py ---
"""Online orders fact group (headers, lines, payments stream), Spark-native.

Mirrors `receipts.py`: all randomness flows through `runtime.seeded_draws`
(xxhash64-based, partition-arrangement independent), money is integer cents,
and daily volume is negative-binomial, allowing overdispersed counts and zero days.

Shared with the live stream (``stream_events.plan_orders``):
- Orders arrive on the online hourly curve (``vocabulary.ONLINE_HOURLY_WEIGHTS``,
  UTC) at the profile's monthly weights, the year-over-year trend and the
  online holiday calendar (``stream_calendar.online_holiday_factor_column``).
- Each order's customer is active (signed up, not churned) on the order day,
  drawn from a home-store pool by store traffic (customers.py), so orders are
  local to the customer's home store.
- One fulfillment mode serves the whole order, by the historical mix. Store
  modes (ship-from-store, BOPIS) use the customer's home store; ship-from-DC
  uses the DC that supplies that store (``network.assign_dcs``).
- Baskets are sized to the profile's ticket target like store receipts
  (``behavior.calibrated_basket_mean``), lines follow the same shopping
  missions, and repeated SKUs merge into one line.
- Tax uses the exact integer basis-point formula from receipts with the state
  taxability rules (``baskets.line_tax_cents``): store modes use the store's
  rate and rules, ship-from-DC the customer's geography (destination).
- Cancelled and voided orders (decline model) have zero totals, zero line
  amounts and no payment, which preserves ``payment.amount == header.total``.
- Backordered orders (never BOPIS) wait for stock between picking and
  shipping (``ONLINE_BACKORDER_SECONDS``), then ship and deliver.
- Statuses and stage timestamps are as of the end of history: an order still
  in fulfillment then is PENDING, PICKED or SHIPPED, and stages that would
  happen later are NULL.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


# Width of the daily order sequence in order IDs; busier days widen it (IDs
# stay unique because the random suffix has a fixed width).
ORDER_SEQUENCE_DIGITS = 5


def stage_seconds(d: seeded_draws, key: list, bounds: tuple[int, int], salt: str) -> Column:
    """A stage delay uniform in ``[low, high)`` seconds, drawn per order."""
    draw = d.u(key, salt)
    return (F.lit(bounds[0]) + draw * F.lit(float(bounds[1] - bounds[0]))).cast("long")


def generate_online_orders(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate fact_online_order_headers, fact_online_order_lines, payments.

    Payments are returned under the "payments" key for the orchestrator to
    union with the in-store stream before writing fact_payments.
    """
    d = seeded_draws(cfg.seed)

    # Destination tax: the mean store tax_rate of the customer's geography,
    # with the network mean as the fallback for geographies without a store.
    mean_row = dims["dim_stores"].agg(F.avg("tax_rate")).first()
    if mean_row is None or mean_row[0] is None:
        raise ValueError("dim_stores has no tax_rate to derive the online tax fallback")
    mean_bps = int(round(float(mean_row[0]) * 10000))
    rules = tax_rules_by_state(load_tax_rules(cfg.resolved_dictionary_root))
    states = dims["dim_geographies"].select(F.col("ID").alias("GeographyID"), "State")
    geo_rate = dims["dim_stores"].groupBy("GeographyID").agg(F.avg("tax_rate").alias("geo_rate"))
    customers = (
        dims["dim_customers"]
        .select(F.col("ID").alias("customer_id"), "GeographyID", "HomeStoreID")
        .join(geo_rate, "GeographyID", "left")
        .join(states, "GeographyID", "left")
        .select(
            "customer_id",
            F.col("HomeStoreID").cast("long").alias("home_store"),
            F.coalesce(F.round(F.col("geo_rate") * 10000), F.lit(mean_bps))
            .cast("long")
            .alias("customer_bps"),
            tax_rule_column(rules, F.col("State")).alias("customer_rule"),
        )
    )
    stores = (
        dims["dim_stores"]
        .join(states, "GeographyID", "left")
        .select(
            F.col("ID").cast("long").alias("home_store"),
            F.round(F.col("tax_rate") * 10000).cast("long").alias("store_bps"),
            tax_rule_column(rules, F.col("State")).alias("store_rule"),
        )
        .join(
            store_dc_map(
                dims["dim_stores"], dims["dim_geographies"], dims["dim_distribution_centers"]
            ).withColumnRenamed("store_id", "home_store"),
            "home_store",
        )
    )

    # --- day grid: network-wide daily volume, monthly-weight scaled, with the
    # online holiday calendar (Black Friday, Cyber Monday, ...)
    mw = profile.monthly_weights
    m_mean = sum(mw) / 12.0
    days = date_grid(spark, cfg.start_date, cfg.end_date)
    monthly_w = F.element_at(F.array(*[F.lit(w / m_mean) for w in mw]), F.month("day"))
    lam = (
        F.lit(float(cfg.online_orders_per_day))
        * monthly_w
        * online_holiday_factor_column(F.col("day"))
        * volume_growth_column(cfg.annual_volume_growth, cfg.anchor_date, F.col("day"))
    )
    # The daily mean is bound once: the draw references it many times, and
    # copying the holiday calendar into each reference bloats the plan.
    n_orders = bind(lam, lambda mean: d.negative_binomial(["day"], "onl_n", mean, 24)).cast("int")

    # --- orders: ids, timestamps, tender, cancellation
    orders = (
        days.withColumn("n_orders", n_orders)
        .filter(F.col("n_orders") > 0)
        .withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n_orders"))))
        .withColumn(
            "hour",
            d.pick_by_weights(
                ["day", "seq"],
                "onl_hour",
                [(str(hour), weight) for hour, weight in enumerate(ONLINE_HOURLY_WEIGHTS)],
            ).cast("int"),
        )
        .withColumn("minute", (d.h64(["day", "seq"], "onl_min") % 60).cast("int"))
        .withColumn("second", (d.h64(["day", "seq"], "onl_sec") % 60).cast("int"))
        .withColumn(
            "event_ts",
            F.make_timestamp(
                F.year("day"),
                F.month("day"),
                F.dayofmonth("day"),
                F.col("hour"),
                F.col("minute"),
                F.col("second"),
            ),
        )
        .withColumn("event_date", F.col("day"))
        # ONL + yyyyMMdd + daily seq (at least 5 digits) + 3-digit draw;
        # unique because (day, seq) is a key and the suffix width is fixed.
        .withColumn(
            "order_id_ext",
            F.concat(
                F.lit("ONL"),
                F.date_format("day", "yyyyMMdd"),
                minimum_width_decimal(F.col("seq"), ORDER_SEQUENCE_DIGITS),
                fixed_width_decimal(
                    d.h64(["day", "seq"], "onl_rand") % 1000,
                    3,
                    "online-order random suffix",
                ),
            ),
        )
        .withColumn(
            "_first_tender",
            d.pick_by_weights(
                ["order_id_ext"], "onl_tender", [(n, w) for n, w, _, _, _ in ONLINE_TENDERS]
            ),
        )
        # Decline model shared with store sales: a declined order is retried on
        # another tender, or voided, which cancels it (zero totals, no payment).
        .withColumn(
            "_outcome",
            payment_outcome_column(
                d, ["order_id_ext"], F.col("_first_tender"), ONLINE_TENDERS, "onl_decline"
            ),
        )
        .withColumn(
            "tender_type",
            F.when(
                F.col("_outcome") == F.lit(RETRIED),
                retry_tender_column(
                    d, ["order_id_ext"], F.col("_first_tender"), ONLINE_TENDERS, "onl_retry"
                ),
            ).otherwise(F.col("_first_tender")),
        )
        .withColumn(
            "is_cancelled",
            (d.u(["order_id_ext"], "onl_cancel") < F.lit(ONLINE_CANCEL_RATE))
            | (F.col("_outcome") == F.lit(VOIDED)),
        )
        # per-order promo intensity: the share of its lines that carry a promo
        .withColumn(
            "promo_rate",
            d.u(["order_id_ext"], "onl_prate")
            * F.lit(ONLINE_PROMO_LINE_SHARE[1] - ONLINE_PROMO_LINE_SHARE[0])
            + F.lit(ONLINE_PROMO_LINE_SHARE[0]),
        )
    )
    # Each order's customer is active that day (signed up, not churned); the
    # rare order whose tries all find inactive customers is not placed.
    lifecycle = lifecycle_for(cfg, profile.resolved_customer_visits_per_year())
    pool, sizes, picker = customer_pools(lifecycle, dims)
    mode = F.col("fulfillment_mode")
    orders = (
        assign_online_customers(
            lifecycle, orders, pool, sizes, picker, "order_id_ext", "event_date"
        )
        .join(customers, "customer_id", "left")
        .join(F.broadcast(stores), "home_store", "left")
        # One fulfillment mode and node per order: the home store for store
        # modes, the DC that supplies it for ship-from-DC.
        .withColumn(
            "fulfillment_mode",
            d.pick_by_weights(["order_id_ext"], "onl_mode", ONLINE_FULFILLMENT_MIX),
        )
        .withColumn("node_type", F.when(mode == "SHIP_FROM_DC", "DC").otherwise("STORE"))
        .withColumn(
            "node_id",
            F.when(F.col("node_type") == "DC", F.col("dc_id")).otherwise(F.col("home_store")),
        )
        .withColumn(
            "rate_bps",
            F.when(F.col("node_type") == "STORE", F.col("store_bps")).otherwise(
                F.coalesce(F.col("customer_bps"), F.lit(mean_bps))
            ),
        )
        .withColumn(
            "tax_rule",
            F.when(F.col("node_type") == "STORE", F.col("store_rule")).otherwise(
                F.col("customer_rule")
            ),
        )
        .withColumn(
            "is_backordered",
            (mode != "BOPIS")
            & ~F.col("is_cancelled")
            & (d.u(["order_id_ext"], "onl_backorder") < F.lit(ONLINE_BACKORDER_RATE)),
        )
        .withColumn("_shopper", shopper_key(F.col("customer_id"), F.col("order_id_ext")))
        .withColumn("_offer", receipt_offer(d, F.col("order_id_ext"), profile.promo_rate))
    )

    # --- baskets sized to the ticket target, lines drawn like store baskets
    # from the shared popularity and shelf prices of launched products
    products = basket_products(dims["dim_products"], cfg.seed)
    order_dates = date_grid(spark, cfg.start_date, cfg.end_date, "event_date")
    catalog = retail_catalog(products, order_dates, d, PriceSchedule(cfg.seed, cfg.anchor_date))
    # The day's calibration is a few numbers per day: collected once, it keeps
    # the catalog out of the order plans.
    calibration = spark.createDataFrame(
        basket_calibration(catalog, profile).collect(),
        "event_date date, _line_value double, _taxable_value double",
    )
    orders = (
        orders.join(F.broadcast(calibration), "event_date")
        .withColumn(
            "basket_n",
            retail_basket_size(
                d,
                ["order_id_ext"],
                F.col("_shopper"),
                calibrated_basket_mean(
                    F.col("_line_value"),
                    F.col("_taxable_value"),
                    F.col("rate_bps") / 10000.0,
                    profile.avg_ticket_target,
                    profile.promo_rate,
                    profile.basket_lambda,
                ),
                F.col("_offer"),
                profile.promo_rate,
            ),
        )
        .drop("_line_value", "_taxable_value")
    )
    # --- order lifecycle, as of the end of history (cancelled orders keep
    # NULL stage times); one plan per order, like the live stream
    key = ["order_id_ext"]
    live = ~F.col("is_cancelled")
    is_bopis = F.col("fulfillment_mode") == "BOPIS"
    pick_secs = F.when(
        is_bopis, stage_seconds(d, key, ONLINE_PICK_SECONDS["BOPIS"], "onl_pick")
    ).otherwise(stage_seconds(d, key, ONLINE_PICK_SECONDS["SHIP"], "onl_pick"))
    wait_secs = F.when(
        F.col("is_backordered"),
        stage_seconds(d, key, ONLINE_BACKORDER_SECONDS, "onl_backorder_wait"),
    ).otherwise(F.lit(0).cast("long"))
    ship_secs = stage_seconds(d, key, ONLINE_SHIP_SECONDS, "onl_ship") + wait_secs
    deliver_secs = stage_seconds(d, key, ONLINE_DELIVER_SECONDS, "onl_dlv")
    pickup_secs = stage_seconds(d, key, ONLINE_PICKUP_SECONDS, "onl_pickup")
    history_end = F.lit(cfg.end_date).cast("timestamp") + F.expr("INTERVAL 1 DAY")

    def after(ts: str, seconds: Column) -> Column:
        return F.timestamp_seconds(F.unix_timestamp(ts) + seconds)

    orders = (
        orders.withColumn("_picked", F.when(live, after("event_ts", pick_secs)))
        # BOPIS never ships: the customer collects the order once it is ready
        .withColumn("_shipped", F.when(~is_bopis, after("_picked", ship_secs)))
        .withColumn(
            "_delivered",
            F.when(is_bopis, after("_picked", pickup_secs)).otherwise(
                after("_shipped", deliver_secs)
            ),
        )
    )
    for stage in ("picked", "shipped", "delivered"):
        orders = orders.withColumn(
            f"{stage}_ts", F.when(F.col(f"_{stage}") < history_end, F.col(f"_{stage}"))
        )
    orders = orders.withColumn(
        "fulfillment_status",
        F.when(F.col("is_cancelled"), "CANCELLED")
        .when(F.col("delivered_ts").isNotNull(), "DELIVERED")
        .when(F.col("shipped_ts").isNotNull(), "SHIPPED")
        .when(F.col("picked_ts").isNotNull(), "PICKED")
        .otherwise("PENDING"),
    ).drop("_picked", "_shipped", "_delivered")

    promo_idx = (d.h64(["order_id_ext", "line_num"], "onl_pidx") % len(ONLINE_PROMOS)).cast("int")
    promo_code_arr = F.array(*[F.lit(c) for c, _ in ONLINE_PROMOS])
    promo_pct_arr = F.array(*[F.lit(p).cast("long") for _, p in ONLINE_PROMOS])
    # Header fields travel with the lines, so headers are summed from the lines
    # instead of joining the order plan a second time (every order has lines).
    header_fields = ["customer_id", "_first_tender", "tender_type", "_outcome"]
    carried = [
        *header_fields,
        "event_ts",
        "is_cancelled",
        "promo_rate",
        "rate_bps",
        "tax_rule",
        "fulfillment_mode",
        "node_type",
        "node_id",
        "fulfillment_status",
        "picked_ts",
        "shipped_ts",
        "delivered_ts",
    ]
    lines = (
        draw_basket_lines(
            orders.select("order_id_ext", "event_date", "basket_n", "_shopper", *carried),
            "order_id_ext",
            catalog,
            d,
            profile,
            carry=["event_date", *carried],
        )
        .withColumn("_lines", merge_lines(F.col("_lines")))
        .select("*", F.explode("_lines").alias("_l"))
        .select(
            "*",
            F.col("_l.line_num").cast("long").alias("line_num"),
            "_l.product_id",
            F.col("_l.quantity").cast("long").alias("quantity"),
            "_l.unit_cents",
            "_l.taxability",
        )
        .drop("_lines", "_l")
        .withColumn("ext_before", F.col("unit_cents") * F.col("quantity"))
        .withColumn(
            "has_promo", d.u(["order_id_ext", "line_num"], "onl_promo") < F.col("promo_rate")
        )
        .withColumn(
            "promo_code", F.when(F.col("has_promo"), F.element_at(promo_code_arr, promo_idx + 1))
        )
        .withColumn(
            "promo_pct",
            F.when(F.col("has_promo"), F.element_at(promo_pct_arr, promo_idx + 1)).otherwise(
                F.lit(0).cast("long")
            ),
        )
        # matching 5/10/20% discount, rounded half-up in integer cents; a
        # cancelled order's lines carry no amount, like its header
        .withColumn(
            "discount_cents",
            F.when(
                live,
                F.floor((F.col("ext_before") * F.col("promo_pct") + F.lit(50)) / F.lit(100)),
            )
            .otherwise(F.lit(0))
            .cast("long"),
        )
        .withColumn(
            "ext_cents",
            F.when(live, F.col("ext_before") - F.col("discount_cents"))
            .otherwise(F.lit(0))
            .cast("long"),
        )
        .withColumn(
            "line_tax_cents",
            line_tax_cents(
                F.col("ext_cents"),
                F.col("taxability"),
                F.col("rate_bps"),
                F.col("tax_rule"),
                F.col("unit_cents"),
            ),
        )
    )

    # TMDL-bound legacy pandas-index column: hash-derived via legacy_index.
    fact_online_order_lines = (
        lines.withColumn("__index_level_0__", legacy_index("order_id_ext", "line_num"))
        .select(
            F.col("order_id_ext").alias("order_id"),
            "product_id",
            F.col("line_num").cast("long").alias("line_num"),
            "quantity",
            _fmt(F.col("unit_cents")).alias("unit_price"),
            "unit_cents",
            _fmt(F.col("ext_cents")).alias("ext_price"),
            "ext_cents",
            "promo_code",
            "fulfillment_mode",
            "fulfillment_status",
            "node_type",
            "node_id",
            "picked_ts",
            "shipped_ts",
            "delivered_ts",
            "event_ts",
            "event_date",
            "__index_level_0__",
        )
        .select(*column_names("fact_online_order_lines"))
    )

    # --- headers: sums of the lines; cancelled orders' lines are zero
    headers_base = (
        lines.groupBy("order_id_ext", "event_ts", "event_date", "is_cancelled", *header_fields)
        .agg(
            F.sum("ext_cents").cast("long").alias("subtotal_cents"),
            F.sum("discount_cents").cast("long").alias("discount_cents"),
            F.sum("line_tax_cents").cast("long").alias("tax_cents"),
        )
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
        .withColumn("payment_method", F.col("tender_type"))
        # IMP-007: subtotal_cents stays net (existing behavior); gross recovers
        # the pre-discount total. Attribution fields default NULL; enriched for
        # the orders an impression converts (attribution.py).
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
    )
    fact_online_order_headers = headers_base.select(
        "order_id_ext",
        "customer_id",
        "subtotal_cents",
        "tax_cents",
        "total_cents",
        _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
        _fmt(F.col("tax_cents")).alias("tax_amount"),
        _fmt(F.col("total_cents")).alias("total_amount"),
        "payment_method",
        "event_ts",
        "event_date",
        "gross_subtotal_cents",
        "discount_cents",
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    ).select(*column_names("fact_online_order_headers"))

    # --- payments: per NON-cancelled order, the first attempt and, for a
    # retried order, the approved second tender (receipt_id_ext/store_id NULL)
    payments = payment_attempts(
        d,
        headers_base.filter(~F.col("is_cancelled")).select(
            "order_id_ext",
            "event_ts",
            F.lit(None).cast("long").alias("store_id"),
            "customer_id",
            "total_cents",
            "_first_tender",
            "tender_type",
            "_outcome",
        ),
        "order_id_ext",
        ONLINE_TENDERS,
        "onl_",
    )

    return {
        "fact_online_order_headers": fact_online_order_headers,
        "fact_online_order_lines": fact_online_order_lines,
        "payments": payments,
    }

# --- retail_setup/generation/promotions.py ---
"""Promotions fact group, derived purely from receipt lines (no draws).

Semantics (datagen promotions_mixin):

- ``fact_promo_lines``: one row per SALE receipt line that has a non-NULL
  ``promo_code`` and a positive line discount
  (``discount_cents = unit_cents * quantity - ext_cents > 0``). Event fields
  come from the line; ``trace_id = 'TRC-PRM-' + receipt_id_ext + '-' +
  promo_code + '-' + line_num``.
- ``fact_promotions``: one row per (receipt_id_ext, promo_code) aggregating
  those lines — discount sums, distinct ``product_count``, ``product_ids`` as
  a comma-joined sorted id list. ``discount_type`` is 'BOGO' for BOGO codes and
  'PERCENTAGE' for all other generated promo codes.
  Store/customer/event fields are joined from the receipt header;
  ``trace_id = 'TRC-PRM-' + receipt_id_ext + '-' + promo_code``.

TMDL contract note: ``fact_promo_lines`` carries dual columns — the
snake_case plan names plus TMDL-bound PascalCase mirrors (ReceiptId,
PromoCode, LineNumber, ProductID, Qty, DiscountAmount, DiscountCents) and the
legacy pandas-index column ``__index_level_0__`` (hash-derived via
legacy_index(receipt_id_ext, promo_code, line_number)). The PascalCase columns
are exact copies of their snake_case twins.

Pure joins/groupBys over the sales group — deterministic, no randomness.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F



def generate_promotions(
    spark: SparkSession,
    sales_group: dict[str, DataFrame],
) -> tuple[DataFrame, DataFrame]:
    """Derive (fact_promotions, fact_promo_lines) from the sales group."""

    sale_ids = (
        sales_group["fact_receipts"]
        .filter(F.col("receipt_type") == "SALE")
        .select("receipt_id_ext")
    )

    # --- promo lines: discounted SALE lines with a promo code
    lines = (
        sales_group["fact_receipt_lines"]
        .join(sale_ids, "receipt_id_ext")
        .filter(F.col("promo_code").isNotNull())
        .withColumn(
            "discount_cents",
            F.col("unit_cents") * F.col("quantity") - F.col("ext_cents"),
        )
        .filter(F.col("discount_cents") > 0)
        .withColumn("line_number", F.col("line_num").cast("long"))
        .withColumn("quantity", F.col("quantity").cast("long"))
        .withColumn("discount_amount", _fmt(F.col("discount_cents")))
        .withColumn(
            "trace_id",
            F.concat(
                F.lit("TRC-PRM-"),
                F.col("receipt_id_ext"),
                F.lit("-"),
                F.col("promo_code"),
                F.lit("-"),
                F.col("line_number").cast("string"),
            ),
        )
    )

    promo_lines = (
        lines
        # TMDL-bound PascalCase mirrors of the snake_case columns
        .withColumn("ReceiptId", F.col("receipt_id_ext"))
        .withColumn("PromoCode", F.col("promo_code"))
        .withColumn("LineNumber", F.col("line_number"))
        .withColumn("ProductID", F.col("product_id"))
        .withColumn("Qty", F.col("quantity"))
        .withColumn("DiscountAmount", F.col("discount_amount"))
        .withColumn("DiscountCents", F.col("discount_cents"))
        # Legacy pandas-index column bound by the semantic model
        .withColumn(
            "__index_level_0__", legacy_index("receipt_id_ext", "promo_code", "line_number")
        )
        .select(*column_names("fact_promo_lines"))
    )

    # --- promotions: aggregate per (receipt_id_ext, promo_code)
    agg = lines.groupBy("receipt_id_ext", "promo_code").agg(
        F.sum("discount_cents").alias("discount_cents"),
        F.countDistinct("product_id").alias("product_count"),
        F.concat_ws(",", F.sort_array(F.collect_set("product_id"))).alias("product_ids"),
    )

    headers = sales_group["fact_receipts"].select(
        "receipt_id_ext", "event_ts", "event_date", "store_id", "customer_id"
    )
    promotions = (
        agg.join(headers, "receipt_id_ext")
        .withColumn("discount_amount", _fmt(F.col("discount_cents")))
        # BOGO codes are buy-one-get-one; everything else is a percentage off.
        .withColumn(
            "discount_type",
            F.when(F.col("promo_code").startswith("BOGO"), F.lit("BOGO")).otherwise(
                F.lit("PERCENTAGE")
            ),
        )
        .withColumn(
            "trace_id",
            F.concat(F.lit("TRC-PRM-"), F.col("receipt_id_ext"), F.lit("-"), F.col("promo_code")),
        )
        # IMP-007: NULL unless the underlying receipt was selected for
        # attribution; enriched in attribution.py.
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .select(*column_names("fact_promotions"))
    )

    return promotions, promo_lines

# --- retail_setup/generation/marketing.py ---
"""Marketing impressions fact, Spark-native.

Ports datagen's marketing_campaign.py constants: four campaign archetypes,
each with its own channel mix and base impressions/day. Daily volumes are
scaled by `cfg.store_count / 86` — 86 is the legacy datagen fleet size, so a
fleet of 86 stores reproduces datagen's nominal volumes. `flash_sale` runs
only ~1 day in 7 (a per-day uniform gate, u < 1/7).

Per impression: channel uniform over the archetype's channels; a creative from
the campaign's small pool (``CREATIVES_PER_CAMPAIGN`` per channel); device
MOBILE/DESKTOP/TABLET at 60/30/10 with cost multipliers 1.2/0.8/0.9; cost a
uniform draw within the channel's CPM band ($0.002-$0.03 an impression) times
the device multiplier, kept to four decimals (``cost_cents`` rounds at random);
a uniformly-assigned customer supplies `customer_ad_id` (dim_customers.AdId),
with `customer_id` populated for ~5% of impressions (CRM match) else NULL. A
CRM match needs the customer to be signed up and not yet churned that day
(generation.customers), so lapsed customers stop showing up as matched.
All draws via `runtime.seeded_draws`, so output is deterministic for a
(config, seed) pair regardless of cluster shape.

Schema note: fact_marketing is TMDL-bound with DUAL columns — PascalCase
`CustomerId`/`CostCents` (the TMDL sourceColumns) are exact mirrors of the
snake_case plan columns `customer_id`/`cost_cents`, and `__index_level_0__`
is the legacy pandas index (hash-derived via legacy_index).
"""

from datetime import date

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


ARCHETYPES = CAMPAIGN_ARCHETYPES

LEGACY_FLEET_SIZE = 86  # datagen's store count; volumes scale store_count/86
# Background (non-journey) impressions carry a customer_id for this CRM-matched share.
CRM_MATCH_RATE = 0.05


def campaign_window_start(day: Column, history_start: date, duration: Column) -> Column:
    """Start of the campaign window a day falls in.

    Campaigns of an archetype run back to back for its duration, anchored on
    the first history day, so the live stream continues the same campaigns.
    """
    return F.date_add(
        F.lit(history_start),
        (F.datediff(day, F.lit(history_start)) / duration).cast("int") * duration,
    )


def impression_device_cost(
    draws: seeded_draws, key: list, channel: Column, device_salt: str, cost_salt: str
) -> tuple[Column, Column]:
    """(device, cost in dollars) of an impression.

    The device is MOBILE/DESKTOP/TABLET at 60/30/10; the cost is uniform in
    the channel's band times the device's multiplier.
    """
    device = draws.pick_by_weights(key, device_salt, DEVICE_WEIGHTS)
    multiplier = case_when((device == name, m) for name, m in DEVICE_MULTIPLIERS.items())
    low = case_when((channel == name, c_lo) for name, (c_lo, _) in CHANNEL_COSTS.items())
    high = case_when((channel == name, c_hi) for name, (_, c_hi) in CHANNEL_COSTS.items())
    return device, (low + draws.u(key, cost_salt) * (high - low)) * multiplier


def generate_marketing(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    profile: StoreTypeProfile | None = None,
) -> DataFrame:
    """Ad-impression facts: day x archetype grid exploded to impressions."""

    d = seeded_draws(cfg.seed)
    if profile is None:
        profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile
    lifecycle = lifecycle_for(cfg, profile.resolved_customer_visits_per_year())
    scale = cfg.store_count / LEGACY_FLEET_SIZE

    # --- Spark-native day x archetype grid
    rows = [
        (idx + 1, name, channels, base, dur)
        for idx, (name, channels, base, dur) in enumerate(ARCHETYPES)
    ]
    archetypes = spark.createDataFrame(
        rows,
        "archetype_idx int, archetype string, "
        "channels array<string>, base_impressions int, duration int",
    )
    grid = (
        date_grid(spark, cfg.start_date, cfg.end_date)
        .withColumn(
            "day_offset",
            F.datediff(F.col("day"), F.lit(cfg.start_date)),
        )
        .crossJoin(archetypes)
        .withColumn(
            # Campaigns span their archetype duration: the day's impressions
            # belong to the window [start + k*duration, ...) it falls in.
            "campaign_start",
            campaign_window_start(F.col("day"), cfg.start_date, F.col("duration")),
        )
    )

    # flash_sale runs ~1 day in 7 (uniform gate per day)
    grid = grid.filter(
        (F.col("archetype") != "flash_sale")
        | (d.u([F.col("day")], "mk_flash_gate") < F.lit(1.0 / 7.0))
    )

    # per-day impression count: clamped normal around the scaled base
    lam = F.col("base_impressions") * F.lit(scale)
    n_imps = F.greatest(
        F.lit(1),
        F.round(lam + d.gauss(["day", "archetype"], "mk_n") * F.sqrt(lam)),
    ).cast("int")
    grid = grid.withColumn("n_impressions", n_imps)

    # --- explode to impressions; build IDs
    imps = (
        grid.withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n_impressions"))))
        .withColumn(
            # campaign_id keyed on the campaign window so it spans multiple days
            "campaign_id",
            campaign_id_column(F.col("campaign_start"), F.col("archetype_idx")),
        )
        .withColumn(
            # impression id stays day-unique (campaign_id now repeats across days)
            "impression_id_ext",
            F.concat(
                F.lit("IMP"),
                F.date_format("day", "yyyyMMdd"),
                fixed_width_decimal(F.col("archetype_idx"), 2, "impression archetype index"),
                fixed_width_decimal(F.col("seq"), 7, "daily impression sequence"),
            ),
        )
    )

    key = [F.col("impression_id_ext")]

    # --- channel uniform over the archetype's channels
    ch_idx = F.floor(d.u(key, "mk_channel") * F.size("channels")).cast("int")
    imps = imps.withColumn("channel", F.element_at("channels", ch_idx + 1)).withColumn(
        "creative_id",
        creative_id_column(d, key, F.col("campaign_id"), F.col("channel")),
    )

    # --- device pick (60/30/10); cost uniform in the channel band times the
    # device multiplier
    device, cost = impression_device_cost(d, key, F.col("channel"), "mk_device", "mk_cost")
    imps = imps.withColumn("device", device).withColumn("_cost_dollars", cost)
    cost_dollars = F.col("_cost_dollars")
    # Costs are fractions of a cent at real CPMs: cost keeps four decimals and
    # cost_cents is rounded at random, so summed spend stays unbiased.
    imps = imps.withColumn("cost_cents", impression_cost_cents(d, key, cost_dollars)).withColumn(
        "cost", F.format_string("%.4f", cost_dollars)
    )

    # --- uniform customer per impression; ~5% CRM-matched (customer_id set).
    # A CRM match needs a customer who is signed up and not churned that day.
    imps = imps.withColumn(
        "customer_pick",
        (d.h64(key, "mk_customer") % F.lit(cfg.customer_count) + 1).cast("long"),
    )
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_pick"), F.col("AdId").alias("customer_ad_id")
    )
    imps = imps.join(customers, "customer_pick", "left")
    imps = imps.withColumn(
        "customer_id",
        F.when(
            (d.u(key, "mk_crm") < F.lit(CRM_MATCH_RATE))
            & (activity_column(lifecycle, F.col("customer_pick"), F.col("day")) > 0),
            F.col("customer_pick").cast("double"),
        ),
    )

    # --- event_ts uniform within the day
    day_ts = F.unix_timestamp(F.col("day").cast("timestamp"))
    event_ts = F.timestamp_seconds(day_ts + F.floor(d.u(key, "mk_ts") * F.lit(86400)))

    out = (
        imps.select(
            event_ts.alias("event_ts"),
            F.concat(F.lit("TRC-MKT-"), F.col("impression_id_ext")).alias("trace_id"),
            "channel",
            "campaign_id",
            "creative_id",
            "customer_ad_id",
            "customer_id",
            "cost_cents",
            # TMDL-bound PascalCase duals: exact mirrors of the snake_case columns
            F.col("customer_id").alias("CustomerId"),
            F.col("cost_cents").alias("CostCents"),
            "impression_id_ext",
            "cost",
            "device",
            F.col("day").alias("event_date"),
        )
        .withColumn(
            "__index_level_0__",
            legacy_index("impression_id_ext"),
        )
        .withColumn(
            # IMP-007: attribution.py sets it on the impressions that make up a
            # converting journey.
            "attribution_journey_id",
            F.lit(None).cast("string"),
        )
    )
    return out.select(*column_names("fact_marketing"))

# --- retail_setup/generation/attribution.py ---
"""Deterministic marketing-attribution fact (IMP-007), Spark-native.

Builds ``ag.fact_marketing_attribution``: one row per in-store SALE receipt or
online order, including attributed, unattributed, declined, and
reconciliation-failed outcomes, using deterministic last-touch attribution
within an inclusive 7-day window.

Conversion is driven by exposure. Every background impression
(``marketing.py``) reaches the customer whose ad ID it carries; an impression
converts with a small per-impression probability weighted by channel
(``vocabulary.CHANNEL_CONVERSION_WEIGHTS``) when that customer makes an
eligible purchase within the 7 days after it. The probability is calibrated
per run so the campaigns return about ``TARGET_ROAS`` times their spend
(realistic retail ROAS), which keeps conversions far below 1% of impressions
at real CPMs. Journeys are therefore made of real impressions: a journey's
touches are indistinguishable from other impressions except for their
``attribution_journey_id``, every campaign family can convert, and each touch
lies inside its own campaign window.

Pipeline:

1. Purchase candidates are built from SALE receipts and online order headers,
   each joined to its final payment (a declined first tender that was retried
   has an approved second payment). A purchase is "reconciled" only when that
   payment is APPROVED and equals the purchase's ``total_cents``; anything else
   is ``PAYMENT_DECLINED`` or ``RECONCILIATION_FAILED``.
2. Candidate impressions (a stable draw ``u`` below ``CANDIDATE_RATE`` x the
   channel weight) are joined to the reached customer's eligible purchases
   within the window; each reaches the customer's next purchase.
3. A purchase's conversion score is the smallest ``u / weight`` of the
   impressions reaching it. Purchases convert in score order until their net
   revenue reaches ``TARGET_ROAS`` x the spend of all impressions; the
   impressions at or under the cut are the journey's touches.
4. Last-touch attribution is DERIVED via ``_last_touch`` (window/rank join
   ordered by touch time then impression ID); ``attributed_revenue_cents`` is
   ``net_subtotal_cents`` for ATTRIBUTED rows and 0 otherwise.
5. Receipts/orders/payments/promotions are enriched with the derived
   ``attribution_journey_id``/``campaign_id``/``impression_id_ext``, and the
   journey's impressions in ``fact_marketing`` carry the journey ID.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from datetime import date


ATTRIBUTION_MODEL = "LAST_TOUCH_7D"
ATTRIBUTION_WINDOW_DAYS = 7
# Campaigns return about this many dollars of net revenue per dollar spent.
TARGET_ROAS = 4.0
# Impressions that may convert at all: a stable draw below this rate times the
# channel's weight relative to the strongest channel. Bounds the conversion
# probability (and the join) well above what TARGET_ROAS needs.
CANDIDATE_RATE = 0.03
# Live stream journeys (with_journey_plan): 1-3 touches, 1 minute to 7 days
# before the purchase, each in the campaign window of its own time.
JOURNEY_MAX_TOUCHES = 3
JOURNEY_TOUCH_MIN_S = 60
FLASH_SALE_DAY_SHARE = 1.0 / 7.0


def _last_touch(touches: DataFrame, purchases: DataFrame) -> DataFrame:
    """Deterministic last-touch join: one output row per input purchase row.

    ``touches`` columns: match_key, touch_ts, impression_id_ext, campaign_id,
    creative_id, channel, customer_ad_id, customer_id.
    ``purchases`` columns: match_key, purchase_ts (any other columns are
    passed through unchanged on the output).

    For each purchase, selects the touch with
    ``touch_ts <= purchase_ts <= touch_ts + ATTRIBUTION_WINDOW_DAYS days``,
    ranked by the greatest ``touch_ts``, ties broken by ``impression_id_ext``
    descending — a real window/rank computation, not a lookup of a
    pre-known "right answer". Purchases with no qualifying touch keep all
    touch columns NULL (left join).
    """
    window_s = ATTRIBUTION_WINDOW_DAYS * 86400
    t = touches.select(
        F.col("match_key").alias("_t_match_key"),
        F.col("touch_ts"),
        F.col("impression_id_ext"),
        F.col("campaign_id"),
        F.col("creative_id"),
        F.col("channel"),
        F.col("customer_ad_id"),
        F.col("customer_id").alias("touch_customer_id"),
    )
    p_cols = purchases.columns
    cond = (
        (F.col("_t_match_key") == F.col("match_key"))
        & (F.col("touch_ts").cast("long") <= F.col("purchase_ts").cast("long"))
        & (F.col("touch_ts").cast("long") >= F.col("purchase_ts").cast("long") - F.lit(window_s))
    )
    joined = purchases.join(t, cond, "left")
    w = Window.partitionBy("match_key").orderBy(
        F.col("touch_ts").desc_nulls_last(),
        F.col("impression_id_ext").desc_nulls_last(),
    )
    ranked = (
        joined.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "_t_match_key")
        .withColumn(
            "lag_seconds",
            F.when(
                F.col("touch_ts").isNotNull(),
                F.col("purchase_ts").cast("long") - F.col("touch_ts").cast("long"),
            ).cast("long"),
        )
    )
    return ranked.select(
        *p_cols,
        "touch_ts",
        "impression_id_ext",
        "campaign_id",
        "creative_id",
        "channel",
        "customer_ad_id",
        "touch_customer_id",
        "lag_seconds",
    )


def reached_customer(customer_ad_id: Column) -> Column:
    """The customer an impression reaches: the ID inside its ad ID (``AD`` + digits)."""
    return F.substring(customer_ad_id, len(AD_ID_PREFIX) + 1, 32).cast("long")


def conversion_score(d: seeded_draws, impression_id: Column, channel: Column) -> Column:
    """An impression's conversion score: a stable draw over its channel weight.

    Lower converts first; candidates are impressions scoring below
    ``CANDIDATE_RATE / max weight``.
    """
    return d.u([impression_id], "attr_conversion") / channel_conversion_weight(channel)


def candidate_score_limit() -> float:
    return CANDIDATE_RATE / max(CHANNEL_CONVERSION_WEIGHTS.values())


def archetype_weights() -> list[tuple[str, float]]:
    """Each campaign archetype's share of impressions (flash sales run one day in 7)."""
    return [
        (str(index + 1), base * (FLASH_SALE_DAY_SHARE if name == "flash_sale" else 1.0))
        for index, (name, _, base, _) in enumerate(CAMPAIGN_ARCHETYPES)
    ]


def archetype_share(name: str) -> float:
    """An archetype's share of the background impression picks (before flash gating)."""
    total = sum(base for _, _, base, _ in CAMPAIGN_ARCHETYPES)
    return sum(base for archetype, _, base, _ in CAMPAIGN_ARCHETYPES if archetype == name) / total


def mean_impression_cost() -> float:
    """Expected dollars per background impression across archetypes, channels and devices."""
    device = sum(weight * DEVICE_MULTIPLIERS[name] for name, weight in DEVICE_WEIGHTS)
    weights = dict(archetype_weights())
    total = sum(weights.values())
    cost = 0.0
    for index, (_, channels, _, _) in enumerate(CAMPAIGN_ARCHETYPES):
        mean = sum(sum(CHANNEL_COSTS[channel]) / 2 for channel in channels) / len(channels)
        cost += weights[str(index + 1)] / total * mean
    return cost * device


def live_journey_rate(
    impressions_per_day: float, eligible_purchases_per_day: float, net_ticket_dollars: float
) -> float:
    """Share of the live stream's eligible purchases it attributes.

    Sized like history: the attributed net revenue is ``TARGET_ROAS`` times the
    day's impression spend.
    """
    if eligible_purchases_per_day <= 0 or net_ticket_dollars <= 0:
        return 0.0
    spend = impressions_per_day * mean_impression_cost()
    return min(1.0, TARGET_ROAS * spend / (eligible_purchases_per_day * net_ticket_dollars))


def with_journey_plan(purchases: DataFrame, d: seeded_draws, history_start: date) -> DataFrame:
    """Add a live purchase's journey: its ID and 1-3 touches, oldest first.

    ``purchases`` needs ``match_key`` (the purchase's own ID) and
    ``purchase_ts``. Each touch (``journey_touches``: ``seq``, ``touch_ts``,
    ``archetype``, ``campaign_id``, ``channel``) is 1 minute to 7 days before
    the purchase, on an archetype drawn by impression volume (a flash sale only
    on its active days), in the campaign window of its own time, on one of the
    archetype's channels, like a background impression. ``campaign_id`` and
    ``touch_ts`` are the last touch's.
    """
    key = F.col("match_key")
    window_s = ATTRIBUTION_WINDOW_DAYS * 86400
    count = (1 + F.floor(d.u([key], "attr_touches") * JOURNEY_MAX_TOUCHES)).cast("int")
    offsets = F.slice(
        F.array(
            *[
                (
                    F.lit(JOURNEY_TOUCH_MIN_S)
                    + d.u([key], f"attr_touch{index}") * (window_s - JOURNEY_TOUCH_MIN_S)
                ).cast("long")
                for index in range(1, JOURNEY_MAX_TOUCHES + 1)
            ]
        ),
        1,
        count,
    )
    oldest_first = F.reverse(F.array_sort(offsets))
    flash = next(
        str(index + 1)
        for index, (name, *_) in enumerate(CAMPAIGN_ARCHETYPES)
        if name == "flash_sale"
    )
    durations = {str(index + 1): entry[3] for index, entry in enumerate(CAMPAIGN_ARCHETYPES)}
    channels = {str(index + 1): entry[1] for index, entry in enumerate(CAMPAIGN_ARCHETYPES)}

    def touch(offset: Column, index: Column) -> Column:
        ts = F.timestamp_seconds(F.col("purchase_ts").cast("long") - offset)
        day = F.to_date(ts)
        drawn = d.pick_by_weights([key, index], "attr_archetype", archetype_weights())
        archetype = F.when(
            (drawn == flash) & ~(d.u([day], "mk_flash_gate") < F.lit(FLASH_SALE_DAY_SHARE)),
            F.lit("2"),
        ).otherwise(drawn)
        duration = case_when((archetype == name, F.lit(days)) for name, days in durations.items())
        channel = case_when(
            (
                archetype == name,
                d.pick_by_weights(
                    [key, index], f"attr_channel_{name}", [(c, 1.0) for c in options]
                ),
            )
            for name, options in channels.items()
        )
        start = campaign_window_start(day, history_start, duration)
        return F.struct(
            (index + 1).alias("seq"),
            ts.alias("touch_ts"),
            archetype.cast("int").alias("archetype"),
            campaign_id_column(start, archetype.cast("int")).alias("campaign_id"),
            channel.alias("channel"),
        )

    last = F.element_at(F.col("journey_touches"), -1)
    return (
        purchases.withColumn(
            "attribution_journey_id",
            F.concat(F.lit("JRN"), F.hex(F.xxhash64(key, F.lit(f"attr_journey|{d.seed}")))),
        )
        .withColumn("journey_touches", F.transform(oldest_first, touch))
        .withColumn("campaign_id", last["campaign_id"])
        .withColumn("touch_ts", last["touch_ts"])
    )


def generate_attribution(
    spark: SparkSession,
    t: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Derive fact_marketing_attribution and enrich receipts/orders/payments/
    promotions/marketing with the selected attribution mapping.

    Must run after fact_receipts, fact_receipt_lines, fact_online_order_headers,
    fact_payments, fact_promotions and the background fact_marketing all exist
    in ``t`` — wired in engine.py right after marketing.generate_marketing.
    """
    d = seeded_draws(cfg.seed)

    # --- 1. purchase candidates: SALE receipts + online orders, each
    # left-joined to its own payment.
    receipts = t["fact_receipts"].filter(F.col("receipt_type") == "SALE")
    # A purchase's payment is its final attempt: a declined first tender that
    # was retried has an approved second payment (vocabulary decline model).
    final = Window.partitionBy(F.coalesce(F.col("receipt_id_ext"), F.col("order_id_ext"))).orderBy(
        (F.col("status") == "APPROVED").desc(), F.col("transaction_id").desc()
    )
    payments = (
        t["fact_payments"]
        .withColumn("_attempt_rank", F.row_number().over(final))
        .filter(F.col("_attempt_rank") == 1)
        .drop("_attempt_rank")
    )
    store_pay = payments.filter(F.col("receipt_id_ext").isNotNull()).select(
        F.col("receipt_id_ext"),
        F.col("status").alias("payment_status"),
        F.col("amount_cents").alias("payment_cents"),
    )
    online_pay = payments.filter(F.col("order_id_ext").isNotNull()).select(
        F.col("order_id_ext"),
        F.col("status").alias("payment_status"),
        F.col("amount_cents").alias("payment_cents"),
    )

    store_purchases = receipts.join(store_pay, "receipt_id_ext", "left").select(
        F.col("receipt_id_ext").alias("match_key"),
        F.lit("STORE").alias("purchase_type"),
        F.col("receipt_id_ext"),
        F.lit(None).cast("string").alias("order_id_ext"),
        F.col("event_ts").alias("purchase_ts"),
        F.col("event_date"),
        F.col("store_id"),
        F.col("customer_id"),
        F.col("gross_subtotal_cents"),
        F.col("discount_cents"),
        F.col("subtotal_cents").alias("net_subtotal_cents"),
        F.col("tax_cents"),
        F.col("total_cents"),
        F.col("payment_status"),
        F.coalesce(F.col("payment_cents"), F.lit(0).cast("long")).alias("payment_cents"),
    )

    online_purchases = (
        t["fact_online_order_headers"]
        .join(online_pay, "order_id_ext", "left")
        .select(
            F.col("order_id_ext").alias("match_key"),
            F.lit("ONLINE").alias("purchase_type"),
            F.lit(None).cast("string").alias("receipt_id_ext"),
            F.col("order_id_ext"),
            F.col("event_ts").alias("purchase_ts"),
            F.col("event_date"),
            F.lit(None).cast("long").alias("store_id"),
            F.col("customer_id"),
            F.col("gross_subtotal_cents"),
            F.col("discount_cents"),
            F.col("subtotal_cents").alias("net_subtotal_cents"),
            F.col("tax_cents"),
            F.col("total_cents"),
            F.col("payment_status"),
            F.coalesce(F.col("payment_cents"), F.lit(0).cast("long")).alias("payment_cents"),
        )
    )

    purchases_all = store_purchases.unionByName(online_purchases)

    # --- 2. reconciliation status: an APPROVED payment whose amount exactly
    # equals the purchase total is the only path to attribution eligibility.
    reconciled = (F.col("payment_status") == "APPROVED") & (
        F.col("payment_cents") == F.col("total_cents")
    )
    purchases_all = purchases_all.withColumn(
        "base_status",
        F.when(F.col("payment_status").isNull(), F.lit("RECONCILIATION_FAILED"))
        .when(F.col("payment_status") == "DECLINED", F.lit("PAYMENT_DECLINED"))
        .when(~reconciled, F.lit("RECONCILIATION_FAILED"))
        .otherwise(F.lit("ELIGIBLE")),
    )

    # --- 3. exposure-driven conversion: candidate impressions reach their
    # customer's next eligible purchase within the window.
    earliest_purchase_ts = F.lit(cfg.start_date).cast("timestamp")
    eligible = purchases_all.filter(
        (F.col("base_status") == "ELIGIBLE")
        & F.col("customer_id").isNotNull()
        & (F.col("purchase_ts") >= earliest_purchase_ts)
    )
    impressions = t["fact_marketing"]
    spend_cents = impressions.agg(F.sum("cost_cents")).collect()[0][0] or 0
    candidates = (
        impressions.withColumn(
            "_score", conversion_score(d, F.col("impression_id_ext"), F.col("channel"))
        )
        .filter(F.col("_score") < F.lit(candidate_score_limit()))
        .select(
            "impression_id_ext",
            F.col("event_ts").alias("touch_ts"),
            "campaign_id",
            "creative_id",
            "channel",
            "customer_ad_id",
            reached_customer(F.col("customer_ad_id")).alias("_reached"),
            "_score",
        )
    )
    window_s = ATTRIBUTION_WINDOW_DAYS * 86400
    reach = candidates.join(
        eligible.select("match_key", "customer_id", "purchase_ts", "net_subtotal_cents"),
        (F.col("_reached") == F.col("customer_id"))
        & (F.col("purchase_ts").cast("long") > F.col("touch_ts").cast("long"))
        & (F.col("purchase_ts").cast("long") <= F.col("touch_ts").cast("long") + window_s),
    ).drop("_reached")
    next_purchase = Window.partitionBy("impression_id_ext").orderBy("purchase_ts", "match_key")
    reach = (
        reach.withColumn("_n", F.row_number().over(next_purchase))
        .filter(F.col("_n") == 1)
        .drop("_n")
    )
    # --- 4. convert purchases in score order until revenue reaches the target.
    scored = reach.groupBy("match_key").agg(
        F.min("_score").alias("_purchase_score"),
        F.first("net_subtotal_cents").alias("_revenue"),
    )
    by_score = Window.orderBy("_purchase_score", "match_key").rowsBetween(
        Window.unboundedPreceding, -1
    )
    target_cents = TARGET_ROAS * spend_cents
    cut = (
        scored.withColumn("_before", F.coalesce(F.sum("_revenue").over(by_score), F.lit(0)))
        .filter(F.col("_before") < F.lit(target_cents))
        .agg(F.max("_purchase_score"))
        .collect()[0][0]
    )
    # No purchase converts when nothing was reached (or nothing was spent).
    touches_all = reach.filter(F.col("_score") <= F.lit(-1.0 if cut is None else float(cut)))
    selected = eligible.join(touches_all.select("match_key").distinct(), "match_key").withColumn(
        "attribution_journey_id",
        F.concat(
            F.lit("JRN"),
            F.hex(F.xxhash64(F.col("match_key"), F.lit(f"attr_journey|{d.seed}"))),
        ),
    )
    touches = touches_all.join(selected.select("match_key", "attribution_journey_id"), "match_key")
    journey_of = touches.select("impression_id_ext", "attribution_journey_id")
    fact_marketing = (
        impressions.drop("attribution_journey_id")
        .join(journey_of, "impression_id_ext", "left")
        .select(*column_names("fact_marketing"))
    )

    # --- 5. derive last-touch attribution via a real window/rank join over
    # each journey's own touches.
    best_touch = _last_touch(
        touches.select(
            F.col("attribution_journey_id").alias("match_key"),
            "touch_ts",
            "impression_id_ext",
            "campaign_id",
            "creative_id",
            "channel",
            "customer_ad_id",
            "customer_id",
        ),
        selected.select(F.col("attribution_journey_id").alias("match_key"), "purchase_ts"),
    ).select(
        F.col("match_key").alias("attribution_journey_id"),
        "touch_ts",
        "impression_id_ext",
        "campaign_id",
        "creative_id",
        "channel",
        "customer_ad_id",
        "lag_seconds",
    )

    attributed = selected.join(best_touch, "attribution_journey_id").withColumn(
        "attribution_status", F.lit("ATTRIBUTED")
    )

    non_attributed = (
        purchases_all.join(selected.select("match_key"), "match_key", "left_anti")
        .withColumn(
            "attribution_status",
            F.when(F.col("base_status") == "ELIGIBLE", F.lit("UNATTRIBUTED_NO_JOURNEY")).otherwise(
                F.col("base_status")
            ),
        )
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("touch_ts", F.lit(None).cast("timestamp"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("creative_id", F.lit(None).cast("string"))
        .withColumn("channel", F.lit(None).cast("string"))
        .withColumn("customer_ad_id", F.lit(None).cast("string"))
        .withColumn("lag_seconds", F.lit(None).cast("long"))
    )

    combined = attributed.unionByName(non_attributed)
    fact_marketing_attribution = (
        combined.withColumn(
            "attribution_id",
            F.concat(
                F.lit("ATTR-"),
                F.col("purchase_type"),
                F.lit("-"),
                F.col("match_key"),
            ),
        )
        .withColumn("attribution_model", F.lit(ATTRIBUTION_MODEL))
        .withColumn("attribution_window_days", F.lit(ATTRIBUTION_WINDOW_DAYS).cast("int"))
        .withColumn(
            "attributed_revenue_cents",
            F.when(
                F.col("attribution_status") == "ATTRIBUTED", F.col("net_subtotal_cents")
            ).otherwise(F.lit(0).cast("long")),
        )
        .select(*column_names("fact_marketing_attribution"))
    )

    # --- 6. enrich receipts/orders/payments/promotions with the DERIVED
    # (not pre-assigned) journey/campaign/impression for attributed purchases.
    enrich_map = attributed.select(
        "match_key", "purchase_type", "attribution_journey_id", "campaign_id", "impression_id_ext"
    )
    store_enrich = enrich_map.filter(F.col("purchase_type") == "STORE").select(
        F.col("match_key").alias("receipt_id_ext"),
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    )
    online_enrich = enrich_map.filter(F.col("purchase_type") == "ONLINE").select(
        F.col("match_key").alias("order_id_ext"),
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    )
    pay_enrich = enrich_map.select(F.col("match_key").alias("pay_key"), "attribution_journey_id")

    fact_receipts = (
        t["fact_receipts"]
        .drop("attribution_journey_id", "campaign_id", "impression_id_ext")
        .join(store_enrich, "receipt_id_ext", "left")
        .select(*column_names("fact_receipts"))
    )
    fact_online_order_headers = (
        t["fact_online_order_headers"]
        .drop("attribution_journey_id", "campaign_id", "impression_id_ext")
        .join(online_enrich, "order_id_ext", "left")
        .select(*column_names("fact_online_order_headers"))
    )
    fact_payments = (
        t["fact_payments"]
        .drop("attribution_journey_id")
        .withColumn("pay_key", F.coalesce(F.col("receipt_id_ext"), F.col("order_id_ext")))
        .join(pay_enrich, "pay_key", "left")
        .drop("pay_key")
        .select(*column_names("fact_payments"))
    )
    fact_promotions = (
        t["fact_promotions"]
        .drop("attribution_journey_id")
        .join(
            store_enrich.select("receipt_id_ext", "attribution_journey_id"),
            "receipt_id_ext",
            "left",
        )
        .select(*column_names("fact_promotions"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_online_order_headers": fact_online_order_headers,
        "fact_payments": fact_payments,
        "fact_promotions": fact_promotions,
        "fact_marketing": fact_marketing,
        "fact_marketing_attribution": fact_marketing_attribution,
    }

# --- retail_setup/generation/sensors.py ---
"""BLE pings + customer zone changes, Spark-native.

A visit walks the store type's zones in order (``vocabulary.visit_zone_path``,
the path model the live stream uses): the entrance, 1-3 department zones,
checkout when it buys, and the exit, with a dwell per zone. Beacons detect the
phones of ``BLE_DETECTED_SHARE`` of visits, buyers and non-buyers alike:

- buyers are SALE receipts; the visit is laid out so the purchase falls in the
  middle of its checkout dwell, and 30% of them carry the customer's BLE ID
  (``dim_customers.BLEId``) with ``customer_id``, the rest an anonymous ID;
- non-buyers are the hour's visitors who did not buy (``traffic``), with
  anonymous IDs and a start time inside the hour.

Each zone gets 2-5 pings spread over its dwell, so pings and zone changes run
entrance -> departments -> checkout -> exit. Times are clamped to the store's
open hours and the requested history (monotone, so the order holds). All
per-visit sequencing is row-local array work, so the ping stream is never
shuffled.

Schema notes (TMDL arbiter results):
- fact_ble_pings: BOTH customer_id (double, snake) AND CustomerId (double,
  TMDL-bound) with mirrored values; __index_level_0__ (long) = hash-derived
  via legacy_index(trace_id).
- fact_customer_zone_changes: BOTH snake_case (store_id, customer_ble_id,
  from_zone, to_zone) AND PascalCase (StoreID, CustomerBLEId, FromZone, ToZone)
  with mirrored values; __index_level_0__ = legacy_index(trace_id).
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


_VISIT_COLUMNS = [
    "visit_key",
    "store_id",
    "converts",
    "customer_ble_id",
    "visit_customer_id",
    "_anchor_epoch",
    "_day",
]


def _buyer_visits(
    receipts: DataFrame, dims: dict[str, DataFrame], d: seeded_draws, cfg
) -> DataFrame:
    """Detected buying visits: SALE receipts, anchored on the purchase time."""
    visits = receipts.filter(
        (F.col("receipt_type") == "SALE")
        & F.col("event_date").between(F.lit(cfg.start_date), F.lit(cfg.end_date))
        & (d.u(["receipt_id_ext"], "ble-detected") < F.lit(BLE_DETECTED_SHARE))
    )
    customers = dims["dim_customers"].select(
        F.col("ID").alias("_cust_key"), F.col("BLEId").alias("_ble_from_dim")
    )
    visits = visits.withColumn("_cust_key", F.col("customer_id").cast("long")).join(
        customers, "_cust_key", "left"
    )
    known = (d.u(["receipt_id_ext"], "known") < F.lit(BLE_KNOWN_SHARE)) & F.col(
        "_ble_from_dim"
    ).isNotNull()
    return visits.select(
        F.col("receipt_id_ext").alias("visit_key"),
        "store_id",
        F.lit(True).alias("converts"),
        F.when(known, F.col("_ble_from_dim"))
        .otherwise(anonymous_ble_id_column(F.col("store_id"), d.h64(["receipt_id_ext"], "anon_id")))
        .alias("customer_ble_id"),
        F.when(known, F.col("customer_id").cast("double"))
        .cast("double")
        .alias("visit_customer_id"),
        F.unix_timestamp("event_ts").alias("_anchor_epoch"),
        F.col("event_date").alias("_day"),
    )


def _browser_visits(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
    d: seeded_draws,
) -> DataFrame:
    """Detected non-buying visits: the hour's visitors who did not buy."""
    grid = generate_hourly_visits(
        spark, dims["dim_stores"], profile, cfg, dims.get("dim_geographies")
    )
    hour = [F.col("store_id"), F.col("day"), F.col("hour")]
    browsers = F.greatest(F.lit(0), F.col("visitors") - F.col("n_receipts"))
    detected = F.floor(browsers * F.lit(BLE_DETECTED_SHARE) + d.u(hour, "ble-browsers")).cast("int")
    visits = (
        grid.withColumn("_detected", detected)
        .filter(F.col("_detected") > 0)
        .withColumn("_seq", F.explode(F.sequence(F.lit(1), F.col("_detected"))))
        .withColumn(
            "visit_key",
            F.concat(
                F.lit("NB"),
                F.col("store_id").cast("string"),
                F.lit("-"),
                F.date_format("hour_ts", "yyyyMMddHH"),
                F.lit("-"),
                F.col("_seq").cast("string"),
            ),
        )
    )
    return visits.select(
        "visit_key",
        "store_id",
        F.lit(False).alias("converts"),
        anonymous_ble_id_column(F.col("store_id"), d.h64(["visit_key"], "anon_id")).alias(
            "customer_ble_id"
        ),
        F.lit(None).cast("double").alias("visit_customer_id"),
        (
            F.unix_timestamp("hour_ts") + F.floor(d.u(["visit_key"], "visit-start") * F.lit(3600))
        ).alias("_anchor_epoch"),
        F.col("day").alias("_day"),
    )


def generate_ble(
    spark: SparkSession,
    receipts: DataFrame,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    profile: StoreTypeProfile | None = None,
) -> tuple[DataFrame, DataFrame]:
    """Generate BLE ping and zone-change facts of detected visits.

    Zones are the store type's (``profile.zones``, upper-cased).

    Returns:
        (pings_df, zone_changes_df)
    """
    d = seeded_draws(cfg.seed)
    if profile is None:
        profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile
    layout = ble_layout(profile)

    visits = (
        _buyer_visits(receipts, dims, d, cfg)
        .select(*_VISIT_COLUMNS)
        .unionByName(_browser_visits(spark, dims, profile, cfg, d).select(*_VISIT_COLUMNS))
    )
    key = [F.col("visit_key")]
    visits = (
        visits.withColumn("zones", visit_zone_path(d, key, F.col("converts"), layout))
        .withColumn("dwell", visit_zone_dwell(d, key, F.col("zones"), layout))
        .withColumn("starts", visit_zone_starts(F.col("dwell")))
        .withColumn("_ping_counts", visit_ping_counts(d, key, F.col("zones")))
    )
    # A buyer pays in the middle of its checkout dwell (as in the live stream);
    # a browser's visit starts at its anchor.
    last = checkout_index(F.col("zones"), layout)
    pay_offset = F.element_at("starts", last) + F.floor(F.element_at("dwell", last) / 2)
    visits = visits.withColumn(
        "_start_epoch",
        F.when(F.col("converts"), F.col("_anchor_epoch") - pay_offset).otherwise(
            F.col("_anchor_epoch")
        ),
    )

    store_hours = dims["dim_stores"].select(F.col("ID").alias("store_id"), "operating_hours")
    visits = visits.join(store_hours, "store_id")
    is_24h = F.col("operating_hours") == "24h"
    hour_parts = F.split(F.col("operating_hours"), "-")
    open_hour = F.when(is_24h, 0).otherwise(hour_parts.getItem(0).cast("int"))
    close_hour = F.when(is_24h, 24).otherwise(hour_parts.getItem(1).cast("int"))
    day_epoch = F.unix_timestamp(F.col("_day").cast("timestamp"))
    history_start = F.unix_timestamp(
        F.to_timestamp(F.lit(f"{cfg.start_date.isoformat()} 00:00:00"))
    )
    history_end = F.unix_timestamp(F.to_timestamp(F.lit(f"{cfg.end_date.isoformat()} 23:59:59")))
    visits = visits.withColumn(
        "_lo", F.greatest(day_epoch + F.coalesce(open_hour, F.lit(6)) * 3600, history_start)
    ).withColumn(
        "_hi", F.least(day_epoch + F.coalesce(close_hour, F.lit(22)) * 3600 - 1, history_end)
    )

    def clamp(epoch: Column) -> Column:
        return F.timestamp_seconds(F.greatest(F.col("_lo"), F.least(epoch, F.col("_hi"))))

    # --- pings: each zone's pings spread over its dwell, in path order
    pings = visits.select(
        "visit_key",
        "store_id",
        "customer_ble_id",
        "visit_customer_id",
        "_start_epoch",
        "_lo",
        "_hi",
        "dwell",
        "starts",
        "_ping_counts",
        F.posexplode("zones").alias("_zone_index", "zone"),
    )
    index = F.col("_zone_index") + 1
    n_pings = F.element_at("_ping_counts", index)
    pings = pings.withColumn("n_pings", n_pings).withColumn(
        "_prior_pings",
        F.aggregate(
            F.slice("_ping_counts", 1, F.col("_zone_index")), F.lit(0), lambda acc, n: acc + n
        ),
    )
    pings = pings.select("*", F.explode(F.sequence(F.lit(1), F.col("n_pings"))).alias("ping_seq"))
    offset = F.element_at("starts", index) + F.floor(
        F.element_at("dwell", index) * F.col("ping_seq") / (F.col("n_pings") + 1)
    )
    ping_key = [
        F.col("visit_key"),
        F.col("_zone_index").cast("string"),
        F.col("ping_seq").cast("string"),
    ]
    pings = (
        pings.withColumn("ping_ts", clamp(F.col("_start_epoch") + offset))
        .withColumn(
            "trace_id",
            F.concat(
                F.lit("TRC-BLE-"),
                F.col("visit_key"),
                F.lit("-"),
                (F.col("_prior_pings") + F.col("ping_seq")).cast("string"),
            ),
        )
        .withColumn(
            "rssi",
            (F.lit(RSSI_FLOOR) + F.floor(d.u(ping_key, "rssi") * F.lit(RSSI_SPAN))).cast("long"),
        )
    )
    pings_df = (
        pings.select(
            F.col("ping_ts").alias("event_ts"),
            "trace_id",
            F.col("store_id").cast("long").alias("store_id"),
            beacon_id_column(F.col("store_id"), F.col("zone")).alias("beacon_id"),
            "customer_ble_id",
            F.col("visit_customer_id").alias("customer_id"),
            # TMDL-bound PascalCase mirror
            F.col("visit_customer_id").alias("CustomerId"),
            "rssi",
            "zone",
            F.to_date("ping_ts").alias("event_date"),
        )
        .withColumn("__index_level_0__", legacy_index("trace_id"))
        .select(*column_names("fact_ble_pings"))
    )

    # --- zone changes: entering each zone after the first, in path order
    changes = visits.select(
        "visit_key",
        "store_id",
        "customer_ble_id",
        "_start_epoch",
        "_lo",
        "_hi",
        "zones",
        "starts",
        F.explode(F.sequence(F.lit(2), F.greatest(F.lit(2), F.size("zones")))).alias("_to"),
    ).filter(F.col("_to") <= F.size("zones"))
    changes = (
        changes.withColumn(
            "change_ts", clamp(F.col("_start_epoch") + F.element_at("starts", F.col("_to")))
        )
        .withColumn("from_zone", F.element_at("zones", F.col("_to") - 1))
        .withColumn("to_zone", F.element_at("zones", F.col("_to")))
    )
    zc_df = (
        changes.select(
            F.col("change_ts").alias("event_ts"),
            F.concat(
                F.lit("TRC-ZC-"),
                F.col("visit_key"),
                F.lit("-"),
                (F.col("_to") - 1).cast("string"),
            ).alias("trace_id"),
            # snake_case (extras)
            F.col("store_id").cast("long").alias("store_id"),
            "customer_ble_id",
            "from_zone",
            "to_zone",
            # TMDL-bound PascalCase mirrors
            F.col("store_id").cast("long").alias("StoreID"),
            F.col("customer_ble_id").alias("CustomerBLEId"),
            F.col("from_zone").alias("FromZone"),
            F.col("to_zone").alias("ToZone"),
            F.to_date("change_ts").alias("event_date"),
        )
        .withColumn("__index_level_0__", legacy_index("trace_id"))
        .select(*column_names("fact_customer_zone_changes"))
    )
    return pings_df, zc_df

# --- retail_setup/generation/inventory_balances.py ---
"""Balance + stockout helpers for the inventory chain (Plan 2b Task 9).

Split out of ``inventory.py`` per the plan's ~400-line guidance. Covers
stages 7-8: day-0 INITIAL seed txns plus the running-balance window, and the
balance-crossing stockout extraction. Shared draw/column primitives used by
both modules live here to keep the import direction one-way
(``inventory`` -> ``inventory_balances``).
"""

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Canonical column layout for raw (pre-balance) inventory txn streams.
TXN_COLS = [
    "node_id",
    "product_id",
    "quantity",
    "txn_type",
    "source",
    "event_ts",
    "event_date",
    "trace_id",
]


def draw_int(u: Column, lo: int, hi: int) -> Column:
    """Uniform integer draw in [lo, hi] from a [0,1) uniform column."""
    return (F.lit(lo) + F.floor(u * F.lit(hi - lo + 1))).cast("long")


# ---------------------------------------------------------------------------
# Stage 7: initial-stock seeds + running balances
# ---------------------------------------------------------------------------


def with_balances(
    txns: DataFrame,
    lo: int,
    hi: int,
    tag: str,
    d: seeded_draws,
    cfg: GenerationConfig,
    *,
    keys: DataFrame | None = None,
) -> DataFrame:
    """Fold a day-0 INITIAL seed txn per (node, product) into the stream and
    compute the running balance ordered by (event_ts, trace_id).

    Opening stock is a direct seeded draw in ``[lo, hi]``. It deliberately
    ignores later transactions: extending the requested history must not
    rewrite day-one stock by backsolving from future demand."""
    keys = txns.select("node_id", "product_id").distinct() if keys is None else keys
    seeds = (
        keys.withColumn(
            "quantity",
            draw_int(
                d.u(["node_id", "product_id"], f"seed-stock-{tag}"),
                lo,
                hi,
            ),
        )
        .withColumn("txn_type", F.lit("INITIAL"))
        .withColumn("source", F.lit("SEED"))
        # String-built timestamp (session-TZ semantics) to match every
        # other event_ts in the chain; a naive Python datetime literal
        # would shift with the driver's local timezone.
        .withColumn("event_ts", F.to_timestamp(F.lit(f"{cfg.start_date.isoformat()} 00:00:00")))
        .withColumn("event_date", F.lit(cfg.start_date).cast("date"))
        .withColumn(
            "trace_id",
            F.concat(
                F.lit(f"TRC-INIT-{tag}-"),
                F.col("node_id").cast("string"),
                F.lit("-"),
                F.col("product_id").cast("string"),
            ),
        )
        .select(*TXN_COLS)
    )
    run_w = (
        Window.partitionBy("node_id", "product_id")
        .orderBy("event_ts", "trace_id")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    return txns.unionByName(seeds).withColumn("balance", F.sum("quantity").over(run_w).cast("long"))


# ---------------------------------------------------------------------------
# Stage 8: stockouts
# ---------------------------------------------------------------------------


def stockouts(balanced: DataFrame, tag: str, node_as: str) -> DataFrame:
    """Balance crossings to <=0 (previous balance > 0); deduped to one per
    (node, product, day). ``node_as`` is 'StoreID' or 'DCID' — the other
    contract column stays NULL (double, per the TMDL contract)."""
    order_w = Window.partitionBy("node_id", "product_id").orderBy("event_ts", "trace_id")
    day_w = Window.partitionBy("node_id", "product_id", "event_date").orderBy(
        "event_ts", "trace_id"
    )
    other = "DCID" if node_as == "StoreID" else "StoreID"
    return (
        balanced.withColumn("_prev", F.lag("balance").over(order_w))
        .filter((F.col("balance") <= 0) & (F.col("_prev") > 0))
        .withColumn("_dup", F.row_number().over(day_w))
        .filter(F.col("_dup") == 1)
        .select(
            "event_ts",
            F.concat(
                F.lit(f"TRC-SO-{tag}-"),
                F.col("node_id").cast("string"),
                F.lit("-"),
                F.col("product_id").cast("string"),
                F.lit("-"),
                F.date_format("event_date", "yyyyMMdd"),
            ).alias("trace_id"),
            F.col("node_id").cast("double").alias(node_as),
            F.lit(None).cast("double").alias(other),
            F.col("product_id").alias("ProductID"),
            F.abs("quantity").cast("long").alias("LastKnownQuantity"),
            "event_date",
        )
    )

# --- retail_setup/generation/inventory.py ---
"""Inventory/logistics ledgers derived from the causal fulfillment plan.

Store stock, reorders and delivery times come from inventory_policy, never
from a second independent draw. Each store's reorders of one day leave on its
next delivery slot as one load (``network.plan_routes``); the load is split
into truck-capacity shipments (``_split_reorders_by_capacity``), and each
shipment is put on a truck that is free for the whole trip: the store's route
truck first, then the DC's other trucks, then the shared pool. Trucks carry one
shipment at a time. Dwell follows the load size, the dock, the arrival hour
and lateness (``network.unload_minutes``, shared with the live stream).

Each load carries its planned and actual arrival (``incidents.plan_delivery``):
truck moves before arrival show the plan, moves from arrival on show the
actual arrival with ``delay_minutes`` and ``delay_reason``. The DC ledger comes
from the DC simulation (``dc_supply``): opening stock, supplier receipts
(short during supplier incidents), ship-from-DC online lines and expedites,
and the truck loads. ``fact_logistics_incidents`` and its store bridge come
from the same incident plan. Truck moves retain the six-stage schedule
(including future scheduled legs); stock-changing ledgers contain only events
inside the requested history.
"""

from datetime import datetime, timedelta, timezone
from functools import partial
from typing import Any, Iterator

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def _with_index(df: DataFrame, table: str) -> DataFrame:
    """Add the TMDL-bound __index_level_0__ column and apply the contract order."""
    return df.withColumn("__index_level_0__", legacy_index("trace_id")).select(*column_names(table))


# ---------------------------------------------------------------------------
# Stage 2: reorders
# ---------------------------------------------------------------------------


def _store_dc_map(dims: dict[str, DataFrame]) -> DataFrame:
    """Route each store to its nearest DC by geography (see ``network.store_dc_map``)."""
    return store_dc_map(
        dims["dim_stores"], dims["dim_geographies"], dims["dim_distribution_centers"]
    )


def _reorders(events: DataFrame, store_dc: DataFrame) -> DataFrame:
    return (
        events.filter(F.col("kind") == "REORDER")
        .select(
            "store_id",
            "product_id",
            "event_ts",
            "event_date",
            "trace_id",
            F.col("balance").alias("current_quantity"),
            F.col("quantity").alias("reorder_quantity"),
            "reorder_point",
            "priority",
        )
        .join(store_dc, "store_id")
    )


def _split_reorders_by_capacity(
    reorders: DataFrame,
    capacity: int,
    load: tuple[str, ...] = ("store_id", "event_date"),
) -> DataFrame:
    """Pack each load's reorder units into capacity-bounded legs, splitting rows as needed."""
    if capacity < 1:
        raise ValueError("capacity must be positive")
    ordering = (
        Window.partitionBy(*load)
        .orderBy("product_id")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    capacity_col = F.lit(capacity).cast("long")
    return (
        reorders.withColumn("_end_unit", F.sum("reorder_quantity").over(ordering))
        .withColumn("_start_unit", F.col("_end_unit") - F.col("reorder_quantity"))
        .withColumn(
            "_first_leg",
            F.floor(F.col("_start_unit") / capacity_col).cast("long"),
        )
        .withColumn(
            "_last_leg",
            F.floor((F.col("_end_unit") - 1) / capacity_col).cast("long"),
        )
        .withColumn(
            "leg",
            F.explode(F.sequence(F.col("_first_leg"), F.col("_last_leg"))),
        )
        .withColumn(
            "reorder_quantity",
            (
                F.least(F.col("_end_unit"), (F.col("leg") + 1) * capacity_col)
                - F.greatest(F.col("_start_unit"), F.col("leg") * capacity_col)
            ).cast("long"),
        )
        .drop("_end_unit", "_start_unit", "_first_leg", "_last_leg")
    )


# ---------------------------------------------------------------------------
# Stage 3: consolidated shipments + truck assignment
# ---------------------------------------------------------------------------

_LOAD_TIMES = ["scheduled_ts", "loading_ts", "depart_ts", "eta", "unloading_ts", "planned_eta"]
_LOAD_DELAY = ["delay_minutes", "delay_reason"]


def delivery_routes(dims: dict[str, DataFrame], cfg: GenerationConfig) -> dict[int, DeliveryRoute]:
    """Each store's DC, route, local delivery slot and lane (shared with fulfillment)."""
    return store_routes(
        cfg.seed, dims["dim_stores"], dims["dim_geographies"], dims["dim_distribution_centers"]
    )


def _fleets(dims: dict[str, DataFrame]) -> tuple[dict[int, tuple[int, ...]], tuple[int, ...]]:
    """(DC fleets, shared pool) from ``dim_trucks``."""
    trucks = [(int(r.ID), r.DCID) for r in dims["dim_trucks"].select("ID", "DCID").collect()]
    dc_ids = [int(r.ID) for r in dims["dim_distribution_centers"].select("ID").collect()]
    pool = tuple(sorted(truck for truck, dc in trucks if dc is None))
    return dc_fleets(trucks, dc_ids), pool


def _shipment_lines(events: DataFrame, capacity: int) -> DataFrame:
    """The policy's shipped lines packed into capacity-bounded shipments.

    Every SKU on a store's load shares its dispatch and timing (``dc_supply``),
    so the lines of one load form one shipment, split into legs of at most
    ``capacity`` units in product order. ``shipment_id`` is the load ID plus
    the 3-digit leg.
    """
    planned = events.filter(F.col("kind") == "SHIPMENT").select(
        "store_id",
        "product_id",
        "event_date",
        F.col("quantity").alias("reorder_quantity"),
        F.col("source").alias("load_id"),
        *_LOAD_TIMES,
        *_LOAD_DELAY,
    )
    return (
        _split_reorders_by_capacity(planned, capacity, ("load_id",))
        .withColumn(
            "shipment_id",
            F.concat(
                F.col("load_id"),
                F.lit("-"),
                fixed_width_decimal(F.col("leg"), 3, "shipment leg"),
            ),
        )
        .withColumnRenamed("reorder_quantity", "quantity")
    )


def _assign_partition(
    batches: Iterator[Any],
    *,
    seed: int,
    fleets: dict[int, tuple[int, ...]],
    pool: tuple[int, ...],
) -> Iterator[Any]:
    """Put shipments, in loading order, on trucks that are free for the whole trip.

    A trip runs from loading until the truck is back at its DC (the dwell from
    ``network.unload_minutes`` plus the lane home). The first leg of a load
    takes the store's route truck, which the route plan keeps free; other legs
    prefer the shared pool. When every candidate is busy the truck that frees
    up first takes it (only possible when overflow outgrows the pool).
    """
    import pyarrow as pa

    free_at: dict[int, int] = {}
    minute_us = 60_000_000
    schema = pa.schema(
        [
            ("shipment_id", pa.string()),
            ("truck_id", pa.int64()),
            ("unload_minutes", pa.int64()),
            ("estimated_unload", pa.int64()),
        ]
    )
    for batch in batches:
        rows = batch.to_pylist()
        out_ids, out_trucks, out_minutes, out_estimates = [], [], [], []
        for row in rows:
            dwell = unload_minutes(
                seed,
                row["store_id"],
                row["units"],
                row["shipment_id"],
                row["arrival_hour"],
                row["delay_minutes"] or 0,
            )
            out_estimates.append(
                estimated_unload_minutes(seed, row["store_id"], row["units"], row["planned_hour"])
            )
            busy_from = row["loading_us"]
            busy_to = row["eta_us"] + (dwell + row["transit_minutes"]) * minute_us
            fleet = fleets.get(row["dc_id"], ())
            if row["leg"] == 0:
                order = [row["route_truck"], *fleet, *pool]
            else:
                order = [*pool, *fleet]
            candidates = list(dict.fromkeys(order)) or [row["route_truck"]]
            truck = next((t for t in candidates if free_at.get(t, 0) <= busy_from), None)
            if truck is None:
                truck = min(candidates, key=lambda t: (free_at.get(t, 0), t))
            free_at[truck] = busy_to
            out_ids.append(row["shipment_id"])
            out_trucks.append(truck)
            out_minutes.append(dwell)
        yield pa.RecordBatch.from_arrays(
            [
                pa.array(out_ids, pa.string()),
                pa.array(out_trucks, pa.int64()),
                pa.array(out_minutes, pa.int64()),
                pa.array(out_estimates, pa.int64()),
            ],
            schema=schema,
        )


def _shipments(
    lines: DataFrame,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    routes: dict[int, DeliveryRoute],
) -> DataFrame:
    """One row per shipment (leg) with its truck, in UTC; local dates kept for filters."""
    fleets, pool = _fleets(dims)
    spark = lines.sparkSession
    plan = spark.createDataFrame(
        [
            (store, route.dc_id, route_truck(route, fleets), route.transit_minutes)
            for store, route in sorted(routes.items())
        ],
        "store_id long, dc_id long, route_truck long, transit_minutes long",
    )
    loads = (
        lines.groupBy("shipment_id", "store_id", "leg", *_LOAD_TIMES, *_LOAD_DELAY)
        .agg(F.sum("quantity").cast("long").alias("units"))
        .join(F.broadcast(plan), "store_id")
        .withColumn("_load_date", F.to_date("loading_ts"))
        .withColumn("_unload_date", F.to_date("unloading_ts"))
        # Store-local arrival hours drive the unload rate.
        .withColumn("arrival_hour", F.hour("eta"))
        .withColumn("planned_hour", F.hour("planned_eta"))
    )
    loads = localize_by_store(loads, dims, _LOAD_TIMES)
    micros = 1_000_000
    worker = partial(_assign_partition, seed=cfg.seed, fleets=fleets, pool=pool)
    assigned = (
        loads.select(
            "shipment_id",
            "store_id",
            "dc_id",
            "leg",
            "route_truck",
            "units",
            "transit_minutes",
            "arrival_hour",
            "planned_hour",
            "delay_minutes",
            (F.unix_timestamp("loading_ts") * micros).alias("loading_us"),
            (F.unix_timestamp("eta") * micros).alias("eta_us"),
        )
        .repartition(1)
        .sortWithinPartitions("loading_us", "shipment_id")
        .mapInArrow(
            worker,
            "shipment_id string, truck_id long, unload_minutes long, estimated_unload long",
        )
    )
    return (
        loads.join(assigned, "shipment_id")
        .withColumn(
            "etd", F.timestamp_seconds(F.unix_timestamp("eta") + F.col("unload_minutes") * 60)
        )
        .withColumn(
            "planned_etd",
            F.timestamp_seconds(F.unix_timestamp("planned_eta") + F.col("estimated_unload") * 60),
        )
        .withColumn("unload_minutes", F.col("unload_minutes").cast("double"))
    )


def _truck_moves(shipments: DataFrame) -> DataFrame:
    stages = F.array(
        *[
            F.struct(F.lit(status).alias("status"), F.col(ts_col).alias("event_ts"))
            for status, ts_col in [
                ("SCHEDULED", "scheduled_ts"),
                ("LOADING", "loading_ts"),
                ("IN_TRANSIT", "depart_ts"),
                ("ARRIVED", "eta"),
                ("UNLOADING", "unloading_ts"),
                ("COMPLETED", "etd"),
            ]
        ]
    )
    done = F.col("status") == "COMPLETED"
    arrived = F.col("status").isin("ARRIVED", "UNLOADING", "COMPLETED")
    return (
        shipments.withColumn("_s", F.explode(stages))
        .select(
            F.col("_s.event_ts").alias("event_ts"),
            F.col("truck_id"),
            F.col("dc_id"),
            F.col("store_id"),
            F.col("shipment_id"),
            F.col("_s.status").alias("status"),
            # Until the truck arrives a move shows the plan (eta, etd); from
            # arrival on it shows the actual arrival and the delay against plan.
            F.when(arrived, F.col("eta")).otherwise(F.col("planned_eta")).alias("eta"),
            F.when(arrived, F.col("etd")).otherwise(F.col("planned_etd")).alias("etd"),
            F.col("unload_minutes"),
            F.col("planned_eta").alias("planned_arrival_time"),
            F.when(arrived, F.col("delay_minutes"))
            .otherwise(F.lit(0))
            .cast("double")
            .alias("delay_minutes"),
            F.when(arrived, F.col("delay_reason")).alias("delay_reason"),
            F.col("units").cast("long").alias("load_units"),
        )
        .withColumn("departure_time", F.when(done, F.col("etd")).cast("timestamp"))
        .withColumn("actual_unload_duration", F.when(done, F.col("unload_minutes")).cast("double"))
        .withColumn(
            "trace_id",
            F.concat(F.lit("TRC-TM-"), F.col("shipment_id"), F.lit("-"), F.col("status")),
        )
        .withColumn("event_date", F.to_date("event_ts"))
    )


def _truck_inventory(shipments: DataFrame, lines: DataFrame, end_date: Any) -> DataFrame:
    """LOAD at the DC and UNLOAD at the store per shipment and product, inside history.

    History ends at ``end_date`` in store-local time, as the policy's ledger does.
    """
    timed = lines.select("shipment_id", "product_id", "quantity").join(
        shipments.select(
            "shipment_id",
            "truck_id",
            "dc_id",
            "store_id",
            "loading_ts",
            "unloading_ts",
            "_load_date",
            "_unload_date",
        ),
        "shipment_id",
    )
    actions = F.array(
        F.struct(
            F.lit("LOAD").alias("action"),
            F.col("loading_ts").alias("event_ts"),
            F.col("_load_date").alias("local_date"),
            F.col("dc_id").alias("location_id"),
            F.lit("DC").alias("location_type"),
        ),
        F.struct(
            F.lit("UNLOAD").alias("action"),
            F.col("unloading_ts").alias("event_ts"),
            F.col("_unload_date").alias("local_date"),
            F.col("store_id").alias("location_id"),
            F.lit("STORE").alias("location_type"),
        ),
    )
    return (
        timed.withColumn("_a", F.explode(actions))
        .filter(F.col("_a.local_date") <= F.lit(end_date))
        .select(
            F.col("_a.event_ts").alias("event_ts"),
            "truck_id",
            "shipment_id",
            "product_id",
            "quantity",
            F.col("_a.action").alias("action"),
            F.col("_a.location_id").alias("location_id"),
            F.col("_a.location_type").alias("location_type"),
            "dc_id",
            "store_id",
        )
        .withColumn(
            "trace_id",
            F.concat(
                F.lit("TRC-TI-"),
                F.col("shipment_id"),
                F.lit("-"),
                F.col("product_id").cast("string"),
                F.lit("-"),
                F.col("action"),
            ),
        )
        .withColumn("event_date", F.to_date("event_ts"))
    )


def _store_inbound_by_leg(events: DataFrame, lines: DataFrame) -> DataFrame:
    """The policy's store receipts, one row per shipment leg that carried the SKU.

    The policy receives a SKU's whole reorder at once; when the load is split,
    the receipt becomes one row per leg at the same time, with the running
    balance before each later leg, so the ledger still conserves stock.
    """
    received = events.filter(F.col("kind") == "INBOUND_SHIPMENT")
    legs = lines.select(
        "store_id",
        "product_id",
        F.col("load_id").alias("source"),
        F.col("shipment_id").alias("_shipment_id"),
        F.col("leg").alias("_leg"),
        F.col("quantity").alias("_quantity"),
    )
    later = (
        Window.partitionBy("store_id", "product_id", "source")
        .orderBy(F.desc("_leg"))
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    return (
        received.join(legs, ["store_id", "product_id", "source"])
        .withColumn(
            "balance", F.col("balance") - F.coalesce(F.sum("_quantity").over(later), F.lit(0))
        )
        .withColumn("quantity", F.col("_quantity"))
        .withColumn("source", F.col("_shipment_id"))
        .withColumn(
            "trace_id",
            F.concat(
                F.lit("TRC-SI-"),
                F.col("_shipment_id"),
                F.lit("-"),
                F.col("product_id").cast("string"),
            ),
        )
        .select(*received.columns)
    )


# ---------------------------------------------------------------------------
# Stages 5-6: DC txns + store inbound mirrors
# ---------------------------------------------------------------------------


# DC simulation rows (``dc_supply``) -> DC ledger transaction types.
_DC_TXN_TYPES = {
    "DC_INITIAL": "INITIAL",
    "DC_RECEIPT": "INBOUND_SHIPMENT",
    "DC_ONLINE": "ONLINE_SHIPMENT",
    "DC_EXPEDITE": "EXPEDITED_RECEIPT",
}


def _dc_txns(truck_inv: DataFrame, events: DataFrame, network: IncidentNetwork) -> DataFrame:
    """The DC ledger: the simulation's own rows (UTC) plus a debit per truck LOAD.

    Opening stock, supplier receipts, online lines and expedites are in the
    DC's local time and are converted with its zone; truck loads are already
    UTC (their destination store's zone).
    """
    spark = events.sparkSession
    zones = spark.createDataFrame(
        [(dc, network.zone_of_dc(dc)) for dc in network.dc_ids()] or [(0, "UTC")],
        "node_id long, _dc_zone string",
    )
    txn_type = case_when(
        (F.col("kind") == kind, F.lit(name)) for kind, name in _DC_TXN_TYPES.items()
    )
    own = (
        events.filter(F.col("kind").isin(*_DC_TXN_TYPES))
        .select(
            F.col("dc_id").alias("node_id"),
            "product_id",
            F.col("quantity").cast("long").alias("quantity"),
            txn_type.alias("txn_type"),
            "source",
            "event_ts",
            "trace_id",
        )
        .join(F.broadcast(zones), "node_id", "left")
        .withColumn(
            "event_ts",
            F.to_utc_timestamp("event_ts", F.coalesce(F.col("_dc_zone"), F.lit("UTC"))),
        )
        .withColumn("event_date", F.to_date("event_ts"))
    )
    outbound = truck_inv.filter(F.col("action") == "LOAD").select(
        F.col("dc_id").alias("node_id"),
        "product_id",
        (-F.col("quantity")).cast("long").alias("quantity"),
        F.lit("OUTBOUND_SHIPMENT").alias("txn_type"),
        F.col("shipment_id").alias("source"),
        "event_ts",
        "event_date",
        F.concat(
            F.lit("TRC-DC-"), F.col("shipment_id"), F.lit("-"), F.col("product_id").cast("string")
        ).alias("trace_id"),
    )
    return own.select(*TXN_COLS).unionByName(outbound.select(*TXN_COLS))


def _dc_balances(dc_raw: DataFrame, dims: dict[str, DataFrame], cfg: GenerationConfig) -> DataFrame:
    """Running DC balances; a DC product the simulation never saw opens from a stable draw."""
    keys = (
        dims["dim_distribution_centers"]
        .select(F.col("ID").cast("long").alias("node_id"))
        .crossJoin(dims["dim_products"].select(F.col("ID").cast("long").alias("product_id")))
    )
    opened = dc_raw.filter(F.col("txn_type") == "INITIAL").select("node_id", "product_id")
    missing = keys.join(opened, ["node_id", "product_id"], "left_anti")
    return with_balances(dc_raw, 500, 2000, "DC", seeded_draws(cfg.seed), cfg, keys=missing)


def _store_inbound(truck_inv: DataFrame) -> DataFrame:
    return truck_inv.filter(F.col("action") == "UNLOAD").select(
        F.col("store_id").alias("node_id"),
        "product_id",
        F.col("quantity").cast("long").alias("quantity"),
        F.lit("INBOUND_SHIPMENT").alias("txn_type"),
        F.col("shipment_id").alias("source"),
        "event_ts",
        "event_date",
        F.concat(
            F.lit("TRC-SI-"), F.col("shipment_id"), F.lit("-"), F.col("product_id").cast("string")
        ).alias("trace_id"),
    )


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------


def generate_inventory_chain(
    spark: SparkSession,
    sales: dict[str, DataFrame],
    rets: dict[str, DataFrame],
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    *,
    fulfillment: FulfillmentResult | None = None,
) -> dict[str, DataFrame]:
    """Generate six ledgers from the plan used to fulfill ``sales``.

    Independent callers must first call ``resolve_fulfillment`` and pass its
    result here, using its sales and return header plan for dependent tables.
    Silently accepting unconstrained sales would recreate the negative-stock bug.
    """
    if fulfillment is None:
        raise ValueError("resolve_fulfillment is required before generate_inventory_chain")
    events = fulfillment.events
    store_dc = _store_dc_map(dims)
    network = incident_network(
        dims["dim_stores"], dims["dim_geographies"], dims["dim_distribution_centers"]
    )
    # The policy runs in store-local time. Outputs are converted to UTC with
    # the store's zone; a shipment's whole lifecycle (and the DC rows that
    # mirror its loads) uses its destination store's zone and is converted
    # before trucks are assigned, so trips never overlap in real time. The
    # DC's own rows use the DC's zone. DC balances are computed after
    # conversion so they follow UTC order.
    reorders = localize_by_store(_reorders(events, store_dc), dims, ["event_ts"])
    lines = _shipment_lines(events, cfg.truck_capacity)
    shipments = _shipments(lines, dims, cfg, delivery_routes(dims, cfg))
    truck_moves = _truck_moves(shipments)
    truck_inv = _truck_inventory(shipments, lines, cfg.end_date)
    dc_raw = _dc_txns(truck_inv, events, network)
    store_bal = localize_by_store(
        events.filter(~F.col("kind").isin("REORDER", "SHIPMENT", "INBOUND_SHIPMENT", *DC_KINDS))
        .unionByName(_store_inbound_by_leg(events, lines))
        .withColumnRenamed("store_id", "node_id")
        .withColumnRenamed("kind", "txn_type"),
        dims,
        ["event_ts"],
        store_id=F.col("node_id"),
    )
    dc_bal = _dc_balances(dc_raw, dims, cfg)

    fact_store_txn = _with_index(
        store_bal.withColumnRenamed("node_id", "store_id"), "fact_store_inventory_txn"
    )
    fact_dc_txn = _with_index(
        dc_bal.withColumnRenamed("node_id", "dc_id")
        # Rename lowercase source -> Source (TMDL-bound PascalCase). Keeping
        # both would be a case-insensitive duplicate that Delta rejects.
        .withColumnRenamed("source", "Source"),
        "fact_dc_inventory_txn",
    )

    stockouts_df = stockouts(store_bal, "ST", "StoreID").unionByName(
        stockouts(dc_bal, "DC", "DCID")
    )
    incidents, incident_stores = logistics_incidents(
        spark,
        network_incidents(spark, cfg.seed, network, cfg.start_date, cfg.end_date),
        network,
        cfg,
    )

    return {
        "fact_store_inventory_txn": fact_store_txn,
        "fact_dc_inventory_txn": fact_dc_txn,
        "fact_truck_moves": _with_index(truck_moves, "fact_truck_moves"),
        "fact_truck_inventory": _with_index(truck_inv, "fact_truck_inventory"),
        "fact_reorders": _with_index(reorders, "fact_reorders"),
        "fact_stockouts": _with_index(stockouts_df, "fact_stockouts"),
        "fact_logistics_incidents": incidents,
        "fact_logistics_incident_stores": incident_stores,
    }


def _utc_text(at: datetime | None) -> str | None:
    return None if at is None else at.astimezone(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def logistics_incidents(
    spark: SparkSession,
    incidents: list[Incident],
    network: IncidentNetwork,
    cfg: GenerationConfig,
) -> tuple[DataFrame, DataFrame]:
    """``fact_logistics_incidents`` and its store bridge over the history.

    An incident belongs to history when it is open at some point between the
    first local midnight and the end of the last local day of its DC; one
    still open at the end has a NULL ``resolved_at``. The bridge lists every
    affected store of each incident.
    """
    headers, stores = [], []
    for incident in incidents:
        zone = network.zone_of_dc(incident.dc_id) if incident.dc_id is not None else "UTC"
        opens = local_instant(cfg.start_date, 0, zone)
        closes = local_instant(cfg.end_date + timedelta(days=1), 0, zone)
        if incident.started_at >= closes or incident.resolved_at <= opens:
            continue
        resolved = None if incident.resolved_at > closes else incident.resolved_at
        started, ended = _utc_text(incident.started_at), _utc_text(resolved)
        headers.append(
            (
                incident.incident_id,
                incident.incident_type,
                incident.dc_id,
                incident.store_id,
                started,
                ended,
                incident.severity,
                len(incident.affected_stores),
                incident.description,
                f"TRC-INC-{incident.incident_id}",
            )
        )
        for store in incident.affected_stores:
            stores.append(
                (
                    incident.incident_id,
                    int(store),
                    incident.dc_id,
                    incident.incident_type,
                    started,
                    ended,
                    f"TRC-INCS-{incident.incident_id}-{int(store)}",
                )
            )
    header_frame = spark.createDataFrame(
        headers,
        "incident_id string, incident_type string, dc_id long, store_id long, "
        "started_at string, resolved_at string, severity string, "
        "affected_store_count long, description string, trace_id string",
    )
    store_frame = spark.createDataFrame(
        stores,
        "incident_id string, store_id long, dc_id long, incident_type string, "
        "started_at string, resolved_at string, trace_id string",
    )

    def typed(frame: DataFrame, table: str) -> DataFrame:
        return (
            frame.withColumn("started_at", F.to_timestamp("started_at"))
            .withColumn("resolved_at", F.to_timestamp("resolved_at"))
            .withColumn("event_date", F.to_date("started_at"))
            .select(*column_names(table))
        )

    return (
        typed(header_frame, "fact_logistics_incidents"),
        typed(store_frame, "fact_logistics_incident_stores"),
    )

# --- retail_setup/generation/gold_rules.py ---
"""Gold aggregate definitions shared by setup-11 and notebook 04.

``gold.py`` (setup-11, in the ``ml-required`` pipeline) and
``fabric/lakehouse/04-streaming-to-gold.ipynb`` (the ``streaming-data-load``
pipeline) both rebuild the same ten Gold tables. This module is their single
definition: setup-11 bundles it in the engine cell and
``scripts/build_notebooks.py`` copies it verbatim into notebook 04 (``--check``
fails on drift), so whichever pipeline ran last publishes the same numbers.

Rules applied to every Gold table:

- Sales aggregates count ``receipt_type = 'SALE'`` receipts that have an
  ``APPROVED`` payment; returns and declined-only purchases never count.
- Online sales count orders with an ``APPROVED`` payment.
- Every source row must be earlier than ``cutoff`` (see ``gold_cutoff``), so
  future-dated rows (scheduled truck moves, deliveries) never reach Gold.
- ``truck_dwell_daily`` is one row per completed shipment lifecycle, dated by
  its arrival, and ``trucks`` counts those lifecycles.

Keep this module free of package imports: notebook 04 runs it as a cell.
"""

from datetime import date, datetime, time, timedelta, timezone

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

GOLD_TABLES: list[str] = [
    "sales_minute_store",
    "top_products_15m",
    "inventory_position_current",
    "dc_inventory_position_current",
    "truck_dwell_daily",
    "online_sales_daily",
    "zone_dwell_minute",
    "marketing_cost_daily",
    "campaign_performance_daily",
    "tender_mix_daily",
]

# Silver inputs read by the Gold builders.
GOLD_SOURCE_TABLES: list[str] = [
    "fact_receipts",
    "fact_receipt_lines",
    "fact_payments",
    "fact_online_order_headers",
    "fact_marketing",
    "fact_marketing_attribution",
    "fact_foot_traffic",
    "fact_truck_moves",
    "fact_store_inventory_txn",
    "fact_dc_inventory_txn",
]

GOLD_TRUCK_LIFECYCLE_KEYS = ["truck_id", "dc_id", "store_id", "shipment_id"]
# History ends at END_DATE in each store's local time; in UTC the last local
# evening runs into the next day (up to UTC-12), so the history cutoff allows it.
GOLD_HISTORY_END_GRACE = timedelta(hours=12)
GOLD_APPROVED_STATUS = "APPROVED"
GOLD_SALE_RECEIPT_TYPE = "SALE"


def gold_cutoff(history_end: date | None, live_started: bool, now: datetime) -> datetime:
    """Exclusive upper bound (naive UTC) for every row that reaches Gold.

    Before live data exists Gold stops at the end of the generated history
    (local midnight after ``END_DATE``, at most ``GOLD_HISTORY_END_GRACE`` past
    UTC midnight), which drops rows the generator scheduled past it.
    Once streaming has started (``_watermarks`` exists) Gold runs up to now.
    """
    current = now.astimezone(timezone.utc).replace(tzinfo=None) if now.tzinfo else now
    if live_started or history_end is None:
        return current
    return datetime.combine(history_end + timedelta(days=1), time()) + GOLD_HISTORY_END_GRACE


def _gold_money(column: str):
    """Explicit double cast for the formatted string money columns."""
    return F.col(column).cast("double")


def _gold_utc(value: datetime):
    """A naive-UTC ``datetime`` as a Spark timestamp in the (UTC) session zone.

    ``F.lit(datetime)`` would read a naive value in the driver's local zone.
    """
    return F.lit(value.isoformat(sep=" ")).cast("timestamp")


def _gold_before(frame: DataFrame, column: str, cutoff: datetime) -> DataFrame:
    return frame.filter(F.col(column) < _gold_utc(cutoff))


def gold_approved_sales(
    fact_receipts: DataFrame, fact_payments: DataFrame, cutoff: datetime
) -> DataFrame:
    """SALE receipts before ``cutoff`` with at least one approved payment."""
    approved = fact_payments.filter(
        (F.col("status") == GOLD_APPROVED_STATUS) & F.col("receipt_id_ext").isNotNull()
    ).select("receipt_id_ext")
    return (
        _gold_before(fact_receipts, "event_ts", cutoff)
        .filter(F.col("receipt_type") == GOLD_SALE_RECEIPT_TYPE)
        .join(approved, "receipt_id_ext", "left_semi")
    )


def gold_approved_online_orders(
    fact_online_order_headers: DataFrame, fact_payments: DataFrame, cutoff: datetime
) -> DataFrame:
    """Online orders before ``cutoff`` with at least one approved payment."""
    approved = fact_payments.filter(
        (F.col("status") == GOLD_APPROVED_STATUS) & F.col("order_id_ext").isNotNull()
    ).select("order_id_ext")
    return _gold_before(fact_online_order_headers, "event_ts", cutoff).join(
        approved, "order_id_ext", "left_semi"
    )


def gold_sales_minute_store(
    fact_receipts: DataFrame, fact_payments: DataFrame, cutoff: datetime
) -> DataFrame:
    return (
        gold_approved_sales(fact_receipts, fact_payments, cutoff)
        .withColumn("ts", F.date_trunc("minute", F.col("event_ts")))
        .groupBy("store_id", "ts")
        .agg(
            F.sum(_gold_money("total_amount")).alias("total_sales"),
            F.count("*").alias("receipts"),
            F.avg(_gold_money("total_amount")).alias("avg_basket"),
        )
        .select("store_id", "ts", "total_sales", "receipts", "avg_basket")
    )


def gold_top_products_15m(
    fact_receipt_lines: DataFrame,
    fact_receipts: DataFrame,
    fact_payments: DataFrame,
    cutoff: datetime,
) -> DataFrame:
    """Revenue and units per product in the latest 15-minute window of sales.

    Only the most recent window is kept (a real-time tile), so the scan is
    limited to the last day of approved sales before the window is chosen.
    """
    sales = gold_approved_sales(fact_receipts, fact_payments, cutoff).select(
        "receipt_id_ext", "event_ts"
    )
    latest_sale = sales.agg(F.max("event_ts")).collect()[0][0]
    lines = _gold_before(fact_receipt_lines, "event_ts", cutoff)
    if latest_sale is not None:
        start = latest_sale - timedelta(days=1)
        lines = lines.filter(F.col("event_ts") >= F.lit(start))
        sales = sales.filter(F.col("event_ts") >= F.lit(start - timedelta(days=1)))
    windows = (
        lines.join(sales.select("receipt_id_ext"), "receipt_id_ext", "left_semi")
        .withColumn("window_15m", F.window(F.col("event_ts"), "15 minutes"))
        .groupBy("product_id", "window_15m")
        .agg(
            F.sum(_gold_money("ext_price")).alias("revenue"),
            F.sum("quantity").alias("units"),
        )
        .withColumn("computed_at", F.col("window_15m.end"))
        .drop("window_15m")
    )
    latest_window = windows.agg(F.max("computed_at")).collect()[0][0]
    if latest_window is not None:
        windows = windows.filter(F.col("computed_at") == F.lit(latest_window))
    return windows.select("product_id", "revenue", "units", "computed_at")


def gold_latest_inventory_position(
    transactions: DataFrame, node_column: str, cutoff: datetime
) -> DataFrame:
    """The final balance before ``cutoff`` for each node/product key."""
    latest = Window.partitionBy(node_column, "product_id").orderBy(
        F.desc("event_ts"),
        F.desc("trace_id"),
    )
    return (
        _gold_before(transactions, "event_ts", cutoff)
        .withColumn("rn", F.row_number().over(latest))
        .filter(F.col("rn") == 1)
        .select(
            node_column,
            "product_id",
            F.col("balance").alias("on_hand"),
            F.col("event_ts").alias("as_of"),
        )
    )


def gold_truck_dwell_daily(fact_truck_moves: DataFrame, cutoff: datetime) -> DataFrame:
    """Average unload dwell per site and arrival day, one row per shipment.

    History writes six status rows per shipment and only the COMPLETED row
    carries the actual arrival (``eta``) and departure; live Silver writes
    COMPLETED rows only. A lifecycle counts once it departed before ``cutoff``.
    """
    lifecycles = (
        fact_truck_moves.filter(F.col("status") == "COMPLETED")
        .groupBy(*GOLD_TRUCK_LIFECYCLE_KEYS)
        .agg(
            F.min("eta").alias("arrival_time"),
            F.max(F.coalesce(F.col("departure_time"), F.col("etd"))).alias("departure_time"),
        )
        .filter(
            (F.col("departure_time") > F.col("arrival_time"))
            & (F.col("departure_time") < _gold_utc(cutoff))
        )
    )
    return (
        lifecycles.withColumn("day", F.to_date("arrival_time"))
        .withColumn(
            "site",
            F.when(
                F.col("store_id").isNotNull(), F.concat(F.lit("STORE_"), F.col("store_id"))
            ).otherwise(F.concat(F.lit("DC_"), F.col("dc_id"))),
        )
        .withColumn(
            "dwell_min",
            (F.unix_timestamp("departure_time") - F.unix_timestamp("arrival_time")) / 60.0,
        )
        .groupBy("site", "day")
        .agg(
            F.avg("dwell_min").alias("avg_dwell_min"),
            F.count("*").alias("trucks"),
        )
        .select("site", "day", "avg_dwell_min", "trucks")
    )


def gold_online_sales_daily(
    fact_online_order_headers: DataFrame, fact_payments: DataFrame, cutoff: datetime
) -> DataFrame:
    return (
        gold_approved_online_orders(fact_online_order_headers, fact_payments, cutoff)
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("day")
        .agg(
            F.count("*").alias("orders"),
            F.sum(_gold_money("subtotal_amount")).alias("subtotal"),
            F.sum(_gold_money("tax_amount")).alias("tax"),
            F.sum(_gold_money("total_amount")).alias("total"),
            F.avg(_gold_money("total_amount")).alias("avg_order_value"),
        )
        .select("day", "orders", "subtotal", "tax", "total", "avg_order_value")
    )


def gold_zone_dwell_minute(fact_foot_traffic: DataFrame, cutoff: datetime) -> DataFrame:
    return (
        _gold_before(fact_foot_traffic, "event_ts", cutoff)
        .withColumn("ts", F.date_trunc("minute", F.col("event_ts")))
        .groupBy("store_id", "zone", "ts")
        .agg(
            F.avg("dwell_seconds").alias("avg_dwell"),
            F.sum("count").alias("customers"),
        )
        .select("store_id", "zone", "ts", "avg_dwell", "customers")
    )


def gold_marketing_cost_daily(fact_marketing: DataFrame, cutoff: datetime) -> DataFrame:
    return (
        _gold_before(fact_marketing, "event_ts", cutoff)
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("campaign_id", "day")
        .agg(
            F.count("*").alias("impressions"),
            F.sum(_gold_money("cost")).alias("cost"),
        )
        .select("campaign_id", "day", "impressions", "cost")
    )


def gold_campaign_performance_daily(
    fact_marketing: DataFrame,
    fact_marketing_attribution: DataFrame,
    cutoff: datetime,
) -> DataFrame:
    """Campaign spend and reconciled conversions by touch day."""
    spend = (
        _gold_before(fact_marketing, "event_ts", cutoff)
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("campaign_id", "channel", "day")
        .agg(
            F.countDistinct("impression_id_ext").alias("impressions"),
            F.sum("cost_cents").cast("long").alias("spend_cents"),
        )
    )
    conversions = (
        _gold_before(fact_marketing_attribution, "purchase_ts", cutoff)
        .filter(F.col("attribution_status") == "ATTRIBUTED")
        .withColumn("day", F.to_date("touch_ts"))
        .groupBy("campaign_id", "channel", "day")
        .agg(
            F.countDistinct("attribution_id").alias("conversions"),
            F.sum("attributed_revenue_cents").cast("long").alias("attributed_revenue_cents"),
            F.sum("discount_cents").cast("long").alias("discount_cents"),
            F.sum("tax_cents").cast("long").alias("tax_cents"),
            F.sum("payment_cents").cast("long").alias("payment_cents"),
        )
    )
    return (
        spend.join(conversions, ["campaign_id", "channel", "day"], "left")
        .fillna(
            0,
            subset=[
                "conversions",
                "attributed_revenue_cents",
                "discount_cents",
                "tax_cents",
                "payment_cents",
            ],
        )
        .withColumn(
            "conversion_rate",
            F.when(F.col("impressions") == 0, F.lit(0.0)).otherwise(
                F.col("conversions") / F.col("impressions")
            ),
        )
        .withColumn(
            "roas",
            F.when(F.col("spend_cents") == 0, F.lit(0.0)).otherwise(
                F.col("attributed_revenue_cents") / F.col("spend_cents")
            ),
        )
        .select(
            "campaign_id",
            "channel",
            "day",
            "impressions",
            "spend_cents",
            "conversions",
            "attributed_revenue_cents",
            "discount_cents",
            "tax_cents",
            "payment_cents",
            "conversion_rate",
            "roas",
        )
    )


def gold_tender_mix_daily(
    fact_receipts: DataFrame, fact_payments: DataFrame, cutoff: datetime
) -> DataFrame:
    return (
        gold_approved_sales(fact_receipts, fact_payments, cutoff)
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("day", "payment_method")
        .agg(
            F.count("*").alias("transactions"),
            F.sum(_gold_money("total_amount")).alias("total_amount"),
        )
        .select("day", "payment_method", "transactions", "total_amount")
    )


def build_gold_tables(tables: dict[str, DataFrame], cutoff: datetime) -> dict[str, DataFrame]:
    """All ten Gold frames from the Silver inputs in ``GOLD_SOURCE_TABLES``."""
    receipts = tables["fact_receipts"]
    payments = tables["fact_payments"]
    return {
        "sales_minute_store": gold_sales_minute_store(receipts, payments, cutoff),
        "top_products_15m": gold_top_products_15m(
            tables["fact_receipt_lines"], receipts, payments, cutoff
        ),
        "inventory_position_current": gold_latest_inventory_position(
            tables["fact_store_inventory_txn"], "store_id", cutoff
        ),
        "dc_inventory_position_current": gold_latest_inventory_position(
            tables["fact_dc_inventory_txn"], "dc_id", cutoff
        ),
        "truck_dwell_daily": gold_truck_dwell_daily(tables["fact_truck_moves"], cutoff),
        "online_sales_daily": gold_online_sales_daily(
            tables["fact_online_order_headers"], payments, cutoff
        ),
        "zone_dwell_minute": gold_zone_dwell_minute(tables["fact_foot_traffic"], cutoff),
        "marketing_cost_daily": gold_marketing_cost_daily(tables["fact_marketing"], cutoff),
        "campaign_performance_daily": gold_campaign_performance_daily(
            tables["fact_marketing"], tables["fact_marketing_attribution"], cutoff
        ),
        "tender_mix_daily": gold_tender_mix_daily(receipts, payments, cutoff),
    }

# --- retail_setup/generation/gold.py ---
"""Gold aggregates for setup-11, built from the shared ``gold_rules`` module.

Setup-11 (``ml-required``) and notebook 04 (``streaming-data-load``) publish
the same ten Gold tables. Both use ``gold_rules`` (notebook 04 embeds its
source; ``scripts/build_notebooks.py --check`` enforces that), so the tables
don't change depending on which pipeline ran last. This module only adds the
contract column order from ``schemas.TABLES``.
"""

from datetime import datetime

from pyspark.sql import DataFrame, SparkSession


__all__ = ["GOLD_TABLES", "generate_gold"]


def generate_gold(
    spark: SparkSession, tables: dict[str, DataFrame], cutoff: datetime
) -> dict[str, DataFrame]:
    """Build the 10 Gold aggregate frames from the Silver fact tables.

    ``cutoff`` is the exclusive upper bound from ``gold_rules.gold_cutoff``.
    """
    gold = build_gold_tables(tables, cutoff)
    return {name: gold[name].select(*column_names(name)) for name in GOLD_TABLES}

# --- retail_setup/generation/job_locks.py ---
"""Lease locks that keep Lakehouse writers from overlapping.

``streaming-data-load`` (03, 04), ``daily-maintenance`` (05 and the ML
refresh), ``ml-required`` (setup-11 and the ML producers) and notebook 90 run
on independent schedules and write the same Delta tables. Fabric has no
cross-pipeline mutex, so each writer claims a lease row in
``<silver>._job_locks`` before it writes:

- ``silver``: notebook 03 and notebook 90 (all Silver tables and ``_watermarks``);
- ``gold``: notebook 04 and setup-11 (the ten Gold aggregates, ``setup_run_log``);
- ``ml:<table>``: the producer of that ML output table.

Notebook 05 claims the same lock for each table it maintains, without
waiting, and skips the table when another job holds it.

A claim is a Delta ``MERGE`` that succeeds only when the row is missing,
expired, or already ours; concurrent claims conflict in Delta and one of them
loses. The holder renews its lease from a background heartbeat, so a crashed
notebook releases its locks once the lease expires (``JOB_LOCK_TTL_SECONDS``).
All times come from Spark's ``current_timestamp()``, never the driver clock.

Keep this module free of package imports: Lakehouse notebooks run it as a
cell (``scripts/build_notebooks.py`` copies it in).
"""

import threading
import time as _time
import uuid
from collections.abc import Callable, Iterable
from datetime import datetime, timedelta, timezone

JOB_LOCK_TABLE = "_job_locks"
JOB_LOCK_TTL_SECONDS = 900
JOB_LOCK_HEARTBEAT_SECONDS = 120
JOB_LOCK_POLL_SECONDS = 30
SILVER_LOCK = "silver"
GOLD_LOCK = "gold"


def ml_output_lock(table: str) -> str:
    """The lock that guards one ML output table."""
    return f"ml:{table}"


def is_concurrent_write_error(exc: BaseException) -> bool:
    """True for Delta optimistic-concurrency conflicts (another writer won)."""
    text = f"{type(exc).__name__}: {exc}"
    return "Concurrent" in text or "DELTA_CONCURRENT" in text


class DeltaJobLockStore:
    """Lease rows in a Delta table: (lock_name, owner, acquired_at, expires_at)."""

    def __init__(self, spark, table_name: str, ttl_seconds: int = JOB_LOCK_TTL_SECONDS):
        self.spark = spark
        self.table_name = table_name
        self.ttl_seconds = int(ttl_seconds)
        self.spark.sql(
            f"CREATE TABLE IF NOT EXISTS {table_name} "
            "(lock_name STRING, owner STRING, acquired_at TIMESTAMP, expires_at TIMESTAMP) "
            "USING DELTA"
        )

    @staticmethod
    def _quote(value: str) -> str:
        return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

    def holder(self, name: str) -> str | None:
        """The owner of a live lease on ``name``, or None."""
        rows = self.spark.sql(
            f"SELECT owner FROM {self.table_name} "
            f"WHERE lock_name = {self._quote(name)} AND expires_at > current_timestamp()"
        ).collect()
        return rows[0]["owner"] if rows else None

    def try_claim(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"""
                MERGE INTO {self.table_name} AS t
                USING (
                    SELECT {self._quote(name)} AS lock_name, {self._quote(owner)} AS owner,
                           current_timestamp() AS acquired_at,
                           current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS AS expires_at
                ) AS s
                ON t.lock_name = s.lock_name
                WHEN MATCHED AND (t.expires_at <= s.acquired_at OR t.owner = s.owner)
                    THEN UPDATE SET t.owner = s.owner, t.acquired_at = s.acquired_at,
                                    t.expires_at = s.expires_at
                WHEN NOT MATCHED THEN INSERT (lock_name, owner, acquired_at, expires_at)
                    VALUES (s.lock_name, s.owner, s.acquired_at, s.expires_at)
                """
            )
        except Exception as exc:
            if is_concurrent_write_error(exc):
                return False
            raise
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"UPDATE {self.table_name} "
                f"SET expires_at = current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS "
                f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
            )
        except Exception as exc:
            if not is_concurrent_write_error(exc):
                raise
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        for _attempt in range(5):
            try:
                self.spark.sql(
                    f"DELETE FROM {self.table_name} "
                    f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
                )
                return
            except Exception as exc:
                if not is_concurrent_write_error(exc):
                    raise
                _time.sleep(1)


class MemoryJobLockStore:
    """The same lease semantics in memory, with an injectable clock (tests)."""

    def __init__(
        self,
        ttl_seconds: int = JOB_LOCK_TTL_SECONDS,
        clock: Callable[[], datetime] | None = None,
    ):
        self.ttl = timedelta(seconds=ttl_seconds)
        self.clock = clock or (lambda: datetime.now(timezone.utc).replace(tzinfo=None))
        self.rows: dict[str, tuple[str, datetime]] = {}

    def holder(self, name: str) -> str | None:
        row = self.rows.get(name)
        return row[0] if row and row[1] > self.clock() else None

    def try_claim(self, name: str, owner: str) -> bool:
        now = self.clock()
        row = self.rows.get(name)
        if row is None or row[1] <= now or row[0] == owner:
            self.rows[name] = (owner, now + self.ttl)
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        row = self.rows.get(name)
        if row is not None and row[0] == owner:
            self.rows[name] = (owner, self.clock() + self.ttl)
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        if self.rows.get(name, ("", None))[0] == owner:
            del self.rows[name]


def new_lock_owner(holder: str) -> str:
    """A unique owner for one notebook run, e.g. ``03-streaming-to-silver:1a2b...``."""
    return f"{holder}:{uuid.uuid4().hex[:12]}"


def acquire_job_locks(
    store,
    names: Iterable[str],
    owner: str,
    *,
    wait_seconds: float,
    poll_seconds: float = JOB_LOCK_POLL_SECONDS,
    sleep: Callable[[float], None] = _time.sleep,
    monotonic: Callable[[], float] = _time.monotonic,
) -> list[str] | None:
    """Claim every lock in ``names`` or none of them.

    Locks are claimed in sorted order, so two jobs that need several locks
    can't deadlock. Retries until ``wait_seconds`` elapse (0 = one attempt)
    and returns the claimed names, or None when another job still holds one.
    """
    ordered = sorted(set(names))
    deadline = monotonic() + max(wait_seconds, 0)
    while True:
        claimed = []
        for name in ordered:
            if not store.try_claim(name, owner):
                break
            claimed.append(name)
        if len(claimed) == len(ordered):
            return claimed
        for name in reversed(claimed):
            store.release(name, owner)
        remaining = deadline - monotonic()
        if remaining <= 0:
            return None
        sleep(min(poll_seconds, remaining))


class JobLocks:
    """A notebook's set of leases, renewed by a heartbeat thread until released."""

    def __init__(
        self,
        store,
        holder: str,
        names: Iterable[str],
        heartbeat_seconds: float = JOB_LOCK_HEARTBEAT_SECONDS,
    ):
        self.store = store
        self.owner = new_lock_owner(holder)
        self.names = sorted(set(names))
        self.heartbeat_seconds = heartbeat_seconds
        self.held: list[str] = []
        self.lost: list[str] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def blockers(self) -> dict[str, str | None]:
        """Current holders of the locks this job needs (for messages)."""
        return {name: self.store.holder(name) for name in self.names}

    def acquire(self, wait_seconds: float, **kwargs) -> bool:
        claimed = acquire_job_locks(
            self.store, self.names, self.owner, wait_seconds=wait_seconds, **kwargs
        )
        if claimed is None:
            return False
        self.held = claimed
        if self.heartbeat_seconds > 0:
            self._thread = threading.Thread(target=self._heartbeat, daemon=True)
            self._thread.start()
        return True

    def _heartbeat(self) -> None:
        while not self._stop.wait(self.heartbeat_seconds):
            self.renew()

    def renew(self) -> None:
        for name in self.held:
            try:
                renewed = self.store.renew(name, self.owner)
            except Exception as exc:
                print(f"Lock {name}: renewal failed ({exc}); retrying at the next heartbeat")
                continue
            if not renewed and name not in self.lost:
                self.lost.append(name)

    def assert_held(self) -> None:
        """Fail before a commit point if a lease expired and another job took it."""
        if self.lost:
            raise RuntimeError(f"Job locks lost before completion: {self.lost}")

    def release(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.heartbeat_seconds + 5)
        for name in reversed(self.held):
            self.store.release(name, self.owner)
        self.held = []

# --- retail_setup/generation/invariants.py ---
"""Cross-table invariant checks. Pure reads; raises nothing — returns a report."""

from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from functools import partial

from pyspark import inheritable_thread_target
from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Each check is one small Spark job; at demo scale their fixed per-job overhead
# dominates. The publish notebook evaluates checks concurrently over Delta
# reads; the default stays sequential because concurrent checks over lineage
# that still runs Python (mapInArrow) can exhaust local Python workers.
PUBLISH_INVARIANT_WORKERS = 8
_ROW_COUNT = object()  # marks queued row counts in the check queue


@dataclass
class InvariantReport:
    """Checks are queued, then evaluated concurrently on first read.

    Reading ``checks``, ``failures``, or ``passed`` evaluates every queued
    check; results are recorded in the order the checks were declared.
    """

    row_counts: dict[str, int] = field(default_factory=dict)
    max_workers: int = 1
    _pending: list[tuple[object, Callable[[], int]]] = field(default_factory=list, repr=False)
    _checks: list[str] = field(default_factory=list, repr=False)
    _failures: list[str] = field(default_factory=list, repr=False)

    @property
    def checks(self) -> list[str]:
        self.evaluate()
        return self._checks

    @property
    def failures(self) -> list[str]:
        self.evaluate()
        return self._failures

    @property
    def passed(self) -> bool:
        return not self.failures

    def evaluate(self) -> None:
        pending, self._pending = self._pending, []
        if not pending:
            return
        if self.max_workers <= 1:
            results = [(name, count()) for name, count in pending]
        else:
            with ThreadPoolExecutor(self.max_workers) as pool:
                futures = [
                    (name, pool.submit(inheritable_thread_target(count))) for name, count in pending
                ]
                results = [(name, future.result()) for name, future in futures]
        for name, bad_count in results:
            if not isinstance(name, str):  # a queued row count
                continue
            self._checks.append(name)
            if bad_count:
                self._failures.append(f"{name}: {bad_count} violations")


def _check(report: InvariantReport, name: str, bad_count: Callable[[], int]) -> None:
    report._pending.append((name, bad_count))


def _single_row(df: DataFrame) -> Row:
    """The one row of a global aggregate."""
    row = df.first()
    if row is None:
        raise RuntimeError("global aggregate returned no row")
    return row


def _check_count(report: InvariantReport, name: str, df: DataFrame) -> None:
    """Queue a row count; it is recorded as data, not as a check."""

    def count() -> int:
        report.row_counts[name] = df.count()
        return 0

    report._pending.append((_ROW_COUNT, count))


# Longest time an online order can stay open: BOPIS picking plus pickup, or
# picking, the backorder wait, shipping and delivery.
ONLINE_MAX_OPEN_SECONDS = max(
    ONLINE_PICK_SECONDS["BOPIS"][1] + ONLINE_PICKUP_SECONDS[1],
    ONLINE_PICK_SECONDS["SHIP"][1]
    + ONLINE_BACKORDER_SECONDS[1]
    + ONLINE_SHIP_SECONDS[1]
    + ONLINE_DELIVER_SECONDS[1],
)


def _stale_open_orders(lines: DataFrame) -> int:
    """Open online lines placed too long before the last order to still be open.

    Also counts lines whose status disagrees with their stage times (a
    DELIVERED line without ``delivered_ts``, an open line with one).
    """
    last = _single_row(lines.agg(F.max("event_ts").alias("last")))["last"]
    if last is None:
        return 0
    status = F.col("fulfillment_status")
    open_status = status.isin("PENDING", "PICKED", "SHIPPED")
    cutoff = F.lit(last) - F.expr(f"INTERVAL {ONLINE_MAX_OPEN_SECONDS} SECONDS")
    return lines.filter(
        (open_status & ((F.col("event_ts") < cutoff) | F.col("delivered_ts").isNotNull()))
        | ((status == "DELIVERED") & F.col("delivered_ts").isNull())
    ).count()


def _online_order_checks(r: InvariantReport, ol: DataFrame) -> None:
    """Order-level fulfillment, zeroed cancellations and as-of-end statuses."""
    _check(
        r,
        "online orders have one fulfillment mode and node",
        lambda: (
            ol.groupBy("order_id")
            .agg(F.countDistinct("fulfillment_mode", "node_type", "node_id").alias("n"))
            .filter(F.col("n") != 1)
            .count()
        ),
    )
    _check(
        r,
        "cancelled online lines carry no amount",
        lambda: ol.filter(
            (F.col("fulfillment_status") == "CANCELLED") & (F.col("ext_cents") != 0)
        ).count(),
    )
    # Statuses are as of the end of history: only orders placed within the
    # longest fulfillment time before the last order may still be open.
    _check(
        r,
        "online orders open only near the end of history",
        lambda: _stale_open_orders(ol),
    )


def run_invariants(
    spark: SparkSession,
    t: dict[str, DataFrame],
    *,
    max_workers: int = 1,
) -> InvariantReport:
    r = InvariantReport(max_workers=max_workers)
    for name, df in t.items():
        _check_count(r, name, df)
    r.evaluate()

    receipts, lines, pay = t["fact_receipts"], t["fact_receipt_lines"], t["fact_payments"]
    _check(
        r,
        "fact_receipts.receipt_id_ext unique",
        lambda: (
            r.row_counts["fact_receipts"] - receipts.select("receipt_id_ext").distinct().count()
        ),
    )
    _check(
        r,
        "fact_receipt_lines -> fact_receipts FK",
        lambda: lines.join(receipts, "receipt_id_ext", "left_anti").count(),
    )
    _check(
        r,
        "fact_payments xor keys",
        lambda: pay.filter(
            F.col("receipt_id_ext").isNotNull() == F.col("order_id_ext").isNotNull()
        ).count(),
    )
    _check(
        r,
        "fact_payments -> receipts FK",
        lambda: (
            pay.filter(F.col("receipt_id_ext").isNotNull())
            .join(receipts, "receipt_id_ext", "left_anti")
            .count()
        ),
    )

    products = t["dim_products"].select(F.col("ID").alias("product_id"))
    for tbl in [
        "fact_receipt_lines",
        "fact_online_order_lines",
        "fact_promo_lines",
        "fact_store_inventory_txn",
        "fact_dc_inventory_txn",
        "fact_reorders",
    ]:
        _check(
            r,
            f"{tbl} -> dim_products FK",
            t[tbl].join(products, "product_id", "left_anti").count,
        )

    stores = t["dim_stores"].select(F.col("ID").alias("store_id"))
    for tbl in [
        "fact_receipts",
        "fact_store_inventory_txn",
        "fact_reorders",
        "fact_store_ops",
        "fact_foot_traffic",
        "fact_ble_pings",
    ]:
        _check(
            r,
            f"{tbl} -> dim_stores FK",
            t[tbl].join(stores, "store_id", "left_anti").count,
        )

    for tbl, df in t.items():
        if tbl.startswith("fact_") and "event_date" in df.columns:
            _check(
                r,
                f"{tbl}.event_date not null",
                df.filter(F.col("event_date").isNull()).count,
            )

    oh, ol = t["fact_online_order_headers"], t["fact_online_order_lines"]
    _check(
        r,
        "online order ids unique",
        lambda: (
            r.row_counts["fact_online_order_headers"] - oh.select("order_id_ext").distinct().count()
        ),
    )
    _check(
        r,
        "online lines -> headers FK",
        lambda: ol.join(oh, ol.order_id == oh.order_id_ext, "left_anti").count(),
    )
    _online_order_checks(r, ol)

    so = t["fact_stockouts"]
    _check(
        r,
        "stockouts StoreID xor DCID",
        lambda: so.filter(F.col("StoreID").isNotNull() == F.col("DCID").isNotNull()).count(),
    )

    # Per-receipt promo discount consistency: for receipts present in
    # fact_promo_lines (SALE only by construction), the summed discount must
    # equal the implied line-level discount sum(unit_cents*quantity - ext_cents).
    promo_sum = (
        t["fact_promo_lines"]
        .groupBy("receipt_id_ext")
        .agg(F.sum("discount_cents").alias("promo_discount"))
    )
    line_sum = lines.groupBy("receipt_id_ext").agg(
        F.sum(F.col("unit_cents") * F.col("quantity") - F.col("ext_cents")).alias("line_discount")
    )
    _check(
        r,
        "promo discount consistency",
        lambda: (
            promo_sum.join(line_sum, "receipt_id_ext", "left")
            .filter(
                F.col("line_discount").isNull()
                | (F.col("promo_discount") != F.col("line_discount"))
            )
            .count()
        ),
    )

    # Basket-level promotions: one code per SALE receipt, whose minimum the
    # receipt's gross subtotal meets, and no SKU on two lines of a receipt.
    minimums = F.create_map(
        *[item for code, *_, minimum, _ in PROMO_CATALOG for item in (F.lit(code), F.lit(minimum))]
    )
    sale_receipts = t["fact_receipts"].filter(F.col("receipt_type") == "SALE")
    sale_lines = lines.join(sale_receipts.select("receipt_id_ext"), "receipt_id_ext", "left_semi")
    receipt_codes = (
        sale_lines.filter(F.col("promo_code").isNotNull())
        .groupBy("receipt_id_ext")
        .agg(F.countDistinct("promo_code").alias("codes"), F.first("promo_code").alias("code"))
    )
    _check(
        r,
        "one promotion code per receipt",
        lambda: receipt_codes.filter(F.col("codes") > 1).count(),
    )
    _check(
        r,
        "promotion minimums met by the basket",
        lambda: (
            receipt_codes.join(
                sale_receipts.select("receipt_id_ext", "gross_subtotal_cents"), "receipt_id_ext"
            )
            .filter(F.coalesce(minimums[F.col("code")], F.lit(0)) > F.col("gross_subtotal_cents"))
            .count()
        ),
    )
    _check(
        r,
        "no SKU repeated within a receipt",
        lambda: (
            sale_lines.groupBy("receipt_id_ext", "product_id").count().filter("count > 1").count()
        ),
    )

    # --- dimension geography FK integrity (datagen foreign_key validator parity)
    geo_ids = t["dim_geographies"].select(F.col("ID").alias("geo_id"))
    for dim in ["dim_stores", "dim_distribution_centers", "dim_customers"]:
        _check(
            r,
            f"{dim} -> dim_geographies FK",
            t[dim]
            .select(F.col("GeographyID").alias("geo_id"))
            .join(geo_ids, "geo_id", "left_anti")
            .count,
        )

    # --- DC coverage on facts that reference a distribution center
    dc_ids = t["dim_distribution_centers"].select(F.col("ID").alias("dc_id"))
    for tbl in ["fact_dc_inventory_txn", "fact_truck_moves", "fact_reorders"]:
        _check(
            r,
            f"{tbl} -> dim_distribution_centers FK",
            t[tbl]
            .filter(F.col("dc_id").isNotNull())
            .select("dc_id")
            .join(dc_ids, "dc_id", "left_anti")
            .count,
        )

    # --- truck coverage on logistics facts
    truck_ids = t["dim_trucks"].select(F.col("ID").alias("truck_id"))
    for tbl in ["fact_truck_moves", "fact_truck_inventory"]:
        _check(
            r,
            f"{tbl} -> dim_trucks FK",
            t[tbl]
            .filter(F.col("truck_id").isNotNull())
            .select("truck_id")
            .join(truck_ids, "truck_id", "left_anti")
            .count,
        )

    incidents = t["fact_logistics_incidents"]
    _check(
        r,
        "fact_logistics_incidents -> dim_distribution_centers FK",
        lambda: (
            incidents.filter(F.col("dc_id").isNotNull()).join(dc_ids, "dc_id", "left_anti").count()
        ),
    )
    _check(
        r,
        "fact_logistics_incidents -> dim_stores FK",
        lambda: (
            incidents.filter(F.col("store_id").isNotNull())
            .join(stores, "store_id", "left_anti")
            .count()
        ),
    )
    _check(
        r,
        "fact_logistics_incidents resolved after start",
        lambda: incidents.filter(F.col("resolved_at") < F.col("started_at")).count(),
    )

    # --- truck timing: arrival (eta) must not be after completion (etd)
    tm = t["fact_truck_moves"]
    _check(
        r,
        "fact_truck_moves etd >= eta",
        lambda: tm.filter(
            F.col("eta").isNotNull() & F.col("etd").isNotNull() & (F.col("etd") < F.col("eta"))
        ).count(),
    )

    # --- customer coverage on facts that resolve a customer (nullable for some)
    customer_ids = t["dim_customers"].select(F.col("ID").alias("customer_id"))
    for tbl in ["fact_receipts", "fact_online_order_headers"]:
        _check(
            r,
            f"{tbl} -> dim_customers FK",
            t[tbl]
            .filter(F.col("customer_id").isNotNull())
            .select("customer_id")
            .join(customer_ids, "customer_id", "left_anti")
            .count,
        )
    # fact_marketing.customer_id is a nullable double (low resolution rate); cast.
    _check(
        r,
        "fact_marketing -> dim_customers FK",
        lambda: (
            t["fact_marketing"]
            .filter(F.col("customer_id").isNotNull())
            .select(F.col("customer_id").cast("long").alias("customer_id"))
            .join(customer_ids, "customer_id", "left_anti")
            .count()
        ),
    )

    # --- dim_products pricing constraints (datagen pricing validator parity)
    prod = t["dim_products"]
    _check(
        r,
        "dim_products pricing Cost<SalePrice<=MSRP",
        lambda: prod.filter(
            ~(
                (F.col("Cost") > 0)
                & (F.col("Cost") < F.col("SalePrice"))
                & (F.col("SalePrice") <= F.col("MSRP"))
            )
        ).count(),
    )

    # --- IMP-007 marketing attribution -------------------------------------
    # gross - discount = net holds for every SALE receipt / online order,
    # attributed or not (discount is always applied before tax).
    _check(
        r,
        "fact_receipts gross - discount = net subtotal",
        lambda: receipts.filter(
            F.col("gross_subtotal_cents") - F.col("discount_cents") != F.col("subtotal_cents")
        ).count(),
    )
    _check(
        r,
        "fact_online_order_headers gross - discount = net subtotal",
        lambda: oh.filter(
            F.col("gross_subtotal_cents") - F.col("discount_cents") != F.col("subtotal_cents")
        ).count(),
    )

    if "fact_marketing_attribution" in t:
        fma = t["fact_marketing_attribution"]
        _check(
            r,
            "fact_marketing_attribution.attribution_id unique",
            lambda: (
                r.row_counts["fact_marketing_attribution"]
                - fma.select("attribution_id").distinct().count()
            ),
        )
        _check(
            r,
            "fact_marketing_attribution xor purchase keys",
            lambda: fma.filter(
                F.col("receipt_id_ext").isNotNull() == F.col("order_id_ext").isNotNull()
            ).count(),
        )
        _check(
            r,
            "fact_marketing_attribution purchase_type matches xor key",
            lambda: fma.filter(
                ((F.col("purchase_type") == "STORE") != F.col("receipt_id_ext").isNotNull())
                | ((F.col("purchase_type") == "ONLINE") != F.col("order_id_ext").isNotNull())
            ).count(),
        )
        # one journey per purchase, one purchase per journey: distinct
        # non-NULL attribution_journey_id count must equal ATTRIBUTED row count.
        attributed = fma.filter(F.col("attribution_status") == "ATTRIBUTED")
        _check(
            r,
            "fact_marketing_attribution one journey per purchase",
            lambda: (
                attributed.count() - attributed.select("attribution_journey_id").distinct().count()
            ),
        )
        _check(
            r,
            "fact_marketing_attribution journey_id set iff ATTRIBUTED",
            lambda: fma.filter(
                (F.col("attribution_status") == "ATTRIBUTED")
                != F.col("attribution_journey_id").isNotNull()
            ).count(),
        )
        _check(
            r,
            "fact_marketing_attribution -> fact_receipts FK",
            lambda: (
                fma.filter(F.col("receipt_id_ext").isNotNull())
                .select("receipt_id_ext")
                .join(receipts.select("receipt_id_ext"), "receipt_id_ext", "left_anti")
                .count()
            ),
        )
        _check(
            r,
            "fact_marketing_attribution -> online headers FK",
            lambda: (
                fma.filter(F.col("order_id_ext").isNotNull())
                .select("order_id_ext")
                .join(oh.select("order_id_ext"), "order_id_ext", "left_anti")
                .count()
            ),
        )
        # financial reconciliation: gross-discount=net, net+tax=total, and for
        # any row that reached a payment decision (not RECONCILIATION_FAILED)
        # the recorded payment equals the purchase total.
        _check(
            r,
            "fact_marketing_attribution gross - discount = net",
            lambda: fma.filter(
                F.col("gross_subtotal_cents") - F.col("discount_cents")
                != F.col("net_subtotal_cents")
            ).count(),
        )
        _check(
            r,
            "fact_marketing_attribution net + tax = total",
            lambda: fma.filter(
                F.col("net_subtotal_cents") + F.col("tax_cents") != F.col("total_cents")
            ).count(),
        )
        _check(
            r,
            "fact_marketing_attribution approved payment = total",
            lambda: (
                fma.filter(
                    F.col("attribution_status").isin("ATTRIBUTED", "UNATTRIBUTED_NO_JOURNEY")
                )
                .filter(F.col("payment_cents") != F.col("total_cents"))
                .count()
            ),
        )
        _check(
            r,
            "fact_marketing_attribution attributed_revenue matches status",
            lambda: fma.filter(
                (
                    (F.col("attribution_status") == "ATTRIBUTED")
                    & (F.col("attributed_revenue_cents") != F.col("net_subtotal_cents"))
                )
                | (
                    (F.col("attribution_status") != "ATTRIBUTED")
                    & (F.col("attributed_revenue_cents") != 0)
                )
            ).count(),
        )
        # window/tie correctness: every ATTRIBUTED row's last touch must be
        # inside the inclusive 7-day window, with a non-negative lag.
        _check(
            r,
            "fact_marketing_attribution touch within 7-day window",
            lambda: attributed.filter(
                F.col("touch_ts").isNull()
                | (F.col("lag_seconds") < 0)
                | (F.col("lag_seconds") > F.col("attribution_window_days") * 86400)
            ).count(),
        )

        # fact_marketing: a journey is the converting impressions that reached
        # its purchase (attribution.py), one touch or more. Journeys and
        # attributed purchases pair 1:1, so no journey is orphaned and no
        # attributed purchase lacks a touch.
        mk = t["fact_marketing"]
        journey_touch_counts = (
            mk.filter(F.col("attribution_journey_id").isNotNull())
            .groupBy("attribution_journey_id")
            .agg(F.count("*").alias("n"))
        )
        _check(
            r,
            "fact_marketing journeys <-> attributed purchases 1:1",
            lambda: (
                journey_touch_counts.join(
                    attributed.select("attribution_journey_id"),
                    "attribution_journey_id",
                    "full_outer",
                )
                .filter(F.col("n").isNull() | F.col("attribution_journey_id").isNull())
                .count()
            ),
        )
    _run_inventory_invariants(r, t)
    _run_business_invariants(r, t)
    r.evaluate()
    return r


def _inventory_balance_checks(
    report: InvariantReport,
    table: str,
    transactions: DataFrame,
    node: str,
) -> None:
    ordered = Window.partitionBy(node, "product_id").orderBy("event_ts", "trace_id")
    running = ordered.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    checked = (
        transactions.withColumn("_expected_balance", F.sum("quantity").over(running))
        .withColumn("_position", F.row_number().over(ordered))
        .withColumn(
            "_initial_count",
            F.sum(F.when(F.col("txn_type") == "INITIAL", 1).otherwise(0)).over(
                Window.partitionBy(node, "product_id")
            ),
        )
    )
    invalid = (
        F.col(node).isNull()
        | F.col("product_id").isNull()
        | F.col("event_ts").isNull()
        | F.col("trace_id").isNull()
        | F.col("quantity").isNull()
        | F.col("balance").isNull()
        | (F.col("balance") < 0)
    )
    counts = _single_row(
        checked.agg(
            F.count(F.when(invalid, 1)).alias("physical"),
            F.count(F.when(~F.col("balance").eqNullSafe(F.col("_expected_balance")), 1)).alias(
                "conservation"
            ),
            F.count(
                F.when(
                    (F.col("_initial_count") != 1)
                    | ((F.col("_position") == 1) & ~F.col("txn_type").eqNullSafe(F.lit("INITIAL"))),
                    1,
                )
            ).alias("opening"),
            (F.count("*") - F.countDistinct("trace_id")).alias("duplicate_traces"),
        )
    )
    _check(report, f"{table} nonnegative physical balances", lambda: counts.physical)
    _check(report, f"{table} running quantity conservation", lambda: counts.conservation)
    _check(report, f"{table} exactly one opening INITIAL", lambda: counts.opening)
    _check(report, f"{table}.trace_id unique", lambda: counts.duplicate_traces)


def _inventory_mirror_violations(actual: DataFrame, expected: DataFrame) -> int:
    """Compare both directions as multisets, so duplicates cannot cancel out."""
    return actual.exceptAll(expected).unionByName(expected.exceptAll(actual)).count()


def _run_inventory_invariants(r: InvariantReport, t: dict[str, DataFrame]) -> None:
    """Check physical stock only; independent online fulfillment is not reserved here."""
    store = t["fact_store_inventory_txn"]
    dc = t["fact_dc_inventory_txn"]
    truck = t["fact_truck_inventory"]
    _inventory_balance_checks(r, "fact_store_inventory_txn", store, "store_id")
    _inventory_balance_checks(r, "fact_dc_inventory_txn", dc, "dc_id")

    requested, unfulfilled = F.col("requested_quantity"), F.col("unfulfilled_quantity")
    valid_demand = (
        F.when(
            F.col("txn_type") == "SALE",
            (F.col("quantity") < 0) & (requested == -F.col("quantity")) & (unfulfilled == 0),
        )
        .when(
            F.col("txn_type") == "LOST_SALE",
            (F.col("quantity") == 0)
            & (requested > 0)
            & (unfulfilled == requested)
            & (requested > F.col("balance")),
        )
        .otherwise((requested == 0) & (unfulfilled == 0))
    )
    _check(
        r,
        "store requested = fulfilled + unfulfilled quantity",
        lambda: store.filter(~F.coalesce(valid_demand, F.lit(False))).count(),
    )

    receipt_stock = (
        t["fact_receipt_lines"]
        .join(
            t["fact_receipts"].select("receipt_id_ext", "store_id", "receipt_type"),
            "receipt_id_ext",
        )
        .select(
            F.concat(F.lit("TRC-INV-"), "receipt_id_ext", F.lit("-"), "line_num").alias("trace_id"),
            "store_id",
            "product_id",
            "event_ts",
            (-F.col("quantity")).cast("long").alias("quantity"),
            "receipt_type",
        )
    )
    receipt_keys = ["trace_id", "store_id", "product_id", "event_ts", "quantity"]
    _check(
        r,
        "fulfilled receipt lines <-> store SALE conservation",
        lambda: _inventory_mirror_violations(
            store.filter(F.col("txn_type") == "SALE").select(*receipt_keys),
            receipt_stock.filter(F.col("receipt_type") == "SALE").select(*receipt_keys),
        ),
    )
    # Refunds may be destroyed rather than restocked. Every actual add-back
    # must match a refunded line, but not every refund must add stock.
    _check(
        r,
        "store RETURN backed by refunded lines",
        lambda: (
            store.filter(F.col("txn_type") == "RETURN")
            .select(*receipt_keys)
            .exceptAll(
                receipt_stock.filter(F.col("receipt_type") == "RETURN").select(*receipt_keys)
            )
            .count()
        ),
    )

    for action, ledger, kind, node, source, sign in [
        ("LOAD", dc, "OUTBOUND_SHIPMENT", "dc_id", "Source", -1),
        ("UNLOAD", store, "INBOUND_SHIPMENT", "store_id", "source", 1),
    ]:
        expected = truck.filter(F.col("action") == action).select(
            F.col("location_id").alias("node_id"),
            "product_id",
            "event_ts",
            (F.col("quantity") * sign).alias("quantity"),
            "shipment_id",
        )
        actual = ledger.filter(F.col("txn_type") == kind).select(
            F.col(node).alias("node_id"),
            "product_id",
            "event_ts",
            "quantity",
            F.col(source).alias("shipment_id"),
        )
        _check(
            r,
            f"truck {action} <-> {node} ledger conservation",
            partial(_inventory_mirror_violations, actual, expected),
        )

    valid_cargo = (
        F.col("quantity").isNotNull()
        & (F.col("quantity") > 0)
        & F.col("shipment_id").isNotNull()
        & F.col("truck_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("event_ts").isNotNull()
        & F.col("location_id").isNotNull()
        & F.col("trace_id").isNotNull()
        & (
            ((F.col("action") == "LOAD") & (F.col("location_type") == "DC"))
            | ((F.col("action") == "UNLOAD") & (F.col("location_type") == "STORE"))
        )
    )
    cargo_order = (
        Window.partitionBy("shipment_id", "product_id", "truck_id")
        .orderBy("event_ts", "trace_id")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    cargo_counts = _single_row(
        truck.withColumn(
            "_cargo_balance",
            F.sum(
                F.when(F.col("action") == "LOAD", F.col("quantity")).otherwise(-F.col("quantity"))
            ).over(cargo_order),
        ).agg(
            F.count(F.when(~F.coalesce(valid_cargo, F.lit(False)), 1)).alias("invalid"),
            F.count(F.when(F.col("_cargo_balance") < 0, 1)).alias("negative"),
            (F.count("*") - F.countDistinct("trace_id")).alias("duplicate_traces"),
            (F.count("*") - F.countDistinct("shipment_id", "product_id", "action")).alias(
                "duplicate_actions"
            ),
        )
    )
    _check(r, "truck cargo positive with valid action and location", lambda: cargo_counts.invalid)
    _check(r, "truck nonnegative cargo balances", lambda: cargo_counts.negative)
    _check(r, "fact_truck_inventory.trace_id unique", lambda: cargo_counts.duplicate_traces)
    _check(r, "truck shipment/product actions unique", lambda: cargo_counts.duplicate_actions)
    loads = truck.filter(F.col("action") == "LOAD").select(
        "shipment_id",
        "product_id",
        "truck_id",
        F.col("quantity").alias("_loaded"),
        F.col("event_ts").alias("_loaded_at"),
    )
    # A final-day LOAD may still be in transit; only UNLOAD requires a prior LOAD.
    _check(
        r,
        "truck UNLOAD backed by prior equal LOAD",
        lambda: (
            truck.filter(F.col("action") == "UNLOAD")
            .join(loads, ["shipment_id", "product_id", "truck_id"], "left")
            .filter(
                F.col("_loaded").isNull()
                | ~F.col("quantity").eqNullSafe(F.col("_loaded"))
                | (F.col("event_ts") < F.col("_loaded_at")),
            )
            .count()
        ),
    )


def _run_business_invariants(r: InvariantReport, t: dict[str, DataFrame]) -> None:
    """IMP-010 shared business invariants: deliberate seeds must not produce
    sales while a store is closed, pre-launch sales, same-day returns, or
    same-time dimension lifecycles."""

    receipts = t["fact_receipts"]

    # --- sales-while-closed: every SALE receipt's store-local hour must fall
    # inside its store's operating window. event_ts is UTC, so the local hour
    # comes from the store's time zone; local_hour must agree with it.
    # RETURN rows fall inside store hours too (a random local time).
    store_hours = t["dim_stores"].select(
        F.col("ID").alias("store_id"), "operating_hours", F.col("TimeZone").alias("_tz")
    )
    open_h, close_h = _open_close_cols(F.col("operating_hours"))
    hour_checked = (
        receipts.select("store_id", "event_ts", "local_hour", "receipt_type")
        .join(store_hours, "store_id")
        .withColumn("_h", local_hour(F.col("event_ts"), F.col("_tz")))
        .withColumn("_open", open_h)
        .withColumn("_close", close_h)
    )
    _check(
        r,
        "fact_receipts within store operating hours",
        lambda: hour_checked.filter(
            (F.col("_h") < F.col("_open")) | (F.col("_h") >= F.col("_close"))
        ).count(),
    )
    _check(
        r,
        "fact_receipts local_hour matches the store time zone",
        lambda: hour_checked.filter(~F.col("local_hour").eqNullSafe(F.col("_h"))).count(),
    )

    # --- pre-launch sales: no receipt line may sell a product before its
    # LaunchDate. Compares the calendar day of the sale to the product launch.
    launch = t["dim_products"].select(
        F.col("ID").alias("product_id"), F.to_date("LaunchDate").alias("_launch_date")
    )
    _check(
        r,
        "fact_receipt_lines no pre-launch sales",
        lambda: (
            t["fact_receipt_lines"]
            .filter(F.col("quantity") > 0)  # SALE lines only; RETURN lines are negative
            .join(launch, "product_id")
            .filter(F.col("event_date") < F.col("_launch_date"))
            .count()
        ),
    )

    # --- returns: every RETURN links the SALE it refunds (same store and
    # customer), posts after it, and refunds no more than it.
    if "original_receipt_id_ext" in receipts.columns:
        _return_link_checks(r, receipts)
    if "fact_payments" in t:
        _payment_attempt_checks(r, t["fact_payments"])

    # --- same-time lifecycles: a product's LaunchDate must be a single instant
    # (no null) and strictly before the far-future horizon; guards against
    # degenerate zero-length lifecycles introduced by adversarial seeds.
    _check(
        r,
        "dim_products LaunchDate present",
        lambda: t["dim_products"].filter(F.col("LaunchDate").isNull()).count(),
    )


def _return_link_checks(r: InvariantReport, receipts: DataFrame) -> None:
    returns = receipts.filter(F.col("receipt_type") == "RETURN")
    sales = receipts.filter(F.col("receipt_type") == "SALE").select(
        F.col("receipt_id_ext").alias("original_receipt_id_ext"),
        F.col("store_id").alias("_sale_store"),
        F.col("customer_id").alias("_sale_customer"),
        F.col("event_ts").alias("_sale_ts"),
        F.col("total_cents").alias("_sale_total"),
    )
    linked = returns.join(sales, "original_receipt_id_ext", "left")
    _check(
        r,
        "fact_receipts returns link their sale",
        lambda: linked.filter(
            F.col("_sale_ts").isNull()
            | (F.col("store_id") != F.col("_sale_store"))
            | ~F.col("customer_id").eqNullSafe(F.col("_sale_customer"))
        ).count(),
    )
    # Stored in UTC, so compare instants; returns.build_return_headers also
    # guarantees a later store-local day.
    _check(
        r,
        "fact_receipts returns post after their sale",
        lambda: linked.filter(
            F.col("_sale_ts").isNotNull() & (F.col("event_ts") <= F.col("_sale_ts"))
        ).count(),
    )
    _check(
        r,
        "fact_receipts returns refund at most the sale",
        lambda: linked.filter(-F.col("total_cents") > F.col("_sale_total")).count(),
    )
    _check(
        r,
        "fact_receipts only returns link a sale",
        lambda: receipts.filter(
            (F.col("receipt_type") == "SALE") & F.col("original_receipt_id_ext").isNotNull()
        ).count(),
    )


def _payment_attempt_checks(r: InvariantReport, payments: DataFrame) -> None:
    # --- declined payments: every receipt and placed order has exactly one
    # approved payment (a declined attempt is retried; voided sales never
    # happen), and transaction IDs are unique.
    keyed = payments.withColumn(
        "_purchase", F.coalesce(F.col("receipt_id_ext"), F.col("order_id_ext"))
    )
    approved = (
        keyed.filter(F.col("status") == "APPROVED")
        .groupBy("_purchase")
        .agg(F.count("*").alias("_approved"))
    )
    _check(
        r,
        "fact_payments one approved payment per purchase",
        lambda: (
            keyed.select("_purchase")
            .distinct()
            .join(approved, "_purchase", "left")
            .filter(~F.col("_approved").eqNullSafe(F.lit(1)))
            .count()
        ),
    )
    _check(
        r,
        "fact_payments.transaction_id unique",
        lambda: payments.count() - payments.select("transaction_id").distinct().count(),
    )

# --- retail_setup/generation/handoff.py ---
"""Hand generated stage outputs between setup notebooks.

Each setup notebook runs a slice of the generation stage graph in its own Spark
session. Stage outputs that a later notebook consumes are written as handoff
tables named ``gen__<stage>__<output>`` in the Silver staging schema
(``<silver_db>_stage``). The publish notebook promotes them into Silver and then
drops every handoff.

Historical setup is single-writer, so handoff names are fixed and an activity
retry simply overwrites them. Every handoff records the generation
fingerprint (the configuration it was generated for) atomically with its data
(Delta commit ``userMetadata``); loading a handoff produced for a different
configuration fails instead of silently mixing runs.

Path mode (``base_path=``) stores parquet plus a fingerprint marker file so
local tests need no Delta dependency.
"""

import hashlib
import shutil
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession


HANDOFF_PREFIX = "gen__"
_FINGERPRINT_MARKER = "_retail_generation"


def delete_lakehouse_path(spark: SparkSession, path: str) -> None:
    """Recursively delete a path on the session's default filesystem (if present)."""
    jvm = spark._jvm
    if jvm is None:
        raise RuntimeError("the Spark session has no JVM gateway")
    hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
    hadoop_path.getFileSystem(spark._jsc.hadoopConfiguration()).delete(hadoop_path, True)


def dictionary_fingerprint(root: Path, store_type: str) -> str:
    """Content hash of the dictionary files generation reads (shared + store type)."""
    digest = hashlib.sha256()
    for folder in ("_shared", store_type):
        base = root / folder
        for path in sorted(base.rglob("*.json")) if base.is_dir() else []:
            relative = path.relative_to(root).as_posix()
            digest.update(relative.encode("utf-8") + b"\0")
            digest.update(hashlib.sha256(path.read_bytes()).digest())
    return digest.hexdigest()


def generation_fingerprint(cfg: GenerationConfig, dictionary_hash: str | None = None) -> str:
    """Stable identity of everything that shapes generated data.

    Covers the configuration and the dictionary content, so a handoff
    generated from older dictionaries (for example after the dictionary ref
    moved) is rejected instead of being mixed with new data.
    """
    payload = cfg.model_dump_json(exclude={"dictionary_root"})
    if dictionary_hash is None:
        dictionary_hash = dictionary_fingerprint(cfg.resolved_dictionary_root, cfg.store_type)
    identity = f"{payload}|dictionaries={dictionary_hash}"
    return hashlib.sha256(identity.encode("utf-8")).hexdigest()[:32]


class HandoffMismatchError(RuntimeError):
    """A handoff is missing or belongs to a different generation."""


class HandoffStore:
    def __init__(
        self,
        spark: SparkSession,
        cfg: GenerationConfig,
        *,
        lakehouse: str | None = None,
        base_path: str | None = None,
    ) -> None:
        if (lakehouse is None) == (base_path is None):
            raise ValueError("Provide exactly one of lakehouse= or base_path=")
        self.spark = spark
        self.fingerprint = generation_fingerprint(cfg)
        self.lakehouse = lakehouse
        self.schema = f"{cfg.silver_db}_stage"
        self.base_path = Path(base_path) if base_path is not None else None

    def name(self, stage: str, output: str) -> str:
        table = f"{HANDOFF_PREFIX}{stage}__{output}"
        if self.base_path is not None:
            return str(self.base_path / self.schema / table)
        return f"{self.lakehouse}.{self.schema}.{table}"

    def ensure_schema(self) -> None:
        """Create the handoff schema (setup-02 does this before any parallel writer)."""
        if self.base_path is None:
            self.spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.lakehouse}.{self.schema}")

    def save(self, stage: str, output: str, frame: DataFrame) -> DataFrame:
        """Write one stage output and return a frame that reads it back."""
        name = self.name(stage, output)
        if self.base_path is None:
            (
                frame.write.format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .option("userMetadata", self.fingerprint)
                .saveAsTable(name)
            )
            return self.spark.table(name)
        frame.write.mode("overwrite").parquet(name)
        (Path(name) / _FINGERPRINT_MARKER).write_text(self.fingerprint, encoding="utf-8")
        return self.spark.read.parquet(name)

    def load(self, stage: str, output: str) -> DataFrame:
        name = self.name(stage, output)
        found = self._fingerprint_of(name)
        if found != self.fingerprint:
            reason = (
                "is missing" if found is None else "was generated for a different configuration"
            )
            raise HandoffMismatchError(
                f"Handoff {name} {reason}; rerun the setup notebook that produces "
                f"stage {stage!r} (or the whole setup pipeline)."
            )
        if self.base_path is None:
            return self.spark.table(name)
        return self.spark.read.parquet(name)

    def has_handoffs(self) -> bool:
        """Whether any handoff (of any generation) exists."""
        if self.base_path is not None:
            root = self.base_path / self.schema
            return root.exists() and any(root.glob(f"{HANDOFF_PREFIX}*"))
        self.ensure_schema()
        return any(
            row.tableName.startswith(HANDOFF_PREFIX)
            for row in self.spark.sql(f"SHOW TABLES IN {self.lakehouse}.{self.schema}").collect()
        )

    def drop_all(self) -> None:
        """Remove every handoff, including leftovers from failed runs."""
        if self.base_path is not None:
            root = self.base_path / self.schema
            for path in root.glob(f"{HANDOFF_PREFIX}*") if root.exists() else []:
                shutil.rmtree(path)
            return
        self.ensure_schema()
        for row in self.spark.sql(f"SHOW TABLES IN {self.lakehouse}.{self.schema}").collect():
            if row.tableName.startswith(HANDOFF_PREFIX):
                self.spark.sql(
                    f"DROP TABLE IF EXISTS {self.lakehouse}.{self.schema}.{row.tableName}"
                )

    def _fingerprint_of(self, name: str) -> str | None:
        if self.base_path is not None:
            marker = Path(name) / _FINGERPRINT_MARKER
            return marker.read_text(encoding="utf-8") if marker.exists() else None
        if not self.spark.catalog.tableExists(name):
            return None
        latest = self.spark.sql(f"DESCRIBE HISTORY {name} LIMIT 1").collect()
        return latest[0]["userMetadata"] if latest else None

# --- retail_setup/generation/engine.py ---
"""Generation stage graph, shared by local runs and the setup pipeline notebooks.

Generation is a dependency graph of stages (``STAGES``). Each stage builds one
or more frames from the dimensions and the outputs of the stages it depends
on. ``generate_all`` runs the whole graph in one Spark session (local tests);
the setup pipeline instead runs each notebook's slice of the graph in its own
session (``run_generation_notebook``), hands outputs between notebooks as
Delta tables, and ``load_publication`` assembles the Silver tables for the
publish notebook. In both modes, stages whose inputs are ready run
concurrently. Generation is deterministic, so neither the split nor the
completion order changes the output.
"""

import warnings
from collections.abc import Callable, Iterable
from concurrent.futures import FIRST_COMPLETED, Future, ThreadPoolExecutor, wait
from dataclasses import dataclass
from datetime import date

from pyspark import inheritable_thread_target
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F



@dataclass
class GenerationResult:
    tables: dict[str, DataFrame]


def _shift_year(d: date, years: int) -> date:
    """Shift a date by whole years; Feb 29 falls back to Feb 28."""
    try:
        return d.replace(year=d.year + years)
    except ValueError:
        return date(d.year + years, d.month, 28)


DICTIONARIES_NOTEBOOK = "setup-01-seed-dictionaries"
DIMENSIONS_NOTEBOOK = "setup-02-generate-dimensions"
PUBLISH_NOTEBOOK = "setup-10-publish-silver"
# Ready stages run concurrently; Spark schedules their jobs side by side so
# narrow stages, the Python inventory policy, and stragglers do not leave the
# pool idle.
MAX_PARALLEL_STAGES = 6

StageOutputs = dict[str, DataFrame]


@dataclass(frozen=True)
class GenerationContext:
    spark: SparkSession
    dicts: DictionarySet
    cfg: GenerationConfig
    dims: dict[str, DataFrame]


@dataclass(frozen=True)
class Stage:
    """One generation step: ``build(ctx, *outputs_of_deps) -> outputs``.

    ``notebook`` is the setup pipeline notebook that runs the stage. Stages
    with ``handoff=False`` are only consumed inside their own notebook and
    are never written as handoff tables.
    """

    name: str
    notebook: str
    deps: tuple[str, ...]
    outputs: tuple[str, ...]
    build: Callable[..., StageOutputs]
    handoff: bool = True


_RECEIPT_GROUP = ("fact_receipts", "fact_receipt_lines", "fact_payments")


def _accepted(plan: StageOutputs) -> StageOutputs:
    return {name: plan[name] for name in _RECEIPT_GROUP}


def _build_candidates(ctx: GenerationContext) -> StageOutputs:
    return generate_receipts_group(ctx.spark, ctx.dims, ctx.dicts.profile, ctx.cfg)


def _build_plan(
    ctx: GenerationContext, candidates: StageOutputs, online: StageOutputs
) -> StageOutputs:
    # Inventory-constrained fulfillment precedes every receipt consumer. Its DC
    # stock also serves the ship-from-DC online lines.
    plan = resolve_fulfillment(
        candidates,
        ctx.dims,
        ctx.cfg,
        profile=ctx.dicts.profile,
        online_lines=online["fact_online_order_lines"],
    )
    return {"return_headers": plan.return_headers, "events": plan.events, **plan.sales}


def _build_online(ctx: GenerationContext) -> StageOutputs:
    return generate_online_orders(ctx.spark, ctx.dims, ctx.dicts.profile, ctx.cfg)


def _build_ads(ctx: GenerationContext) -> StageOutputs:
    return {
        "fact_marketing": generate_marketing(
            ctx.spark, ctx.dims, ctx.cfg, ctx.dicts.profile
        )
    }


def _build_refunds(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    return generate_returns(
        ctx.spark, _accepted(plan), ctx.dims, ctx.cfg, return_headers=plan["return_headers"]
    )


# Stages generate in store-local time. Each published output is converted to
# UTC by the stage that produces it (see timezones.py); outputs that later
# stages consume in local time are converted by their final consumer or at
# assembly (fact_receipt_lines).


def _build_promos(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    promos, promo_lines = generate_promotions(ctx.spark, _accepted(plan))
    # fact_promotions is converted by the attribution stage, its final producer.
    return {
        "fact_promotions": promos,
        "fact_promo_lines": localize_receipt_lines(promo_lines, ctx.dims),
    }


def _build_store_ops(ctx: GenerationContext) -> StageOutputs:
    ops = generate_store_ops(ctx.spark, ctx.dims, ctx.cfg, ctx.dicts.profile)
    return {"fact_store_ops": localize_by_store(ops, ctx.dims, ["event_ts"])}


def _build_foot_traffic(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    traffic = generate_foot_traffic(
        ctx.spark, plan["fact_receipts"], ctx.dims, ctx.cfg
    )
    return {"fact_foot_traffic": localize_by_store(traffic, ctx.dims, ["event_ts"])}


def _build_ble(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    pings, zone_changes = generate_ble(
        ctx.spark, plan["fact_receipts"], ctx.dims, ctx.cfg, ctx.dicts.profile
    )
    return {
        "fact_ble_pings": localize_by_store(pings, ctx.dims, ["event_ts"]),
        "fact_customer_zone_changes": localize_by_store(zone_changes, ctx.dims, ["event_ts"]),
    }


def _build_ledgers(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    result = FulfillmentResult(_accepted(plan), plan["return_headers"], plan["events"])
    # Restocking returns are already events in the fulfillment plan, so the
    # ledgers need no refund frames.
    return generate_inventory_chain(
        ctx.spark, _accepted(plan), {}, ctx.dims, ctx.cfg, fulfillment=result
    )


def _union(frames: list[DataFrame]) -> DataFrame:
    combined = frames[0]
    for frame in frames[1:]:
        combined = combined.unionByName(frame)
    return combined


def _build_attributed(
    ctx: GenerationContext,
    plan: StageOutputs,
    refunds: StageOutputs,
    online: StageOutputs,
    promos: StageOutputs,
    ads: StageOutputs,
) -> StageOutputs:
    # IMP-007: deterministic last-touch attribution over every purchase stream.
    # Attribution runs in UTC: store purchases, their payments and promotions
    # are converted first; online orders and impressions are already UTC.
    dims = ctx.dims
    receipts = _union([plan["fact_receipts"], refunds["fact_receipts"]])
    payments = _union([plan["fact_payments"], refunds["fact_payments"], online["payments"]])
    inputs = {
        **dims,
        **ads,
        "fact_promotions": localize_by_store(promos["fact_promotions"], dims, ["event_ts"]),
        "fact_receipts": localize_by_store(receipts, dims, ["event_ts"]),
        "fact_online_order_headers": online["fact_online_order_headers"],
        "fact_payments": localize_by_store(
            payments, dims, ["event_ts"], convert=F.col("receipt_id_ext").isNotNull()
        ),
    }
    out = generate_attribution(ctx.spark, inputs, ctx.cfg)
    out["fact_receipts"] = with_local_hour(out["fact_receipts"], dims)
    return {name: out[name] for name in _ATTRIBUTED}


_ATTRIBUTED = (
    "fact_receipts",
    "fact_online_order_headers",
    "fact_payments",
    "fact_promotions",
    "fact_marketing",
    "fact_marketing_attribution",
)
_LEDGERS = (
    "fact_store_inventory_txn",
    "fact_dc_inventory_txn",
    "fact_truck_moves",
    "fact_truck_inventory",
    "fact_reorders",
    "fact_stockouts",
    "fact_logistics_incidents",
    "fact_logistics_incident_stores",
)

STAGES: tuple[Stage, ...] = (
    Stage(
        "candidates",
        "setup-03-generate-sales",
        (),
        _RECEIPT_GROUP,
        _build_candidates,
        handoff=False,
    ),
    Stage(
        "plan",
        "setup-03-generate-sales",
        ("candidates", "online"),
        ("return_headers", "events", *_RECEIPT_GROUP),
        _build_plan,
    ),
    Stage(
        "online",
        "setup-04-generate-online-orders",
        (),
        ("fact_online_order_headers", "fact_online_order_lines", "payments"),
        _build_online,
    ),
    Stage("ads", "setup-05-generate-marketing", (), ("fact_marketing",), _build_ads),
    Stage(
        "refunds",
        "setup-06-generate-returns-and-promotions",
        ("plan",),
        _RECEIPT_GROUP,
        _build_refunds,
    ),
    Stage(
        "promos",
        "setup-06-generate-returns-and-promotions",
        ("plan",),
        ("fact_promotions", "fact_promo_lines"),
        _build_promos,
    ),
    Stage(
        "store_ops", "setup-07-generate-store-activity", (), ("fact_store_ops",), _build_store_ops
    ),
    Stage(
        "foot_traffic",
        "setup-07-generate-store-activity",
        ("plan",),
        ("fact_foot_traffic",),
        _build_foot_traffic,
    ),
    Stage(
        "ble",
        "setup-07-generate-store-activity",
        ("plan",),
        ("fact_ble_pings", "fact_customer_zone_changes"),
        _build_ble,
    ),
    Stage("ledgers", "setup-08-generate-inventory", ("plan",), _LEDGERS, _build_ledgers),
    Stage(
        "attributed",
        "setup-09-attribute-marketing",
        ("plan", "refunds", "online", "promos", "ads"),
        _ATTRIBUTED,
        _build_attributed,
    ),
)
_STAGE_BY_NAME = {stage.name: stage for stage in STAGES}
STAGE_NOTEBOOKS = tuple(dict.fromkeys(stage.notebook for stage in STAGES))

# Published Silver fact table -> the stage outputs it is made of (unioned in order).
SILVER_SOURCES: dict[str, tuple[tuple[str, str], ...]] = {
    "fact_receipts": (("attributed", "fact_receipts"),),
    "fact_receipt_lines": (("plan", "fact_receipt_lines"), ("refunds", "fact_receipt_lines")),
    "fact_online_order_headers": (("attributed", "fact_online_order_headers"),),
    "fact_online_order_lines": (("online", "fact_online_order_lines"),),
    "fact_payments": (("attributed", "fact_payments"),),
    "fact_promotions": (("attributed", "fact_promotions"),),
    "fact_promo_lines": (("promos", "fact_promo_lines"),),
    "fact_marketing": (("attributed", "fact_marketing"),),
    "fact_marketing_attribution": (("attributed", "fact_marketing_attribution"),),
    "fact_store_ops": (("store_ops", "fact_store_ops"),),
    "fact_foot_traffic": (("foot_traffic", "fact_foot_traffic"),),
    "fact_ble_pings": (("ble", "fact_ble_pings"),),
    "fact_customer_zone_changes": (("ble", "fact_customer_zone_changes"),),
    **{name: (("ledgers", name),) for name in _LEDGERS},
}
_PUBLISHED_STAGES = {stage for sources in SILVER_SOURCES.values() for stage, _ in sources}
# Published tables whose stage outputs are still store-local; they are
# converted to UTC when the sources are assembled.
_LOCAL_AT_ASSEMBLY = {"fact_receipt_lines": localize_receipt_lines}


def _assemble(name: str, frames: list[DataFrame], dims: dict[str, DataFrame]) -> DataFrame:
    table = _union(frames)
    localize = _LOCAL_AT_ASSEMBLY.get(name)
    return table if localize is None else localize(table, dims)


def setup_notebook_dependencies() -> dict[str, tuple[str, ...]]:
    """Upstream notebooks of each generation and publish notebook.

    The setup pipeline's activity graph must equal this mapping (tested), so
    the pipeline is the stage graph drawn at notebook granularity. Generation
    regenerates dimensions in memory, so root generation notebooks need only
    the dictionaries; setup-02's dimension validation runs alongside them and
    gates publication instead.
    """
    graph: dict[str, tuple[str, ...]] = {}
    for notebook in STAGE_NOTEBOOKS:
        upstream = {
            _STAGE_BY_NAME[dep].notebook
            for stage in STAGES
            if stage.notebook == notebook
            for dep in stage.deps
        } - {notebook}
        graph[notebook] = tuple(sorted(upstream)) or (DICTIONARIES_NOTEBOOK,)
    graph[PUBLISH_NOTEBOOK] = tuple(
        sorted(
            {DIMENSIONS_NOTEBOOK} | {_STAGE_BY_NAME[stage].notebook for stage in _PUBLISHED_STAGES}
        )
    )
    return graph


def generate_dimension_frames(
    spark: SparkSession,
    dicts: DictionarySet,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    dims = generate_dimensions(spark, dicts, cfg)
    # dim_date is padded ±5 years — the semantic model's date relationships
    # need the headroom around the generated fact window.
    dims["dim_date"] = generate_dim_date(
        spark, _shift_year(cfg.start_date, -5), _shift_year(cfg.end_date, 5)
    )
    return dims


def generate_all(
    spark: SparkSession,
    dicts: DictionarySet,
    cfg: GenerationConfig,
    *,
    checkpoint_dir: str | None = None,
) -> GenerationResult:
    """Run the whole stage graph in one session and return every Silver table.

    Each stage's outputs are checkpointed before dependants consume them (see
    ``_checkpoint_frames``). With ``checkpoint_dir`` (a shared filesystem
    path) checkpoints are written there and survive executor loss; without
    it they stay on the executors, which suits local runs only. The caller
    owns deleting the directory once the tables are no longer needed.
    """
    reliable = checkpoint_dir is not None
    if checkpoint_dir is not None:
        spark.sparkContext.setCheckpointDir(checkpoint_dir)
    # Every builder reads the dimensions; materialize them once instead of
    # re-serializing their driver-built rows from Python for each consumer.
    dims = _checkpoint_frames(generate_dimension_frames(spark, dicts, cfg), reliable)
    ctx = GenerationContext(spark, dicts, cfg, dims)
    out = run_stages(ctx, STAGES, {}, lambda _stage, frames: _checkpoint_frames(frames, reliable))
    tables = {
        **dims,
        **{
            name: _assemble(name, [out[stage][output] for stage, output in sources], dims)
            for name, sources in SILVER_SOURCES.items()
        },
    }
    # Release stage data the published tables no longer read.
    referenced = {
        id(out[stage][output]) for sources in SILVER_SOURCES.values() for stage, output in sources
    }
    _release(
        frame for frames in out.values() for frame in frames.values() if id(frame) not in referenced
    )
    return GenerationResult(tables=tables)


def run_generation_notebook(
    spark: SparkSession,
    dicts: DictionarySet,
    cfg: GenerationConfig,
    notebook: str,
    store: HandoffStore,
    *,
    checkpoint_dir: str | None = None,
) -> dict[str, StageOutputs]:
    """Run one setup notebook's stages, reading and writing handoff tables.

    Upstream stage outputs are loaded from handoffs (fingerprint-checked);
    this notebook's handoff stages are written to handoffs, and internal
    stages are checkpointed (reliably under ``checkpoint_dir`` when given,
    as the setup notebooks always do).
    """
    stages = [stage for stage in STAGES if stage.notebook == notebook]
    if not stages:
        raise ValueError(f"no generation stages run in notebook {notebook!r}")
    reliable = checkpoint_dir is not None
    if checkpoint_dir is not None:
        spark.sparkContext.setCheckpointDir(checkpoint_dir)
    dims = _checkpoint_frames(generate_dimension_frames(spark, dicts, cfg), reliable)
    ctx = GenerationContext(spark, dicts, cfg, dims)
    local = {stage.name for stage in stages}
    upstream = {dep for stage in stages for dep in stage.deps} - local
    inputs = {
        dep: {output: store.load(dep, output) for output in _STAGE_BY_NAME[dep].outputs}
        for dep in sorted(upstream)
    }

    def materialize(stage: Stage, frames: StageOutputs) -> StageOutputs:
        if not stage.handoff:
            return _checkpoint_frames(frames, reliable)
        return _persist_frames(frames, lambda output, frame: store.save(stage.name, output, frame))

    out = run_stages(ctx, stages, inputs, materialize)
    _release(dims.values())
    return {name: out[name] for name in local if _STAGE_BY_NAME[name].handoff}


def load_publication(
    spark: SparkSession,
    dicts: DictionarySet,
    cfg: GenerationConfig,
    store: HandoffStore,
    *,
    checkpoint_dir: str | None = None,
) -> tuple[dict[str, DataFrame], dict[str, str]]:
    """Assemble every Silver table from handoffs for the publish notebook.

    Returns ``(tables, prestaged)``: ``prestaged`` maps each table that is a
    single handoff to that handoff's name, so publication promotes it directly
    instead of copying it to a second staging table.
    """
    reliable = checkpoint_dir is not None
    if checkpoint_dir is not None:
        spark.sparkContext.setCheckpointDir(checkpoint_dir)
    tables = _checkpoint_frames(generate_dimension_frames(spark, dicts, cfg), reliable)
    prestaged: dict[str, str] = {}
    for name, sources in SILVER_SOURCES.items():
        frames = [store.load(stage, output) for stage, output in sources]
        tables[name] = _assemble(name, frames, tables)
        if len(sources) == 1 and name not in _LOCAL_AT_ASSEMBLY:
            prestaged[name] = store.name(*sources[0])
    return tables, prestaged


def run_stages(
    ctx: GenerationContext,
    stages: Iterable[Stage],
    inputs: dict[str, StageOutputs],
    materialize: Callable[[Stage, StageOutputs], StageOutputs],
) -> dict[str, StageOutputs]:
    """Run each stage once its dependencies finish, up to MAX_PARALLEL_STAGES at once.

    ``inputs`` holds already-available outputs of stages outside ``stages``.
    Each stage builds and materializes its frames in its own thread. A stage
    that publishes nothing is released once every consumer in this run has
    finished. After a failure no further stage starts; stages already
    running finish (cancelling their jobs mid-cache can deadlock Spark's cache
    manager), then the first error is raised.
    """
    pending = {stage.name: stage for stage in stages}
    results: dict[str, StageOutputs] = dict(inputs)
    consumers = {
        name: {other.name for other in pending.values() if name in other.deps} for name in pending
    }
    running: dict[Future, str] = {}
    failure: BaseException | None = None

    def run(stage: Stage) -> StageOutputs:
        frames = stage.build(ctx, *[results[dep] for dep in stage.deps])
        if set(frames) != set(stage.outputs):
            raise ValueError(
                f"stage {stage.name!r} built {sorted(frames)}, declared {sorted(stage.outputs)}"
            )
        return materialize(stage, frames)

    def release_finished() -> None:
        for name, users in consumers.items():
            if (
                name in results
                and name not in _PUBLISHED_STAGES
                and name not in inputs
                and users
                and all(user in results for user in users)
            ):
                _release(results[name].values())
                users.clear()

    with ThreadPoolExecutor(MAX_PARALLEL_STAGES) as pool:
        while pending or running:
            for name, stage in list(pending.items()):
                if failure is not None:
                    break
                if len(running) < MAX_PARALLEL_STAGES and all(d in results for d in stage.deps):
                    del pending[name]
                    # Copies the caller's Spark local properties (job group,
                    # Fabric run tagging) into the worker thread.
                    running[pool.submit(_inheriting(run), stage)] = name
            if not running:
                if failure is not None:
                    break
                raise ValueError(f"generation stages have unmet dependencies: {sorted(pending)}")
            done, _ = wait(running, return_when=FIRST_COMPLETED)
            for future in done:
                name = running.pop(future)
                try:
                    results[name] = future.result()
                except BaseException as exc:  # noqa: BLE001 - re-raised below
                    failure = failure or exc
            if failure is None:
                release_finished()
    if failure is not None:
        raise failure
    return {name: frames for name, frames in results.items() if name not in inputs}


def _inheriting(fn: Callable) -> Callable:
    """Wrap ``fn`` so its thread inherits the caller's Spark local properties.

    Uses the function-only form, which every supported PySpark accepts (the
    session form is not available on all Fabric runtimes). Newer PySpark warns
    that job tags are not inherited; generation does not use job tags.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Spark session is not provided")
        return inheritable_thread_target(fn)


def _persist_frames(
    frames: StageOutputs,
    persist: Callable[[str, DataFrame], DataFrame],
) -> StageOutputs:
    """Persist sibling frames one at a time, computing shared work once.

    Siblings are cached first so a later sibling reuses work an earlier one
    computed. They are persisted sequentially: concurrent jobs over related
    cached plans can deadlock Spark's cache manager when one of them fails.
    Stages, and the notebooks around them, still run concurrently.
    """
    cached = {name: frame.cache() for name, frame in frames.items()}
    try:
        return {name: persist(name, frame) for name, frame in cached.items()}
    finally:
        for frame in cached.values():
            frame.unpersist()


def _checkpoint_frames(frames: StageOutputs, reliable: bool) -> StageOutputs:
    """Materialize each frame once and cut its lineage.

    Generation reuses frames heavily, so without truncation every downstream
    plan embeds the full upstream lineage: at full-demo scale the driver spent
    most of its time re-analyzing and cache-matching those plans for every
    action. Reliable checkpoints are written to the SparkContext checkpoint
    directory and survive executor loss; local checkpoints keep rows on the
    executors that computed them and are only safe for local runs.
    """

    def checkpoint(_name: str, frame: DataFrame) -> DataFrame:
        return frame.checkpoint(eager=True) if reliable else frame.localCheckpoint(eager=True)

    return _persist_frames(frames, checkpoint)


def _release(frames: Iterable[DataFrame]) -> None:
    """Free checkpointed rows: executor blocks and, if reliable, their files.

    Frames that are not checkpoints (for example handoff table scans) are
    left alone.
    """
    for frame in frames:
        plan = frame._jdf.queryExecution().logical()
        if plan.getClass().getSimpleName() != "LogicalRDD":
            continue
        rdd = plan.rdd()
        rdd.unpersist(False)
        checkpoint_file = rdd.getCheckpointFile()
        if checkpoint_file.isDefined():
            delete_lakehouse_path(frame.sparkSession, checkpoint_file.get())

# --- retail_setup/generation/publication.py ---
"""Stage -> validate -> promote -> (rollback) coordinator for historical
setup publication (IMP-002).

Fabric Lakehouse targets are plain Delta tables; there is no cross-table
transaction spanning several ``saveAsTable``/``CREATE OR REPLACE TABLE``
calls. This module implements the staged-promotion pattern used by
setup-02/03/04 instead:

1. Every candidate table is first written to a run-scoped **staging**
   table/location — the final target is never touched during staging.
2. Each staged table is validated (schema + row count) against its source
   before any promotion happens. All staging and validation must succeed
   before a single final target changes.
3. Promotion swaps each staged table into its fixed final name. If the Nth
   promotion fails, every already-promoted target *and the Nth target
   itself* is rolled back: targets that existed before this run are
   restored to their pre-run Delta version (``RESTORE TABLE ... TO VERSION
   AS OF``); targets that did not exist before this run are dropped. The
   failed target is included because a Delta command can commit before a
   client/network error is observed by the caller — restoring/dropping it
   anyway is safe and idempotent (a no-op if it never actually committed)
   and keeps rollback fail-closed instead of trusting the exception.
   Rollback keeps trying every changed target even if one restore/drop
   fails, and the run is marked ROLLED_BACK (all recovered) or
   ROLLBACK_FAILED (some did not recover — this is never reported as
   success).
4. Staging is cleaned up once the run reaches a terminal state that no
   longer needs it (COMPLETED; FAILED before any promotion; ROLLED_BACK with
   a clean rollback). ROLLBACK_FAILED preserves staging for manual recovery
   and logs the preserved names. Cleanup itself never raises: it is
   attempted for every staged table and any failures are collected. A
   cleanup failure after an otherwise-successful promotion or rollback is
   never reported as plain COMPLETED/ROLLED_BACK (that would hide leftover
   staging) but is also never conflated with a data failure — it is
   reported as COMPLETED_CLEANUP_FAILED / ROLLED_BACK_CLEANUP_FAILED, with
   the message explicitly stating the data operation succeeded and only
   staging cleanup needs manual attention.

The coordinator itself has no Spark/Delta dependency: every side effect
(stage, validate, capture target state, promote, restore, drop, cleanup,
log) is injected through the ``PublicationBackend`` protocol, so the state
machine can be exercised deterministically in tests without a Spark session
or a Fabric workspace. ``writer.py`` supplies the concrete Spark/Delta
catalog backend for notebooks and a filesystem backend for local
path/parquet tests.
"""

from dataclasses import dataclass, field
from typing import Any, Callable, Protocol, Sequence

# Bound error text kept in setup_run_log so one runaway exception can't blow
# past the log column budget.
ERROR_TEXT_LIMIT = 1000

# Terminal coordinator states. STAGED/VALIDATED/PROMOTING/ROLLING_BACK are
# intermediate/log-only states; cleanup warnings preserve data success.
COMPLETED = "COMPLETED"
FAILED = "FAILED"
ROLLED_BACK = "ROLLED_BACK"
ROLLBACK_FAILED = "ROLLBACK_FAILED"
# The data operation (promote or rollback) succeeded, but cleaning up staging
# afterward failed. Deliberately distinct from COMPLETED/ROLLED_BACK — never
# silently reported as success — and distinct from FAILED/ROLLBACK_FAILED —
# never conflated with a data problem.
COMPLETED_CLEANUP_FAILED = "COMPLETED_CLEANUP_FAILED"
ROLLED_BACK_CLEANUP_FAILED = "ROLLED_BACK_CLEANUP_FAILED"


def bound_error(exc: BaseException) -> str:
    """Render an exception as ``Type: message``, truncated for log storage."""
    return f"{type(exc).__name__}: {exc}"[:ERROR_TEXT_LIMIT]


@dataclass(frozen=True)
class TableTarget:
    """One candidate table to publish.

    ``name`` is the bare logical table name (e.g. ``fact_receipts``) used as
    the ``setup_run_log.table_name`` value. ``db`` is the schema/database it
    belongs to (silver or gold). ``staging_name`` is the run-scoped staging
    identifier the backend actually writes to — it is opaque to the
    coordinator (a qualified catalog table name, a filesystem path, ...).
    ``prestaged`` marks a staging identifier that already holds the candidate
    (a generation handoff): the backend neither writes nor cleans it up.
    """

    name: str
    db: str
    staging_name: str
    prestaged: bool = False


@dataclass(frozen=True)
class TargetState:
    """Pre-promotion snapshot of one target, captured before any promotion.

    ``existed`` distinguishes a target that must be *restored* on rollback
    from one that must be *dropped*. ``restore_token`` is backend-defined (a
    Delta version int for catalog targets, a backup path for filesystem
    targets) and is handed back to ``backend.restore()`` unchanged.
    """

    existed: bool
    restore_token: Any = None


@dataclass
class PublicationOutcome:
    """Terminal result of one ``PublicationCoordinator.publish()`` call."""

    # COMPLETED | FAILED | ROLLED_BACK | ROLLBACK_FAILED |
    # COMPLETED_CLEANUP_FAILED | ROLLED_BACK_CLEANUP_FAILED
    state: str
    promoted: list[str] = field(default_factory=list)
    error: str | None = None
    rollback_failures: list[tuple[str, str]] = field(default_factory=list)
    # Staging identifiers left behind for manual recovery: every staged
    # target on ROLLBACK_FAILED, or just the ones whose cleanup() call
    # itself failed on *_CLEANUP_FAILED.
    preserved_staging: list[str] = field(default_factory=list)
    cleanup_failures: list[tuple[str, str]] = field(default_factory=list)

    @property
    def ok(self) -> bool:
        """Whether promotion succeeded; cleanup warnings remain explicit in state."""
        return self.state in {COMPLETED, COMPLETED_CLEANUP_FAILED}


class PublicationBackend(Protocol):
    """Side effects the coordinator needs; implemented per storage mode."""

    def stage(self, target: TableTarget) -> int:
        """Write the candidate to ``target.staging_name``; return its row count."""
        ...

    def validate(self, target: TableTarget, staged_row_count: int) -> None:
        """Raise if the staged table's schema/row count disagree with the source."""
        ...

    def target_state(self, target: TableTarget) -> TargetState:
        """Snapshot whether the final target exists and its restore token."""
        ...

    def promote(self, target: TableTarget) -> int:
        """Swap staged data into the final target; return the promoted row count."""
        ...

    def restore(self, target: TableTarget, state: TargetState) -> None:
        """Undo a promotion of a pre-existing target using its captured state."""
        ...

    def drop(self, target: TableTarget) -> None:
        """Undo a promotion of a target that did not exist before this run."""
        ...

    def cleanup(self, target: TableTarget) -> None:
        """Remove the run-scoped staging artifact for ``target``."""
        ...


# (table_name, status, row_count, error) — mirrors setup_run_log's columns.
LogFn = Callable[[str, str, "int | None", "str | None"], None]


def _same_thread(function: Callable[..., Any]) -> Callable[..., Any]:
    return function


def _first_error(
    results: Sequence[tuple[TableTarget, Any, Exception | None]],
) -> tuple[TableTarget, Exception] | None:
    """The first target that failed a concurrent phase, with its error."""
    return next(((target, error) for target, _, error in results if error is not None), None)


class PublicationCoordinator:
    """Runs the stage -> validate -> promote (-> rollback) state machine.

    ``targets`` are processed in the given order for staging, validation and
    promotion; rollback undoes them in reverse (LIFO) order. All logging
    goes through the injected ``log`` callback so callers can persist an
    append-only ``setup_run_log``-shaped history.
    """

    def __init__(
        self,
        backend: PublicationBackend,
        log: LogFn,
        *,
        max_workers: int = 1,
    ) -> None:
        self.backend = backend
        self.log = log
        # >1 runs each phase's per-table operations concurrently. Phases stay
        # strictly ordered (nothing is promoted until every table staged and
        # validated), so the all-or-nothing guarantees are unchanged.
        self.max_workers = max_workers

    def publish(self, targets: Sequence[TableTarget]) -> PublicationOutcome:
        if self.max_workers > 1:
            return self._publish_concurrently(targets)
        self.log("__run__", "STARTED", None, None)

        staged: list[TableTarget] = []
        # Keyed by the full TableTarget (name+db+staging_name), not the bare
        # name: a silver and a gold table can legitimately share a name, and
        # keying by name alone would let one target's staged count/state
        # silently shadow the other's.
        staged_counts: dict[TableTarget, int] = {}

        # Phase 1 — stage every candidate. Final targets are untouched here.
        for target in targets:
            try:
                count = self.backend.stage(target)
            except Exception as exc:  # noqa: BLE001 — recorded, not swallowed
                return self._fail(target, exc, staged)
            staged.append(target)
            staged_counts[target] = count
            self.log(target.name, "STAGED", count, None)

        # Phase 2 — validate every staged table before any promotion.
        for target in targets:
            try:
                self.backend.validate(target, staged_counts[target])
            except Exception as exc:  # noqa: BLE001
                return self._fail(target, exc, staged)
        self.log("__run__", "VALIDATED", None, None)

        # Phase 3 — snapshot pre-promotion state (existed + restore token) for
        # every target before touching any of them. Keyed by TableTarget for
        # the same same-name-different-db reason as staged_counts above.
        target_states = {t: self.backend.target_state(t) for t in targets}

        # Phase 4 — promote in order; roll back everything already promoted
        # (plus the target whose promotion just failed) if any promotion
        # fails.
        self.log("__run__", "PROMOTING", None, None)
        promoted: list[TableTarget] = []
        for target in targets:
            try:
                count = self.backend.promote(target)
            except Exception as exc:  # noqa: BLE001
                return self._roll_back(target, exc, promoted, target_states, staged)
            promoted.append(target)
            self.log(target.name, "COMPLETED", count, None)

        cleanup_failures = self._cleanup_all(staged)
        if cleanup_failures:
            return self._completed_with_cleanup_failure(promoted, cleanup_failures)
        self.log("__run__", "COMPLETED", len(promoted), None)
        return PublicationOutcome(state=COMPLETED, promoted=[t.name for t in promoted])

    def _run_all(
        self,
        operation: Callable[[TableTarget], Any],
        targets: Sequence[TableTarget],
    ) -> list[tuple[TableTarget, Any, Exception | None]]:
        """Run ``operation`` for every target concurrently; results in target order."""
        from concurrent.futures import ThreadPoolExecutor

        in_thread: Callable[[Callable[..., Any]], Callable[..., Any]]
        try:
            # Keep the caller's Spark local properties (job group, Fabric run
            # tagging) on jobs started from worker threads.
            from pyspark import inheritable_thread_target

            in_thread = inheritable_thread_target
        except ImportError:  # the coordinator itself does not need Spark
            in_thread = _same_thread

        with ThreadPoolExecutor(self.max_workers) as pool:
            futures = [(target, pool.submit(in_thread(operation), target)) for target in targets]
            results: list[tuple[TableTarget, Any, Exception | None]] = []
            for target, future in futures:
                try:
                    results.append((target, future.result(), None))
                except Exception as exc:  # noqa: BLE001 — returned to the caller
                    results.append((target, None, exc))
        return results

    def _publish_concurrently(self, targets: Sequence[TableTarget]) -> PublicationOutcome:
        """Same state machine as ``publish`` with each phase run concurrently.

        Every phase finishes for all targets before the next starts. A staging
        or validation failure fails the run before any promotion and cleans up
        everything that did stage. A promotion failure rolls back every target
        (promoted or failed), exactly as the sequential path rolls back
        everything attempted.
        """
        self.log("__run__", "STARTED", None, None)

        # Phase 1 — stage every candidate concurrently; finals untouched.
        stage_results = self._run_all(self.backend.stage, targets)
        staged = [target for target, _, error in stage_results if error is None]
        staged_counts = {target: count for target, count, error in stage_results if error is None}
        for target, count, error in stage_results:
            if error is None:
                self.log(target.name, "STAGED", count, None)
        first_failure = _first_error(stage_results)
        if first_failure is not None:
            return self._fail(*first_failure, staged)

        # Phase 2 — validate every staged table before any promotion.
        validate_results = self._run_all(
            lambda target: self.backend.validate(target, staged_counts[target]),
            targets,
        )
        first_failure = _first_error(validate_results)
        if first_failure is not None:
            return self._fail(*first_failure, staged)
        self.log("__run__", "VALIDATED", None, None)

        # Phase 3 — snapshot every target's pre-promotion state.
        state_results = self._run_all(self.backend.target_state, targets)
        first_failure = _first_error(state_results)
        if first_failure is not None:
            return self._fail(*first_failure, staged)
        target_states = {target: state for target, state, _ in state_results}

        # Phase 4 — promote concurrently; any failure rolls back every target.
        self.log("__run__", "PROMOTING", None, None)
        promote_results = self._run_all(self.backend.promote, targets)
        promoted = [target for target, _, error in promote_results if error is None]
        for target, count, error in promote_results:
            if error is None:
                self.log(target.name, "COMPLETED", count, None)
        failed = [(target, error) for target, _, error in promote_results if error is not None]
        if failed:
            first_target, first_error = failed[0]
            # Other failed targets may also have partially committed; include
            # them so rollback restores or drops every target of the run.
            others = [target for target, _ in failed[1:]]
            return self._roll_back(
                first_target,
                first_error,
                [*promoted, *others],
                target_states,
                staged,
            )

        cleanup_failures = self._cleanup_all(staged)
        if cleanup_failures:
            return self._completed_with_cleanup_failure(promoted, cleanup_failures)
        self.log("__run__", "COMPLETED", len(promoted), None)
        return PublicationOutcome(state=COMPLETED, promoted=[t.name for t in promoted])

    def _fail(
        self, target: TableTarget, exc: Exception, staged: list[TableTarget]
    ) -> PublicationOutcome:
        error = bound_error(exc)
        self.log(target.name, "FAILED", None, error)
        self.log("__run__", "FAILED", None, error)
        # Evidence (the STAGED/FAILED rows above) is logged before cleanup
        # runs. No data was touched, so the run stays FAILED regardless of
        # whether cleanup itself succeeds — but a cleanup failure is still
        # logged (not silently swallowed) and its staging left in place.
        cleanup_failures = self._cleanup_all(staged)
        preserved = [name for name, _ in cleanup_failures]
        if cleanup_failures:
            failure_text = "; ".join(f"{name}: {err}" for name, err in cleanup_failures)
            self.log(
                "__run__",
                "FAILED",
                None,
                f"staging cleanup also failed: {failure_text}"[:ERROR_TEXT_LIMIT],
            )
        return PublicationOutcome(
            state=FAILED,
            error=error,
            cleanup_failures=cleanup_failures,
            preserved_staging=preserved,
        )

    def _roll_back(
        self,
        failed_target: TableTarget,
        exc: Exception,
        promoted: list[TableTarget],
        target_states: dict[TableTarget, TargetState],
        staged: list[TableTarget],
    ) -> PublicationOutcome:
        error = bound_error(exc)
        self.log(failed_target.name, "FAILED", None, error)
        self.log("__run__", "ROLLING_BACK", None, error)

        # Roll back everything already promoted *and* the target whose
        # promote() call just raised: a Delta command can commit before the
        # client observes a network/driver error, so the failed target may
        # have partially or fully landed. Restoring/dropping it anyway is
        # safe and idempotent (RESTORE to the pre-run version is a no-op if
        # nothing committed; DROP TABLE IF EXISTS is a no-op if it was never
        # created) and keeps rollback fail-closed instead of trusting the
        # exception.
        attempted = [*promoted, failed_target]
        failures = self._attempt_rollback(attempted, target_states)

        if failures:
            preserved = [t.staging_name for t in staged]
            names = ", ".join(preserved)
            failure_text = "; ".join(f"{name}: {err}" for name, err in failures)
            self.log(
                "__run__",
                ROLLBACK_FAILED,
                None,
                f"{error}; rollback failures: {failure_text}; "
                f"staging preserved for manual recovery: {names}"[:ERROR_TEXT_LIMIT],
            )
            return PublicationOutcome(
                state=ROLLBACK_FAILED,
                error=error,
                rollback_failures=failures,
                preserved_staging=preserved,
            )

        cleanup_failures = self._cleanup_all(staged)
        if cleanup_failures:
            return self._rolled_back_with_cleanup_failure(error, cleanup_failures)
        self.log("__run__", ROLLED_BACK, None, error)
        return PublicationOutcome(state=ROLLED_BACK, error=error)

    def _attempt_rollback(
        self, targets: list[TableTarget], target_states: dict[TableTarget, TargetState]
    ) -> list[tuple[str, str]]:
        """Undo every given target in reverse (attempted) order.

        ``targets`` is every target that was promoted *plus* the one whose
        promotion just failed (see ``_roll_back``); continues past a
        restore/drop failure so one bad target can't stop the rest from
        being rolled back.
        """
        failures: list[tuple[str, str]] = []
        for target in reversed(targets):
            state = target_states[target]
            try:
                if state.existed:
                    self.backend.restore(target, state)
                else:
                    self.backend.drop(target)
                self.log(target.name, ROLLED_BACK, None, None)
            except Exception as exc:  # noqa: BLE001 — keep rolling back the rest
                error = bound_error(exc)
                failures.append((target.name, error))
                self.log(target.name, ROLLBACK_FAILED, None, error)
        return failures

    def _cleanup_all(self, staged: list[TableTarget]) -> list[tuple[str, str]]:
        """Best-effort cleanup of every staged artifact; never raises.

        Attempts every staged target even if one fails, and returns the
        failures instead of raising: an unhandled exception here would blow
        past an already-successful promotion or rollback and could mask it
        (the caller would never see the terminal log/outcome it earned).
        """
        failures: list[tuple[str, str]] = []
        for target in staged:
            try:
                self.backend.cleanup(target)
            except Exception as exc:  # noqa: BLE001 — collected, not swallowed
                failures.append((target.staging_name, bound_error(exc)))
        return failures

    def _completed_with_cleanup_failure(
        self, promoted: list[TableTarget], cleanup_failures: list[tuple[str, str]]
    ) -> PublicationOutcome:
        """Promotion fully succeeded, but staging cleanup did not.

        Never reported as plain COMPLETED (that would hide leftover
        staging) and never conflated with a data failure — the message is
        explicit that the promoted data is fine and only cleanup needs
        manual attention.
        """
        preserved = [name for name, _ in cleanup_failures]
        failure_text = "; ".join(f"{name}: {err}" for name, err in cleanup_failures)
        message = (f"data promoted successfully; staging cleanup failed: {failure_text}")[
            :ERROR_TEXT_LIMIT
        ]
        self.log("__run__", COMPLETED_CLEANUP_FAILED, len(promoted), message)
        return PublicationOutcome(
            state=COMPLETED_CLEANUP_FAILED,
            promoted=[t.name for t in promoted],
            error=message,
            cleanup_failures=cleanup_failures,
            preserved_staging=preserved,
        )

    def _rolled_back_with_cleanup_failure(
        self, error: str, cleanup_failures: list[tuple[str, str]]
    ) -> PublicationOutcome:
        """Rollback fully restored/dropped every target, but cleanup did not.

        Never reported as plain ROLLED_BACK (that would hide leftover
        staging) and never mislabeled as a data problem — the message states
        the rollback itself succeeded and only cleanup needs manual
        attention.
        """
        preserved = [name for name, _ in cleanup_failures]
        failure_text = "; ".join(f"{name}: {err}" for name, err in cleanup_failures)
        message = (
            f"{error}; data rolled back successfully; staging cleanup failed: {failure_text}"
        )[:ERROR_TEXT_LIMIT]
        self.log("__run__", ROLLED_BACK_CLEANUP_FAILED, None, message)
        return PublicationOutcome(
            state=ROLLED_BACK_CLEANUP_FAILED,
            error=message,
            cleanup_failures=cleanup_failures,
            preserved_staging=preserved,
        )

# --- retail_setup/generation/writer.py ---
"""Write layer. Notebooks call write_to_lakehouse/write_all; tests use
write_table with a format/location override (no delta-spark dependency
locally).

``write_all`` publishes every candidate table through the stage -> validate
-> promote (-> rollback) state machine in ``publication.py``: nothing final
is touched until every candidate has staged and validated successfully, and
any promotion failure rolls back everything already promoted in this run.
Catalog mode (``lakehouse=``) uses real Delta staging tables and
``RESTORE TABLE ... TO VERSION AS OF``/``DROP TABLE`` for rollback; local
path mode (``base_path=``) uses a filesystem staging/backup strategy so
tests don't need delta-spark.
"""

import re
import shutil
import warnings
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


_UNSAFE_IDENTIFIER = re.compile(r"[^0-9A-Za-z_]")
PUBLICATION_WORKERS = 8


def sanitize_identifier(value: str) -> str:
    """Make ``value`` safe to embed in a Spark/Delta table identifier.

    Used to derive run-scoped staging table names from ``run_id`` (which may
    contain ``-`` and other characters legal in a run id but not desirable in
    an unquoted identifier segment).
    """
    return _UNSAFE_IDENTIFIER.sub("_", value)


def assert_historical_replacement_safe(spark: SparkSession, lakehouse: str, silver_db: str) -> None:
    """Refuse to replace shared Silver data after streaming has initialized."""

    checkpoint_table = f"{lakehouse}.{silver_db}._watermarks"
    if spark.catalog.tableExists(checkpoint_table):
        raise RuntimeError(
            f"Historical replacement is blocked because {checkpoint_table} exists. "
            "Stop streaming and use the documented coordinated Silver/checkpoint "
            "reset before rebuilding history; never delete checkpoints alone."
        )


def write_table(df: DataFrame, location: str, fmt: str = "delta") -> None:
    df.write.format(fmt).mode("overwrite").save(location)


def write_to_lakehouse(df: DataFrame, lakehouse: str, schema: str, table: str) -> None:
    """Overwrite-by-design, matching 02-historical-data-load semantics."""
    df.write.format("delta").mode("overwrite").saveAsTable(f"{lakehouse}.{schema}.{table}")


def _schema_signature(df: DataFrame) -> list[tuple[str, str]]:
    return [(f.name, f.dataType.simpleString()) for f in df.schema.fields]


class _LakehouseBackend:
    """PublicationBackend for Fabric Lakehouse Delta tables.

    Every candidate is staged to ``{lakehouse}.{db}_stage.{run_token}__{name}``
    (a schema distinct from and never colliding with the final ``{db}``
    schema, or with another run's staging tables — ``run_token`` is derived
    from the already-unique ``run_id``). Promotion uses
    ``CREATE OR REPLACE TABLE ... USING DELTA AS SELECT * FROM <staging>``.
    Rollback restores pre-existing targets with
    ``RESTORE TABLE ... TO VERSION AS OF`` (version captured via
    ``DESCRIBE HISTORY ... LIMIT 1`` before promotion) and drops targets that
    were newly created by this run.
    """

    def __init__(
        self, spark, lakehouse: str, run_id: str, sources: dict[tuple[str, str], DataFrame]
    ) -> None:
        self.spark = spark
        self.lakehouse = lakehouse
        self.run_token = sanitize_identifier(run_id)
        self.sources = sources
        self._staging_dbs_created: set[str] = set()

    def _final(self, target: TableTarget) -> str:
        return f"{self.lakehouse}.{target.db}.{target.name}"

    def _ensure_staging_db(self, db: str) -> None:
        stage_db = f"{db}_stage"
        if stage_db not in self._staging_dbs_created:
            self.spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.lakehouse}.{stage_db}")
            self._staging_dbs_created.add(stage_db)

    def stage(self, target: TableTarget) -> int:
        if not target.prestaged:
            df = self.sources[(target.db, target.name)]
            self._ensure_staging_db(target.db)
            df.write.format("delta").mode("overwrite").saveAsTable(target.staging_name)
        return self.spark.table(target.staging_name).count()

    def validate(self, target: TableTarget, staged_row_count: int) -> None:
        df = self.sources[(target.db, target.name)]
        staged = self.spark.table(target.staging_name)
        staged_schema = _schema_signature(staged)
        source_schema = _schema_signature(df)
        if staged_schema != source_schema:
            raise ValueError(
                f"schema mismatch staging {target.staging_name!r}: "
                f"{staged_schema} != {source_schema}"
            )
        source_count = df.count()
        if staged_row_count != source_count:
            raise ValueError(
                f"row count mismatch staging {target.staging_name!r}: "
                f"staged={staged_row_count} source={source_count}"
            )

    def target_state(self, target: TableTarget) -> TargetState:
        final = self._final(target)
        if not self.spark.catalog.tableExists(final):
            return TargetState(existed=False)
        version = self.spark.sql(f"DESCRIBE HISTORY {final} LIMIT 1").collect()[0]["version"]
        return TargetState(existed=True, restore_token=version)

    def promote(self, target: TableTarget) -> int:
        final = self._final(target)
        self.spark.sql(
            f"CREATE OR REPLACE TABLE {final} USING DELTA AS SELECT * FROM {target.staging_name}"
        )
        return self.spark.table(final).count()

    def restore(self, target: TableTarget, state: TargetState) -> None:
        final = self._final(target)
        self.spark.sql(f"RESTORE TABLE {final} TO VERSION AS OF {state.restore_token}")

    def drop(self, target: TableTarget) -> None:
        self.spark.sql(f"DROP TABLE IF EXISTS {self._final(target)}")

    def cleanup(self, target: TableTarget) -> None:
        # Prestaged handoffs belong to the generation run; the publish
        # notebook drops them only after a successful publication.
        if not target.prestaged:
            self.spark.sql(f"DROP TABLE IF EXISTS {target.staging_name}")


class _FilesystemBackend:
    """PublicationBackend for local path/parquet test mode.

    No Delta dependency: staging writes to a run-scoped directory under
    ``<base_path>/.setup_staging/<run_token>/``, promotion replaces the final
    directory outright, and rollback restores from a pre-promotion backup
    copy (pre-existing targets, backed up under ``.setup_backup/<run_token>/``
    before the first promotion touches them) or simply removes the directory
    (targets created by this run).
    """

    def __init__(
        self,
        spark,
        base_path: Path,
        fmt: str,
        run_id: str,
        sources: dict[tuple[str, str], DataFrame],
    ) -> None:
        self.spark = spark
        self.base_path = base_path
        self.fmt = fmt
        self.run_token = sanitize_identifier(run_id)
        self.sources = sources
        self._staging_root = base_path / ".setup_staging" / self.run_token
        self._backup_root = base_path / ".setup_backup" / self.run_token

    def _final_path(self, target: TableTarget) -> Path:
        return self.base_path / target.db / target.name

    def _staging_path(self, target: TableTarget) -> Path:
        if target.prestaged:
            return Path(target.staging_name)
        return self._staging_root / target.db / target.name

    def _backup_path(self, target: TableTarget) -> Path:
        return self._backup_root / target.db / target.name

    def stage(self, target: TableTarget) -> int:
        path = self._staging_path(target)
        if not target.prestaged:
            df = self.sources[(target.db, target.name)]
            df.write.format(self.fmt).mode("overwrite").save(str(path))
        return self.spark.read.format(self.fmt).load(str(path)).count()

    def validate(self, target: TableTarget, staged_row_count: int) -> None:
        df = self.sources[(target.db, target.name)]
        staged = self.spark.read.format(self.fmt).load(str(self._staging_path(target)))
        staged_schema = _schema_signature(staged)
        source_schema = _schema_signature(df)
        if staged_schema != source_schema:
            raise ValueError(
                f"schema mismatch staging {target.name!r}: {staged_schema} != {source_schema}"
            )
        source_count = df.count()
        if staged_row_count != source_count:
            raise ValueError(
                f"row count mismatch staging {target.name!r}: "
                f"staged={staged_row_count} source={source_count}"
            )

    def target_state(self, target: TableTarget) -> TargetState:
        final = self._final_path(target)
        if not final.exists():
            return TargetState(existed=False)
        backup = self._backup_path(target)
        backup.parent.mkdir(parents=True, exist_ok=True)
        if backup.exists():
            shutil.rmtree(backup)
        shutil.copytree(final, backup)
        return TargetState(existed=True, restore_token=str(backup))

    def promote(self, target: TableTarget) -> int:
        final = self._final_path(target)
        if final.exists():
            shutil.rmtree(final)
        final.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(self._staging_path(target), final)
        return self.spark.read.format(self.fmt).load(str(final)).count()

    def restore(self, target: TableTarget, state: TargetState) -> None:
        final = self._final_path(target)
        if final.exists():
            shutil.rmtree(final)
        shutil.copytree(Path(str(state.restore_token)), final)

    def drop(self, target: TableTarget) -> None:
        final = self._final_path(target)
        if final.exists():
            shutil.rmtree(final)

    def cleanup(self, target: TableTarget) -> None:
        staged = self._staging_path(target)
        if staged.exists() and not target.prestaged:
            shutil.rmtree(staged)
        backup = self._backup_path(target)
        if backup.exists():
            shutil.rmtree(backup)


def _run_log(
    spark: SparkSession,
    cfg: GenerationConfig,
    lakehouse: str | None,
    base_path: str | None,
    fmt: str,
) -> DataFrame | None:
    if lakehouse is not None:
        name = f"{lakehouse}.{cfg.silver_db}.setup_run_log"
        return spark.table(name) if spark.catalog.tableExists(name) else None
    path = Path(str(base_path)) / cfg.silver_db / "setup_run_log"
    return spark.read.format(fmt).load(str(path)) if path.exists() else None


def completed_publication(
    spark: SparkSession,
    cfg: GenerationConfig,
    fingerprint: str,
    *,
    lakehouse: str | None = None,
    base_path: str | None = None,
    fmt: str = "delta",
) -> str | None:
    """The run_id of the latest Silver publication, when it holds this generation.

    The latest completed run with a logged fingerprint (Silver publications)
    counts only if its fingerprint matches and every table it promoted still
    exists; otherwise None.
    """
    if (lakehouse is None) == (base_path is None):
        raise ValueError("Provide exactly one of lakehouse= or base_path=")
    log = _run_log(spark, cfg, lakehouse, base_path, fmt)
    if log is None or "generation_fingerprint" not in log.columns:
        return None
    done = F.col("status").isin(COMPLETED, COMPLETED_CLEANUP_FAILED)
    latest = (
        log.filter(
            (F.col("table_name") == "__run__") & done & F.col("generation_fingerprint").isNotNull()
        )
        .orderBy(F.col("generated_at").desc(), F.col("run_id").desc())
        .select("run_id", "generation_fingerprint")
        .first()
    )
    if latest is None or latest["generation_fingerprint"] != fingerprint:
        return None
    run_id = latest["run_id"]
    promoted = [
        row["table_name"]
        for row in log.filter(
            (F.col("run_id") == run_id) & done & (F.col("table_name") != "__run__")
        )
        .select("table_name")
        .distinct()
        .collect()
    ]
    for table in promoted:
        if lakehouse is not None:
            exists = spark.catalog.tableExists(f"{lakehouse}.{cfg.silver_db}.{table}")
        else:
            exists = (Path(str(base_path)) / cfg.silver_db / table).exists()
        if not exists:
            return None
    return run_id if promoted else None


def already_published(
    spark: SparkSession,
    cfg: GenerationConfig,
    store: HandoffStore,
    *,
    lakehouse: str | None = None,
    base_path: str | None = None,
    fmt: str = "delta",
) -> str | None:
    """The run that already published this generation when a retry has nothing to do.

    Setup-10 drops the handoffs once Silver is published. A retry that finds
    no handoffs and Silver holding this generation (:func:`completed_publication`)
    succeeds without publishing again; with handoffs present it publishes them.
    """
    if store.has_handoffs():
        return None
    return completed_publication(
        spark, cfg, store.fingerprint, lakehouse=lakehouse, base_path=base_path, fmt=fmt
    )


def write_all(
    tables: dict[str, DataFrame],
    gold: dict[str, DataFrame],
    cfg: GenerationConfig,
    run_id: str,
    *,
    lakehouse: str | None = None,
    base_path: str | None = None,
    fmt: str = "delta",
    prestaged: dict[str, str] | None = None,
    fingerprint: str | None = None,
) -> list[str]:
    """Publish dims+facts to silver, gold to gold, then setup_run_log.

    Two modes: catalog (lakehouse=...) used by notebooks, or path
    (base_path=...) used by local tests/E2E. Either ``tables`` or ``gold``
    (or both) may be non-empty; at least one of the two must contain a table.
    dim_date is written as generated by the engine (±5y padding preserved —
    the semantic model's date relationships need the headroom).

    ``prestaged`` maps Silver table names to an existing staging identifier
    (a generation handoff table, or path in path mode) that already holds
    exactly that table's rows. Those targets skip the staging copy, are
    validated and promoted from the handoff, and are never dropped here.

    Every candidate table is staged to a run-scoped location, validated
    against its source, and only then promoted to its fixed final name (see
    ``retail_setup.generation.publication``). If any promotion fails, every
    target already promoted in this run is rolled back and a
    ``RuntimeError`` is raised — this function never returns after a partial
    or rolled-back publish; a return only happens once every candidate has
    been promoted.

    ``fingerprint`` (``handoff.generation_fingerprint``) is logged with every
    row of a Silver publication, so a retry can tell that Silver already holds
    this generation (:func:`completed_publication`).

    Returns the list of written table names (silver + gold); the
    setup_run_log table itself is not included in the returned list.

    The Spark session is derived from the first DataFrame in ``tables`` or
    ``gold`` (``df.sparkSession``) — no explicit session parameter is needed.
    """
    if (lakehouse is None) == (base_path is None):
        raise ValueError("Provide exactly one of lakehouse= or base_path=")

    first_df = next(iter(tables.values()), None) or next(iter(gold.values()), None)
    if first_df is None:
        raise ValueError("write_all requires at least one table in tables or gold")
    spark = first_df.sparkSession
    if lakehouse is not None and tables:
        assert_historical_replacement_safe(spark, lakehouse, cfg.silver_db)

    log_name = "setup_run_log"
    log_table = f"{lakehouse}.{cfg.silver_db}.{log_name}" if lakehouse is not None else None
    log_path = f"{base_path}/{cfg.silver_db}/{log_name}" if base_path is not None else None

    def _log_exists() -> bool:
        if log_table is not None:
            return spark.catalog.tableExists(log_table)
        return Path(str(log_path)).exists()

    def _read_log() -> DataFrame:
        if log_table is not None:
            return spark.table(log_table)
        return spark.read.format(fmt).load(str(log_path))

    def _append_log(
        table_name: str,
        row_count: int | None,
        status: str,
        error: str | None = None,
    ) -> None:
        row = [
            (
                run_id,
                cfg.store_type,
                cfg.seed,
                cfg.start_date,
                cfg.end_date,
                table_name,
                row_count,
                status,
                error,
            )
        ]
        log_df = spark.createDataFrame(
            row,
            "run_id string, store_type string, seed long, start_date date, "
            "end_date date, table_name string, row_count long, status string, "
            "error string",
        ).withColumn("generated_at", F.current_timestamp())
        log_df = log_df.withColumn("generation_fingerprint", F.lit(fingerprint).cast("string"))
        writer = log_df.write.format("delta" if log_table is not None else fmt)
        writer = writer.mode("append").option("mergeSchema", "true")
        if log_table is not None:
            writer.saveAsTable(log_table)
        else:
            writer.save(str(log_path))

    if _log_exists() and _read_log().filter(F.col("run_id") == run_id).limit(1).count():
        raise ValueError(f"setup run_id already exists: {run_id!r}")

    run_token = sanitize_identifier(run_id)
    prestaged = prestaged or {}
    unknown = sorted(set(prestaged) - set(tables))
    if unknown:
        raise ValueError(f"prestaged tables are not Silver candidates: {unknown}")
    sources: dict[tuple[str, str], DataFrame] = {}
    targets: list[TableTarget] = []
    for db, frames in ((cfg.silver_db, tables), (cfg.gold_db, gold)):
        for name, df in frames.items():
            sources[(db, name)] = df
            handoff = prestaged.get(name) if frames is tables else None
            if handoff is not None:
                targets.append(TableTarget(name=name, db=db, staging_name=handoff, prestaged=True))
                continue
            if lakehouse is not None:
                staging_name = f"{lakehouse}.{db}_stage.{run_token}__{name}"
            else:
                staging_name = f"{db}/{name}"
            targets.append(TableTarget(name=name, db=db, staging_name=staging_name))

    backend: PublicationBackend
    if lakehouse is not None:
        for db in (cfg.silver_db, cfg.gold_db):
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {lakehouse}.{db}")
            # Created up front: staging runs concurrently per table.
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {lakehouse}.{db}_stage")
        backend = _LakehouseBackend(spark, lakehouse, run_id, sources)
    else:
        assert base_path is not None  # enforced by the exactly-one-of check above
        backend = _FilesystemBackend(spark, Path(base_path), fmt, run_id, sources)

    def _log(table_name: str, status: str, row_count: int | None, error: str | None) -> None:
        _append_log(table_name, row_count, status, error)

    # Per-table staging, validation, and promotion are independent and each is
    # a separate Spark job; running them concurrently removes per-table
    # overhead without changing the phase ordering or rollback guarantees.
    coordinator = PublicationCoordinator(backend, _log, max_workers=PUBLICATION_WORKERS)
    outcome = coordinator.publish(targets)

    if not outcome.ok:
        raise RuntimeError(
            f"setup publication {outcome.state} for run_id={run_id!r}: {outcome.error}"
        )
    if outcome.state == COMPLETED_CLEANUP_FAILED:
        warnings.warn(
            f"setup publication completed but staging cleanup needs attention: {outcome.error}",
            RuntimeWarning,
            stacklevel=2,
        )

    return outcome.promoted

## Run this notebook's generation stages

In [ ]:
from datetime import date

SETUP_NOTEBOOK = "setup-06-generate-returns-and-promotions"

cfg = GenerationConfig(
    store_type=STORE_TYPE,
    start_date=date.fromisoformat(START_DATE),
    end_date=date.fromisoformat(END_DATE),
    anchor_date=date.fromisoformat(ANCHOR_DATE),
    store_count=STORE_COUNT,
    seed=SEED,
    silver_db=SILVER_DB,
    gold_db=GOLD_DB,
    dictionary_root="/lakehouse/default/Files/setup/dictionaries",
)
dicts = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type)

# Size shuffles to the generated volume (Spark's default 200 partitions makes each
# billion-row full-demo shuffle task spill millions of rows); adaptive execution
# coalesces small shuffles back down. Output does not depend on partitioning.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set(
    "spark.sql.shuffle.partitions",
    str(shuffle_partitions_for(cfg, dicts.profile.basket_lambda)),
)
print(f"spark.sql.shuffle.partitions={spark.conf.get('spark.sql.shuffle.partitions')}")

# Dimensions are regenerated in memory in every notebook: generation is fully
# deterministic (same seed => identical dims), so this is cheaper than a
# catalog round-trip. Checkpoints live in the Lakehouse, not on executors, so
# autoscaling or a lost executor cannot drop rows a later step still needs
# (CHECKPOINT_RDD_BLOCK_ID_NOT_FOUND). Each notebook owns its folder, so
# concurrent notebooks never touch each other's files; clearing it first
# removes whatever a failed attempt left behind.
checkpoint_dir = f"Files/setup/_checkpoints/{SETUP_NOTEBOOK}"
delete_lakehouse_path(spark, checkpoint_dir)
store = HandoffStore(spark, cfg, lakehouse=LAKEHOUSE_NAME)
outputs = run_generation_notebook(
    spark, dicts, cfg, SETUP_NOTEBOOK, store, checkpoint_dir=checkpoint_dir,
)
delete_lakehouse_path(spark, checkpoint_dir)
for stage_name, frames in sorted(outputs.items()):
    for name, frame in sorted(frames.items()):
        print(f"{stage_name}/{name}: {frame.count():,} rows -> {store.name(stage_name, name)}")